In [1]:
# ============================================================
# OIP v1.0.33 — ESS4
# C00 — CANONICAL ENVIRONMENT & SOURCE SCOPE LOCK
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C00 — CANONICAL ENVIRONMENT & SOURCE SCOPE LOCK")
print("=" * 78)

# ------------------------------------------------------------
# 1. CANONICAL SOURCE
# ------------------------------------------------------------

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/sudharsandas27/"
    "oip-v1-0-33-ethiopia-ess4-evidence-audit"
)

WORK_ROOT = Path("/kaggle/working/oip_v1_0_33")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Source root : {SOURCE_ROOT}")
print(f"Working dir : {WORK_ROOT}")

# ------------------------------------------------------------
# 2. SOURCE EXISTENCE CHECK
# ------------------------------------------------------------

if not SOURCE_ROOT.exists():
    raise RuntimeError(
        "FAIL-CLOSED: Canonical source root does not exist."
    )

if not SOURCE_ROOT.is_dir():
    raise RuntimeError(
        "FAIL-CLOSED: Canonical source root is not a directory."
    )

# ------------------------------------------------------------
# 3. DISCOVER RAW SOURCE FILES
# ------------------------------------------------------------

source_files = sorted(
    [
        p for p in SOURCE_ROOT.rglob("*")
        if p.is_file()
    ],
    key=lambda p: str(p.relative_to(SOURCE_ROOT)).lower()
)

if not source_files:
    raise RuntimeError(
        "FAIL-CLOSED: No raw source files discovered."
    )

# ------------------------------------------------------------
# 4. FILE INVENTORY + SHA256
# ------------------------------------------------------------

inventory_rows = []

for path in source_files:

    relative_path = str(
        path.relative_to(SOURCE_ROOT)
    )

    suffix = path.suffix.lower()
    size_bytes = path.stat().st_size

    if suffix == ".dta":
        file_type = "STATA"
    elif suffix == ".pdf":
        file_type = "PDF"
    else:
        file_type = "OTHER"

    sha256 = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    inventory_rows.append({
        "relative_path": relative_path,
        "file_name": path.name,
        "suffix": suffix,
        "file_type": file_type,
        "size_bytes": int(size_bytes),
        "sha256": sha256.hexdigest(),
        "zero_byte": bool(size_bytes == 0)
    })

inventory = pd.DataFrame(inventory_rows)

# ------------------------------------------------------------
# 5. DESCRIPTIVE COUNTS
# ------------------------------------------------------------

total_files = int(len(inventory))

stata_files = int(
    (inventory["file_type"] == "STATA").sum()
)

pdf_files = int(
    (inventory["file_type"] == "PDF").sum()
)

other_files = int(
    (inventory["file_type"] == "OTHER").sum()
)

zero_byte_files = int(
    inventory["zero_byte"].sum()
)

duplicate_relative_paths = int(
    inventory["relative_path"].duplicated().sum()
)

duplicate_file_names = int(
    inventory["file_name"].duplicated().sum()
)

# ------------------------------------------------------------
# 6. BASIC SOURCE INTEGRITY STATE
# ------------------------------------------------------------

integrity_pass = bool(
    total_files > 0
    and zero_byte_files == 0
    and duplicate_relative_paths == 0
    and duplicate_file_names == 0
)

# ------------------------------------------------------------
# 7. CANONICAL C00 STATE
# ------------------------------------------------------------

c00_state = {
    "oip_version": "v1.0.33",
    "cell": "C00",
    "cell_name": "Canonical Environment and Source Scope Lock",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "status": "PASS" if integrity_pass else "BLOCKED",

    "fail_closed_policy_active": True,
    "fail_closed_triggered": not integrity_pass,

    "authorization": {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY"
    },

    "payload": {
        "source_root": str(SOURCE_ROOT),
        "working_root": str(WORK_ROOT),

        "source_authority": "RAW_SOURCE_ONLY",
        "source_scope_locked": True,

        "total_files": total_files,
        "stata_files": stata_files,
        "pdf_files": pdf_files,
        "other_files": other_files,

        "zero_byte_files": zero_byte_files,
        "duplicate_relative_paths": duplicate_relative_paths,
        "duplicate_file_names": duplicate_file_names,

        "historical_artifacts_used": False,

        "semantic_interpretation": "NOT_PERFORMED",
        "semantic_inference_allowed": False,

        "integrity_check": (
            "PASS" if integrity_pass else "BLOCKED"
        )
    }
}

# ------------------------------------------------------------
# 8. WRITE CANONICAL INVENTORY
# ------------------------------------------------------------

inventory_path = (
    WORK_ROOT / "c00_source_inventory.csv"
)

inventory.to_csv(
    inventory_path,
    index=False
)

# ------------------------------------------------------------
# 9. WRITE CANONICAL C00 JSON
# ------------------------------------------------------------

scope_path = (
    WORK_ROOT / "c00_source_scope.json"
)

with scope_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c00_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 10. CONSOLE OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("RAW SOURCE INVENTORY")
print("-" * 78)

print(f"Total files              : {total_files}")
print(f"Stata files              : {stata_files}")
print(f"PDF files                : {pdf_files}")
print(f"Other files              : {other_files}")
print(f"Zero-byte files          : {zero_byte_files}")
print(f"Duplicate paths          : {duplicate_relative_paths}")
print(f"Duplicate file names     : {duplicate_file_names}")

print()
print("-" * 78)
print("SOURCE SCOPE")
print("-" * 78)

print(
    f"Source authority         : "
    f"{c00_state['payload']['source_authority']}"
)

print(
    f"Source scope locked      : "
    f"{c00_state['payload']['source_scope_locked']}"
)

print(
    f"Historical artifacts used: "
    f"{c00_state['payload']['historical_artifacts_used']}"
)

print(
    f"Semantic interpretation  : "
    f"{c00_state['payload']['semantic_interpretation']}"
)

print(
    f"Semantic inference       : "
    f"{c00_state['payload']['semantic_inference_allowed']}"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in c00_state["authorization"].items():
    print(f"{key:24s}: {value}")

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    f"Policy active            : "
    f"{c00_state['fail_closed_policy_active']}"
)

print(
    f"Triggered                : "
    f"{c00_state['fail_closed_triggered']}"
)

print(
    f"C00 STATUS               : "
    f"{c00_state['status']}"
)

print()
print("-" * 78)
print("CANONICAL ARTIFACTS")
print("-" * 78)

print(inventory_path)
print(scope_path)

print("=" * 78)

OIP v1.0.33 — ESS4
C00 — CANONICAL ENVIRONMENT & SOURCE SCOPE LOCK
Source root : /kaggle/input/datasets/sudharsandas27/oip-v1-0-33-ethiopia-ess4-evidence-audit
Working dir : /kaggle/working/oip_v1_0_33

------------------------------------------------------------------------------
RAW SOURCE INVENTORY
------------------------------------------------------------------------------
Total files              : 75
Stata files              : 68
PDF files                : 7
Other files              : 0
Zero-byte files          : 0
Duplicate paths          : 0
Duplicate file names     : 0

------------------------------------------------------------------------------
SOURCE SCOPE
------------------------------------------------------------------------------
Source authority         : RAW_SOURCE_ONLY
Source scope locked      : True
Historical artifacts used: False
Semantic interpretation  : NOT_PERFORMED
Semantic inference       : False

----------------------------------------------------------

In [2]:
# ============================================================
# OIP v1.0.33 — ESS4
# C01 — C00 ACTUAL SCHEMA VERIFICATION + VOCABULARY LOCK
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C01 — C00 ACTUAL SCHEMA VERIFICATION + VOCABULARY LOCK")
print("=" * 78)

# ------------------------------------------------------------
# 1. PATH
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working/oip_v1_0_33")
C00_PATH = WORK_ROOT / "c00_source_scope.json"

if not C00_PATH.exists():
    raise RuntimeError(
        "FAIL-CLOSED: c00_source_scope.json is missing."
    )

# ------------------------------------------------------------
# 2. LOAD ACTUAL C00 ARTIFACT
# ------------------------------------------------------------

with C00_PATH.open("r", encoding="utf-8") as f:
    c00 = json.load(f)

if not isinstance(c00, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C00 JSON root is not an object."
    )

actual_top_level_keys = list(c00.keys())

print()
print("-" * 78)
print("ACTUAL C00 TOP-LEVEL SCHEMA")
print("-" * 78)

for i, key in enumerate(actual_top_level_keys, 1):
    print(f"{i:03d}. {key}")

# ------------------------------------------------------------
# 3. MINIMUM C00 IDENTITY CONTRACT
# ------------------------------------------------------------

identity_keys = [
    "oip_version",
    "cell",
    "cell_name",
    "timestamp_utc",
    "status",
    "fail_closed_policy_active",
    "fail_closed_triggered"
]

missing_identity_keys = [
    key for key in identity_keys
    if key not in c00
]

if missing_identity_keys:
    raise RuntimeError(
        "FAIL-CLOSED: Required C00 identity keys missing: "
        + ", ".join(missing_identity_keys)
    )

# ------------------------------------------------------------
# 4. VERIFY C00 IDENTITY
# ------------------------------------------------------------

if c00["oip_version"] != "v1.0.33":
    raise RuntimeError(
        "FAIL-CLOSED: C00 OIP version mismatch."
    )

if c00["cell"] != "C00":
    raise RuntimeError(
        "FAIL-CLOSED: C00 cell identity mismatch."
    )

if c00["status"] != "PASS":
    raise RuntimeError(
        "FAIL-CLOSED: C00 status is not PASS."
    )

if c00["fail_closed_policy_active"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C00 fail-closed policy is not active."
    )

if c00["fail_closed_triggered"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C00 fail-closed state is triggered."
    )

# ------------------------------------------------------------
# 5. DETERMINE ACTUAL AUTHORIZATION REPRESENTATION
#    No aliasing. No silent conversion.
# ------------------------------------------------------------

nested_authorization = (
    "authorization" in c00
)

flat_authorization_keys = [
    "construct_authorization",
    "predictor_authorization",
    "outcome_authorization",
    "key_authorization",
    "relationship_authorization",
    "cohort_authorization",
    "score_authorization",
    "empirical_evaluation"
]

flat_authorization_present = [
    key for key in flat_authorization_keys
    if key in c00
]

nested_payload = (
    "payload" in c00
)

print()
print("-" * 78)
print("ACTUAL AUTHORIZATION STRUCTURE")
print("-" * 78)

print(
    f"Nested authorization object : {nested_authorization}"
)

print(
    f"Flat authorization fields   : "
    f"{len(flat_authorization_present)}"
)

print(
    f"Payload object              : {nested_payload}"
)

# ------------------------------------------------------------
# 6. SCHEMA AMBIGUITY CHECK
# ------------------------------------------------------------

if nested_authorization and flat_authorization_present:
    raise RuntimeError(
        "FAIL-CLOSED: C00 contains both nested and flat "
        "authorization representations. Schema is ambiguous."
    )

if nested_authorization:

    if not isinstance(c00["authorization"], dict):
        raise RuntimeError(
            "FAIL-CLOSED: C00 authorization is not an object."
        )

    actual_authorization_keys = list(
        c00["authorization"].keys()
    )

    wrapper_style = "NESTED"

elif len(flat_authorization_present) == len(flat_authorization_keys):

    actual_authorization_keys = flat_authorization_present

    wrapper_style = "FLAT"

else:

    raise RuntimeError(
        "FAIL-CLOSED: C00 authorization structure is incomplete "
        "or unrecognized. No assumption permitted."
    )

# ------------------------------------------------------------
# 7. VOCABULARY
# ------------------------------------------------------------

STATUS_TERMS = [
    "PASS",
    "FAIL",
    "BLOCKED"
]

COMPLETION_TERMS = [
    "COMPLETE"
]

EVIDENCE_TERMS = [
    "OBSERVED",
    "DOCUMENTED",
    "NOT_DOCUMENTED",
    "VERIFIED",
    "NOT_VERIFIED",
    "CANDIDATE",
    "NOT_ESTABLISHED"
]

AUTHORIZATION_TERMS = [
    "AUTHORIZED",
    "NOT_AUTHORIZED"
]

READINESS_TERMS = [
    "READY",
    "NOT_READY"
]

LIFECYCLE_TERMS = [
    "SUPERSEDED",
    "MISSING"
]

FORBIDDEN_TERMS = [
    "DOCUMENTARY_SOURCE_AVAILABLE",
    "PRESENT_HASH_RECORDED_NOT_VERIFIED",
    "ESTABLISHED_FOR_NARROW_MEANING"
]

# ------------------------------------------------------------
# 8. VOCABULARY INTEGRITY
# ------------------------------------------------------------

all_terms = (
    STATUS_TERMS
    + COMPLETION_TERMS
    + EVIDENCE_TERMS
    + AUTHORIZATION_TERMS
    + READINESS_TERMS
    + LIFECYCLE_TERMS
)

duplicate_terms = sorted({
    term for term in all_terms
    if all_terms.count(term) > 1
})

if duplicate_terms:
    raise RuntimeError(
        "FAIL-CLOSED: Duplicate vocabulary terms: "
        + ", ".join(duplicate_terms)
    )

forbidden_overlap = sorted(
    set(all_terms).intersection(FORBIDDEN_TERMS)
)

if forbidden_overlap:
    raise RuntimeError(
        "FAIL-CLOSED: Forbidden term entered vocabulary: "
        + ", ".join(forbidden_overlap)
    )

# ------------------------------------------------------------
# 9. SEMANTIC BOUNDARY
# ------------------------------------------------------------

semantic_rules = {
    "PASS_is_not_AUTHORIZED": True,
    "COMPLETE_is_not_AUTHORIZED": True,
    "OBSERVED_is_not_AUTHORIZED": True,
    "DOCUMENTED_is_not_AUTHORIZED": True,
    "VERIFIED_is_not_AUTHORIZED": True,
    "CANDIDATE_is_not_AUTHORIZED": True,
    "NOT_ESTABLISHED_blocks_authorization": True,
    "NOT_DOCUMENTED_blocks_authorization": True,
    "NOT_VERIFIED_blocks_authorization": True,
    "unsupported_inference_blocks": True,
    "AI_suggestion_is_candidate_only": True,
    "missing_evidence_blocks": True
}

# ------------------------------------------------------------
# 10. CANONICAL C01 AUTHORIZATION
# ------------------------------------------------------------

authorization = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY"
}

# ------------------------------------------------------------
# 11. C01 STATE
# ------------------------------------------------------------

c01_state = {
    "oip_version": "v1.0.33",
    "cell": "C01",
    "cell_name": "C00 Actual Schema Verification and Vocabulary Lock",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "status": "PASS",

    "fail_closed_policy_active": True,
    "fail_closed_triggered": False,

    "authorization": authorization,

    "payload": {
        "upstream_cell": "C00",

        "upstream_status": c00["status"],

        "actual_c00_top_level_keys": actual_top_level_keys,

        "actual_authorization_wrapper_style": wrapper_style,

        "actual_c00_authorization_keys": (
            actual_authorization_keys
        ),

        "actual_payload_object_present": nested_payload,

        "schema_assumption_used": False,

        "schema_alias_used": False,

        "schema_conversion_performed": False,

        "vocabulary_locked": True,

        "status_terms": STATUS_TERMS,
        "completion_terms": COMPLETION_TERMS,
        "evidence_terms": EVIDENCE_TERMS,
        "authorization_terms": AUTHORIZATION_TERMS,
        "readiness_terms": READINESS_TERMS,
        "lifecycle_terms": LIFECYCLE_TERMS,

        "forbidden_terms": FORBIDDEN_TERMS,

        "semantic_boundary_locked": True,
        "semantic_rules": semantic_rules,

        "historical_artifacts_used": False,
        "raw_source_only": True,

        "semantic_inference_allowed": False
    }
}

# ------------------------------------------------------------
# 12. WRITE CANONICAL C01 ARTIFACT
# ------------------------------------------------------------

output_path = WORK_ROOT / "c01_vocabulary_lock.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(
        c01_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 13. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("C00 VERIFICATION")
print("-" * 78)

print(f"C00 STATUS               : {c00['status']}")
print(f"C00 schema style         : {wrapper_style}")
print(
    f"Schema assumption used   : "
    f"{c01_state['payload']['schema_assumption_used']}"
)
print(
    f"Schema alias used        : "
    f"{c01_state['payload']['schema_alias_used']}"
)
print(
    f"Schema conversion        : "
    f"{c01_state['payload']['schema_conversion_performed']}"
)

print()
print("-" * 78)
print("VOCABULARY LOCK")
print("-" * 78)

print(
    f"Vocabulary locked        : "
    f"{c01_state['payload']['vocabulary_locked']}"
)

print(
    f"Semantic boundary locked : "
    f"{c01_state['payload']['semantic_boundary_locked']}"
)

print(
    f"Forbidden terms          : "
    f"{len(FORBIDDEN_TERMS)}"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in authorization.items():
    print(f"{key:24s}: {value}")

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    f"Policy active            : "
    f"{c01_state['fail_closed_policy_active']}"
)

print(
    f"Triggered                : "
    f"{c01_state['fail_closed_triggered']}"
)

print(
    f"C01 STATUS               : "
    f"{c01_state['status']}"
)

print()
print("Canonical artifact:")
print(output_path)

print("=" * 78)

OIP v1.0.33 — ESS4
C01 — C00 ACTUAL SCHEMA VERIFICATION + VOCABULARY LOCK

------------------------------------------------------------------------------
ACTUAL C00 TOP-LEVEL SCHEMA
------------------------------------------------------------------------------
001. oip_version
002. cell
003. cell_name
004. timestamp_utc
005. status
006. fail_closed_policy_active
007. fail_closed_triggered
008. authorization
009. payload

------------------------------------------------------------------------------
ACTUAL AUTHORIZATION STRUCTURE
------------------------------------------------------------------------------
Nested authorization object : True
Flat authorization fields   : 0
Payload object              : True

------------------------------------------------------------------------------
C00 VERIFICATION
------------------------------------------------------------------------------
C00 STATUS               : PASS
C00 schema style         : NESTED
Schema assumption used   : False
Schema al

In [3]:
import json
from pathlib import Path

c00_path = Path("/kaggle/working/oip_v1_0_33/c00_source_scope.json")

if not c00_path.exists():
    raise RuntimeError("FAIL-CLOSED: c00_source_scope.json not found.")

c00 = json.loads(c00_path.read_text(encoding="utf-8"))

print("=" * 78)
print("C00 PAYLOAD SCHEMA VERIFICATION")
print("=" * 78)

print()
print("C00 top-level keys:")
print(list(c00.keys()))

print()
print("C00 payload keys:")

payload = c00.get("payload")

if not isinstance(payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C00 payload is not a dictionary."
    )

print(list(payload.keys()))

print()
print("source_root present in payload:")
print("source_root" in payload)

print("=" * 78)

C00 PAYLOAD SCHEMA VERIFICATION

C00 top-level keys:
['oip_version', 'cell', 'cell_name', 'timestamp_utc', 'status', 'fail_closed_policy_active', 'fail_closed_triggered', 'authorization', 'payload']

C00 payload keys:
['source_root', 'working_root', 'source_authority', 'source_scope_locked', 'total_files', 'stata_files', 'pdf_files', 'other_files', 'zero_byte_files', 'duplicate_relative_paths', 'duplicate_file_names', 'historical_artifacts_used', 'semantic_interpretation', 'semantic_inference_allowed', 'integrity_check']

source_root present in payload:
True


In [4]:
# ============================================================
# OIP v1.0.33 — ESS4
# C02 — CANONICAL RAW FILE INTEGRITY VERIFICATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C02 — CANONICAL RAW FILE INTEGRITY VERIFICATION")
print("=" * 78)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working/oip_v1_0_33")

C00_JSON = WORK_ROOT / "c00_source_scope.json"
C00_CSV = WORK_ROOT / "c00_source_inventory.csv"
C01_JSON = WORK_ROOT / "c01_vocabulary_lock.json"

required_upstream = [
    C00_JSON,
    C00_CSV,
    C01_JSON
]

for path in required_upstream:
    if not path.exists():
        raise RuntimeError(
            f"FAIL-CLOSED: Required upstream artifact missing: {path}"
        )

# ------------------------------------------------------------
# 2. LOAD C00
# ------------------------------------------------------------

with C00_JSON.open("r", encoding="utf-8") as f:
    c00 = json.load(f)

# ------------------------------------------------------------
# 3. LOAD C01
# ------------------------------------------------------------

with C01_JSON.open("r", encoding="utf-8") as f:
    c01 = json.load(f)

# ------------------------------------------------------------
# 4. VERIFY C00 IDENTITY
# ------------------------------------------------------------

if c00.get("oip_version") != "v1.0.33":
    raise RuntimeError(
        "FAIL-CLOSED: C00 OIP version mismatch."
    )

if c00.get("cell") != "C00":
    raise RuntimeError(
        "FAIL-CLOSED: C00 cell identity mismatch."
    )

if c00.get("status") != "PASS":
    raise RuntimeError(
        "FAIL-CLOSED: C00 status is not PASS."
    )

# ------------------------------------------------------------
# 5. VERIFY C01 IDENTITY
# ------------------------------------------------------------

if c01.get("oip_version") != "v1.0.33":
    raise RuntimeError(
        "FAIL-CLOSED: C01 OIP version mismatch."
    )

if c01.get("cell") != "C01":
    raise RuntimeError(
        "FAIL-CLOSED: C01 cell identity mismatch."
    )

if c01.get("status") != "PASS":
    raise RuntimeError(
        "FAIL-CLOSED: C01 status is not PASS."
    )

# ------------------------------------------------------------
# 6. VERIFY C00 PAYLOAD STRUCTURE
# ------------------------------------------------------------

c00_payload = c00.get("payload")

if not isinstance(c00_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C00 payload is not a dictionary."
    )

if "source_root" not in c00_payload:
    raise RuntimeError(
        "FAIL-CLOSED: C00 payload.source_root missing."
    )

if "source_scope_locked" not in c00_payload:
    raise RuntimeError(
        "FAIL-CLOSED: C00 payload.source_scope_locked missing."
    )

if c00_payload["source_scope_locked"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C00 source scope is not locked."
    )

# ------------------------------------------------------------
# 7. VERIFY C01 PAYLOAD
# ------------------------------------------------------------

c01_payload = c01.get("payload")

if not isinstance(c01_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C01 payload is not a dictionary."
    )

if c01_payload.get("vocabulary_locked") is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C01 vocabulary is not locked."
    )

if c01_payload.get("semantic_boundary_locked") is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C01 semantic boundary is not locked."
    )

if c01_payload.get("semantic_inference_allowed") is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C01 semantic inference boundary invalid."
    )

# ------------------------------------------------------------
# 8. LOAD C00 INVENTORY
# ------------------------------------------------------------

inventory = pd.read_csv(C00_CSV)

required_inventory_columns = [
    "relative_path",
    "file_name",
    "suffix",
    "file_type",
    "size_bytes",
    "sha256",
    "zero_byte"
]

missing_columns = [
    col
    for col in required_inventory_columns
    if col not in inventory.columns
]

if missing_columns:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C00 inventory columns: "
        + ", ".join(missing_columns)
    )

# ------------------------------------------------------------
# 9. SOURCE ROOT
# ------------------------------------------------------------

SOURCE_ROOT = Path(
    c00_payload["source_root"]
)

if not SOURCE_ROOT.exists():
    raise RuntimeError(
        "FAIL-CLOSED: C00 source root does not exist."
    )

if not SOURCE_ROOT.is_dir():
    raise RuntimeError(
        "FAIL-CLOSED: C00 source root is not a directory."
    )

# ------------------------------------------------------------
# 10. REBUILD CURRENT RAW INVENTORY
# ------------------------------------------------------------

actual_files = sorted(
    [
        p
        for p in SOURCE_ROOT.rglob("*")
        if p.is_file()
    ],
    key=lambda p: str(
        p.relative_to(SOURCE_ROOT)
    ).lower()
)

actual_records = []

for path in actual_files:

    relative_path = str(
        path.relative_to(SOURCE_ROOT)
    )

    suffix = path.suffix.lower()

    if suffix == ".dta":
        file_type = "STATA"
    elif suffix == ".pdf":
        file_type = "PDF"
    else:
        file_type = "OTHER"

    size_bytes = int(
        path.stat().st_size
    )

    sha256 = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            sha256.update(chunk)

    actual_records.append({
        "relative_path": relative_path,
        "file_name": path.name,
        "suffix": suffix,
        "file_type": file_type,
        "size_bytes": size_bytes,
        "sha256": sha256.hexdigest(),
        "zero_byte": bool(
            size_bytes == 0
        )
    })

actual_inventory = pd.DataFrame(
    actual_records
)

# ------------------------------------------------------------
# 11. NORMALIZE INVENTORIES
# ------------------------------------------------------------

expected = inventory[
    required_inventory_columns
].copy()

observed = actual_inventory[
    required_inventory_columns
].copy()

expected = expected.sort_values(
    ["relative_path", "file_name"]
).reset_index(drop=True)

observed = observed.sort_values(
    ["relative_path", "file_name"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 12. COUNT CHECK
# ------------------------------------------------------------

expected_count = int(
    len(expected)
)

observed_count = int(
    len(observed)
)

count_match = (
    expected_count == observed_count
)

# ------------------------------------------------------------
# 13. EXACT INVENTORY CHECK
# ------------------------------------------------------------

exact_inventory_match = bool(
    expected.equals(observed)
)

# ------------------------------------------------------------
# 14. PATH RECONCILIATION
# ------------------------------------------------------------

expected_paths = set(
    expected["relative_path"]
)

observed_paths = set(
    observed["relative_path"]
)

missing_since_c00 = sorted(
    expected_paths - observed_paths
)

new_since_c00 = sorted(
    observed_paths - expected_paths
)

# ------------------------------------------------------------
# 15. HASH RECONCILIATION
# ------------------------------------------------------------

expected_hashes = dict(
    zip(
        expected["relative_path"],
        expected["sha256"]
    )
)

observed_hashes = dict(
    zip(
        observed["relative_path"],
        observed["sha256"]
    )
)

all_paths = sorted(
    expected_paths | observed_paths
)

hash_mismatches = []

for rel_path in all_paths:

    expected_hash = expected_hashes.get(
        rel_path
    )

    observed_hash = observed_hashes.get(
        rel_path
    )

    if expected_hash != observed_hash:
        hash_mismatches.append(
            rel_path
        )

# ------------------------------------------------------------
# 16. SIZE RECONCILIATION
# ------------------------------------------------------------

expected_sizes = dict(
    zip(
        expected["relative_path"],
        expected["size_bytes"]
    )
)

observed_sizes = dict(
    zip(
        observed["relative_path"],
        observed["size_bytes"]
    )

)

size_mismatches = []

for rel_path in all_paths:

    expected_size = expected_sizes.get(
        rel_path
    )

    observed_size = observed_sizes.get(
        rel_path
    )

    if expected_size != observed_size:
        size_mismatches.append(
            rel_path
        )

# ------------------------------------------------------------
# 17. TYPE COUNT RECONCILIATION
# ------------------------------------------------------------

expected_type_counts = {
    str(k): int(v)
    for k, v in
    expected["file_type"].value_counts().items()
}

observed_type_counts = {
    str(k): int(v)
    for k, v in
    observed["file_type"].value_counts().items()
}

type_counts_match = (
    expected_type_counts
    == observed_type_counts
)

# ------------------------------------------------------------
# 18. ZERO BYTE CHECK
# ------------------------------------------------------------

zero_byte_files = int(
    observed["zero_byte"].sum()
)

# ------------------------------------------------------------
# 19. DUPLICATE PATH CHECK
# ------------------------------------------------------------

duplicate_relative_paths = int(
    observed["relative_path"]
    .duplicated()
    .sum()
)

duplicate_file_names = int(
    observed["file_name"]
    .duplicated()
    .sum()
)

# ------------------------------------------------------------
# 20. FINAL INTEGRITY DECISION
# ------------------------------------------------------------

integrity_pass = bool(
    count_match
    and exact_inventory_match
    and len(missing_since_c00) == 0
    and len(new_since_c00) == 0
    and len(hash_mismatches) == 0
    and len(size_mismatches) == 0
    and type_counts_match
    and zero_byte_files == 0
    and duplicate_relative_paths == 0
)

# ------------------------------------------------------------
# 21. C02 CANONICAL STATE
# ------------------------------------------------------------

c02_state = {

    "oip_version": "v1.0.33",

    "cell": "C02",

    "cell_name":
        "Canonical Raw File Integrity Verification",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "PASS"
        if integrity_pass
        else "BLOCKED",

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        not integrity_pass,

    "authorization": {

        "construct":
            "NOT_AUTHORIZED",

        "predictor":
            "NOT_AUTHORIZED",

        "outcome":
            "NOT_AUTHORIZED",

        "key":
            "NOT_AUTHORIZED",

        "relationship":
            "NOT_AUTHORIZED",

        "cohort":
            "NOT_AUTHORIZED",

        "score":
            "NOT_AUTHORIZED",

        "empirical_evaluation":
            "NOT_READY"
    },

    "payload": {

        "upstream_c00_status":
            c00["status"],

        "upstream_c01_status":
            c01["status"],

        "source_root":
            str(SOURCE_ROOT),

        "expected_file_count":
            expected_count,

        "observed_file_count":
            observed_count,

        "inventory_count_match":
            count_match,

        "exact_inventory_match":
            exact_inventory_match,

        "expected_type_counts":
            expected_type_counts,

        "observed_type_counts":
            observed_type_counts,

        "type_counts_match":
            type_counts_match,

        "missing_since_c00_count":
            len(missing_since_c00),

        "new_since_c00_count":
            len(new_since_c00),

        "hash_mismatch_count":
            len(hash_mismatches),

        "size_mismatch_count":
            len(size_mismatches),

        "zero_byte_file_count":
            zero_byte_files,

        "duplicate_relative_path_count":
            duplicate_relative_paths,

        "duplicate_file_name_count":
            duplicate_file_names,

        "historical_artifacts_used":
            False,

        "raw_source_only":
            True,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_inference_allowed":
            False,

        "integrity_verification":
            "PASS"
            if integrity_pass
            else "BLOCKED"
    }
}

# ------------------------------------------------------------
# 22. DETAILED AUDIT TABLE
# ------------------------------------------------------------

audit_rows = []

for rel_path in all_paths:

    audit_rows.append({

        "relative_path":
            rel_path,

        "expected_sha256":
            expected_hashes.get(
                rel_path
            ),

        "observed_sha256":
            observed_hashes.get(
                rel_path
            ),

        "hash_match":
            (
                expected_hashes.get(
                    rel_path
                )
                ==
                observed_hashes.get(
                    rel_path
                )
            ),

        "expected_size_bytes":
            expected_sizes.get(
                rel_path
            ),

        "observed_size_bytes":
            observed_sizes.get(
                rel_path
            ),

        "size_match":
            (
                expected_sizes.get(
                    rel_path
                )
                ==
                observed_sizes.get(
                    rel_path
                )
            ),

        "present_in_c00":
            rel_path in expected_paths,

        "present_now":
            rel_path in observed_paths
    })

audit_df = pd.DataFrame(
    audit_rows
)

# ------------------------------------------------------------
# 23. WRITE ARTIFACTS
# ------------------------------------------------------------

audit_path = (
    WORK_ROOT
    / "c02_integrity_audit.csv"
)

status_path = (
    WORK_ROOT
    / "c02_integrity_status.json"
)

audit_df.to_csv(
    audit_path,
    index=False
)

with status_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c02_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 24. OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("UPSTREAM VERIFICATION")
print("-" * 78)

print(
    f"C00 STATUS               : "
    f"{c00['status']}"
)

print(
    f"C01 STATUS               : "
    f"{c01['status']}"
)

print(
    f"Vocabulary locked        : "
    f"{c01_payload['vocabulary_locked']}"
)

print(
    f"Semantic boundary locked : "
    f"{c01_payload['semantic_boundary_locked']}"
)

print()
print("-" * 78)
print("RAW FILE RECONCILIATION")
print("-" * 78)

print(
    f"Expected files           : "
    f"{expected_count}"
)

print(
    f"Observed files           : "
    f"{observed_count}"
)

print(
    f"Count match              : "
    f"{count_match}"
)

print(
    f"Exact inventory match    : "
    f"{exact_inventory_match}"
)

print()
print("-" * 78)
print("TYPE RECONCILIATION")
print("-" * 78)

print(
    f"Expected type counts     : "
    f"{expected_type_counts}"
)

print(
    f"Observed type counts     : "
    f"{observed_type_counts}"
)

print(
    f"Type counts match        : "
    f"{type_counts_match}"
)

print()
print("-" * 78)
print("INTEGRITY AUDIT")
print("-" * 78)

print(
    f"Hash mismatches          : "
    f"{len(hash_mismatches)}"
)

print(
    f"Size mismatches          : "
    f"{len(size_mismatches)}"
)

print(
    f"Missing since C00        : "
    f"{len(missing_since_c00)}"
)

print(
    f"New since C00            : "
    f"{len(new_since_c00)}"
)

print(
    f"Zero-byte files          : "
    f"{zero_byte_files}"
)

print(
    f"Duplicate paths          : "
    f"{duplicate_relative_paths}"
)

print(
    f"Duplicate file names     : "
    f"{duplicate_file_names}"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in c02_state[
    "authorization"
].items():

    print(
        f"{key:24s}: {value}"
    )

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    f"Policy active            : "
    f"{c02_state['fail_closed_policy_active']}"
)

print(
    f"Triggered                : "
    f"{c02_state['fail_closed_triggered']}"
)

print(
    f"C02 STATUS               : "
    f"{c02_state['status']}"
)

print()
print("Canonical artifacts:")
print(audit_path)
print(status_path)

print("=" * 78)

OIP v1.0.33 — ESS4
C02 — CANONICAL RAW FILE INTEGRITY VERIFICATION

------------------------------------------------------------------------------
UPSTREAM VERIFICATION
------------------------------------------------------------------------------
C00 STATUS               : PASS
C01 STATUS               : PASS
Vocabulary locked        : True
Semantic boundary locked : True

------------------------------------------------------------------------------
RAW FILE RECONCILIATION
------------------------------------------------------------------------------
Expected files           : 75
Observed files           : 75
Count match              : True
Exact inventory match    : True

------------------------------------------------------------------------------
TYPE RECONCILIATION
------------------------------------------------------------------------------
Expected type counts     : {'STATA': 68, 'PDF': 7}
Observed type counts     : {'STATA': 68, 'PDF': 7}
Type counts match        : True

---

In [5]:
!pip install -q pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 27.3 MB/s eta 0:00:00


In [6]:
import json
from pathlib import Path

path = Path(
    "/kaggle/working/oip_v1_0_33/c02_integrity_status.json"
)

if not path.exists():
    raise RuntimeError(
        "FAIL-CLOSED: c02_integrity_status.json not found."
    )

c02 = json.loads(
    path.read_text(encoding="utf-8")
)

print("=" * 78)
print("C02 ACTUAL SCHEMA VERIFICATION")
print("=" * 78)

print()
print("C02 top-level keys:")
for key in c02.keys():
    print(f"  {key}")

print()
print("C02 payload keys:")

payload = c02.get("payload")

if not isinstance(payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C02 payload is not a dictionary."
    )

for key in payload.keys():
    print(f"  {key}")

print()
print("C02 authorization keys:")

authorization = c02.get("authorization")

if not isinstance(authorization, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C02 authorization is not a dictionary."
    )

for key in authorization.keys():
    print(f"  {key}")

print("=" * 78)

C02 ACTUAL SCHEMA VERIFICATION

C02 top-level keys:
  oip_version
  cell
  cell_name
  timestamp_utc
  status
  fail_closed_policy_active
  fail_closed_triggered
  authorization
  payload

C02 payload keys:
  upstream_c00_status
  upstream_c01_status
  source_root
  expected_file_count
  observed_file_count
  inventory_count_match
  exact_inventory_match
  expected_type_counts
  observed_type_counts
  type_counts_match
  missing_since_c00_count
  new_since_c00_count
  hash_mismatch_count
  size_mismatch_count
  zero_byte_file_count
  duplicate_relative_path_count
  duplicate_file_name_count
  historical_artifacts_used
  raw_source_only
  semantic_interpretation
  semantic_inference_allowed
  integrity_verification

C02 authorization keys:
  construct
  predictor
  outcome
  key
  relationship
  cohort
  score
  empirical_evaluation


In [7]:
import json
from pathlib import Path

c02_path = Path(
    "/kaggle/working/oip_v1_0_33/c02_integrity_status.json"
)

c02 = json.loads(
    c02_path.read_text(encoding="utf-8")
)

print("=" * 78)
print("C02 ACTUAL PAYLOAD KEYS — FINAL VERIFICATION")
print("=" * 78)

payload = c02.get("payload")

if not isinstance(payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C02 payload is not a dictionary."
    )

for key in payload.keys():
    print(key)

print("=" * 78)

C02 ACTUAL PAYLOAD KEYS — FINAL VERIFICATION
upstream_c00_status
upstream_c01_status
source_root
expected_file_count
observed_file_count
inventory_count_match
exact_inventory_match
expected_type_counts
observed_type_counts
type_counts_match
missing_since_c00_count
new_since_c00_count
hash_mismatch_count
size_mismatch_count
zero_byte_file_count
duplicate_relative_path_count
duplicate_file_name_count
historical_artifacts_used
raw_source_only
semantic_interpretation
semantic_inference_allowed
integrity_verification


In [8]:
# ============================================================
# OIP v1.0.33 — ESS4
# C03 — CANONICAL STATA SCHEMA AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

try:
    import pyreadstat
except ImportError:
    raise RuntimeError(
        "FAIL-CLOSED: pyreadstat is required but not available."
    )

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C03 — CANONICAL STATA SCHEMA AUDIT")
print("=" * 78)

WORK_ROOT = Path(
    "/kaggle/working/oip_v1_0_33"
)

C00_JSON = WORK_ROOT / "c00_source_scope.json"
C00_CSV = WORK_ROOT / "c00_source_inventory.csv"
C01_JSON = WORK_ROOT / "c01_vocabulary_lock.json"
C02_JSON = WORK_ROOT / "c02_integrity_status.json"

for path in [
    C00_JSON,
    C00_CSV,
    C01_JSON,
    C02_JSON
]:
    if not path.exists():
        raise RuntimeError(
            f"FAIL-CLOSED: Required artifact missing: {path}"
        )

# ------------------------------------------------------------
# LOAD UPSTREAM
# ------------------------------------------------------------

with C00_JSON.open("r", encoding="utf-8") as f:
    c00 = json.load(f)

with C01_JSON.open("r", encoding="utf-8") as f:
    c01 = json.load(f)

with C02_JSON.open("r", encoding="utf-8") as f:
    c02 = json.load(f)

c00_inventory = pd.read_csv(C00_CSV)

# ------------------------------------------------------------
# IDENTITY CHECK
# ------------------------------------------------------------

for expected_cell, artifact in [
    ("C00", c00),
    ("C01", c01),
    ("C02", c02)
]:

    if artifact.get("oip_version") != "v1.0.33":
        raise RuntimeError(
            f"FAIL-CLOSED: {expected_cell} version mismatch."
        )

    if artifact.get("cell") != expected_cell:
        raise RuntimeError(
            f"FAIL-CLOSED: {expected_cell} cell identity mismatch."
        )

    if artifact.get("status") != "PASS":
        raise RuntimeError(
            f"FAIL-CLOSED: {expected_cell} status is not PASS."
        )

# ------------------------------------------------------------
# C00 ACTUAL PAYLOAD
# ------------------------------------------------------------

c00_payload = c00.get("payload")

if not isinstance(c00_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C00 payload is not a dictionary."
    )

for key in [
    "source_root",
    "source_scope_locked"
]:
    if key not in c00_payload:
        raise RuntimeError(
            f"FAIL-CLOSED: Missing C00 payload key: {key}"
        )

if c00_payload["source_scope_locked"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C00 source scope is not locked."
    )

# ------------------------------------------------------------
# C01 ACTUAL PAYLOAD
# ------------------------------------------------------------

c01_payload = c01.get("payload")

if not isinstance(c01_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C01 payload is not a dictionary."
    )

for key in [
    "vocabulary_locked",
    "semantic_boundary_locked",
    "semantic_inference_allowed"
]:
    if key not in c01_payload:
        raise RuntimeError(
            f"FAIL-CLOSED: Missing C01 payload key: {key}"
        )

if c01_payload["vocabulary_locked"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C01 vocabulary is not locked."
    )

if c01_payload["semantic_boundary_locked"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C01 semantic boundary is not locked."
    )

if c01_payload["semantic_inference_allowed"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C01 semantic inference boundary invalid."
    )

# ------------------------------------------------------------
# C02 ACTUAL PAYLOAD
# VERIFIED DIRECTLY FROM C02 ARTIFACT
# ------------------------------------------------------------

c02_payload = c02.get("payload")

if not isinstance(c02_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C02 payload is not a dictionary."
    )

verified_c02_keys = [
    "integrity_verification",
    "historical_artifacts_used",
    "raw_source_only"
]

for key in verified_c02_keys:
    if key not in c02_payload:
        raise RuntimeError(
            f"FAIL-CLOSED: Missing verified C02 payload key: {key}"
        )

if c02_payload["integrity_verification"] != "PASS":
    raise RuntimeError(
        "FAIL-CLOSED: C02 integrity verification is not PASS."
    )

if c02_payload["historical_artifacts_used"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: Historical artifacts detected."
    )

if c02_payload["raw_source_only"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C02 raw-source-only boundary invalid."
    )

# ------------------------------------------------------------
# SOURCE ROOT
# ------------------------------------------------------------

SOURCE_ROOT = Path(
    c00_payload["source_root"]
)

if not SOURCE_ROOT.exists():
    raise RuntimeError(
        "FAIL-CLOSED: C00 source root does not exist."
    )

if not SOURCE_ROOT.is_dir():
    raise RuntimeError(
        "FAIL-CLOSED: C00 source root is not a directory."
    )

# ------------------------------------------------------------
# C00 INVENTORY SCHEMA
# ------------------------------------------------------------

required_inventory_columns = [
    "relative_path",
    "file_name",
    "suffix",
    "file_type",
    "size_bytes",
    "sha256",
    "zero_byte"
]

missing_inventory_columns = [
    col
    for col in required_inventory_columns
    if col not in c00_inventory.columns
]

if missing_inventory_columns:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C00 inventory columns: "
        + ", ".join(missing_inventory_columns)
    )

# ------------------------------------------------------------
# STATA INVENTORY
# ------------------------------------------------------------

stata_inventory = c00_inventory[
    c00_inventory["file_type"]
    .astype(str)
    .str.upper()
    == "STATA"
].copy()

if stata_inventory.empty:
    raise RuntimeError(
        "FAIL-CLOSED: No STATA files identified."
    )

# ------------------------------------------------------------
# FILE EXISTENCE
# ------------------------------------------------------------

missing_stata_files = []

for rel_path in stata_inventory["relative_path"]:

    file_path = SOURCE_ROOT / str(rel_path)

    if not file_path.exists():
        missing_stata_files.append(
            str(rel_path)
        )

if missing_stata_files:
    raise RuntimeError(
        "FAIL-CLOSED: Missing STATA source files: "
        + ", ".join(missing_stata_files[:10])
    )

# ------------------------------------------------------------
# PYREADSTAT METADATA EXTRACTION
# ------------------------------------------------------------

metadata_rows = []
file_summary_rows = []
read_errors = []

for _, inventory_row in stata_inventory.iterrows():

    relative_path = str(
        inventory_row["relative_path"]
    )

    file_name = str(
        inventory_row["file_name"]
    )

    file_path = SOURCE_ROOT / relative_path

    try:

        _, meta = pyreadstat.read_dta(
            str(file_path),
            metadataonly=True
        )

        column_names = list(
            meta.column_names
        )

        column_labels = list(
            meta.column_labels
        )

        if len(column_names) != len(column_labels):
            raise RuntimeError(
                "Variable-name and variable-label counts differ."
            )

        variable_count = len(column_names)

        value_labels = getattr(
            meta,
            "value_labels",
            {}
        )

        variable_to_label = getattr(
            meta,
            "variable_to_label",
            {}
        )

        if not isinstance(value_labels, dict):
            value_labels = {}

        if not isinstance(variable_to_label, dict):
            variable_to_label = {}

        variable_label_count = 0
        value_label_variable_count = 0

        for position, variable in enumerate(
            column_names,
            start=1
        ):

            variable_label = column_labels[
                position - 1
            ]

            has_variable_label = bool(
                variable_label is not None
                and
                str(variable_label).strip() != ""
            )

            if has_variable_label:
                variable_label_count += 1

            value_label_name = variable_to_label.get(
                variable
            )

            has_value_labels = bool(
                value_label_name
                and
                value_label_name in value_labels
            )

            if has_value_labels:
                value_label_variable_count += 1

            value_label_count = 0

            if has_value_labels:
                value_label_count = len(
                    value_labels[value_label_name]
                )

            metadata_rows.append({

                "file_name":
                    file_name,

                "relative_path":
                    relative_path,

                "variable":
                    str(variable),

                "position":
                    int(position),

                "variable_label":
                    (
                        ""
                        if variable_label is None
                        else str(variable_label)
                    ),

                "has_variable_label":
                    has_variable_label,

                "has_value_labels":
                    has_value_labels,

                "value_label_count":
                    int(value_label_count)
            })

        file_summary_rows.append({

            "file_name":
                file_name,

            "relative_path":
                relative_path,

            "read_status":
                "READ_OK",

            "variable_count":
                variable_count,

            "variable_label_count":
                variable_label_count,

            "value_label_variable_count":
                value_label_variable_count
        })

    except Exception as exc:

        read_errors.append({

            "file_name":
                file_name,

            "relative_path":
                relative_path,

            "error_type":
                type(exc).__name__,

            "error_message":
                str(exc)
        })

        file_summary_rows.append({

            "file_name":
                file_name,

            "relative_path":
                relative_path,

            "read_status":
                "READ_ERROR",

            "variable_count":
                None,

            "variable_label_count":
                None,

            "value_label_variable_count":
                None
        })

# ------------------------------------------------------------
# DATAFRAMES
# ------------------------------------------------------------

metadata_df = pd.DataFrame(
    metadata_rows
)

file_summary_df = pd.DataFrame(
    file_summary_rows
)

# ------------------------------------------------------------
# READ AUDIT
# ------------------------------------------------------------

read_ok_count = int(
    (
        file_summary_df["read_status"]
        == "READ_OK"
    ).sum()
)

read_error_count = int(
    (
        file_summary_df["read_status"]
        == "READ_ERROR"
    ).sum()
)

file_count_match = bool(
    len(file_summary_df)
    ==
    len(stata_inventory)
)

all_stata_read_ok = bool(
    read_error_count == 0
)

# ------------------------------------------------------------
# METADATA STRUCTURAL AUDIT
# ------------------------------------------------------------

duplicate_file_variable_pairs = int(
    metadata_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .duplicated()
    .sum()
)

duplicate_file_variable_position = int(
    metadata_df[
        [
            "relative_path",
            "position"
        ]
    ]
    .duplicated()
    .sum()
)

file_variable_records = int(
    len(metadata_df)
)

unique_variable_names = int(
    metadata_df["variable"].nunique()
)

unique_file_variable_pairs = int(
    metadata_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

unique_file_position_pairs = int(
    metadata_df[
        [
            "relative_path",
            "position"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

metadata_pair_reconciliation = bool(
    file_variable_records
    ==
    unique_file_variable_pairs
    ==
    unique_file_position_pairs
)

variable_label_records = int(
    metadata_df[
        "has_variable_label"
    ].sum()
)

value_label_pairs = int(
    metadata_df[
        "has_value_labels"
    ].sum()
)

value_label_rows = int(
    metadata_df[
        "value_label_count"
    ].sum()
)

# ------------------------------------------------------------
# AUTHORIZATION BOUNDARY
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}

semantic_interpretation = "NOT_PERFORMED"
semantic_inference_allowed = False

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

schema_pass = bool(

    file_count_match

    and
    all_stata_read_ok

    and
    metadata_pair_reconciliation

    and
    duplicate_file_variable_pairs == 0

    and
    duplicate_file_variable_position == 0
)

# ------------------------------------------------------------
# CANONICAL STATE
# ------------------------------------------------------------

c03_state = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C03",

    "cell_name":
        "Canonical Stata Schema Audit",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        (
            "PASS"
            if schema_pass
            else "BLOCKED"
        ),

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        not schema_pass,

    "authorization":
        authorization,

    "payload": {

        "upstream_c00_status":
            c00["status"],

        "upstream_c01_status":
            c01["status"],

        "upstream_c02_status":
            c02["status"],

        "c02_integrity_verification":
            c02_payload[
                "integrity_verification"
            ],

        "stata_files_in_c00":
            int(len(stata_inventory)),

        "stata_files_processed":
            int(len(file_summary_df)),

        "stata_files_read_ok":
            read_ok_count,

        "stata_files_read_error":
            read_error_count,

        "file_count_match":
            file_count_match,

        "all_stata_read_ok":
            all_stata_read_ok,

        "file_variable_records":
            file_variable_records,

        "unique_variable_names":
            unique_variable_names,

        "unique_file_variable_pairs":
            unique_file_variable_pairs,

        "unique_file_position_pairs":
            unique_file_position_pairs,

        "metadata_pair_reconciliation":
            metadata_pair_reconciliation,

        "duplicate_file_variable_pairs":
            duplicate_file_variable_pairs,

        "duplicate_file_variable_position":
            duplicate_file_variable_position,

        "variable_label_records":
            variable_label_records,

        "value_label_variable_pairs":
            value_label_pairs,

        "value_label_rows":
            value_label_rows,

        "metadata_source":
            "PYREADSTAT",

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False,

        "semantic_interpretation":
            semantic_interpretation,

        "semantic_inference_allowed":
            semantic_inference_allowed,

        "schema_frozen":
            schema_pass,

        "canonical_schema":
            schema_pass
    }
}

# ------------------------------------------------------------
# WRITE ARTIFACTS
# ------------------------------------------------------------

metadata_path = (
    WORK_ROOT
    / "c03_authoritative_variable_metadata.csv"
)

file_summary_path = (
    WORK_ROOT
    / "c03_file_schema_summary.csv"
)

errors_path = (
    WORK_ROOT
    / "c03_read_errors.json"
)

status_path = (
    WORK_ROOT
    / "c03_final_status.json"
)

metadata_df.to_csv(
    metadata_path,
    index=False
)

file_summary_df.to_csv(
    file_summary_path,
    index=False
)

with errors_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        read_errors,
        f,
        indent=2,
        ensure_ascii=False
    )

with status_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c03_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("UPSTREAM")
print("-" * 78)

print(
    f"C00 STATUS               : {c00['status']}"
)

print(
    f"C01 STATUS               : {c01['status']}"
)

print(
    f"C02 STATUS               : {c02['status']}"
)

print()
print("-" * 78)
print("VERIFIED C02 FIELD")
print("-" * 78)

print(
    "integrity_verification  : "
    f"{c02_payload['integrity_verification']}"
)

print()
print("-" * 78)
print("STATA SCHEMA AUDIT")
print("-" * 78)

print(
    f"STATA files in C00       : "
    f"{len(stata_inventory)}"
)

print(
    f"STATA files processed    : "
    f"{len(file_summary_df)}"
)

print(
    f"READ_OK                  : "
    f"{read_ok_count}"
)

print(
    f"READ_ERROR               : "
    f"{read_error_count}"
)

print(
    f"File count match         : "
    f"{file_count_match}"
)

print(
    f"All STATA read OK        : "
    f"{all_stata_read_ok}"
)

print()
print("-" * 78)
print("METADATA")
print("-" * 78)

print(
    f"File-variable records    : "
    f"{file_variable_records}"
)

print(
    f"Unique variable names    : "
    f"{unique_variable_names}"
)

print(
    f"Unique file-variable     : "
    f"{unique_file_variable_pairs}"
)

print(
    f"Unique file-position     : "
    f"{unique_file_position_pairs}"
)

print(
    f"Metadata reconciliation  : "
    f"{metadata_pair_reconciliation}"
)

print(
    f"Duplicate file-variable : "
    f"{duplicate_file_variable_pairs}"
)

print(
    f"Duplicate file-position  : "
    f"{duplicate_file_variable_position}"
)

print(
    f"Variable-label records   : "
    f"{variable_label_records}"
)

print(
    f"Value-label pairs        : "
    f"{value_label_pairs}"
)

print(
    f"Value-label rows         : "
    f"{value_label_rows}"
)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    f"Metadata source          : PYREADSTAT"
)

print(
    f"Raw source only          : True"
)

print(
    f"Historical artifacts     : False"
)

print(
    f"Semantic interpretation  : "
    f"{semantic_interpretation}"
)

print(
    f"Semantic inference       : "
    f"{semantic_inference_allowed}"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in authorization.items():

    print(
        f"{key:24s}: {value}"
    )

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    f"Policy active            : "
    f"{c03_state['fail_closed_policy_active']}"
)

print(
    f"Triggered                : "
    f"{c03_state['fail_closed_triggered']}"
)

print(
    f"C03 STATUS               : "
    f"{c03_state['status']}"
)

print()
print("Canonical artifacts:")
print(metadata_path)
print(file_summary_path)
print(errors_path)
print(status_path)

print("=" * 78)

OIP v1.0.33 — ESS4
C03 — CANONICAL STATA SCHEMA AUDIT

------------------------------------------------------------------------------
UPSTREAM
------------------------------------------------------------------------------
C00 STATUS               : PASS
C01 STATUS               : PASS
C02 STATUS               : PASS

------------------------------------------------------------------------------
VERIFIED C02 FIELD
------------------------------------------------------------------------------
integrity_verification  : PASS

------------------------------------------------------------------------------
STATA SCHEMA AUDIT
------------------------------------------------------------------------------
STATA files in C00       : 68
STATA files processed    : 68
READ_OK                  : 68
READ_ERROR               : 0
File count match         : True
All STATA read OK        : True

------------------------------------------------------------------------------
METADATA
-----------------------

In [9]:
# ============================================================
# OIP v1.0.33 — ESS4
# C04 — VALUE-LABEL EVIDENCE LEDGER
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

try:
    import pyreadstat
except ImportError:
    raise RuntimeError(
        "FAIL-CLOSED: pyreadstat is required but not available."
    )

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C04 — VALUE-LABEL EVIDENCE LEDGER")
print("=" * 78)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORK_ROOT = Path(
    "/kaggle/working/oip_v1_0_33"
)

C03_STATUS = (
    WORK_ROOT / "c03_final_status.json"
)

C03_METADATA = (
    WORK_ROOT / "c03_authoritative_variable_metadata.csv"
)

C00_SCOPE = (
    WORK_ROOT / "c00_source_scope.json"
)

C04_LEDGER = (
    WORK_ROOT / "c04_value_label_ledger.csv"
)

C04_SUMMARY = (
    WORK_ROOT / "c04_value_label_summary.json"
)

C04_STATUS = (
    WORK_ROOT / "c04_final_status.json"
)

# ------------------------------------------------------------
# REQUIRED ARTIFACT CHECK
# ------------------------------------------------------------

required_paths = [
    C03_STATUS,
    C03_METADATA,
    C00_SCOPE
]

for path in required_paths:
    if not path.exists():
        raise RuntimeError(
            f"FAIL-CLOSED: Required artifact missing: {path}"
        )

# ------------------------------------------------------------
# LOAD UPSTREAM
# ------------------------------------------------------------

with C03_STATUS.open(
    "r",
    encoding="utf-8"
) as f:
    c03 = json.load(f)

with C00_SCOPE.open(
    "r",
    encoding="utf-8"
) as f:
    c00 = json.load(f)

c03_payload = c03.get("payload")

if not isinstance(c03_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C03 payload is not a dictionary."
    )

# ------------------------------------------------------------
# C03 IDENTITY
# ------------------------------------------------------------

if c03.get("oip_version") != "v1.0.33":
    raise RuntimeError(
        "FAIL-CLOSED: C03 version mismatch."
    )

if c03.get("cell") != "C03":
    raise RuntimeError(
        "FAIL-CLOSED: C03 cell identity mismatch."
    )

if c03.get("status") != "PASS":
    raise RuntimeError(
        "FAIL-CLOSED: C03 is not PASS."
    )

# ------------------------------------------------------------
# EXACT C03 PAYLOAD KEYS REQUIRED
# ------------------------------------------------------------

required_c03_keys = [
    "stata_files_processed",
    "stata_files_read_ok",
    "file_variable_records",
    "unique_variable_names",
    "value_label_variable_pairs",
    "value_label_rows",
    "raw_source_only",
    "historical_artifacts_used",
    "semantic_interpretation",
    "semantic_inference_allowed"
]

missing_c03_keys = [
    key
    for key in required_c03_keys
    if key not in c03_payload
]

if missing_c03_keys:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C03 payload keys: "
        + ", ".join(missing_c03_keys)
    )

# ------------------------------------------------------------
# UPSTREAM BOUNDARY
# ------------------------------------------------------------

if c03_payload["raw_source_only"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C03 raw-source-only boundary invalid."
    )

if c03_payload["historical_artifacts_used"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: Historical artifacts used by C03."
    )

if c03_payload["semantic_inference_allowed"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C03 semantic inference boundary invalid."
    )

# ------------------------------------------------------------
# C03 METADATA SCHEMA
# ------------------------------------------------------------

metadata_df = pd.read_csv(
    C03_METADATA
)

required_metadata_columns = [
    "file_name",
    "relative_path",
    "variable",
    "position",
    "variable_label",
    "has_variable_label",
    "has_value_labels",
    "value_label_count"
]

missing_metadata_columns = [
    col
    for col in required_metadata_columns
    if col not in metadata_df.columns
]

if missing_metadata_columns:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C03 metadata columns: "
        + ", ".join(missing_metadata_columns)
    )

# ------------------------------------------------------------
# SOURCE ROOT
# ------------------------------------------------------------

c00_payload = c00.get("payload")

if not isinstance(c00_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C00 payload is not a dictionary."
    )

if "source_root" not in c00_payload:
    raise RuntimeError(
        "FAIL-CLOSED: C00 source_root not found."
    )

SOURCE_ROOT = Path(
    c00_payload["source_root"]
)

if not SOURCE_ROOT.exists():
    raise RuntimeError(
        "FAIL-CLOSED: Source root does not exist."
    )

# ------------------------------------------------------------
# C03 REFERENCE VALUES
# ------------------------------------------------------------

c03_file_count = int(
    c03_payload["stata_files_processed"]
)

c03_read_ok = int(
    c03_payload["stata_files_read_ok"]
)

c03_file_variable_records = int(
    c03_payload["file_variable_records"]
)

c03_unique_variable_names = int(
    c03_payload["unique_variable_names"]
)

c03_value_label_pairs = int(
    c03_payload["value_label_variable_pairs"]
)

c03_value_label_rows = int(
    c03_payload["value_label_rows"]
)

# ------------------------------------------------------------
# C03 METADATA VALUE-LABEL VARIABLES
# ------------------------------------------------------------

value_label_metadata = metadata_df[
    metadata_df["has_value_labels"] == True
].copy()

metadata_value_label_pair_count = int(
    len(value_label_metadata)
)

metadata_unique_variables = int(
    value_label_metadata["variable"].nunique()
)

metadata_files_with_value_labels = int(
    value_label_metadata["relative_path"].nunique()
)

if metadata_value_label_pair_count != c03_value_label_pairs:
    raise RuntimeError(
        "FAIL-CLOSED: C03 value-label pair count "
        "does not reconcile with its metadata artifact."
    )

# ------------------------------------------------------------
# LEDGER EXTRACTION
# ------------------------------------------------------------

ledger_rows = []
read_errors = []

processed_files = 0
read_ok_files = 0

for relative_path in (
    metadata_df["relative_path"]
    .drop_duplicates()
    .tolist()
):

    relative_path = str(
        relative_path
    )

    file_path = SOURCE_ROOT / relative_path

    if not file_path.exists():
        read_errors.append({
            "relative_path":
                relative_path,
            "error_type":
                "FILE_NOT_FOUND",
            "error_message":
                "Source file does not exist."
        })
        continue

    processed_files += 1

    try:

        _, meta = pyreadstat.read_dta(
            str(file_path),
            metadataonly=True
        )

        read_ok_files += 1

        value_labels = getattr(
            meta,
            "value_labels",
            {}
        )

        variable_to_label = getattr(
            meta,
            "variable_to_label",
            {}
        )

        if not isinstance(
            value_labels,
            dict
        ):
            value_labels = {}

        if not isinstance(
            variable_to_label,
            dict
        ):
            variable_to_label = {}

        file_metadata = value_label_metadata[
            value_label_metadata[
                "relative_path"
            ].astype(str)
            == relative_path
        ]

        for _, row in file_metadata.iterrows():

            variable = str(
                row["variable"]
            )

            label_set_name = (
                variable_to_label.get(
                    variable
                )
            )

            if not label_set_name:
                raise RuntimeError(
                    "FAIL-CLOSED: C03 says variable has "
                    "value labels but pyreadstat does not "
                    "provide a linked value-label set."
                )

            if label_set_name not in value_labels:
                raise RuntimeError(
                    "FAIL-CLOSED: Linked value-label set "
                    "not present in pyreadstat metadata."
                )

            label_set = value_labels[
                label_set_name
            ]

            if not isinstance(
                label_set,
                dict
            ):
                raise RuntimeError(
                    "FAIL-CLOSED: Value-label set is not a dictionary."
                )

            for value, label in label_set.items():

                ledger_rows.append({

                    "relative_path":
                        relative_path,

                    "file_name":
                        str(row["file_name"]),

                    "variable":
                        variable,

                    "position":
                        int(row["position"]),

                    "value_label_set":
                        str(label_set_name),

                    "value":
                        value,

                    "label":
                        (
                            ""
                            if label is None
                            else str(label)
                        )
                })

    except Exception as exc:

        read_errors.append({
            "relative_path":
                relative_path,
            "error_type":
                type(exc).__name__,
            "error_message":
                str(exc)
        })

# ------------------------------------------------------------
# LEDGER DATAFRAME
# ------------------------------------------------------------

ledger_df = pd.DataFrame(
    ledger_rows
)

ledger_columns = [
    "relative_path",
    "file_name",
    "variable",
    "position",
    "value_label_set",
    "value",
    "label"
]

if ledger_df.empty:

    ledger_df = pd.DataFrame(
        columns=ledger_columns
    )

else:

    ledger_df = ledger_df[
        ledger_columns
    ]

# ------------------------------------------------------------
# LEDGER STRUCTURAL CHECKS
# ------------------------------------------------------------

ledger_rows_count = int(
    len(ledger_df)
)

ledger_duplicate_rows = int(
    ledger_df[
        [
            "relative_path",
            "variable",
            "value",
            "label"
        ]
    ]
    .duplicated()
    .sum()
)

ledger_file_variable_pairs = int(
    ledger_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

ledger_unique_variables = int(
    ledger_df[
        "variable"
    ].nunique()
)

ledger_files = int(
    ledger_df[
        "relative_path"
    ].nunique()
)

# ------------------------------------------------------------
# RECONCILIATION
# ------------------------------------------------------------

ledger_rows_match_c03 = bool(
    ledger_rows_count
    ==
    c03_value_label_rows
)

ledger_pairs_match_c03 = bool(
    ledger_file_variable_pairs
    ==
    c03_value_label_pairs
)

ledger_no_duplicates = bool(
    ledger_duplicate_rows == 0
)

files_match_c03 = bool(
    processed_files
    ==
    c03_file_count
)

read_ok_match_c03 = bool(
    read_ok_files
    ==
    c03_read_ok
)

read_errors_zero = bool(
    len(read_errors) == 0
)

# ------------------------------------------------------------
# IMPORTANT SEMANTIC BOUNDARY
# ------------------------------------------------------------

semantic_interpretation = (
    "NOT_PERFORMED"
)

semantic_inference_allowed = False

# ------------------------------------------------------------
# AUTHORIZATION
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

c04_pass = bool(

    files_match_c03

    and
    read_ok_match_c03

    and
    read_errors_zero

    and
    ledger_rows_match_c03

    and
    ledger_pairs_match_c03

    and
    ledger_no_duplicates

    and
    semantic_inference_allowed is False

)

# ------------------------------------------------------------
# CANONICAL ARTIFACT
# ------------------------------------------------------------

c04_state = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C04",

    "cell_name":
        "Value-Label Evidence Ledger",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        (
            "PASS"
            if c04_pass
            else "BLOCKED"
        ),

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        not c04_pass,

    "authorization":
        authorization,

    "payload": {

        "upstream_c03_status":
            c03["status"],

        "stata_files_processed":
            processed_files,

        "stata_files_read_ok":
            read_ok_files,

        "stata_files_read_error":
            len(read_errors),

        "c03_stata_files_processed":
            c03_file_count,

        "c03_stata_files_read_ok":
            c03_read_ok,

        "c03_file_variable_records":
            c03_file_variable_records,

        "c03_unique_variable_names":
            c03_unique_variable_names,

        "c03_value_label_variable_pairs":
            c03_value_label_pairs,

        "c03_value_label_rows":
            c03_value_label_rows,

        "metadata_value_label_pairs":
            metadata_value_label_pair_count,

        "metadata_unique_variables":
            metadata_unique_variables,

        "metadata_files_with_value_labels":
            metadata_files_with_value_labels,

        "ledger_rows":
            ledger_rows_count,

        "ledger_file_variable_pairs":
            ledger_file_variable_pairs,

        "ledger_unique_variables":
            ledger_unique_variables,

        "ledger_files":
            ledger_files,

        "ledger_duplicate_rows":
            ledger_duplicate_rows,

        "ledger_rows_match_c03":
            ledger_rows_match_c03,

        "ledger_pairs_match_c03":
            ledger_pairs_match_c03,

        "files_match_c03":
            files_match_c03,

        "read_ok_match_c03":
            read_ok_match_c03,

        "read_errors_zero":
            read_errors_zero,

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False,

        "semantic_interpretation":
            semantic_interpretation,

        "semantic_inference_allowed":
            semantic_inference_allowed,

        "ledger_unit":
            "FILE_VARIABLE_VALUE_LABEL_RECORD"
    }
}

# ------------------------------------------------------------
# WRITE ARTIFACTS
# ------------------------------------------------------------

ledger_df.to_csv(
    C04_LEDGER,
    index=False
)

with C04_SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "oip_version":
                "v1.0.33",

            "cell":
                "C04",

            "cell_name":
                "Value-Label Evidence Ledger",

            "status":
                c04_state["status"],

            "payload":
                c04_state["payload"]
        },
        f,
        indent=2,
        ensure_ascii=False
    )

with C04_STATUS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c04_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("UPSTREAM")
print("-" * 78)

print(
    f"C03 STATUS               : {c03['status']}"
)

print(
    f"C03 STATA files          : {c03_file_count}"
)

print(
    f"C03 READ_OK              : {c03_read_ok}"
)

print()
print("-" * 78)
print("VALUE-LABEL EXTRACTION")
print("-" * 78)

print(
    f"STATA files processed    : {processed_files}"
)

print(
    f"READ_OK                  : {read_ok_files}"
)

print(
    f"READ_ERROR               : {len(read_errors)}"
)

print(
    f"Metadata label pairs     : "
    f"{metadata_value_label_pair_count}"
)

print(
    f"Ledger rows              : "
    f"{ledger_rows_count}"
)

print(
    f"Ledger file-variable     : "
    f"{ledger_file_variable_pairs}"
)

print(
    f"Ledger unique variables  : "
    f"{ledger_unique_variables}"
)

print(
    f"Ledger files             : "
    f"{ledger_files}"
)

print(
    f"Ledger duplicate rows    : "
    f"{ledger_duplicate_rows}"
)

print()
print("-" * 78)
print("C03 RECONCILIATION")
print("-" * 78)

print(
    f"Rows vs C03              : "
    f"{ledger_rows_match_c03}"
)

print(
    f"Pairs vs C03             : "
    f"{ledger_pairs_match_c03}"
)

print(
    f"Files vs C03             : "
    f"{files_match_c03}"
)

print(
    f"READ_OK vs C03           : "
    f"{read_ok_match_c03}"
)

print(
    f"Read errors zero         : "
    f"{read_errors_zero}"
)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    f"Raw source only          : True"
)

print(
    f"Historical artifacts     : False"
)

print(
    f"Semantic interpretation  : "
    f"{semantic_interpretation}"
)

print(
    f"Semantic inference       : "
    f"{semantic_inference_allowed}"
)

print(
    "Ledger unit              : "
    "FILE_VARIABLE_VALUE_LABEL_RECORD"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in authorization.items():

    print(
        f"{key:24s}: {value}"
    )

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    f"Policy active            : True"
)

print(
    f"Triggered                : "
    f"{not c04_pass}"
)

print(
    f"C04 STATUS               : "
    f"{c04_state['status']}"
)

print()
print("Canonical artifacts:")
print(C04_LEDGER)
print(C04_SUMMARY)
print(C04_STATUS)

print("=" * 78)

OIP v1.0.33 — ESS4
C04 — VALUE-LABEL EVIDENCE LEDGER

------------------------------------------------------------------------------
UPSTREAM
------------------------------------------------------------------------------
C03 STATUS               : PASS
C03 STATA files          : 68
C03 READ_OK              : 68

------------------------------------------------------------------------------
VALUE-LABEL EXTRACTION
------------------------------------------------------------------------------
STATA files processed    : 68
READ_OK                  : 68
READ_ERROR               : 0
Metadata label pairs     : 959
Ledger rows              : 7077
Ledger file-variable     : 959
Ledger unique variables  : 767
Ledger files             : 67
Ledger duplicate rows    : 0

------------------------------------------------------------------------------
C03 RECONCILIATION
------------------------------------------------------------------------------
Rows vs C03              : True
Pairs vs C03          

In [10]:
# ============================================================
# OIP v1.0.33 — ESS4
# C05 — METADATA AND VALUE-LABEL RECONCILIATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C05 — METADATA AND VALUE-LABEL RECONCILIATION")
print("=" * 78)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORK_ROOT = Path(
    "/kaggle/working/oip_v1_0_33"
)

C03_STATUS = WORK_ROOT / "c03_final_status.json"
C03_METADATA = WORK_ROOT / "c03_authoritative_variable_metadata.csv"

C04_STATUS = WORK_ROOT / "c04_final_status.json"
C04_SUMMARY = WORK_ROOT / "c04_value_label_summary.json"
C04_LEDGER = WORK_ROOT / "c04_value_label_ledger.csv"

C05_AUDIT = WORK_ROOT / "c05_metadata_reconciliation.csv"
C05_SUMMARY = WORK_ROOT / "c05_metadata_reconciliation_summary.json"
C05_STATUS = WORK_ROOT / "c05_final_status.json"

# ------------------------------------------------------------
# REQUIRED ARTIFACTS
# ------------------------------------------------------------

required_paths = [
    C03_STATUS,
    C03_METADATA,
    C04_STATUS,
    C04_SUMMARY,
    C04_LEDGER
]

for path in required_paths:
    if not path.exists():
        raise RuntimeError(
            f"FAIL-CLOSED: Required artifact missing: {path}"
        )

# ------------------------------------------------------------
# LOAD ARTIFACTS
# ------------------------------------------------------------

with C03_STATUS.open(
    "r",
    encoding="utf-8"
) as f:
    c03 = json.load(f)

with C04_STATUS.open(
    "r",
    encoding="utf-8"
) as f:
    c04 = json.load(f)

with C04_SUMMARY.open(
    "r",
    encoding="utf-8"
) as f:
    c04_summary = json.load(f)

c03_payload = c03.get("payload")
c04_payload = c04.get("payload")

if not isinstance(c03_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C03 payload is not a dictionary."
    )

if not isinstance(c04_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C04 payload is not a dictionary."
    )

# ------------------------------------------------------------
# UPSTREAM IDENTITY
# ------------------------------------------------------------

for artifact, expected_cell in [
    (c03, "C03"),
    (c04, "C04")
]:

    if artifact.get("oip_version") != "v1.0.33":
        raise RuntimeError(
            f"FAIL-CLOSED: {expected_cell} version mismatch."
        )

    if artifact.get("cell") != expected_cell:
        raise RuntimeError(
            f"FAIL-CLOSED: {expected_cell} identity mismatch."
        )

    if artifact.get("status") != "PASS":
        raise RuntimeError(
            f"FAIL-CLOSED: {expected_cell} is not PASS."
        )

# ------------------------------------------------------------
# EXACT UPSTREAM PAYLOAD KEYS
# ------------------------------------------------------------

required_c03_keys = [
    "stata_files_processed",
    "stata_files_read_ok",
    "file_variable_records",
    "unique_variable_names",
    "value_label_variable_pairs",
    "value_label_rows",
    "raw_source_only",
    "historical_artifacts_used",
    "semantic_interpretation",
    "semantic_inference_allowed"
]

required_c04_keys = [
    "stata_files_processed",
    "stata_files_read_ok",
    "stata_files_read_error",
    "c03_stata_files_processed",
    "c03_stata_files_read_ok",
    "c03_file_variable_records",
    "c03_unique_variable_names",
    "c03_value_label_variable_pairs",
    "c03_value_label_rows",
    "metadata_value_label_pairs",
    "metadata_unique_variables",
    "metadata_files_with_value_labels",
    "ledger_rows",
    "ledger_file_variable_pairs",
    "ledger_unique_variables",
    "ledger_files",
    "ledger_duplicate_rows",
    "ledger_rows_match_c03",
    "ledger_pairs_match_c03",
    "files_match_c03",
    "read_ok_match_c03",
    "read_errors_zero",
    "raw_source_only",
    "historical_artifacts_used",
    "semantic_interpretation",
    "semantic_inference_allowed",
    "ledger_unit"
]

missing_c03_keys = [
    key
    for key in required_c03_keys
    if key not in c03_payload
]

missing_c04_keys = [
    key
    for key in required_c04_keys
    if key not in c04_payload
]

if missing_c03_keys:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C03 payload keys: "
        + ", ".join(missing_c03_keys)
    )

if missing_c04_keys:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C04 payload keys: "
        + ", ".join(missing_c04_keys)
    )

# ------------------------------------------------------------
# UPSTREAM BOUNDARY
# ------------------------------------------------------------

if c03_payload["raw_source_only"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C03 raw-source boundary invalid."
    )

if c04_payload["raw_source_only"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C04 raw-source boundary invalid."
    )

if c03_payload["historical_artifacts_used"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C03 used historical artifacts."
    )

if c04_payload["historical_artifacts_used"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C04 used historical artifacts."
    )

if c03_payload["semantic_inference_allowed"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C03 semantic inference boundary invalid."
    )

if c04_payload["semantic_inference_allowed"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: C04 semantic inference boundary invalid."
    )

# ------------------------------------------------------------
# LOAD TABLES
# ------------------------------------------------------------

metadata_df = pd.read_csv(
    C03_METADATA
)

ledger_df = pd.read_csv(
    C04_LEDGER
)

# ------------------------------------------------------------
# VERIFY ACTUAL C03 METADATA SCHEMA
# ------------------------------------------------------------

required_metadata_columns = [
    "file_name",
    "relative_path",
    "variable",
    "position",
    "variable_label",
    "has_variable_label",
    "has_value_labels",
    "value_label_count"
]

missing_metadata_columns = [
    col
    for col in required_metadata_columns
    if col not in metadata_df.columns
]

if missing_metadata_columns:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C03 metadata columns: "
        + ", ".join(missing_metadata_columns)
    )

# ------------------------------------------------------------
# VERIFY ACTUAL C04 LEDGER SCHEMA
# ------------------------------------------------------------

required_ledger_columns = [
    "relative_path",
    "file_name",
    "variable",
    "position",
    "value_label_set",
    "value",
    "label"
]

missing_ledger_columns = [
    col
    for col in required_ledger_columns
    if col not in ledger_df.columns
]

if missing_ledger_columns:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C04 ledger columns: "
        + ", ".join(missing_ledger_columns)
    )

# ------------------------------------------------------------
# C03 METADATA COUNTS
# ------------------------------------------------------------

metadata_rows = int(
    len(metadata_df)
)

metadata_files = int(
    metadata_df[
        "relative_path"
    ].nunique()
)

metadata_unique_variables = int(
    metadata_df[
        "variable"
    ].nunique()
)

metadata_file_variable_pairs = int(
    metadata_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

metadata_file_position_pairs = int(
    metadata_df[
        [
            "relative_path",
            "position"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

metadata_duplicate_file_variable = int(
    metadata_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .duplicated()
    .sum()
)

metadata_duplicate_file_position = int(
    metadata_df[
        [
            "relative_path",
            "position"
        ]
    ]
    .duplicated()
    .sum()
)

metadata_variable_label_records = int(
    metadata_df[
        "has_variable_label"
    ].sum()
)

metadata_value_label_pairs = int(
    metadata_df[
        "has_value_labels"
    ].sum()
)

metadata_value_label_rows = int(
    metadata_df[
        "value_label_count"
    ].sum()
)

# ------------------------------------------------------------
# C04 LEDGER COUNTS
# ------------------------------------------------------------

ledger_rows = int(
    len(ledger_df)
)

ledger_files = int(
    ledger_df[
        "relative_path"
    ].nunique()
)

ledger_unique_variables = int(
    ledger_df[
        "variable"
    ].nunique()
)

ledger_file_variable_pairs = int(
    ledger_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

ledger_duplicate_rows = int(
    ledger_df[
        [
            "relative_path",
            "variable",
            "value",
            "label"
        ]
    ]
    .duplicated()
    .sum()
)

# ------------------------------------------------------------
# C04 PAYLOAD RECONCILIATION
# ------------------------------------------------------------

c04_payload_ledger_rows = int(
    c04_payload["ledger_rows"]
)

c04_payload_ledger_pairs = int(
    c04_payload["ledger_file_variable_pairs"]
)

c04_payload_ledger_files = int(
    c04_payload["ledger_files"]
)

c04_payload_unique_variables = int(
    c04_payload["ledger_unique_variables"]
)

c04_payload_duplicate_rows = int(
    c04_payload["ledger_duplicate_rows"]
)

c04_payload_rows_reconciled = bool(
    ledger_rows
    ==
    c04_payload_ledger_rows
)

c04_payload_pairs_reconciled = bool(
    ledger_file_variable_pairs
    ==
    c04_payload_ledger_pairs
)

c04_payload_files_reconciled = bool(
    ledger_files
    ==
    c04_payload_ledger_files
)

c04_payload_unique_variables_reconciled = bool(
    ledger_unique_variables
    ==
    c04_payload_unique_variables
)

c04_payload_duplicates_reconciled = bool(
    ledger_duplicate_rows
    ==
    c04_payload_duplicate_rows
)

# ------------------------------------------------------------
# C03 ↔ C04 RECONCILIATION
# ------------------------------------------------------------

files_match = bool(
    c04_payload["stata_files_processed"]
    ==
    c03_payload["stata_files_processed"]
)

read_ok_match = bool(
    c04_payload["stata_files_read_ok"]
    ==
    c03_payload["stata_files_read_ok"]
)

file_variable_records_match = bool(
    c04_payload["c03_file_variable_records"]
    ==
    c03_payload["file_variable_records"]
)

unique_variable_names_match = bool(
    c04_payload["c03_unique_variable_names"]
    ==
    c03_payload["unique_variable_names"]
)

value_label_pairs_match = bool(
    c04_payload["c03_value_label_variable_pairs"]
    ==
    c03_payload["value_label_variable_pairs"]
)

value_label_rows_match = bool(
    c04_payload["c03_value_label_rows"]
    ==
    c03_payload["value_label_rows"]
)

metadata_pairs_match_ledger = bool(
    metadata_value_label_pairs
    ==
    ledger_file_variable_pairs
)

metadata_rows_match_ledger = bool(
    metadata_value_label_rows
    ==
    ledger_rows
)

# ------------------------------------------------------------
# STRUCTURAL RECONCILIATION
# ------------------------------------------------------------

metadata_unique_pair_integrity = bool(
    metadata_rows
    ==
    metadata_file_variable_pairs
)

metadata_position_integrity = bool(
    metadata_rows
    ==
    metadata_file_position_pairs
)

no_metadata_duplicates = bool(
    metadata_duplicate_file_variable == 0
    and
    metadata_duplicate_file_position == 0
)

no_ledger_duplicates = bool(
    ledger_duplicate_rows == 0
)

# ------------------------------------------------------------
# C04 INTERNAL RECONCILIATION
# ------------------------------------------------------------

c04_internal_reconciliation = bool(
    c04_payload["ledger_rows_match_c03"] is True
    and
    c04_payload["ledger_pairs_match_c03"] is True
    and
    c04_payload["files_match_c03"] is True
    and
    c04_payload["read_ok_match_c03"] is True
    and
    c04_payload["read_errors_zero"] is True
)

# ------------------------------------------------------------
# FINAL RECONCILIATION
# ------------------------------------------------------------

c05_pass = bool(

    files_match

    and
    read_ok_match

    and
    file_variable_records_match

    and
    unique_variable_names_match

    and
    value_label_pairs_match

    and
    value_label_rows_match

    and
    metadata_pairs_match_ledger

    and
    metadata_rows_match_ledger

    and
    metadata_unique_pair_integrity

    and
    metadata_position_integrity

    and
    no_metadata_duplicates

    and
    no_ledger_duplicates

    and
    c04_internal_reconciliation

    and
    c04_payload_rows_reconciled

    and
    c04_payload_pairs_reconciled

    and
    c04_payload_files_reconciled

    and
    c04_payload_unique_variables_reconciled

    and
    c04_payload_duplicates_reconciled
)

# ------------------------------------------------------------
# AUTHORIZATION BOUNDARY
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}

# ------------------------------------------------------------
# CANONICAL STATE
# ------------------------------------------------------------

c05_state = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C05",

    "cell_name":
        "Metadata and Value-Label Reconciliation",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        (
            "PASS"
            if c05_pass
            else "BLOCKED"
        ),

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        not c05_pass,

    "authorization":
        authorization,

    "payload": {

        "upstream_c03_status":
            c03["status"],

        "upstream_c04_status":
            c04["status"],

        "metadata_rows":
            metadata_rows,

        "metadata_files":
            metadata_files,

        "metadata_unique_variables":
            metadata_unique_variables,

        "metadata_file_variable_pairs":
            metadata_file_variable_pairs,

        "metadata_file_position_pairs":
            metadata_file_position_pairs,

        "metadata_duplicate_file_variable":
            metadata_duplicate_file_variable,

        "metadata_duplicate_file_position":
            metadata_duplicate_file_position,

        "metadata_variable_label_records":
            metadata_variable_label_records,

        "metadata_value_label_pairs":
            metadata_value_label_pairs,

        "metadata_value_label_rows":
            metadata_value_label_rows,

        "ledger_rows":
            ledger_rows,

        "ledger_files":
            ledger_files,

        "ledger_unique_variables":
            ledger_unique_variables,

        "ledger_file_variable_pairs":
            ledger_file_variable_pairs,

        "ledger_duplicate_rows":
            ledger_duplicate_rows,

        "files_match":
            files_match,

        "read_ok_match":
            read_ok_match,

        "file_variable_records_match":
            file_variable_records_match,

        "unique_variable_names_match":
            unique_variable_names_match,

        "value_label_pairs_match":
            value_label_pairs_match,

        "value_label_rows_match":
            value_label_rows_match,

        "metadata_pairs_match_ledger":
            metadata_pairs_match_ledger,

        "metadata_rows_match_ledger":
            metadata_rows_match_ledger,

        "metadata_unique_pair_integrity":
            metadata_unique_pair_integrity,

        "metadata_position_integrity":
            metadata_position_integrity,

        "no_metadata_duplicates":
            no_metadata_duplicates,

        "no_ledger_duplicates":
            no_ledger_duplicates,

        "c04_internal_reconciliation":
            c04_internal_reconciliation,

        "c04_payload_rows_reconciled":
            c04_payload_rows_reconciled,

        "c04_payload_pairs_reconciled":
            c04_payload_pairs_reconciled,

        "c04_payload_files_reconciled":
            c04_payload_files_reconciled,

        "c04_payload_unique_variables_reconciled":
            c04_payload_unique_variables_reconciled,

        "c04_payload_duplicates_reconciled":
            c04_payload_duplicates_reconciled,

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_inference_allowed":
            False
    }
}

# ------------------------------------------------------------
# AUDIT TABLE
# ------------------------------------------------------------

audit_rows = [

    {
        "check":
            "C03 metadata rows",
        "value":
            metadata_rows,
        "status":
            "PASS"
    },

    {
        "check":
            "C03 unique variables",
        "value":
            metadata_unique_variables,
        "status":
            "PASS"
    },

    {
        "check":
            "C03 value-label pairs",
        "value":
            metadata_value_label_pairs,
        "status":
            "PASS"
            if metadata_value_label_pairs
            ==
            c03_value_label_pairs
            else "FAIL"
    },

    {
        "check":
            "C03 value-label rows",
        "value":
            metadata_value_label_rows,
        "status":
            "PASS"
            if metadata_value_label_rows
            ==
            c03_value_label_rows
            else "FAIL"
    },

    {
        "check":
            "C04 ledger rows",
        "value":
            ledger_rows,
        "status":
            "PASS"
            if ledger_rows
            ==
            c03_value_label_rows
            else "FAIL"
    },

    {
        "check":
            "C04 ledger file-variable pairs",
        "value":
            ledger_file_variable_pairs,
        "status":
            "PASS"
            if ledger_file_variable_pairs
            ==
            c03_value_label_pairs
            else "FAIL"
    },

    {
        "check":
            "Ledger duplicate rows",
        "value":
            ledger_duplicate_rows,
        "status":
            "PASS"
            if ledger_duplicate_rows == 0
            else "FAIL"
    }
]

audit_df = pd.DataFrame(
    audit_rows
)

# ------------------------------------------------------------
# WRITE ARTIFACTS
# ------------------------------------------------------------

audit_df.to_csv(
    C05_AUDIT,
    index=False
)

with C05_SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c05_state,
        f,
        indent=2,
        ensure_ascii=False
    )

with C05_STATUS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c05_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("UPSTREAM")
print("-" * 78)

print(
    f"C03 STATUS               : {c03['status']}"
)

print(
    f"C04 STATUS               : {c04['status']}"
)

print()
print("-" * 78)
print("METADATA")
print("-" * 78)

print(
    f"Metadata rows            : {metadata_rows}"
)

print(
    f"Metadata files           : {metadata_files}"
)

print(
    f"Unique variables         : {metadata_unique_variables}"
)

print(
    f"File-variable pairs      : "
    f"{metadata_file_variable_pairs}"
)

print(
    f"File-position pairs      : "
    f"{metadata_file_position_pairs}"
)

print(
    f"Variable-label records   : "
    f"{metadata_variable_label_records}"
)

print(
    f"Value-label pairs        : "
    f"{metadata_value_label_pairs}"
)

print(
    f"Value-label rows         : "
    f"{metadata_value_label_rows}"
)

print()
print("-" * 78)
print("LEDGER")
print("-" * 78)

print(
    f"Ledger rows              : "
    f"{ledger_rows}"
)

print(
    f"Ledger files             : "
    f"{ledger_files}"
)

print(
    f"Ledger unique variables  : "
    f"{ledger_unique_variables}"
)

print(
    f"Ledger file-variable     : "
    f"{ledger_file_variable_pairs}"
)

print(
    f"Ledger duplicate rows    : "
    f"{ledger_duplicate_rows}"
)

print()
print("-" * 78)
print("RECONCILIATION")
print("-" * 78)

print(
    f"Files match              : {files_match}"
)

print(
    f"READ_OK match            : {read_ok_match}"
)

print(
    f"File-variable match      : "
    f"{file_variable_records_match}"
)

print(
    f"Unique-variable match    : "
    f"{unique_variable_names_match}"
)

print(
    f"Value-label pairs match  : "
    f"{value_label_pairs_match}"
)

print(
    f"Value-label rows match   : "
    f"{value_label_rows_match}"
)

print(
    f"Metadata ↔ ledger pairs  : "
    f"{metadata_pairs_match_ledger}"
)

print(
    f"Metadata ↔ ledger rows   : "
    f"{metadata_rows_match_ledger}"
)

print(
    f"Metadata pair integrity  : "
    f"{metadata_unique_pair_integrity}"
)

print(
    f"Metadata position check  : "
    f"{metadata_position_integrity}"
)

print(
    f"No metadata duplicates   : "
    f"{no_metadata_duplicates}"
)

print(
    f"No ledger duplicates     : "
    f"{no_ledger_duplicates}"
)

print(
    f"C04 internal reconcil.   : "
    f"{c04_internal_reconciliation}"
)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    "Raw source only          : True"
)

print(
    "Historical artifacts     : False"
)

print(
    "Semantic interpretation  : NOT_PERFORMED"
)

print(
    "Semantic inference       : False"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in authorization.items():

    print(
        f"{key:24s}: {value}"
    )

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    "Policy active            : True"
)

print(
    f"Triggered                : "
    f"{not c05_pass}"
)

print(
    f"C05 STATUS               : "
    f"{c05_state['status']}"
)

print()
print("Canonical artifacts:")
print(C05_AUDIT)
print(C05_SUMMARY)
print(C05_STATUS)

print("=" * 78)

OIP v1.0.33 — ESS4
C05 — METADATA AND VALUE-LABEL RECONCILIATION

------------------------------------------------------------------------------
UPSTREAM
------------------------------------------------------------------------------
C03 STATUS               : PASS
C04 STATUS               : PASS

------------------------------------------------------------------------------
METADATA
------------------------------------------------------------------------------
Metadata rows            : 2550
Metadata files           : 68
Unique variables         : 1625
File-variable pairs      : 2550
File-position pairs      : 2550
Variable-label records   : 2550
Value-label pairs        : 959
Value-label rows         : 7077

------------------------------------------------------------------------------
LEDGER
------------------------------------------------------------------------------
Ledger rows              : 7077
Ledger files             : 67
Ledger unique variables  : 767
Ledger file-variable   

In [11]:
# ============================================================
# OIP v1.0.33 — ESS4
# C06 — IDENTIFIER CANDIDATE DISCOVERY
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C06 — IDENTIFIER CANDIDATE DISCOVERY")
print("=" * 78)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORK_ROOT = Path(
    "/kaggle/working/oip_v1_0_33"
)

C05_STATUS = WORK_ROOT / "c05_final_status.json"
C05_METADATA = WORK_ROOT / "c03_authoritative_variable_metadata.csv"

C06_LEDGER = WORK_ROOT / "c06_identifier_candidate_ledger.csv"
C06_SUMMARY = WORK_ROOT / "c06_identifier_discovery_summary.json"
C06_STATUS = WORK_ROOT / "c06_final_status.json"

# ------------------------------------------------------------
# REQUIRED ARTIFACTS
# ------------------------------------------------------------

for path in [
    C05_STATUS,
    C05_METADATA
]:
    if not path.exists():
        raise RuntimeError(
            f"FAIL-CLOSED: Required artifact missing: {path}"
        )

# ------------------------------------------------------------
# LOAD C05
# ------------------------------------------------------------

with C05_STATUS.open(
    "r",
    encoding="utf-8"
) as f:
    c05 = json.load(f)

if c05.get("oip_version") != "v1.0.33":
    raise RuntimeError(
        "FAIL-CLOSED: C05 version mismatch."
    )

if c05.get("cell") != "C05":
    raise RuntimeError(
        "FAIL-CLOSED: C05 cell identity mismatch."
    )

if c05.get("status") != "PASS":
    raise RuntimeError(
        "FAIL-CLOSED: C05 is not PASS."
    )

c05_payload = c05.get("payload")

if not isinstance(c05_payload, dict):
    raise RuntimeError(
        "FAIL-CLOSED: C05 payload is not a dictionary."
    )

# ------------------------------------------------------------
# ACTUAL C05 PAYLOAD KEY VALIDATION
# ------------------------------------------------------------

required_c05_keys = [
    "metadata_rows",
    "metadata_files",
    "metadata_unique_variables",
    "metadata_file_variable_pairs",
    "metadata_file_position_pairs",
    "metadata_duplicate_file_variable",
    "metadata_duplicate_file_position",
    "metadata_variable_label_records",
    "metadata_value_label_pairs",
    "metadata_value_label_rows",
    "ledger_rows",
    "ledger_files",
    "ledger_unique_variables",
    "ledger_file_variable_pairs",
    "ledger_duplicate_rows",
    "raw_source_only",
    "historical_artifacts_used",
    "semantic_interpretation",
    "semantic_inference_allowed"
]

missing_c05_keys = [
    key
    for key in required_c05_keys
    if key not in c05_payload
]

if missing_c05_keys:
    raise RuntimeError(
        "FAIL-CLOSED: Missing C05 payload keys: "
        + ", ".join(missing_c05_keys)
    )

# ------------------------------------------------------------
# UPSTREAM BOUNDARY
# ------------------------------------------------------------

if c05_payload["raw_source_only"] is not True:
    raise RuntimeError(
        "FAIL-CLOSED: C05 raw-source boundary invalid."
    )

if c05_payload["historical_artifacts_used"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: Historical artifacts used."
    )

if c05_payload["semantic_inference_allowed"] is not False:
    raise RuntimeError(
        "FAIL-CLOSED: Semantic inference boundary invalid."
    )

# ------------------------------------------------------------
# LOAD C03 METADATA
# ------------------------------------------------------------

metadata_df = pd.read_csv(
    C05_METADATA
)

# ------------------------------------------------------------
# ACTUAL METADATA SCHEMA VALIDATION
# ------------------------------------------------------------

required_metadata_columns = [
    "file_name",
    "relative_path",
    "variable",
    "position",
    "variable_label",
    "has_variable_label",
    "has_value_labels",
    "value_label_count"
]

missing_metadata_columns = [
    col
    for col in required_metadata_columns
    if col not in metadata_df.columns
]

if missing_metadata_columns:
    raise RuntimeError(
        "FAIL-CLOSED: Missing metadata columns: "
        + ", ".join(missing_metadata_columns)
    )

# ------------------------------------------------------------
# CANDIDATE DEFINITION
#
# Basis:
#   VARIABLE NAME + VARIABLE LABEL
#
# Matching:
#   PREDEFINED TERM SUBSTRING
#
# Interpretation:
#   DISCOVERY ONLY
#
# No semantic authorization is performed.
# ------------------------------------------------------------

IDENTIFIER_TERMS = [
    "identifier",
    "id",
    "code",
    "key"
]

# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

def normalize_text(value):

    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return str(value).strip().lower()

# ------------------------------------------------------------
# DISCOVERY
# ------------------------------------------------------------

candidate_rows = []

for _, row in metadata_df.iterrows():

    variable = str(
        row["variable"]
    )

    variable_label = normalize_text(
        row["variable_label"]
    )

    variable_name_normalized = normalize_text(
        variable
    )

    combined_text = (
        variable_name_normalized
        + " "
        + variable_label
    )

    matched_terms = []

    for term in IDENTIFIER_TERMS:

        if term in combined_text:

            matched_terms.append(
                term
            )

    if not matched_terms:
        continue

    candidate_rows.append({

        "relative_path":
            str(row["relative_path"]),

        "file_name":
            str(row["file_name"]),

        "variable":
            variable,

        "position":
            int(row["position"]),

        "variable_label":
            (
                ""
                if pd.isna(row["variable_label"])
                else str(row["variable_label"])
            ),

        "identifier_terms_matched":
            "|".join(
                matched_terms
            ),

        "candidate_basis":
            "VARIABLE_NAME_AND_VARIABLE_LABEL",

        "matching_rule":
            "PREDEFINED_TERM_SUBSTRING",

        "discovery_only":
            True,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "identifier_authorization":
            "NOT_AUTHORIZED",

        "key_authorization":
            "NOT_AUTHORIZED",

        "relationship_authorization":
            "NOT_AUTHORIZED",

        "cohort_authorization":
            "NOT_AUTHORIZED",

        "construct_authorization":
            "NOT_AUTHORIZED",

        "semantic_inference_allowed":
            False
    })

# ------------------------------------------------------------
# DATAFRAME
# ------------------------------------------------------------

candidate_df = pd.DataFrame(
    candidate_rows
)

candidate_columns = [
    "relative_path",
    "file_name",
    "variable",
    "position",
    "variable_label",
    "identifier_terms_matched",
    "candidate_basis",
    "matching_rule",
    "discovery_only",
    "semantic_interpretation",
    "identifier_authorization",
    "key_authorization",
    "relationship_authorization",
    "cohort_authorization",
    "construct_authorization",
    "semantic_inference_allowed"
]

if candidate_df.empty:

    candidate_df = pd.DataFrame(
        columns=candidate_columns
    )

else:

    candidate_df = candidate_df[
        candidate_columns
    ]

# ------------------------------------------------------------
# COUNTS
# ------------------------------------------------------------

candidate_records = int(
    len(candidate_df)
)

unique_candidate_variables = int(
    candidate_df[
        "variable"
    ].nunique()
)

candidate_files = int(
    candidate_df[
        "relative_path"
    ].nunique()
)

# ------------------------------------------------------------
# DUPLICATE CHECK
# ------------------------------------------------------------

duplicate_candidate_records = int(
    candidate_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .duplicated()
    .sum()
)

# ------------------------------------------------------------
# TERM MATCH AUDIT
# ------------------------------------------------------------

empty_term_matches = int(
    (
        candidate_df[
            "identifier_terms_matched"
        ]
        .astype(str)
        .str.strip()
        .eq("")
    ).sum()
)

discovery_only_violations = int(
    (
        candidate_df[
            "discovery_only"
        ]
        != True
    ).sum()
)

semantic_inference_leaks = int(
    (
        candidate_df[
            "semantic_inference_allowed"
        ]
        != False
    ).sum()
)

authorization_leaks = 0

authorization_columns = [
    "identifier_authorization",
    "key_authorization",
    "relationship_authorization",
    "cohort_authorization",
    "construct_authorization"
]

for column in authorization_columns:

    authorization_leaks += int(
        (
            candidate_df[column]
            .astype(str)
            .str.upper()
            !=
            "NOT_AUTHORIZED"
        ).sum()
    )

# ------------------------------------------------------------
# C05 METADATA RECONCILIATION
# ------------------------------------------------------------

c05_metadata_rows = int(
    c05_payload["metadata_rows"]
)

c05_metadata_unique_variables = int(
    c05_payload[
        "metadata_unique_variables"
    ]
)

metadata_source_rows_match = bool(
    len(metadata_df)
    ==
    c05_metadata_rows
)

metadata_source_unique_variables_match = bool(
    metadata_df["variable"].nunique()
    ==
    c05_metadata_unique_variables
)

# ------------------------------------------------------------
# STATUS
# ------------------------------------------------------------

c06_pass = bool(

    metadata_source_rows_match

    and
    metadata_source_unique_variables_match

    and
    duplicate_candidate_records == 0

    and
    empty_term_matches == 0

    and
    discovery_only_violations == 0

    and
    semantic_inference_leaks == 0

    and
    authorization_leaks == 0
)

# ------------------------------------------------------------
# AUTHORIZATION
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}

# ------------------------------------------------------------
# CANONICAL STATE
# ------------------------------------------------------------

c06_state = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C06",

    "cell_name":
        "Identifier Candidate Discovery",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        (
            "PASS"
            if c06_pass
            else "BLOCKED"
        ),

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        not c06_pass,

    "authorization":
        authorization,

    "payload": {

        "upstream_c05_status":
            c05["status"],

        "candidate_definition":
            "VARIABLE_NAME_AND_VARIABLE_LABEL",

        "matching_rule":
            "PREDEFINED_TERM_SUBSTRING",

        "identifier_terms":
            IDENTIFIER_TERMS,

        "candidate_records":
            candidate_records,

        "unique_candidate_variables":
            unique_candidate_variables,

        "candidate_files":
            candidate_files,

        "duplicate_candidate_records":
            duplicate_candidate_records,

        "empty_term_matches":
            empty_term_matches,

        "discovery_only_violations":
            discovery_only_violations,

        "semantic_inference_leaks":
            semantic_inference_leaks,

        "authorization_leaks":
            authorization_leaks,

        "c05_metadata_rows":
            c05_metadata_rows,

        "c05_metadata_unique_variables":
            c05_metadata_unique_variables,

        "metadata_source_rows_match":
            metadata_source_rows_match,

        "metadata_source_unique_variables_match":
            metadata_source_unique_variables_match,

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_inference_allowed":
            False,

        "discovery_only":
            True
    }
}

# ------------------------------------------------------------
# WRITE ARTIFACTS
# ------------------------------------------------------------

candidate_df.to_csv(
    C06_LEDGER,
    index=False
)

with C06_SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c06_state,
        f,
        indent=2,
        ensure_ascii=False
    )

with C06_STATUS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c06_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("UPSTREAM")
print("-" * 78)

print(
    f"C05 STATUS               : {c05['status']}"
)

print(
    f"C05 metadata rows        : "
    f"{c05_metadata_rows}"
)

print(
    f"C05 unique variables     : "
    f"{c05_metadata_unique_variables}"
)

print()
print("-" * 78)
print("CANDIDATE DISCOVERY")
print("-" * 78)

print(
    "Candidate definition     : "
    "VARIABLE_NAME_AND_VARIABLE_LABEL"
)

print(
    "Matching rule            : "
    "PREDEFINED_TERM_SUBSTRING"
)

print(
    "Identifier terms         : "
    + ", ".join(IDENTIFIER_TERMS)
)

print(
    f"Candidate records        : "
    f"{candidate_records}"
)

print(
    f"Unique candidate vars    : "
    f"{unique_candidate_variables}"
)

print(
    f"Candidate files          : "
    f"{candidate_files}"
)

print(
    f"Duplicate candidates     : "
    f"{duplicate_candidate_records}"
)

print(
    f"Empty term matches       : "
    f"{empty_term_matches}"
)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    f"Discovery only           : True"
)

print(
    f"Semantic interpretation  : NOT_PERFORMED"
)

print(
    f"Semantic inference       : False"
)

print(
    f"Authorization leaks     : "
    f"{authorization_leaks}"
)

print(
    f"Discovery-only violations: "
    f"{discovery_only_violations}"
)

print(
    f"Semantic inference leaks : "
    f"{semantic_inference_leaks}"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in authorization.items():

    print(
        f"{key:24s}: {value}"
    )

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    "Policy active            : True"
)

print(
    f"Triggered                : "
    f"{not c06_pass}"
)

print(
    f"C06 STATUS               : "
    f"{c06_state['status']}"
)

print()
print("Canonical artifacts:")
print(C06_LEDGER)
print(C06_SUMMARY)
print(C06_STATUS)

print("=" * 78)

OIP v1.0.33 — ESS4
C06 — IDENTIFIER CANDIDATE DISCOVERY

------------------------------------------------------------------------------
UPSTREAM
------------------------------------------------------------------------------
C05 STATUS               : PASS
C05 metadata rows        : 2550
C05 unique variables     : 1625

------------------------------------------------------------------------------
CANDIDATE DISCOVERY
------------------------------------------------------------------------------
Candidate definition     : VARIABLE_NAME_AND_VARIABLE_LABEL
Matching rule            : PREDEFINED_TERM_SUBSTRING
Identifier terms         : identifier, id, code, key
Candidate records        : 1044
Unique candidate vars    : 378
Candidate files          : 67
Duplicate candidates     : 0
Empty term matches       : 0

------------------------------------------------------------------------------
BOUNDARY
------------------------------------------------------------------------------
Discovery only  

In [12]:
# ============================================================
# OIP v1.0.33 — ESS4
# C07 — IDENTIFIER STRUCTURAL UNIQUENESS AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C07 — IDENTIFIER STRUCTURAL UNIQUENESS AUDIT")
print("=" * 78)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORK_ROOT = Path(
    "/kaggle/working/oip_v1_0_33"
)

C06_STATUS = (
    WORK_ROOT / "c06_final_status.json"
)

C06_LEDGER = (
    WORK_ROOT / "c06_identifier_candidate_ledger.csv"
)

C07_PROFILE = (
    WORK_ROOT / "c07_identifier_structural_profile.csv"
)

C07_UNIQUE = (
    WORK_ROOT / "c07_structurally_unique_candidates.csv"
)

C07_SUMMARY = (
    WORK_ROOT / "c07_identifier_structural_summary.json"
)

C07_STATUS = (
    WORK_ROOT / "c07_final_status.json"
)

# ------------------------------------------------------------
# REQUIRED ARTIFACTS
# ------------------------------------------------------------

for path in [
    C06_STATUS,
    C06_LEDGER
]:

    if not path.exists():

        raise RuntimeError(
            f"FAIL-CLOSED: Required artifact missing: {path}"
        )

# ------------------------------------------------------------
# LOAD C06
# ------------------------------------------------------------

with C06_STATUS.open(
    "r",
    encoding="utf-8"
) as f:

    c06 = json.load(f)

if c06.get("oip_version") != "v1.0.33":

    raise RuntimeError(
        "FAIL-CLOSED: C06 version mismatch."
    )

if c06.get("cell") != "C06":

    raise RuntimeError(
        "FAIL-CLOSED: C06 cell identity mismatch."
    )

if c06.get("status") != "PASS":

    raise RuntimeError(
        "FAIL-CLOSED: C06 is not PASS."
    )

c06_payload = c06.get("payload")

if not isinstance(c06_payload, dict):

    raise RuntimeError(
        "FAIL-CLOSED: C06 payload is not a dictionary."
    )

# ------------------------------------------------------------
# EXACT C06 PAYLOAD KEYS
# ------------------------------------------------------------

required_c06_keys = [

    "candidate_definition",
    "matching_rule",
    "identifier_terms",
    "candidate_records",
    "unique_candidate_variables",
    "candidate_files",
    "duplicate_candidate_records",
    "empty_term_matches",
    "discovery_only_violations",
    "semantic_inference_leaks",
    "authorization_leaks",
    "raw_source_only",
    "historical_artifacts_used",
    "semantic_interpretation",
    "semantic_inference_allowed",
    "discovery_only"
]

missing_c06_keys = [

    key
    for key in required_c06_keys
    if key not in c06_payload
]

if missing_c06_keys:

    raise RuntimeError(
        "FAIL-CLOSED: Missing C06 payload keys: "
        + ", ".join(missing_c06_keys)
    )

# ------------------------------------------------------------
# UPSTREAM BOUNDARY
# ------------------------------------------------------------

if c06_payload["raw_source_only"] is not True:

    raise RuntimeError(
        "FAIL-CLOSED: C06 raw-source boundary invalid."
    )

if c06_payload["historical_artifacts_used"] is not False:

    raise RuntimeError(
        "FAIL-CLOSED: Historical artifacts used."
    )

if c06_payload["semantic_inference_allowed"] is not False:

    raise RuntimeError(
        "FAIL-CLOSED: Semantic inference boundary invalid."
    )

if c06_payload["discovery_only"] is not True:

    raise RuntimeError(
        "FAIL-CLOSED: C06 discovery-only boundary invalid."
    )

# ------------------------------------------------------------
# LOAD CANDIDATE LEDGER
# ------------------------------------------------------------

candidate_df = pd.read_csv(
    C06_LEDGER
)

# ------------------------------------------------------------
# ACTUAL C06 SCHEMA
# ------------------------------------------------------------

required_columns = [

    "relative_path",
    "file_name",
    "variable",
    "position",
    "variable_label",
    "identifier_terms_matched",
    "candidate_basis",
    "matching_rule",
    "discovery_only",
    "semantic_interpretation",
    "identifier_authorization",
    "key_authorization",
    "relationship_authorization",
    "cohort_authorization",
    "construct_authorization",
    "semantic_inference_allowed"
]

missing_columns = [

    col
    for col in required_columns
    if col not in candidate_df.columns
]

if missing_columns:

    raise RuntimeError(
        "FAIL-CLOSED: Missing C06 columns: "
        + ", ".join(missing_columns)
    )

# ------------------------------------------------------------
# SOURCE RECONCILIATION
# ------------------------------------------------------------

candidate_records = int(
    len(candidate_df)
)

c06_candidate_records = int(
    c06_payload["candidate_records"]
)

candidate_record_count_match = bool(
    candidate_records
    ==
    c06_candidate_records
)

if not candidate_record_count_match:

    raise RuntimeError(
        "FAIL-CLOSED: C06 candidate ledger count "
        "does not match C06 payload."
    )

# ------------------------------------------------------------
# STRUCTURAL DUPLICATE CHECK
#
# One candidate record is identified by:
#   relative_path + variable
#
# This is structural only.
# It does NOT establish identifier meaning.
# ------------------------------------------------------------

duplicate_file_variable = int(

    candidate_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .duplicated()
    .sum()
)

if duplicate_file_variable != 0:

    raise RuntimeError(
        "FAIL-CLOSED: Duplicate file-variable "
        "candidate records detected."
    )

# ------------------------------------------------------------
# STRUCTURAL UNIQUENESS
# ------------------------------------------------------------

grouped = (

    candidate_df
    .groupby(
        "variable",
        dropna=False
    )
    .agg(
        structurally_unique_record_count=(
            "relative_path",
            "nunique"
        ),
        file_count=(
            "relative_path",
            "nunique"
        ),
        files=(
            "file_name",
            lambda x:
                "|".join(
                    sorted(
                        set(
                            x.astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
)

grouped[
    "structural_uniqueness"
] = grouped[
    "structurally_unique_record_count"
].eq(1).map({

    True:
        "OBSERVED_STRUCTURALLY_UNIQUE",

    False:
        "OBSERVED_STRUCTURALLY_NONUNIQUE"
})

# ------------------------------------------------------------
# STRUCTURALLY UNIQUE FILE-VARIABLE RECORDS
# ------------------------------------------------------------

unique_records = (

    candidate_df[
        candidate_df[
            "variable"
        ].isin(

            grouped.loc[
                grouped[
                    "structurally_unique_record_count"
                ]
                == 1,
                "variable"
            ]
        )
    ]
    .copy()
)

unique_records[
    "structural_uniqueness"
] = (
    "OBSERVED_STRUCTURALLY_UNIQUE"
)

# ------------------------------------------------------------
# STRUCTURAL COUNTS
# ------------------------------------------------------------

unique_variable_names = int(
    grouped[
        "variable"
    ].nunique()
)

structurally_unique_variables = int(

    grouped.loc[
        grouped[
            "structurally_unique_record_count"
        ]
        == 1,
        "variable"
    ].nunique()
)

structurally_nonunique_variables = int(

    grouped.loc[
        grouped[
            "structurally_unique_record_count"
        ]
        > 1,
        "variable"
    ].nunique()
)

structurally_unique_records = int(
    len(unique_records)
)

structurally_nonunique_records = (

    candidate_records
    -
    structurally_unique_records
)

candidate_files = int(
    candidate_df[
        "relative_path"
    ].nunique()
)

# ------------------------------------------------------------
# ACCOUNTING
# ------------------------------------------------------------

structural_accounting_match = bool(

    structurally_unique_records
    +
    structurally_nonunique_records
    ==
    candidate_records
)

if not structural_accounting_match:

    raise RuntimeError(
        "FAIL-CLOSED: Structural accounting mismatch."
    )

# ------------------------------------------------------------
# BOUNDARY CHECKS
# ------------------------------------------------------------

semantic_inference_leaks = int(

    (
        candidate_df[
            "semantic_inference_allowed"
        ]
        != False
    )
    .sum()
)

discovery_only_violations = int(

    (
        candidate_df[
            "discovery_only"
        ]
        != True
    )
    .sum()
)

authorization_leaks = 0

authorization_columns = [

    "identifier_authorization",
    "key_authorization",
    "relationship_authorization",
    "cohort_authorization",
    "construct_authorization"
]

for column in authorization_columns:

    authorization_leaks += int(

        (
            candidate_df[column]
            .astype(str)
            .str.upper()
            !=
            "NOT_AUTHORIZED"
        )
        .sum()
    )

structural_semantic_violations = int(

    (
        candidate_df[
            "semantic_interpretation"
        ]
        .astype(str)
        .str.upper()
        !=
        "NOT_PERFORMED"
    )
    .sum()
)

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

c07_pass = bool(

    candidate_record_count_match

    and
    duplicate_file_variable == 0

    and
    structural_accounting_match

    and
    semantic_inference_leaks == 0

    and
    discovery_only_violations == 0

    and
    authorization_leaks == 0

    and
    structural_semantic_violations == 0
)

# ------------------------------------------------------------
# AUTHORIZATION
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}

# ------------------------------------------------------------
# CANONICAL STATE
# ------------------------------------------------------------

c07_state = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C07",

    "cell_name":
        "Identifier Structural Uniqueness Audit",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        (
            "PASS"
            if c07_pass
            else "BLOCKED"
        ),

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        not c07_pass,

    "authorization":
        authorization,

    "payload": {

        "upstream_c06_status":
            c06["status"],

        "candidate_records":
            candidate_records,

        "c06_candidate_records":
            c06_candidate_records,

        "candidate_record_count_match":
            candidate_record_count_match,

        "unique_variable_names":
            unique_variable_names,

        "candidate_files":
            candidate_files,

        "structurally_unique_variables":
            structurally_unique_variables,

        "structurally_nonunique_variables":
            structurally_nonunique_variables,

        "structurally_unique_records":
            structurally_unique_records,

        "structurally_nonunique_records":
            structurally_nonunique_records,

        "duplicate_file_variable":
            duplicate_file_variable,

        "structural_accounting_match":
            structural_accounting_match,

        "structural_uniqueness":
            "OBSERVED_ONLY",

        "semantic_identification":
            "NOT_PERFORMED",

        "semantic_inference_allowed":
            False,

        "semantic_inference_leaks":
            semantic_inference_leaks,

        "discovery_only":
            True,

        "discovery_only_violations":
            discovery_only_violations,

        "structural_semantic_violations":
            structural_semantic_violations,

        "authorization_leaks":
            authorization_leaks,

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False,

        "semantic_interpretation":
            "NOT_PERFORMED"
    }
}

# ------------------------------------------------------------
# WRITE ARTIFACTS
# ------------------------------------------------------------

grouped.to_csv(
    C07_PROFILE,
    index=False
)

unique_records.to_csv(
    C07_UNIQUE,
    index=False
)

with C07_SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c07_state,
        f,
        indent=2,
        ensure_ascii=False
    )

with C07_STATUS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        c07_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("UPSTREAM")
print("-" * 78)

print(
    f"C06 STATUS               : "
    f"{c06['status']}"
)

print(
    f"C06 candidate records    : "
    f"{c06_candidate_records}"
)

print()
print("-" * 78)
print("STRUCTURAL AUDIT")
print("-" * 78)

print(
    f"Candidate records        : "
    f"{candidate_records}"
)

print(
    f"Candidate files          : "
    f"{candidate_files}"
)

print(
    f"Unique variable names    : "
    f"{unique_variable_names}"
)

print(
    f"Structurally unique vars : "
    f"{structurally_unique_variables}"
)

print(
    f"Structurally nonunique   : "
    f"{structurally_nonunique_variables}"
)

print(
    f"Structurally unique rec. : "
    f"{structurally_unique_records}"
)

print(
    f"Structurally nonunique rec.: "
    f"{structurally_nonunique_records}"
)

print(
    f"Duplicate file-variable  : "
    f"{duplicate_file_variable}"
)

print(
    f"Accounting match         : "
    f"{structural_accounting_match}"
)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    "Structural uniqueness    : OBSERVED_ONLY"
)

print(
    "Semantic identification  : NOT_PERFORMED"
)

print(
    "Discovery only           : True"
)

print(
    f"Semantic inference leaks : "
    f"{semantic_inference_leaks}"
)

print(
    f"Authorization leaks      : "
    f"{authorization_leaks}"
)

print(
    f"Structural semantic leaks: "
    f"{structural_semantic_violations}"
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in authorization.items():

    print(
        f"{key:24s}: {value}"
    )

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    "Policy active            : True"
)

print(
    f"Triggered                : "
    f"{not c07_pass}"
)

print(
    f"C07 STATUS               : "
    f"{c07_state['status']}"
)

print()
print("Canonical artifacts:")
print(C07_PROFILE)
print(C07_UNIQUE)
print(C07_SUMMARY)
print(C07_STATUS)

print("=" * 78)

OIP v1.0.33 — ESS4
C07 — IDENTIFIER STRUCTURAL UNIQUENESS AUDIT

------------------------------------------------------------------------------
UPSTREAM
------------------------------------------------------------------------------
C06 STATUS               : PASS
C06 candidate records    : 1044

------------------------------------------------------------------------------
STRUCTURAL AUDIT
------------------------------------------------------------------------------
Candidate records        : 1044
Candidate files          : 67
Unique variable names    : 378
Structurally unique vars : 353
Structurally nonunique   : 25
Structurally unique rec. : 353
Structurally nonunique rec.: 691
Duplicate file-variable  : 0
Accounting match         : True

------------------------------------------------------------------------------
BOUNDARY
------------------------------------------------------------------------------
Structural uniqueness    : OBSERVED_ONLY
Semantic identification  : NOT_PERFORMED

In [13]:
# ============================================================
# OIP v1.0.33 — ESS4
# C08 — DOCUMENTARY EVIDENCE READINESS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C08 — DOCUMENTARY EVIDENCE READINESS")
print("=" * 78)

WORK_ROOT = Path("/kaggle/working/oip_v1_0_33")

C00 = WORK_ROOT / "c00_source_scope.json"
C02 = WORK_ROOT / "c02_integrity_status.json"
C04 = WORK_ROOT / "c04_final_status.json"
C06 = WORK_ROOT / "c06_final_status.json"
C07 = WORK_ROOT / "c07_final_status.json"

C08_INVENTORY = WORK_ROOT / "c08_documentary_evidence_inventory.csv"
C08_SUMMARY = WORK_ROOT / "c08_documentary_evidence_summary.json"
C08_STATUS = WORK_ROOT / "c08_final_status.json"

for p in [C00, C02, C04, C06, C07]:
    if not p.exists():
        raise RuntimeError(
            f"FAIL-CLOSED: Required upstream artifact missing: {p}"
        )

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

c00 = load_json(C00)
c02 = load_json(C02)
c04 = load_json(C04)
c06 = load_json(C06)
c07 = load_json(C07)

for obj, cell in [
    (c00, "C00"),
    (c02, "C02"),
    (c04, "C04"),
    (c06, "C06"),
    (c07, "C07")
]:
    if obj.get("status") != "PASS":
        raise RuntimeError(
            f"FAIL-CLOSED: {cell} is not PASS."
        )

# ------------------------------------------------------------
# SOURCE ROOT
# ------------------------------------------------------------

source_root = Path(
    c00["payload"]["source_root"]
)

if not source_root.exists():
    raise RuntimeError(
        "FAIL-CLOSED: C00 source root does not exist."
    )

# ------------------------------------------------------------
# ACTUAL PDF INVENTORY
# ------------------------------------------------------------

pdf_files = sorted(
    [
        p for p in source_root.rglob("*")
        if p.is_file()
        and p.suffix.lower() == ".pdf"
    ]
)

if len(pdf_files) == 0:
    raise RuntimeError(
        "FAIL-CLOSED: No documentary PDF files found."
    )

records = []

for p in pdf_files:

    records.append({
        "relative_path":
            str(p.relative_to(source_root)),

        "file_name":
            p.name,

        "suffix":
            p.suffix.lower(),

        "size_bytes":
            int(p.stat().st_size),

        "nonzero":
            bool(p.stat().st_size > 0),

        "documentary_source":
            True,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "construct_authorization":
            "NOT_AUTHORIZED",

        "predictor_authorization":
            "NOT_AUTHORIZED",

        "outcome_authorization":
            "NOT_AUTHORIZED",

        "key_authorization":
            "NOT_AUTHORIZED",

        "relationship_authorization":
            "NOT_AUTHORIZED",

        "cohort_authorization":
            "NOT_AUTHORIZED",

        "score_authorization":
            "NOT_AUTHORIZED",

        "empirical_evaluation":
            "NOT_READY",

        "semantic_inference_allowed":
            False
    })

df = pd.DataFrame(records)

# ------------------------------------------------------------
# INTEGRITY CHECKS
# ------------------------------------------------------------

pdf_count = int(len(df))
zero_byte = int((~df["nonzero"]).sum())
duplicate_paths = int(df["relative_path"].duplicated().sum())
duplicate_names = int(df["file_name"].duplicated().sum())

if zero_byte != 0:
    raise RuntimeError(
        "FAIL-CLOSED: Zero-byte documentary file detected."
    )

if duplicate_paths != 0:
    raise RuntimeError(
        "FAIL-CLOSED: Duplicate documentary paths detected."
    )

if duplicate_names != 0:
    raise RuntimeError(
        "FAIL-CLOSED: Duplicate documentary filenames detected."
    )

# ------------------------------------------------------------
# UPSTREAM C02 RECONCILIATION
# ------------------------------------------------------------

c02_pdf_count = int(
    c02["payload"]["observed_type_counts"]["PDF"]
)

pdf_count_match = bool(
    pdf_count == c02_pdf_count
)

if not pdf_count_match:
    raise RuntimeError(
        "FAIL-CLOSED: PDF count does not reconcile with C02."
    )

# ------------------------------------------------------------
# BOUNDARY AUDIT
# ------------------------------------------------------------

semantic_inference_leaks = int(
    (
        df["semantic_inference_allowed"]
        != False
    ).sum()
)

authorization_leaks = 0

for col in [
    "construct_authorization",
    "predictor_authorization",
    "outcome_authorization",
    "key_authorization",
    "relationship_authorization",
    "cohort_authorization",
    "score_authorization"
]:

    authorization_leaks += int(
        (
            df[col]
            .astype(str)
            .str.upper()
            != "NOT_AUTHORIZED"
        ).sum()
    )

semantic_interpretation_leaks = int(
    (
        df["semantic_interpretation"]
        .astype(str)
        .str.upper()
        != "NOT_PERFORMED"
    ).sum()
)

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

status = "PASS"

if (
    pdf_count == 0
    or
    zero_byte != 0
    or
    duplicate_paths != 0
    or
    duplicate_names != 0
    or
    not pdf_count_match
    or
    semantic_inference_leaks != 0
    or
    authorization_leaks != 0
    or
    semantic_interpretation_leaks != 0
):

    status = "BLOCKED"

state = {
    "oip_version": "v1.0.33",
    "cell": "C08",
    "cell_name": "Documentary Evidence Readiness",
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),
    "status": status,
    "fail_closed_policy_active": True,
    "fail_closed_triggered": status != "PASS",

    "authorization": {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY"
    },

    "payload": {

        "source_root": str(source_root),

        "pdf_count": pdf_count,

        "c02_pdf_count": c02_pdf_count,

        "pdf_count_match": pdf_count_match,

        "zero_byte_files": zero_byte,

        "duplicate_relative_paths": duplicate_paths,

        "duplicate_file_names": duplicate_names,

        "documentary_source":
            "AVAILABLE_FOR_TARGETED_REVIEW",

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_inference_allowed":
            False,

        "semantic_inference_leaks":
            semantic_inference_leaks,

        "semantic_interpretation_leaks":
            semantic_interpretation_leaks,

        "authorization_leaks":
            authorization_leaks,

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False
    }
}

df.to_csv(
    C08_INVENTORY,
    index=False
)

with C08_SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        state,
        f,
        indent=2,
        ensure_ascii=False
    )

with C08_STATUS.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        state,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print("-" * 78)
print("UPSTREAM")
print("-" * 78)
print("C00 STATUS               :", c00["status"])
print("C02 STATUS               :", c02["status"])
print("C04 STATUS               :", c04["status"])
print("C06 STATUS               :", c06["status"])
print("C07 STATUS               :", c07["status"])

print()
print("-" * 78)
print("DOCUMENTARY SOURCE")
print("-" * 78)
print("PDF files                :", pdf_count)
print("C02 PDF count            :", c02_pdf_count)
print("PDF count match          :", pdf_count_match)
print("Zero-byte files          :", zero_byte)
print("Duplicate paths          :", duplicate_paths)
print("Duplicate names          :", duplicate_names)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)
print("Documentary source       : AVAILABLE_FOR_TARGETED_REVIEW")
print("Semantic interpretation  : NOT_PERFORMED")
print("Semantic inference       : False")
print("Semantic inference leaks :", semantic_inference_leaks)
print("Authorization leaks      :", authorization_leaks)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for k, v in state["authorization"].items():
    print(f"{k:24s}: {v}")

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)
print("Policy active            : True")
print("Triggered                :", status != "PASS")
print("C08 STATUS               :", status)

print()
print("Canonical artifacts:")
print(C08_INVENTORY)
print(C08_SUMMARY)
print(C08_STATUS)

print("=" * 78)

OIP v1.0.33 — ESS4
C08 — DOCUMENTARY EVIDENCE READINESS

------------------------------------------------------------------------------
UPSTREAM
------------------------------------------------------------------------------
C00 STATUS               : PASS
C02 STATUS               : PASS
C04 STATUS               : PASS
C06 STATUS               : PASS
C07 STATUS               : PASS

------------------------------------------------------------------------------
DOCUMENTARY SOURCE
------------------------------------------------------------------------------
PDF files                : 7
C02 PDF count            : 7
PDF count match          : True
Zero-byte files          : 0
Duplicate paths          : 0
Duplicate names          : 0

------------------------------------------------------------------------------
BOUNDARY
------------------------------------------------------------------------------
Documentary source       : AVAILABLE_FOR_TARGETED_REVIEW
Semantic interpretation  : NOT_PERFO

In [14]:
from pathlib import Path

WORK_ROOT = Path("/kaggle/working/oip_v1_0_33")

print("=" * 78)
print("OIP v1.0.33 — ACTUAL ARTIFACT INVENTORY")
print("=" * 78)

files = sorted(
    p for p in WORK_ROOT.iterdir()
    if p.is_file()
)

print("Working directory:", WORK_ROOT)
print("Total files      :", len(files))
print()

for p in files:
    print(p.name)

print("=" * 78)

OIP v1.0.33 — ACTUAL ARTIFACT INVENTORY
Working directory: /kaggle/working/oip_v1_0_33
Total files      : 25

c00_source_inventory.csv
c00_source_scope.json
c01_vocabulary_lock.json
c02_integrity_audit.csv
c02_integrity_status.json
c03_authoritative_variable_metadata.csv
c03_file_schema_summary.csv
c03_final_status.json
c03_read_errors.json
c04_final_status.json
c04_value_label_ledger.csv
c04_value_label_summary.json
c05_final_status.json
c05_metadata_reconciliation.csv
c05_metadata_reconciliation_summary.json
c06_final_status.json
c06_identifier_candidate_ledger.csv
c06_identifier_discovery_summary.json
c07_final_status.json
c07_identifier_structural_profile.csv
c07_identifier_structural_summary.json
c07_structurally_unique_candidates.csv
c08_documentary_evidence_inventory.csv
c08_documentary_evidence_summary.json
c08_final_status.json


In [15]:
# ============================================================
# OIP v1.0.33 — ESS4
# C09 — STRUCTURAL RECONCILIATION & LINEAGE AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C09 — STRUCTURAL RECONCILIATION & LINEAGE AUDIT")
print("=" * 78)

WORK_ROOT = Path("/kaggle/working/oip_v1_0_33")

# ------------------------------------------------------------
# ACTUAL ARTIFACTS FROM CLEAN RUN
# ------------------------------------------------------------

PATHS = {
    "C01": WORK_ROOT / "c01_vocabulary_lock.json",
    "C03": WORK_ROOT / "c03_final_status.json",
    "C04": WORK_ROOT / "c04_final_status.json",
    "C05": WORK_ROOT / "c05_final_status.json",
    "C06": WORK_ROOT / "c06_final_status.json",
    "C07": WORK_ROOT / "c07_final_status.json",

    "C06_LEDGER":
        WORK_ROOT / "c06_identifier_candidate_ledger.csv",

    "C07_PROFILE":
        WORK_ROOT / "c07_identifier_structural_profile.csv",

    "C07_UNIQUE":
        WORK_ROOT / "c07_structurally_unique_candidates.csv",

    "C07_SUMMARY":
        WORK_ROOT / "c07_identifier_structural_summary.json"
}

for name, path in PATHS.items():

    if not path.exists():

        raise RuntimeError(
            f"FAIL-CLOSED: Required artifact missing: "
            f"{name} -> {path}"
        )


def load_json(path):

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


c01 = load_json(PATHS["C01"])
c03 = load_json(PATHS["C03"])
c04 = load_json(PATHS["C04"])
c05 = load_json(PATHS["C05"])
c06 = load_json(PATHS["C06"])
c07 = load_json(PATHS["C07"])
c07_summary = load_json(PATHS["C07_SUMMARY"])


# ------------------------------------------------------------
# 1. ACTUAL UPSTREAM STATUS
# ------------------------------------------------------------

upstream = {
    "C01": c01.get("status"),
    "C03": c03.get("status"),
    "C04": c04.get("status"),
    "C05": c05.get("status"),
    "C06": c06.get("status"),
    "C07": c07.get("status")
}

lineage_pass = all(
    status == "PASS"
    for status in upstream.values()
)


# ------------------------------------------------------------
# 2. LOAD ACTUAL C06 / C07 DATA
# ------------------------------------------------------------

c06_df = pd.read_csv(
    PATHS["C06_LEDGER"]
)

c07_profile = pd.read_csv(
    PATHS["C07_PROFILE"]
)

c07_unique = pd.read_csv(
    PATHS["C07_UNIQUE"]
)


# ------------------------------------------------------------
# 3. COLUMN INVENTORY
# ------------------------------------------------------------

c06_columns = list(
    c06_df.columns
)

c07_profile_columns = list(
    c07_profile.columns
)

c07_unique_columns = list(
    c07_unique.columns
)


# ------------------------------------------------------------
# 4. C06 ACTUAL COUNTS
# ------------------------------------------------------------

if "variable" not in c06_df.columns:

    raise RuntimeError(
        "FAIL-CLOSED: C06 ledger does not contain "
        "required 'variable' column."
    )

if "relative_path" not in c06_df.columns:

    raise RuntimeError(
        "FAIL-CLOSED: C06 ledger does not contain "
        "required 'relative_path' column."
    )


c06_candidate_records = len(c06_df)

c06_unique_variables = (
    c06_df["variable"]
    .astype(str)
    .nunique()
)

c06_candidate_files = (
    c06_df["relative_path"]
    .astype(str)
    .nunique()
)

c06_duplicate_file_variable = int(
    c06_df.duplicated(
        subset=[
            "relative_path",
            "variable"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 5. C07 ACTUAL COUNTS
# ------------------------------------------------------------

if "variable" not in c07_profile.columns:

    raise RuntimeError(
        "FAIL-CLOSED: C07 profile does not contain "
        "required 'variable' column."
    )

c07_profile_rows = len(
    c07_profile
)

c07_profile_unique_variables = (
    c07_profile["variable"]
    .astype(str)
    .nunique()
)

c07_unique_output_rows = len(
    c07_unique
)


# ------------------------------------------------------------
# 6. C06 ↔ C07 VARIABLE COMPARISON
# ------------------------------------------------------------

c06_variables = set(
    c06_df["variable"]
    .astype(str)
)

c07_variables = set(
    c07_profile["variable"]
    .astype(str)
)

only_c06_variables = sorted(
    c06_variables - c07_variables
)

only_c07_variables = sorted(
    c07_variables - c06_variables
)


# ------------------------------------------------------------
# 7. C06 ↔ C07 FILE-VARIABLE COMPARISON
# ------------------------------------------------------------

c06_pairs = set(
    zip(
        c06_df["relative_path"].astype(str),
        c06_df["variable"].astype(str)
    )
)

if {
    "relative_path",
    "variable"
}.issubset(
    c07_profile.columns
):

    c07_pairs = set(
        zip(
            c07_profile["relative_path"].astype(str),
            c07_profile["variable"].astype(str)
        )
    )

else:

    c07_pairs = set()


only_c06_pairs = sorted(
    c06_pairs - c07_pairs
)

only_c07_pairs = sorted(
    c07_pairs - c06_pairs
)


# ------------------------------------------------------------
# 8. CHECK WHETHER C07 IS USING FILE OCCURRENCE
#    OR ACTUAL VALUE UNIQUENESS
# ------------------------------------------------------------

file_occurrence_columns = [
    col
    for col in c07_profile.columns
    if any(
        term in col.lower()
        for term in [
            "file_count",
            "file_count",
            "n_files",
            "occurrence",
            "file"
        ]
    )
]

value_uniqueness_columns = [
    col
    for col in c07_profile.columns
    if any(
        term in col.lower()
        for term in [
            "row_unique",
            "value_unique",
            "unique_value",
            "distinct",
            "nonunique"
        ]
    )
]


# ------------------------------------------------------------
# 9. EXTRACT C07 SUMMARY DEFINITION EVIDENCE
# ------------------------------------------------------------

definition_candidates = {}

if isinstance(
    c07_summary,
    dict
):

    for key, value in c07_summary.items():

        key_lower = str(key).lower()

        if any(
            term in key_lower
            for term in [
                "definition",
                "criteria",
                "rule",
                "meaning",
                "uniqu",
                "structur"
            ]
        ):

            definition_candidates[key] = value


# ------------------------------------------------------------
# 10. HISTORICAL 1045 / 379 STATUS
# ------------------------------------------------------------

# The clean working directory contains no old canonical
# identifier artifact with the 1045 / 379 result.
# Therefore this cell does NOT manufacture a reconciliation.

historical_reference_available = False

historical_reference_note = (
    "Historical 1045/379 identifier artifact was not "
    "present in the clean working directory. "
    "No historical value was substituted or inferred."
)


# ------------------------------------------------------------
# 11. STRUCTURAL DEFINITION STATUS
# ------------------------------------------------------------

if value_uniqueness_columns:

    c07_value_uniqueness_evidence = (
        "PRESENT_IN_C07_SCHEMA"
    )

else:

    c07_value_uniqueness_evidence = (
        "NOT_IDENTIFIED_IN_C07_SCHEMA"
    )


if file_occurrence_columns:

    c07_file_occurrence_evidence = (
        "PRESENT_IN_C07_SCHEMA"
    )

else:

    c07_file_occurrence_evidence = (
        "NOT_IDENTIFIED_IN_C07_SCHEMA"
    )


# ------------------------------------------------------------
# 12. RECONCILIATION RESULT
# ------------------------------------------------------------

# C06 and C07 should preserve the same candidate
# file-variable universe for this structural audit.

candidate_universe_match = (
    len(only_c06_pairs) == 0
    and
    len(only_c07_pairs) == 0
)


# Diagnostic reconciliation does not authorize anything.

status = "PASS"

if not lineage_pass:

    status = "BLOCKED"

if not candidate_universe_match:

    status = "BLOCKED"


# ------------------------------------------------------------
# 13. FINAL STATE
# ------------------------------------------------------------

final_state = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C09",

    "cell_name":
        "Structural Reconciliation & Lineage Audit",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        status != "PASS",

    "authorization": {

        "construct":
            "NOT_AUTHORIZED",

        "predictor":
            "NOT_AUTHORIZED",

        "outcome":
            "NOT_AUTHORIZED",

        "key":
            "NOT_AUTHORIZED",

        "relationship":
            "NOT_AUTHORIZED",

        "cohort":
            "NOT_AUTHORIZED",

        "score":
            "NOT_AUTHORIZED",

        "empirical_evaluation":
            "NOT_READY"
    },

    "payload": {

        "upstream_lineage":
            upstream,

        "lineage_pass":
            lineage_pass,

        "c06_candidate_records":
            int(c06_candidate_records),

        "c06_unique_variables":
            int(c06_unique_variables),

        "c06_candidate_files":
            int(c06_candidate_files),

        "c06_duplicate_file_variable":
            int(c06_duplicate_file_variable),

        "c07_profile_rows":
            int(c07_profile_rows),

        "c07_profile_unique_variables":
            int(c07_profile_unique_variables),

        "c07_unique_output_rows":
            int(c07_unique_output_rows),

        "only_c06_variables":
            only_c06_variables,

        "only_c07_variables":
            only_c07_variables,

        "only_c06_variable_count":
            len(only_c06_variables),

        "only_c07_variable_count":
            len(only_c07_variables),

        "only_c06_file_variable_pairs":
            only_c06_pairs[:100],

        "only_c07_file_variable_pairs":
            only_c07_pairs[:100],

        "only_c06_file_variable_pair_count":
            len(only_c06_pairs),

        "only_c07_file_variable_pair_count":
            len(only_c07_pairs),

        "candidate_universe_match":
            candidate_universe_match,

        "c06_columns":
            c06_columns,

        "c07_profile_columns":
            c07_profile_columns,

        "c07_unique_columns":
            c07_unique_columns,

        "c07_file_occurrence_columns":
            file_occurrence_columns,

        "c07_value_uniqueness_columns":
            value_uniqueness_columns,

        "c07_file_occurrence_evidence":
            c07_file_occurrence_evidence,

        "c07_value_uniqueness_evidence":
            c07_value_uniqueness_evidence,

        "c07_definition_candidates":
            definition_candidates,

        "historical_reference_available":
            historical_reference_available,

        "historical_reference_note":
            historical_reference_note,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_inference_allowed":
            False,

        "diagnostic_only":
            True,

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False
    }
}


# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

OUTPUT = (
    WORK_ROOT /
    "c09_structural_reconciliation.json"
)

with OUTPUT.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_state,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 15. OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("UPSTREAM LINEAGE")
print("-" * 78)

for cell, value in upstream.items():

    print(
        f"{cell:24s}: {value}"
    )

print()
print("-" * 78)
print("C06 ACTUAL")
print("-" * 78)

print(
    "Candidate records       :",
    c06_candidate_records
)

print(
    "Unique variables        :",
    c06_unique_variables
)

print(
    "Candidate files         :",
    c06_candidate_files
)

print(
    "Duplicate file-variable :",
    c06_duplicate_file_variable
)

print()
print("-" * 78)
print("C07 ACTUAL")
print("-" * 78)

print(
    "Profile rows             :",
    c07_profile_rows
)

print(
    "Profile unique variables :",
    c07_profile_unique_variables
)

print(
    "Unique output rows       :",
    c07_unique_output_rows
)

print()
print("-" * 78)
print("C06 ↔ C07 VARIABLE RECONCILIATION")
print("-" * 78)

print(
    "Only in C06             :",
    len(only_c06_variables)
)

print(
    "Only in C07             :",
    len(only_c07_variables)
)

if only_c06_variables:

    print(
        "C06-only variables      :",
        only_c06_variables
    )

if only_c07_variables:

    print(
        "C07-only variables      :",
        only_c07_variables
    )

print()
print("-" * 78)
print("C06 ↔ C07 FILE-VARIABLE RECONCILIATION")
print("-" * 78)

print(
    "Only in C06             :",
    len(only_c06_pairs)
)

print(
    "Only in C07             :",
    len(only_c07_pairs)
)

print(
    "Candidate universe match:",
    candidate_universe_match
)

print()
print("-" * 78)
print("C07 DEFINITION EVIDENCE")
print("-" * 78)

print(
    "File occurrence evidence :",
    c07_file_occurrence_evidence
)

print(
    "Value uniqueness evidence:",
    c07_value_uniqueness_evidence
)

print(
    "Definition candidates    :",
    definition_candidates
)

print()
print("-" * 78)
print("HISTORICAL REFERENCE")
print("-" * 78)

print(
    "1045/379 artifact available:",
    historical_reference_available
)

print(
    historical_reference_note
)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    "Semantic interpretation : NOT_PERFORMED"
)

print(
    "Semantic inference      : False"
)

print(
    "Diagnostic only         : True"
)

print(
    "Authorization           : NOT_AUTHORIZED"
)

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    "Policy active           : True"
)

print(
    "Triggered               :",
    status != "PASS"
)

print(
    "C09 STATUS              :",
    status
)

print()
print("Artifact:")
print(OUTPUT)

print("=" * 78)

OIP v1.0.33 — ESS4
C09 — STRUCTURAL RECONCILIATION & LINEAGE AUDIT

------------------------------------------------------------------------------
UPSTREAM LINEAGE
------------------------------------------------------------------------------
C01                     : PASS
C03                     : PASS
C04                     : PASS
C05                     : PASS
C06                     : PASS
C07                     : PASS

------------------------------------------------------------------------------
C06 ACTUAL
------------------------------------------------------------------------------
Candidate records       : 1044
Unique variables        : 378
Candidate files         : 67
Duplicate file-variable : 0

------------------------------------------------------------------------------
C07 ACTUAL
------------------------------------------------------------------------------
Profile rows             : 378
Profile unique variables : 378
Unique output rows       : 353

-------------------

In [16]:
# ============================================================
# OIP v1.0.33 — ESS4
# C07R — IDENTIFIER STRUCTURAL UNIQUENESS AUDIT
# CORRECTED FILE-VARIABLE LEVEL VERSION
# PATH-SAFE VERSION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import pyreadstat

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C07R — IDENTIFIER STRUCTURAL UNIQUENESS AUDIT")
print("=" * 78)

WORK_ROOT = Path(
    "/kaggle/working/oip_v1_0_33"
)

C00_STATUS = WORK_ROOT / "c00_source_scope.json"
C06_STATUS = WORK_ROOT / "c06_final_status.json"
C06_LEDGER = (
    WORK_ROOT /
    "c06_identifier_candidate_ledger.csv"
)

if not C00_STATUS.exists():
    raise RuntimeError(
        f"FAIL-CLOSED: Missing C00 artifact: {C00_STATUS}"
    )

if not C06_STATUS.exists():
    raise RuntimeError(
        f"FAIL-CLOSED: Missing C06 artifact: {C06_STATUS}"
    )

if not C06_LEDGER.exists():
    raise RuntimeError(
        f"FAIL-CLOSED: Missing C06 ledger: {C06_LEDGER}"
    )


def load_json(path):

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


c00 = load_json(C00_STATUS)
c06 = load_json(C06_STATUS)


# ------------------------------------------------------------
# 1. UPSTREAM STATUS
# ------------------------------------------------------------

if c00.get("status") != "PASS":

    raise RuntimeError(
        "FAIL-CLOSED: C00 is not PASS."
    )

if c06.get("status") != "PASS":

    raise RuntimeError(
        "FAIL-CLOSED: C06 is not PASS."
    )


# ------------------------------------------------------------
# 2. ACTUAL SOURCE ROOT FROM C00
# ------------------------------------------------------------

try:

    SOURCE_ROOT = Path(
        c00["payload"]["source_root"]
    )

except Exception:

    raise RuntimeError(
        "FAIL-CLOSED: C00 source_root could not "
        "be resolved from actual artifact."
    )


if not SOURCE_ROOT.exists():

    raise RuntimeError(
        "FAIL-CLOSED: C00 source_root does not exist:\n"
        + str(SOURCE_ROOT)
    )


# ------------------------------------------------------------
# 3. LOAD C06 LEDGER
# ------------------------------------------------------------

c06_df = pd.read_csv(
    C06_LEDGER
)

required_columns = {
    "relative_path",
    "variable"
}

missing_columns = (
    required_columns
    - set(c06_df.columns)
)

if missing_columns:

    raise RuntimeError(
        "FAIL-CLOSED: C06 ledger missing columns: "
        + str(sorted(missing_columns))
    )


# ------------------------------------------------------------
# 4. PRESERVE C06 FILE-VARIABLE RECORDS
# ------------------------------------------------------------

candidate_df = (
    c06_df[
        [
            "relative_path",
            "variable"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

candidate_records = len(candidate_df)

if candidate_records == 0:

    raise RuntimeError(
        "FAIL-CLOSED: No candidate records found."
    )


# ------------------------------------------------------------
# 5. PATH RESOLUTION TEST
# ------------------------------------------------------------

# C06 relative_path already contains the Stata directory.
# Therefore SOURCE_ROOT is the dataset root from C00.
#
# Example:
#
# SOURCE_ROOT/
# ETH_2018_ESS4_Stata/cons_agg_w4.dta
#
# No manual suffix is added.

path_test = SOURCE_ROOT / str(
    candidate_df.iloc[0]["relative_path"]
)

if not path_test.exists():

    raise RuntimeError(
        "FAIL-CLOSED: Resolved sample path does not exist.\n"
        f"SOURCE_ROOT: {SOURCE_ROOT}\n"
        f"relative_path: {candidate_df.iloc[0]['relative_path']}\n"
        f"resolved_path: {path_test}"
    )


print()
print("-" * 78)
print("UPSTREAM")
print("-" * 78)

print(
    "C00 STATUS               :",
    c00.get("status")
)

print(
    "C06 STATUS               :",
    c06.get("status")
)

print(
    "Candidate records       :",
    candidate_records
)

print()
print("-" * 78)
print("PATH RESOLUTION")
print("-" * 78)

print(
    "SOURCE_ROOT              :",
    SOURCE_ROOT
)

print(
    "Sample relative_path     :",
    candidate_df.iloc[0]["relative_path"]
)

print(
    "Resolved sample path     :",
    path_test
)

print(
    "Sample path exists       :",
    path_test.exists()
)


# ------------------------------------------------------------
# 6. STRUCTURAL AUDIT
# ------------------------------------------------------------

results = []
read_errors = []

for idx, row in candidate_df.iterrows():

    relative_path = str(
        row["relative_path"]
    )

    variable = str(
        row["variable"]
    )

    dta_path = (
        SOURCE_ROOT /
        relative_path
    )

    record = {

        "candidate_index":
            int(idx),

        "relative_path":
            relative_path,

        "variable":
            variable,

        "source_exists":
            False,

        "read_status":
            "NOT_ATTEMPTED",

        "row_count":
            None,

        "nonmissing_count":
            None,

        "distinct_nonmissing_values":
            None,

        "missing_count":
            None,

        "structural_uniqueness":
            "NOT_ESTABLISHED",

        "semantic_interpretation":
            "NOT_PERFORMED",

        "authorization":
            "NOT_AUTHORIZED"
    }


    if not dta_path.exists():

        record["read_status"] = (
            "SOURCE_MISSING"
        )

        read_errors.append({

            "relative_path":
                relative_path,

            "variable":
                variable,

            "error":
                "SOURCE_MISSING"
        })

        results.append(record)

        continue


    record["source_exists"] = True


    try:

        df, meta = pyreadstat.read_dta(
            str(dta_path),
            usecols=[variable]
        )

        series = df[variable]

        row_count = len(series)

        missing_count = int(
            series.isna().sum()
        )

        nonmissing = (
            series.dropna()
        )

        nonmissing_count = len(
            nonmissing
        )

        distinct_nonmissing = (
            nonmissing.nunique(
                dropna=True
            )
        )


        record["read_status"] = (
            "READ_OK"
        )

        record["row_count"] = int(
            row_count
        )

        record["nonmissing_count"] = int(
            nonmissing_count
        )

        record[
            "distinct_nonmissing_values"
        ] = int(
            distinct_nonmissing
        )

        record["missing_count"] = int(
            missing_count
        )


        # ----------------------------------------------------
        # STRUCTURAL UNIQUENESS
        # ----------------------------------------------------
        #
        # Only observed nonmissing values are evaluated.
        #
        # Missing values are NOT converted to values.
        # No imputation is performed.
        #
        # Structural uniqueness:
        #
        # nonmissing_count ==
        # distinct_nonmissing_values
        #
        # This remains an observed data property only.
        # It does NOT authorize an identifier or key.
        # ----------------------------------------------------

        if (
            nonmissing_count > 0
            and
            distinct_nonmissing
            ==
            nonmissing_count
        ):

            record[
                "structural_uniqueness"
            ] = (
                "OBSERVED_STRUCTURALLY_UNIQUE"
            )

        elif nonmissing_count == 0:

            record[
                "structural_uniqueness"
            ] = (
                "NOT_ESTABLISHED_NO_OBSERVED_VALUES"
            )

        else:

            record[
                "structural_uniqueness"
            ] = (
                "OBSERVED_NONUNIQUE"
            )


    except Exception as e:

        record["read_status"] = (
            "READ_ERROR"
        )

        read_errors.append({

            "relative_path":
                relative_path,

            "variable":
                variable,

            "error":
                repr(e)
        })


    results.append(record)


result_df = pd.DataFrame(
    results
)


# ------------------------------------------------------------
# 7. ACCOUNTING
# ------------------------------------------------------------

read_ok = int(
    (
        result_df["read_status"]
        ==
        "READ_OK"
    ).sum()
)

read_errors_count = int(
    (
        result_df["read_status"]
        ==
        "READ_ERROR"
    ).sum()
)

source_missing_count = int(
    (
        result_df["read_status"]
        ==
        "SOURCE_MISSING"
    ).sum()
)

structurally_unique = int(
    (
        result_df[
            "structural_uniqueness"
        ]
        ==
        "OBSERVED_STRUCTURALLY_UNIQUE"
    ).sum()
)

structurally_nonunique = int(
    (
        result_df[
            "structural_uniqueness"
        ]
        ==
        "OBSERVED_NONUNIQUE"
    ).sum()
)

not_established = int(
    result_df[
        "structural_uniqueness"
    ]
    .astype(str)
    .str.startswith(
        "NOT_ESTABLISHED"
    )
    .sum()
)

accounting_match = (
    read_ok
    +
    read_errors_count
    +
    source_missing_count
    ==
    candidate_records
)


# ------------------------------------------------------------
# 8. DUPLICATE FILE-VARIABLE CHECK
# ------------------------------------------------------------

duplicate_file_variable = int(
    c06_df.duplicated(
        subset=[
            "relative_path",
            "variable"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 9. BOUNDARY CHECK
# ------------------------------------------------------------

semantic_inference_leaks = 0

authorization_leaks = int(
    (
        result_df["authorization"]
        !=
        "NOT_AUTHORIZED"
    ).sum()
)

semantic_interpretation_leaks = int(
    (
        result_df[
            "semantic_interpretation"
        ]
        !=
        "NOT_PERFORMED"
    ).sum()
)


# ------------------------------------------------------------
# 10. FAIL-CLOSED
# ------------------------------------------------------------

status = "PASS"

if not accounting_match:
    status = "BLOCKED"

if read_errors_count != 0:
    status = "BLOCKED"

if source_missing_count != 0:
    status = "BLOCKED"

if duplicate_file_variable != 0:
    status = "BLOCKED"

if semantic_inference_leaks != 0:
    status = "BLOCKED"

if authorization_leaks != 0:
    status = "BLOCKED"

if semantic_interpretation_leaks != 0:
    status = "BLOCKED"


# ------------------------------------------------------------
# 11. OUTPUT PATHS
# ------------------------------------------------------------

PROFILE = (
    WORK_ROOT /
    "c07r_identifier_structural_profile.csv"
)

UNIQUE = (
    WORK_ROOT /
    "c07r_structurally_unique_candidates.csv"
)

ERRORS = (
    WORK_ROOT /
    "c07r_read_errors.json"
)

SUMMARY = (
    WORK_ROOT /
    "c07r_identifier_structural_summary.json"
)

STATUS = (
    WORK_ROOT /
    "c07r_final_status.json"
)


# ------------------------------------------------------------
# 12. SAVE DATA
# ------------------------------------------------------------

result_df.to_csv(
    PROFILE,
    index=False
)

result_df[
    result_df[
        "structural_uniqueness"
    ]
    ==
    "OBSERVED_STRUCTURALLY_UNIQUE"
].to_csv(
    UNIQUE,
    index=False
)

with ERRORS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        read_errors,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 13. FINAL STATE
# ------------------------------------------------------------

final_state = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C07R",

    "cell_name":
        "Identifier Structural Uniqueness Audit "
        "— File Variable Level",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        status != "PASS",

    "authorization": {

        "construct":
            "NOT_AUTHORIZED",

        "predictor":
            "NOT_AUTHORIZED",

        "outcome":
            "NOT_AUTHORIZED",

        "key":
            "NOT_AUTHORIZED",

        "relationship":
            "NOT_AUTHORIZED",

        "cohort":
            "NOT_AUTHORIZED",

        "score":
            "NOT_AUTHORIZED",

        "empirical_evaluation":
            "NOT_READY"
    },

    "payload": {

        "upstream_c00_status":
            c00.get("status"),

        "upstream_c06_status":
            c06.get("status"),

        "source_root":
            str(SOURCE_ROOT),

        "candidate_records":
            candidate_records,

        "read_ok":
            read_ok,

        "read_errors":
            read_errors_count,

        "source_missing":
            source_missing_count,

        "structurally_unique_records":
            structurally_unique,

        "structurally_nonunique_records":
            structurally_nonunique,

        "not_established_records":
            not_established,

        "accounting_match":
            accounting_match,

        "duplicate_file_variable":
            duplicate_file_variable,

        "structural_unit":
            "FILE_VARIABLE_RECORD",

        "structural_definition":
            "Observed uniqueness of nonmissing values "
            "within each candidate source-file and "
            "variable record.",

        "missing_values_treated_as":
            "MISSING_NOT_VALUE",

        "imputation":
            False,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_inference_allowed":
            False,

        "semantic_inference_leaks":
            semantic_inference_leaks,

        "semantic_interpretation_leaks":
            semantic_interpretation_leaks,

        "authorization_leaks":
            authorization_leaks,

        "discovery_only":
            True,

        "raw_source_only":
            True,

        "historical_artifacts_used":
            False
    }
}


with SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_state,
        f,
        indent=2,
        ensure_ascii=False
    )


with STATUS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_state,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 14. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("-" * 78)
print("STRUCTURAL AUDIT")
print("-" * 78)

print(
    "Candidate records        :",
    candidate_records
)

print(
    "READ_OK                  :",
    read_ok
)

print(
    "Source missing           :",
    source_missing_count
)

print(
    "Read errors              :",
    read_errors_count
)

print(
    "Structurally unique rec. :",
    structurally_unique
)

print(
    "Structurally nonunique   :",
    structurally_nonunique
)

print(
    "Not established          :",
    not_established
)

print(
    "Accounting match         :",
    accounting_match
)

print(
    "Duplicate file-variable  :",
    duplicate_file_variable
)

print()
print("-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    "Structural unit          : FILE_VARIABLE_RECORD"
)

print(
    "Semantic interpretation  : NOT_PERFORMED"
)

print(
    "Semantic inference       : False"
)

print(
    "Discovery only           : True"
)

print(
    "Authorization leaks      :",
    authorization_leaks
)

print()
print("-" * 78)
print("AUTHORIZATION")
print("-" * 78)

for key, value in final_state[
    "authorization"
].items():

    print(
        f"{key:24s}: {value}"
    )

print()
print("-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    "Policy active            : True"
)

print(
    "Triggered                :",
    status != "PASS"
)

print(
    "C07R STATUS              :",
    status
)

print()
print("Canonical artifacts:")

print(PROFILE)
print(UNIQUE)
print(ERRORS)
print(SUMMARY)
print(STATUS)

print("=" * 78)

OIP v1.0.33 — ESS4
C07R — IDENTIFIER STRUCTURAL UNIQUENESS AUDIT

------------------------------------------------------------------------------
UPSTREAM
------------------------------------------------------------------------------
C00 STATUS               : PASS
C06 STATUS               : PASS
Candidate records       : 1044

------------------------------------------------------------------------------
PATH RESOLUTION
------------------------------------------------------------------------------
SOURCE_ROOT              : /kaggle/input/datasets/sudharsandas27/oip-v1-0-33-ethiopia-ess4-evidence-audit
Sample relative_path     : ETH_2018_ESS_v04_M_Stata/cons_agg_w4.dta
Resolved sample path     : /kaggle/input/datasets/sudharsandas27/oip-v1-0-33-ethiopia-ess4-evidence-audit/ETH_2018_ESS_v04_M_Stata/cons_agg_w4.dta
Sample path exists       : True

------------------------------------------------------------------------------
STRUCTURAL AUDIT
-----------------------------------------------

In [17]:
# ============================================================
# OIP v1.0.33 — ESS4
# C09R — CORRECTED STRUCTURAL RECONCILIATION & LINEAGE AUDIT
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C09R — CORRECTED STRUCTURAL RECONCILIATION & LINEAGE AUDIT")
print("=" * 78)

# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C01 = WORKING_DIR / "c01_vocabulary_lock.json"
C03 = WORKING_DIR / "c03_final_status.json"
C04 = WORKING_DIR / "c04_final_status.json"
C05 = WORKING_DIR / "c05_final_status.json"
C06 = WORKING_DIR / "c06_final_status.json"
C07R = WORKING_DIR / "c07r_final_status.json"

C06_LEDGER = WORKING_DIR / "c06_identifier_candidate_ledger.csv"
C07R_PROFILE = WORKING_DIR / "c07r_identifier_structural_profile.csv"

OUT_JSON = WORKING_DIR / "c09r_structural_reconciliation.json"


# ------------------------------------------------------------
# 1. LOAD JSON
# ------------------------------------------------------------

def load_json(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_status(obj):
    return obj.get("status", "MISSING")


c01 = load_json(C01)
c03 = load_json(C03)
c04 = load_json(C04)
c05 = load_json(C05)
c06 = load_json(C06)
c07r = load_json(C07R)


# ------------------------------------------------------------
# 2. UPSTREAM LINEAGE
# ------------------------------------------------------------

upstream = {
    "C01": get_status(c01),
    "C03": get_status(c03),
    "C04": get_status(c04),
    "C05": get_status(c05),
    "C06": get_status(c06),
    "C07R": get_status(c07r)
}

upstream_ok = all(v == "PASS" for v in upstream.values())


# ------------------------------------------------------------
# 3. LOAD CSV ARTIFACTS
# ------------------------------------------------------------

if not C06_LEDGER.exists():
    raise FileNotFoundError(str(C06_LEDGER))

if not C07R_PROFILE.exists():
    raise FileNotFoundError(str(C07R_PROFILE))

c06_df = pd.read_csv(C06_LEDGER)
c07r_df = pd.read_csv(C07R_PROFILE)


# ------------------------------------------------------------
# 4. SCHEMA CHECK
# ------------------------------------------------------------

required_columns = {
    "relative_path",
    "file_name",
    "variable"
}

missing_c06_columns = sorted(
    required_columns - set(c06_df.columns)
)

missing_c07r_columns = sorted(
    required_columns - set(c07r_df.columns)
)

schema_ok = (
    len(missing_c06_columns) == 0 and
    len(missing_c07r_columns) == 0
)


# ------------------------------------------------------------
# 5. FILE-VARIABLE KEY
# ------------------------------------------------------------

def make_file_variable_keys(df):

    x = df[
        ["relative_path", "variable"]
    ].copy()

    x["relative_path"] = (
        x["relative_path"]
        .astype(str)
        .str.strip()
    )

    x["variable"] = (
        x["variable"]
        .astype(str)
        .str.strip()
    )

    return set(
        zip(
            x["relative_path"],
            x["variable"]
        )
    )


if schema_ok:

    c06_keys = make_file_variable_keys(c06_df)
    c07r_keys = make_file_variable_keys(c07r_df)

else:

    c06_keys = set()
    c07r_keys = set()


# ------------------------------------------------------------
# 6. UNIVERSE RECONCILIATION
# ------------------------------------------------------------

only_in_c06 = sorted(
    c06_keys - c07r_keys
)

only_in_c07r = sorted(
    c07r_keys - c06_keys
)

candidate_universe_match = (
    len(only_in_c06) == 0 and
    len(only_in_c07r) == 0
)


# ------------------------------------------------------------
# 7. DUPLICATE FILE-VARIABLE AUDIT
# ------------------------------------------------------------

if schema_ok:

    c06_duplicate_count = int(
        c06_df.duplicated(
            subset=[
                "relative_path",
                "variable"
            ],
            keep=False
        ).sum()
    )

    c07r_duplicate_count = int(
        c07r_df.duplicated(
            subset=[
                "relative_path",
                "variable"
            ],
            keep=False
        ).sum()
    )

else:

    c06_duplicate_count = -1
    c07r_duplicate_count = -1


duplicates_ok = (
    c06_duplicate_count == 0 and
    c07r_duplicate_count == 0
)


# ------------------------------------------------------------
# 8. RECORD ACCOUNTING
# ------------------------------------------------------------

c06_candidate_records = int(
    len(c06_df)
)

c07r_profile_records = int(
    len(c07r_df)
)

c06_unique_file_variable = int(
    len(c06_keys)
)

c07r_unique_file_variable = int(
    len(c07r_keys)
)

c06_accounting_match = (
    c06_candidate_records ==
    c06_unique_file_variable
)

c07r_accounting_match = (
    c07r_profile_records ==
    c07r_unique_file_variable
)

record_accounting_match = (
    c06_accounting_match and
    c07r_accounting_match
)


# ------------------------------------------------------------
# 9. STRUCTURAL UNIQUENESS RECONCILIATION
# ------------------------------------------------------------

if "structural_uniqueness" in c07r_df.columns:

    structural_state = (
        c07r_df[
            "structural_uniqueness"
        ]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    structurally_unique_records = int(
        (structural_state == "UNIQUE").sum()
    )

    structurally_nonunique_records = int(
        (structural_state == "NONUNIQUE").sum()
    )

    not_established_records = int(
        (structural_state == "NOT_ESTABLISHED").sum()
    )

    structural_accounting_match = (
        structurally_unique_records
        + structurally_nonunique_records
        + not_established_records
        ==
        c07r_profile_records
    )

else:

    structurally_unique_records = None
    structurally_nonunique_records = None
    not_established_records = None
    structural_accounting_match = False


# ------------------------------------------------------------
# 10. HISTORICAL ARTIFACT DISCIPLINE
# ------------------------------------------------------------

historical_artifact_detected = False

for p in WORKING_DIR.iterdir():

    name = p.name.lower()

    if (
        "1045" in name
        or "379" in name
    ):
        historical_artifact_detected = True
        break


# Historical reference is never substituted
historical_substitution_used = False


# ------------------------------------------------------------
# 11. SEMANTIC BOUNDARY
# ------------------------------------------------------------

semantic_interpretation = "NOT_PERFORMED"
semantic_inference_allowed = False
diagnostic_only = True

authorization = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY"
}

authorization_leaks = sum(
    1
    for value in authorization.values()
    if value in {
        "AUTHORIZED",
        "APPROVED"
    }
)


# ------------------------------------------------------------
# 12. FAIL-CLOSED CHECKS
# ------------------------------------------------------------

checks = {

    "upstream_ok":
        upstream_ok,

    "schema_ok":
        schema_ok,

    "c06_accounting_match":
        c06_accounting_match,

    "c07r_accounting_match":
        c07r_accounting_match,

    "candidate_universe_match":
        candidate_universe_match,

    "duplicates_ok":
        duplicates_ok,

    "structural_accounting_match":
        structural_accounting_match,

    "historical_substitution_used":
        historical_substitution_used,

    "semantic_inference_allowed":
        semantic_inference_allowed,

    "authorization_leaks_zero":
        authorization_leaks == 0
}


fail_closed_triggered = not all(
    checks.values()
)

status = (
    "BLOCKED"
    if fail_closed_triggered
    else "PASS"
)


# ------------------------------------------------------------
# 13. FINAL ARTIFACT
# ------------------------------------------------------------

result = {

    "oip_version": "v1.0.33",

    "cell": "C09R",

    "cell_name":
        "CORRECTED_STRUCTURAL_RECONCILIATION_AND_LINEAGE_AUDIT",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        authorization,

    "payload": {

        "upstream_lineage":
            upstream,

        "c06_actual": {

            "candidate_records":
                c06_candidate_records,

            "unique_file_variable_records":
                c06_unique_file_variable,

            "duplicate_file_variable_records":
                c06_duplicate_count
        },

        "c07r_actual": {

            "profile_records":
                c07r_profile_records,

            "unique_file_variable_records":
                c07r_unique_file_variable,

            "duplicate_file_variable_records":
                c07r_duplicate_count
        },

        "file_variable_reconciliation": {

            "only_in_c06_count":
                len(only_in_c06),

            "only_in_c07r_count":
                len(only_in_c07r),

            "candidate_universe_match":
                candidate_universe_match
        },

        "structural_reconciliation": {

            "structurally_unique_records":
                structurally_unique_records,

            "structurally_nonunique_records":
                structurally_nonunique_records,

            "not_established_records":
                not_established_records,

            "accounting_match":
                structural_accounting_match
        },

        "historical_reference": {

            "historical_1045_379_artifact_detected":
                historical_artifact_detected,

            "historical_substitution_used":
                historical_substitution_used,

            "historical_reference_authoritative":
                False
        },

        "boundary": {

            "structural_unit":
                "FILE_VARIABLE_RECORD",

            "semantic_interpretation":
                semantic_interpretation,

            "semantic_inference_allowed":
                semantic_inference_allowed,

            "diagnostic_only":
                diagnostic_only,

            "authorization_leaks":
                authorization_leaks
        },

        "checks":
            checks
    }
}


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 14. OUTPUT
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("UPSTREAM LINEAGE")
print("-" * 78)

for key, value in upstream.items():
    print(f"{key:<24}: {value}")


print("\n" + "-" * 78)
print("C06 ↔ C07R FILE-VARIABLE RECONCILIATION")
print("-" * 78)

print(
    f"C06 candidate records       : "
    f"{c06_candidate_records}"
)

print(
    f"C06 unique file-variable    : "
    f"{c06_unique_file_variable}"
)

print(
    f"C07R profile records        : "
    f"{c07r_profile_records}"
)

print(
    f"C07R unique file-variable   : "
    f"{c07r_unique_file_variable}"
)

print(
    f"Only in C06                 : "
    f"{len(only_in_c06)}"
)

print(
    f"Only in C07R                : "
    f"{len(only_in_c07r)}"
)

print(
    f"Candidate universe match    : "
    f"{candidate_universe_match}"
)


print("\n" + "-" * 78)
print("STRUCTURAL RECONCILIATION")
print("-" * 78)

print(
    f"Structurally unique records : "
    f"{structurally_unique_records}"
)

print(
    f"Structurally nonunique      : "
    f"{structurally_nonunique_records}"
)

print(
    f"Not established             : "
    f"{not_established_records}"
)

print(
    f"Accounting match             : "
    f"{structural_accounting_match}"
)


print("\n" + "-" * 78)
print("HISTORICAL REFERENCE")
print("-" * 78)

print(
    f"Historical 1045/379 detected: "
    f"{historical_artifact_detected}"
)

print(
    f"Historical substitution     : "
    f"{historical_substitution_used}"
)


print("\n" + "-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    f"Structural unit             : "
    f"FILE_VARIABLE_RECORD"
)

print(
    f"Semantic interpretation     : "
    f"{semantic_interpretation}"
)

print(
    f"Semantic inference          : "
    f"{semantic_inference_allowed}"
)

print(
    f"Diagnostic only             : "
    f"{diagnostic_only}"
)

print(
    f"Authorization leaks         : "
    f"{authorization_leaks}"
)


print("\n" + "-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    f"Policy active               : True"
)

print(
    f"Triggered                   : "
    f"{fail_closed_triggered}"
)

print(
    f"C09R STATUS                 : "
    f"{status}"
)

print("\nArtifact:")
print(OUT_JSON)

print("=" * 78)

OIP v1.0.33 — ESS4
C09R — CORRECTED STRUCTURAL RECONCILIATION & LINEAGE AUDIT

------------------------------------------------------------------------------
UPSTREAM LINEAGE
------------------------------------------------------------------------------
C01                     : PASS
C03                     : PASS
C04                     : PASS
C05                     : PASS
C06                     : PASS
C07R                    : PASS

------------------------------------------------------------------------------
C06 ↔ C07R FILE-VARIABLE RECONCILIATION
------------------------------------------------------------------------------
C06 candidate records       : 1044
C06 unique file-variable    : 0
C07R profile records        : 1044
C07R unique file-variable   : 0
Only in C06                 : 0
Only in C07R                : 0
Candidate universe match    : True

------------------------------------------------------------------------------
STRUCTURAL RECONCILIATION
----------------------

In [18]:
# ============================================================
# OIP v1.0.33 — ESS4
# C09R v2 — CORRECTED STRUCTURAL RECONCILIATION
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C09R v2 — CORRECTED STRUCTURAL RECONCILIATION")
print("=" * 78)

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C01 = WORKING_DIR / "c01_vocabulary_lock.json"
C03 = WORKING_DIR / "c03_final_status.json"
C04 = WORKING_DIR / "c04_final_status.json"
C05 = WORKING_DIR / "c05_final_status.json"
C06 = WORKING_DIR / "c06_final_status.json"
C07R = WORKING_DIR / "c07r_final_status.json"

C06_LEDGER = WORKING_DIR / "c06_identifier_candidate_ledger.csv"
C07R_PROFILE = WORKING_DIR / "c07r_identifier_structural_profile.csv"

OUT_JSON = WORKING_DIR / "c09r_v2_structural_reconciliation.json"


# ------------------------------------------------------------
# 1. LOAD JSON
# ------------------------------------------------------------

def load_json(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_status(obj):
    return obj.get("status", "MISSING")


c01 = load_json(C01)
c03 = load_json(C03)
c04 = load_json(C04)
c05 = load_json(C05)
c06 = load_json(C06)
c07r = load_json(C07R)


upstream = {
    "C01": get_status(c01),
    "C03": get_status(c03),
    "C04": get_status(c04),
    "C05": get_status(c05),
    "C06": get_status(c06),
    "C07R": get_status(c07r)
}

upstream_ok = all(
    value == "PASS"
    for value in upstream.values()
)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

c06_df = pd.read_csv(C06_LEDGER)
c07r_df = pd.read_csv(C07R_PROFILE)


# ------------------------------------------------------------
# 3. ACTUAL SCHEMA
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("ACTUAL C06 SCHEMA")
print("-" * 78)

print(list(c06_df.columns))

print("\n" + "-" * 78)
print("ACTUAL C07R SCHEMA")
print("-" * 78)

print(list(c07r_df.columns))


required = {
    "relative_path",
    "variable"
}

c06_schema_ok = required.issubset(
    set(c06_df.columns)
)

c07r_schema_ok = required.issubset(
    set(c07r_df.columns)
)

schema_ok = (
    c06_schema_ok
    and c07r_schema_ok
)


# ------------------------------------------------------------
# 4. RAW FIELD DIAGNOSTICS
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("FIELD DIAGNOSTICS")
print("-" * 78)

if schema_ok:

    print(
        "C06 relative_path null      :",
        int(c06_df["relative_path"].isna().sum())
    )

    print(
        "C06 variable null           :",
        int(c06_df["variable"].isna().sum())
    )

    print(
        "C07R relative_path null     :",
        int(c07r_df["relative_path"].isna().sum())
    )

    print(
        "C07R variable null          :",
        int(c07r_df["variable"].isna().sum())
    )

    print(
        "C06 sample relative_path   :",
        repr(c06_df["relative_path"].iloc[0])
    )

    print(
        "C06 sample variable        :",
        repr(c06_df["variable"].iloc[0])
    )

    print(
        "C07R sample relative_path  :",
        repr(c07r_df["relative_path"].iloc[0])
    )

    print(
        "C07R sample variable       :",
        repr(c07r_df["variable"].iloc[0])
    )


# ------------------------------------------------------------
# 5. CANONICAL KEY CONSTRUCTION
# ------------------------------------------------------------

def canonical_file_variable_key(df):

    x = df[
        ["relative_path", "variable"]
    ].copy()

    x["relative_path"] = (
        x["relative_path"]
        .astype("string")
        .str.strip()
    )

    x["variable"] = (
        x["variable"]
        .astype("string")
        .str.strip()
    )

    # Explicitly reject missing/empty components
    valid = (
        x["relative_path"].notna()
        &
        x["variable"].notna()
        &
        (x["relative_path"] != "")
        &
        (x["variable"] != "")
    )

    invalid_count = int((~valid).sum())

    valid_x = x.loc[valid].copy()

    keys = set(
        zip(
            valid_x["relative_path"].tolist(),
            valid_x["variable"].tolist()
        )
    )

    return keys, invalid_count


if schema_ok:

    c06_keys, c06_invalid = canonical_file_variable_key(
        c06_df
    )

    c07r_keys, c07r_invalid = canonical_file_variable_key(
        c07r_df
    )

else:

    c06_keys = set()
    c07r_keys = set()
    c06_invalid = -1
    c07r_invalid = -1


# ------------------------------------------------------------
# 6. RECONCILIATION
# ------------------------------------------------------------

only_in_c06 = sorted(
    c06_keys - c07r_keys
)

only_in_c07r = sorted(
    c07r_keys - c06_keys
)

candidate_universe_match = (
    len(only_in_c06) == 0
    and
    len(only_in_c07r) == 0
)


# ------------------------------------------------------------
# 7. DUPLICATES
# ------------------------------------------------------------

if schema_ok:

    c06_duplicate_rows = int(
        c06_df.duplicated(
            subset=[
                "relative_path",
                "variable"
            ],
            keep=False
        ).sum()
    )

    c07r_duplicate_rows = int(
        c07r_df.duplicated(
            subset=[
                "relative_path",
                "variable"
            ],
            keep=False
        ).sum()
    )

else:

    c06_duplicate_rows = -1
    c07r_duplicate_rows = -1


# ------------------------------------------------------------
# 8. RECORD ACCOUNTING
# ------------------------------------------------------------

c06_records = int(len(c06_df))
c07r_records = int(len(c07r_df))

c06_unique_records = int(len(c06_keys))
c07r_unique_records = int(len(c07r_keys))

c06_accounting = (
    c06_records
    ==
    c06_unique_records
    and
    c06_invalid == 0
)

c07r_accounting = (
    c07r_records
    ==
    c07r_unique_records
    and
    c07r_invalid == 0
)


# ------------------------------------------------------------
# 9. STRUCTURAL STATE FROM C07R
# ------------------------------------------------------------

if "structural_uniqueness" in c07r_df.columns:

    state = (
        c07r_df["structural_uniqueness"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    unique_records = int(
        (state == "UNIQUE").sum()
    )

    nonunique_records = int(
        (state == "NONUNIQUE").sum()
    )

    not_established_records = int(
        (state == "NOT_ESTABLISHED").sum()
    )

    structural_accounting = (
        unique_records
        +
        nonunique_records
        +
        not_established_records
        ==
        c07r_records
    )

else:

    unique_records = None
    nonunique_records = None
    not_established_records = None
    structural_accounting = False


# ------------------------------------------------------------
# 10. HISTORICAL SUBSTITUTION
# ------------------------------------------------------------

historical_substitution_used = False


# ------------------------------------------------------------
# 11. BOUNDARY
# ------------------------------------------------------------

authorization = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY"
}

authorization_leaks = sum(
    1
    for value in authorization.values()
    if value in {
        "AUTHORIZED",
        "APPROVED"
    }
)

semantic_inference_allowed = False
semantic_interpretation = "NOT_PERFORMED"
diagnostic_only = True


# ------------------------------------------------------------
# 12. FAIL-CLOSED
# ------------------------------------------------------------

checks = {

    "upstream_ok":
        upstream_ok,

    "schema_ok":
        schema_ok,

    "c06_accounting":
        c06_accounting,

    "c07r_accounting":
        c07r_accounting,

    "candidate_universe_match":
        candidate_universe_match,

    "c06_duplicate_rows_zero":
        c06_duplicate_rows == 0,

    "c07r_duplicate_rows_zero":
        c07r_duplicate_rows == 0,

    "structural_accounting":
        structural_accounting,

    "historical_substitution_used":
        historical_substitution_used,

    "semantic_inference_allowed":
        semantic_inference_allowed,

    "authorization_leaks_zero":
        authorization_leaks == 0
}

fail_closed_triggered = not all(
    checks.values()
)

status = (
    "BLOCKED"
    if fail_closed_triggered
    else "PASS"
)


# ------------------------------------------------------------
# 13. ARTIFACT
# ------------------------------------------------------------

result = {

    "oip_version": "v1.0.33",

    "cell": "C09R",

    "cell_name":
        "CORRECTED_STRUCTURAL_RECONCILIATION",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        authorization,

    "payload": {

        "upstream_lineage":
            upstream,

        "schema": {

            "c06_schema_ok":
                c06_schema_ok,

            "c07r_schema_ok":
                c07r_schema_ok,

            "missing_c06_columns":
                sorted(required - set(c06_df.columns)),

            "missing_c07r_columns":
                sorted(required - set(c07r_df.columns))
        },

        "record_counts": {

            "c06_records":
                c06_records,

            "c06_unique_file_variable":
                c06_unique_records,

            "c07r_records":
                c07r_records,

            "c07r_unique_file_variable":
                c07r_unique_records
        },

        "invalid_components": {

            "c06_invalid_records":
                c06_invalid,

            "c07r_invalid_records":
                c07r_invalid
        },

        "file_variable_reconciliation": {

            "only_in_c06_count":
                len(only_in_c06),

            "only_in_c07r_count":
                len(only_in_c07r),

            "candidate_universe_match":
                candidate_universe_match
        },

        "duplicate_audit": {

            "c06_duplicate_rows":
                c06_duplicate_rows,

            "c07r_duplicate_rows":
                c07r_duplicate_rows
        },

        "structural_reconciliation": {

            "structurally_unique_records":
                unique_records,

            "structurally_nonunique_records":
                nonunique_records,

            "not_established_records":
                not_established_records,

            "accounting_match":
                structural_accounting
        },

        "historical_reference": {

            "historical_substitution_used":
                historical_substitution_used,

            "authoritative":
                False
        },

        "boundary": {

            "structural_unit":
                "FILE_VARIABLE_RECORD",

            "semantic_interpretation":
                semantic_interpretation,

            "semantic_inference_allowed":
                semantic_inference_allowed,

            "diagnostic_only":
                diagnostic_only,

            "authorization_leaks":
                authorization_leaks
        },

        "checks":
            checks
    }
}


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 14. OUTPUT
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("C06 ↔ C07R RECONCILIATION")
print("-" * 78)

print(
    f"C06 records                : {c06_records}"
)

print(
    f"C06 unique file-variable   : {c06_unique_records}"
)

print(
    f"C07R records               : {c07r_records}"
)

print(
    f"C07R unique file-variable  : {c07r_unique_records}"
)

print(
    f"Only in C06               : {len(only_in_c06)}"
)

print(
    f"Only in C07R              : {len(only_in_c07r)}"
)

print(
    f"Candidate universe match   : {candidate_universe_match}"
)


print("\n" + "-" * 78)
print("STRUCTURAL RECONCILIATION")
print("-" * 78)

print(
    f"Structurally unique       : {unique_records}"
)

print(
    f"Structurally nonunique    : {nonunique_records}"
)

print(
    f"Not established           : {not_established_records}"
)

print(
    f"Accounting match           : {structural_accounting}"
)


print("\n" + "-" * 78)
print("BOUNDARY")
print("-" * 78)

print(
    f"Semantic interpretation   : {semantic_interpretation}"
)

print(
    f"Semantic inference        : {semantic_inference_allowed}"
)

print(
    f"Authorization leaks       : {authorization_leaks}"
)

print(
    f"Diagnostic only           : {diagnostic_only}"
)


print("\n" + "-" * 78)
print("FAIL-CLOSED")
print("-" * 78)

print(
    f"Policy active             : True"
)

print(
    f"Triggered                 : {fail_closed_triggered}"
)

print(
    f"C09R STATUS               : {status}"
)

print("\nArtifact:")
print(OUT_JSON)

print("=" * 78)

OIP v1.0.33 — ESS4
C09R v2 — CORRECTED STRUCTURAL RECONCILIATION

------------------------------------------------------------------------------
ACTUAL C06 SCHEMA
------------------------------------------------------------------------------
['relative_path', 'file_name', 'variable', 'position', 'variable_label', 'identifier_terms_matched', 'candidate_basis', 'matching_rule', 'discovery_only', 'semantic_interpretation', 'identifier_authorization', 'key_authorization', 'relationship_authorization', 'cohort_authorization', 'construct_authorization', 'semantic_inference_allowed']

------------------------------------------------------------------------------
ACTUAL C07R SCHEMA
------------------------------------------------------------------------------
['candidate_index', 'relative_path', 'variable', 'source_exists', 'read_status', 'row_count', 'nonmissing_count', 'distinct_nonmissing_values', 'missing_count', 'structural_uniqueness', 'semantic_interpretation', 'authorization']

-------

In [19]:
from pathlib import Path
import pandas as pd

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")
C07R_PROFILE = WORKING_DIR / "c07r_identifier_structural_profile.csv"

df = pd.read_csv(C07R_PROFILE)

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C09R-DIAG — ACTUAL C07R STRUCTURAL STATE AUDIT")
print("=" * 78)

print("\nC07R COLUMNS")
print("-" * 78)
print(list(df.columns))

print("\nSTRUCTURAL UNIQUENESS")
print("-" * 78)
print("dtype:", df["structural_uniqueness"].dtype)
print("null count:", int(df["structural_uniqueness"].isna().sum()))
print("\nvalue_counts:")
print(df["structural_uniqueness"].value_counts(dropna=False).to_string())

print("\nRAW VALUES")
print("-" * 78)
for v in df["structural_uniqueness"].drop_duplicates().tolist():
    print(repr(v))

print("\nFIRST 20 RECORDS")
print("-" * 78)
print(
    df[
        [
            "candidate_index",
            "relative_path",
            "variable",
            "read_status",
            "row_count",
            "nonmissing_count",
            "distinct_nonmissing_values",
            "missing_count",
            "structural_uniqueness",
        ]
    ].head(20).to_string(index=False)
)

print("\nDIRECT COUNTS")
print("-" * 78)
print("Total records:", len(df))
print("Number of distinct structural states:", df["structural_uniqueness"].nunique(dropna=False))

print("=" * 78)

OIP v1.0.33 — ESS4
C09R-DIAG — ACTUAL C07R STRUCTURAL STATE AUDIT

C07R COLUMNS
------------------------------------------------------------------------------
['candidate_index', 'relative_path', 'variable', 'source_exists', 'read_status', 'row_count', 'nonmissing_count', 'distinct_nonmissing_values', 'missing_count', 'structural_uniqueness', 'semantic_interpretation', 'authorization']

STRUCTURAL UNIQUENESS
------------------------------------------------------------------------------
dtype: object
null count: 0

value_counts:
structural_uniqueness
OBSERVED_NONUNIQUE              1021
OBSERVED_STRUCTURALLY_UNIQUE      23

RAW VALUES
------------------------------------------------------------------------------
'OBSERVED_STRUCTURALLY_UNIQUE'
'OBSERVED_NONUNIQUE'

FIRST 20 RECORDS
------------------------------------------------------------------------------
 candidate_index                                             relative_path     variable read_status  row_count  nonmissing_count  

In [20]:
# ============================================================
# OIP v1.0.33 — ESS4
# C09R v3 — STRUCTURAL RECONCILIATION GATE
# ============================================================

from pathlib import Path
import json
import pandas as pd

# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C06_STATUS = WORKING_DIR / "c06_final_status.json"
C06_LEDGER = WORKING_DIR / "c06_identifier_candidate_ledger.csv"

C07R_STATUS = WORKING_DIR / "c07r_final_status.json"
C07R_PROFILE = WORKING_DIR / "c07r_identifier_structural_profile.csv"

OUT_STATUS = WORKING_DIR / "c09r_final_status.json"
OUT_SUMMARY = WORKING_DIR / "c09r_structural_reconciliation_summary.json"


# ------------------------------------------------------------
# 1. LOAD UPSTREAM STATUS
# ------------------------------------------------------------

with open(C06_STATUS, "r", encoding="utf-8") as f:
    c06_status = json.load(f)

with open(C07R_STATUS, "r", encoding="utf-8") as f:
    c07r_status = json.load(f)

c06_upstream = c06_status.get("status")
c07r_upstream = c07r_status.get("status")

if c06_upstream != "PASS":
    raise RuntimeError(
        f"C09R BLOCKED: C06 status is {c06_upstream}"
    )

if c07r_upstream != "PASS":
    raise RuntimeError(
        f"C09R BLOCKED: C07R status is {c07r_upstream}"
    )


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

c06 = pd.read_csv(C06_LEDGER)
c07r = pd.read_csv(C07R_PROFILE)


# ------------------------------------------------------------
# 3. REQUIRED COLUMN CHECK
# ------------------------------------------------------------

required_c06 = {
    "relative_path",
    "variable"
}

required_c07r = {
    "relative_path",
    "variable",
    "read_status",
    "structural_uniqueness"
}

missing_c06 = sorted(required_c06 - set(c06.columns))
missing_c07r = sorted(required_c07r - set(c07r.columns))

if missing_c06:
    raise RuntimeError(
        f"C09R BLOCKED: missing C06 columns: {missing_c06}"
    )

if missing_c07r:
    raise RuntimeError(
        f"C09R BLOCKED: missing C07R columns: {missing_c07r}"
    )


# ------------------------------------------------------------
# 4. FILE-VARIABLE RECORD KEYS
# ------------------------------------------------------------

c06_key = (
    c06["relative_path"].astype(str)
    + "||"
    + c06["variable"].astype(str)
)

c07r_key = (
    c07r["relative_path"].astype(str)
    + "||"
    + c07r["variable"].astype(str)
)

c06_keys = set(c06_key)
c07r_keys = set(c07r_key)

only_c06 = c06_keys - c07r_keys
only_c07r = c07r_keys - c06_keys


# ------------------------------------------------------------
# 5. DUPLICATE KEY CHECK
# ------------------------------------------------------------

c06_duplicate_keys = int(c06_key.duplicated().sum())
c07r_duplicate_keys = int(c07r_key.duplicated().sum())


# ------------------------------------------------------------
# 6. ACTUAL STRUCTURAL STATE DOMAIN
# ------------------------------------------------------------

actual_states = set(
    c07r["structural_uniqueness"]
    .dropna()
    .astype(str)
    .unique()
)

# These are the exact states observed in C07R-DIAG.
EXPECTED_STATES = {
    "OBSERVED_STRUCTURALLY_UNIQUE",
    "OBSERVED_NONUNIQUE"
}

unexpected_states = actual_states - EXPECTED_STATES


# ------------------------------------------------------------
# 7. NULL / MISSING STATE CHECK
# ------------------------------------------------------------

structural_null_count = int(
    c07r["structural_uniqueness"].isna().sum()
)


# ------------------------------------------------------------
# 8. STRUCTURAL COUNTS
# ------------------------------------------------------------

unique_count = int(
    (
        c07r["structural_uniqueness"]
        == "OBSERVED_STRUCTURALLY_UNIQUE"
    ).sum()
)

nonunique_count = int(
    (
        c07r["structural_uniqueness"]
        == "OBSERVED_NONUNIQUE"
    ).sum()
)

not_established_count = 0


# ------------------------------------------------------------
# 9. READ STATUS CHECK
# ------------------------------------------------------------

read_ok_count = int(
    (c07r["read_status"] == "READ_OK").sum()
)

non_read_ok_count = int(
    (c07r["read_status"] != "READ_OK").sum()
)


# ------------------------------------------------------------
# 10. ACCOUNTING
# ------------------------------------------------------------

total_c07r = len(c07r)

structural_accounting_match = (
    unique_count
    + nonunique_count
    + not_established_count
    == total_c07r
)


# ------------------------------------------------------------
# 11. CANDIDATE UNIVERSE RECONCILIATION
# ------------------------------------------------------------

candidate_universe_match = (
    len(c06_keys) == len(c07r_keys)
    and len(only_c06) == 0
    and len(only_c07r) == 0
)

file_variable_reconciliation = (
    c06_duplicate_keys == 0
    and c07r_duplicate_keys == 0
    and candidate_universe_match
)


# ------------------------------------------------------------
# 12. FAIL-CLOSED CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if c06_duplicate_keys != 0:
    fail_conditions.append("C06_DUPLICATE_FILE_VARIABLE_KEY")

if c07r_duplicate_keys != 0:
    fail_conditions.append("C07R_DUPLICATE_FILE_VARIABLE_KEY")

if not candidate_universe_match:
    fail_conditions.append("CANDIDATE_UNIVERSE_MISMATCH")

if structural_null_count != 0:
    fail_conditions.append("STRUCTURAL_STATE_NULL")

if unexpected_states:
    fail_conditions.append("UNEXPECTED_STRUCTURAL_STATE")

if non_read_ok_count != 0:
    fail_conditions.append("NON_READ_OK_RECORDS")

if not structural_accounting_match:
    fail_conditions.append("STRUCTURAL_ACCOUNTING_MISMATCH")


# ------------------------------------------------------------
# 13. STATUS
# ------------------------------------------------------------

status = "PASS" if not fail_conditions else "BLOCKED"

fail_closed_triggered = bool(fail_conditions)


# ------------------------------------------------------------
# 14. AUTHORIZATION BOUNDARY
# ------------------------------------------------------------

authorization = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY"
}


# ------------------------------------------------------------
# 15. FINAL WRAPPER
# ------------------------------------------------------------

result = {
    "oip_version": "v1.0.33",
    "cell": "C09R",
    "cell_name": "STRUCTURAL_RECONCILIATION_GATE",
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "status": status,
    "fail_closed_policy_active": True,
    "fail_closed_triggered": fail_closed_triggered,
    "authorization": authorization,
    "payload": {
        "upstream_c06_status": c06_upstream,
        "upstream_c07r_status": c07r_upstream,

        "c06_records": len(c06),
        "c07r_records": total_c07r,

        "c06_unique_file_variable_keys": len(c06_keys),
        "c07r_unique_file_variable_keys": len(c07r_keys),

        "only_in_c06": len(only_c06),
        "only_in_c07r": len(only_c07r),

        "candidate_universe_match": candidate_universe_match,
        "file_variable_reconciliation": file_variable_reconciliation,

        "c06_duplicate_file_variable_keys": c06_duplicate_keys,
        "c07r_duplicate_file_variable_keys": c07r_duplicate_keys,

        "actual_structural_states": sorted(actual_states),
        "unexpected_structural_states": sorted(unexpected_states),
        "structural_state_null_count": structural_null_count,

        "read_ok_count": read_ok_count,
        "non_read_ok_count": non_read_ok_count,

        "structurally_unique": unique_count,
        "structurally_nonunique": nonunique_count,
        "not_established": not_established_count,

        "structural_accounting_match": structural_accounting_match,

        "fail_conditions": fail_conditions,

        "structural_unit": "FILE_VARIABLE_RECORD",
        "semantic_interpretation": "NOT_PERFORMED",
        "semantic_inference_allowed": False,
        "discovery_only": True
    }
}


# ------------------------------------------------------------
# 16. SAVE
# ------------------------------------------------------------

with open(OUT_STATUS, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    json.dump(result["payload"], f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------
# 17. OUTPUT
# ------------------------------------------------------------

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C09R — STRUCTURAL RECONCILIATION GATE")
print("=" * 78)

print("\nUPSTREAM")
print("-" * 78)
print("C06 status :", c06_upstream)
print("C07R status:", c07r_upstream)

print("\nCANDIDATE UNIVERSE")
print("-" * 78)
print("C06 records :", len(c06))
print("C07R records:", total_c07r)
print("C06 unique file-variable keys :", len(c06_keys))
print("C07R unique file-variable keys:", len(c07r_keys))
print("Only in C06  :", len(only_c06))
print("Only in C07R :", len(only_c07r))
print("Universe match:", candidate_universe_match)

print("\nSTRUCTURAL STATES")
print("-" * 78)
print("Actual states:")
for state in sorted(actual_states):
    print(" ", repr(state))

print("Unexpected states:", len(unexpected_states))
print("Null states:", structural_null_count)

print("\nSTRUCTURAL COUNTS")
print("-" * 78)
print("Structurally unique    :", unique_count)
print("Structurally nonunique :", nonunique_count)
print("Not established        :", not_established_count)
print("Total                   :", total_c07r)
print("Accounting match        :", structural_accounting_match)

print("\nREAD STATUS")
print("-" * 78)
print("READ_OK     :", read_ok_count)
print("Non-READ_OK :", non_read_ok_count)

print("\nFAIL-CLOSED")
print("-" * 78)
print("Fail conditions:", fail_conditions)
print("Fail-closed triggered:", fail_closed_triggered)

print("\nAUTHORIZATION")
print("-" * 78)
print("Construct :", authorization["construct"])
print("Key       :", authorization["key"])
print("Relationship:", authorization["relationship"])
print("Cohort    :", authorization["cohort"])

print("\nFINAL STATUS")
print("-" * 78)
print("C09R STATUS:", status)

print("=" * 78)

OIP v1.0.33 — ESS4
C09R — STRUCTURAL RECONCILIATION GATE

UPSTREAM
------------------------------------------------------------------------------
C06 status : PASS
C07R status: PASS

CANDIDATE UNIVERSE
------------------------------------------------------------------------------
C06 records : 1044
C07R records: 1044
C06 unique file-variable keys : 1044
C07R unique file-variable keys: 1044
Only in C06  : 0
Only in C07R : 0
Universe match: True

STRUCTURAL STATES
------------------------------------------------------------------------------
Actual states:
  'OBSERVED_NONUNIQUE'
  'OBSERVED_STRUCTURALLY_UNIQUE'
Unexpected states: 0
Null states: 0

STRUCTURAL COUNTS
------------------------------------------------------------------------------
Structurally unique    : 23
Structurally nonunique : 1021
Not established        : 0
Total                   : 1044
Accounting match        : True

READ STATUS
------------------------------------------------------------------------------
READ_OK   

In [21]:
from pathlib import Path
import json

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C09R_STATUS = WORKING_DIR / "c09r_final_status.json"

if not C09R_STATUS.exists():
    raise FileNotFoundError(
        f"C09R status artifact not found: {C09R_STATUS}"
    )

c09r = json.loads(
    C09R_STATUS.read_text(encoding="utf-8")
)

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C09R — EXACT ARTIFACT SCHEMA AUDIT")
print("=" * 78)

print("\nC09R FILE")
print("-" * 78)
print(C09R_STATUS)

print("\nTOP-LEVEL KEYS")
print("-" * 78)
print(list(c09r.keys()))

print("\nPAYLOAD KEYS")
print("-" * 78)
print(list(c09r.get("payload", {}).keys()))

print("\nAUTHORIZATION KEYS")
print("-" * 78)
print(list(c09r.get("authorization", {}).keys()))

print("\nC09R STATUS")
print("-" * 78)
print(c09r.get("status"))

print("\nFAIL-CLOSED")
print("-" * 78)
print("Policy active :", c09r.get("fail_closed_policy_active"))
print("Triggered     :", c09r.get("fail_closed_triggered"))

print("=" * 78)

OIP v1.0.33 — ESS4
C09R — EXACT ARTIFACT SCHEMA AUDIT

C09R FILE
------------------------------------------------------------------------------
/kaggle/working/oip_v1_0_33/c09r_final_status.json

TOP-LEVEL KEYS
------------------------------------------------------------------------------
['oip_version', 'cell', 'cell_name', 'timestamp_utc', 'status', 'fail_closed_policy_active', 'fail_closed_triggered', 'authorization', 'payload']

PAYLOAD KEYS
------------------------------------------------------------------------------
['upstream_c06_status', 'upstream_c07r_status', 'c06_records', 'c07r_records', 'c06_unique_file_variable_keys', 'c07r_unique_file_variable_keys', 'only_in_c06', 'only_in_c07r', 'candidate_universe_match', 'file_variable_reconciliation', 'c06_duplicate_file_variable_keys', 'c07r_duplicate_file_variable_keys', 'actual_structural_states', 'unexpected_structural_states', 'structural_state_null_count', 'read_ok_count', 'non_read_ok_count', 'structurally_unique', 'structur

In [22]:
# ============================================================
# OIP v1.0.33 — ESS4
# C10 — STRUCTURAL CO-OCCURRENCE & COMPOSITE-KEY DISCOVERY
# ============================================================

from pathlib import Path
import json
import itertools
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C09R_STATUS = WORKING_DIR / "c09r_final_status.json"
C07R_PROFILE = WORKING_DIR / "c07r_identifier_structural_profile.csv"
C06_LEDGER = WORKING_DIR / "c06_identifier_candidate_ledger.csv"

OUT_LEDGER = WORKING_DIR / "c10_composite_key_discovery.csv"
OUT_SUMMARY = WORKING_DIR / "c10_composite_key_discovery_summary.json"
OUT_STATUS = WORKING_DIR / "c10_final_status.json"


# ------------------------------------------------------------
# 1. LOAD C09R
# ------------------------------------------------------------

if not C09R_STATUS.exists():
    raise FileNotFoundError(f"Missing artifact: {C09R_STATUS}")

with open(C09R_STATUS, "r", encoding="utf-8") as f:
    c09r = json.load(f)

if c09r.get("status") != "PASS":
    raise RuntimeError(
        f"C10 BLOCKED: C09R status = {c09r.get('status')}"
    )

c09r_payload = c09r.get("payload", {})

required_c09r_keys = {
    "c06_records",
    "c07r_records",
    "candidate_universe_match",
    "structurally_unique",
    "structurally_nonunique",
    "not_established",
    "structural_accounting_match",
    "actual_structural_states",
    "unexpected_structural_states",
    "structural_state_null_count",
    "semantic_interpretation",
    "semantic_inference_allowed",
    "discovery_only"
}

missing_c09r_keys = sorted(
    required_c09r_keys - set(c09r_payload.keys())
)

if missing_c09r_keys:
    raise RuntimeError(
        f"C10 BLOCKED: missing C09R payload keys: {missing_c09r_keys}"
    )


# ------------------------------------------------------------
# 2. LOAD C07R AND C06
# ------------------------------------------------------------

if not C07R_PROFILE.exists():
    raise FileNotFoundError(f"Missing artifact: {C07R_PROFILE}")

if not C06_LEDGER.exists():
    raise FileNotFoundError(f"Missing artifact: {C06_LEDGER}")

c07r = pd.read_csv(C07R_PROFILE)
c06 = pd.read_csv(C06_LEDGER)


# ------------------------------------------------------------
# 3. REQUIRED COLUMN CHECK
# ------------------------------------------------------------

required_c07r = {
    "candidate_index",
    "relative_path",
    "variable",
    "read_status",
    "row_count",
    "nonmissing_count",
    "distinct_nonmissing_values",
    "missing_count",
    "structural_uniqueness"
}

required_c06 = {
    "relative_path",
    "variable"
}

missing_c07r = sorted(required_c07r - set(c07r.columns))
missing_c06 = sorted(required_c06 - set(c06.columns))

if missing_c07r:
    raise RuntimeError(
        f"C10 BLOCKED: missing C07R columns: {missing_c07r}"
    )

if missing_c06:
    raise RuntimeError(
        f"C10 BLOCKED: missing C06 columns: {missing_c06}"
    )


# ------------------------------------------------------------
# 4. VERIFY ACTUAL STRUCTURAL STATE DOMAIN
# ------------------------------------------------------------

actual_states = set(
    c07r["structural_uniqueness"]
    .dropna()
    .astype(str)
    .unique()
)

expected_states = {
    "OBSERVED_STRUCTURALLY_UNIQUE",
    "OBSERVED_NONUNIQUE"
}

unexpected_states = actual_states - expected_states

structural_null_count = int(
    c07r["structural_uniqueness"].isna().sum()
)

if unexpected_states or structural_null_count != 0:
    raise RuntimeError(
        "C10 BLOCKED: unexpected or null structural states."
    )


# ------------------------------------------------------------
# 5. VERIFY C09R COUNTS AGAINST C07R
# ------------------------------------------------------------

c07r_unique_count = int(
    (
        c07r["structural_uniqueness"]
        == "OBSERVED_STRUCTURALLY_UNIQUE"
    ).sum()
)

c07r_nonunique_count = int(
    (
        c07r["structural_uniqueness"]
        == "OBSERVED_NONUNIQUE"
    ).sum()
)

c07r_total = len(c07r)

c09r_counts_match = (
    c07r_total == c09r_payload["c07r_records"]
    and
    c07r_unique_count == c09r_payload["structurally_unique"]
    and
    c07r_nonunique_count == c09r_payload["structurally_nonunique"]
    and
    c09r_payload["not_established"] == 0
    and
    c09r_payload["structural_accounting_match"] is True
)

if not c09r_counts_match:
    raise RuntimeError(
        "C10 BLOCKED: C09R ↔ C07R structural counts do not reconcile."
    )


# ------------------------------------------------------------
# 6. VERIFY C06 ↔ C07R CANDIDATE UNIVERSE
# ------------------------------------------------------------

c06_key = (
    c06["relative_path"].astype(str)
    + "||"
    + c06["variable"].astype(str)
)

c07r_key = (
    c07r["relative_path"].astype(str)
    + "||"
    + c07r["variable"].astype(str)
)

c06_keys = set(c06_key)
c07r_keys = set(c07r_key)

only_c06 = c06_keys - c07r_keys
only_c07r = c07r_keys - c06_keys

candidate_universe_match = (
    len(only_c06) == 0
    and len(only_c07r) == 0
    and len(c06_keys) == len(c07r_keys)
    and c09r_payload["candidate_universe_match"] is True
)

if not candidate_universe_match:
    raise RuntimeError(
        "C10 BLOCKED: C06 ↔ C07R candidate universe mismatch."
    )


# ------------------------------------------------------------
# 7. DISCOVERY UNIVERSE
# ------------------------------------------------------------

# Only file-variable records observed as structurally unique
# are selected for structural composite discovery.
#
# This is NOT key authorization.

unique_candidates = c07r[
    c07r["structural_uniqueness"]
    == "OBSERVED_STRUCTURALLY_UNIQUE"
].copy()

unique_candidates = unique_candidates[
    ["candidate_index", "relative_path", "variable",
     "row_count", "nonmissing_count",
     "distinct_nonmissing_values", "missing_count"]
].sort_values(
    ["relative_path", "candidate_index", "variable"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 8. FILE-LEVEL CO-OCCURRENCE
# ------------------------------------------------------------

# Group by actual source file.
# Candidate pairs are generated only when both variables
# occur in the same source file.

records = []

for relative_path, group in unique_candidates.groupby(
    "relative_path",
    sort=True
):

    vars_in_file = group["variable"].astype(str).tolist()

    # deterministic ordering
    vars_in_file = sorted(set(vars_in_file))

    if len(vars_in_file) < 2:
        continue

    for var_a, var_b in itertools.combinations(vars_in_file, 2):

        records.append({
            "relative_path": relative_path,
            "variable_a": var_a,
            "variable_b": var_b,
            "co_occurrence": "OBSERVED_WITHIN_FILE",
            "structural_key_status": "NOT_AUTHORIZED",
            "relationship_status": "NOT_AUTHORIZED",
            "cohort_status": "NOT_AUTHORIZED",
            "construct_status": "NOT_AUTHORIZED",
            "discovery_only": True,
            "semantic_interpretation": "NOT_PERFORMED",
            "semantic_inference_allowed": False
        })


# ------------------------------------------------------------
# 9. DISCOVERY LEDGER
# ------------------------------------------------------------

cooccurrence_df = pd.DataFrame(records)

if len(cooccurrence_df) == 0:

    cooccurrence_df = pd.DataFrame(
        columns=[
            "relative_path",
            "variable_a",
            "variable_b",
            "co_occurrence",
            "structural_key_status",
            "relationship_status",
            "cohort_status",
            "construct_status",
            "discovery_only",
            "semantic_interpretation",
            "semantic_inference_allowed"
        ]
    )


# ------------------------------------------------------------
# 10. DUPLICATE PAIR CHECK
# ------------------------------------------------------------

if len(cooccurrence_df) > 0:

    pair_key = (
        cooccurrence_df["relative_path"].astype(str)
        + "||"
        + cooccurrence_df["variable_a"].astype(str)
        + "||"
        + cooccurrence_df["variable_b"].astype(str)
    )

    duplicate_pair_count = int(pair_key.duplicated().sum())

else:
    duplicate_pair_count = 0


# ------------------------------------------------------------
# 11. PAIR ACCOUNTING
# ------------------------------------------------------------

expected_pair_count = 0

for _, group in unique_candidates.groupby(
    "relative_path",
    sort=True
):
    n = len(set(group["variable"].astype(str)))
    if n >= 2:
        expected_pair_count += n * (n - 1) // 2

observed_pair_count = len(cooccurrence_df)

pair_accounting_match = (
    expected_pair_count == observed_pair_count
    and duplicate_pair_count == 0
)


# ------------------------------------------------------------
# 12. AUTHORIZATION LEAK AUDIT
# ------------------------------------------------------------

authorization_columns = [
    "structural_key_status",
    "relationship_status",
    "cohort_status",
    "construct_status"
]

authorization_leaks = 0

for col in authorization_columns:
    if len(cooccurrence_df) == 0:
        continue

    authorization_leaks += int(
        (
            cooccurrence_df[col]
            != "NOT_AUTHORIZED"
        ).sum()
    )


# ------------------------------------------------------------
# 13. BOUNDARY AUDIT
# ------------------------------------------------------------

semantic_inference_leaks = 0

if len(cooccurrence_df) > 0:
    semantic_inference_leaks = int(
        (
            cooccurrence_df["semantic_inference_allowed"]
            != False
        ).sum()
    )

discovery_violations = 0

if len(cooccurrence_df) > 0:
    discovery_violations = int(
        (
            cooccurrence_df["discovery_only"]
            != True
        ).sum()
    )


# ------------------------------------------------------------
# 14. STRUCTURAL INTERPRETATION BOUNDARY
# ------------------------------------------------------------

# IMPORTANT:
# Co-occurrence means only that two candidate variables
# are present in the same source file.
#
# It does NOT establish:
# - composite key validity
# - relational integrity
# - joinability
# - identifier semantics
# - causal relationship
# - cohort membership

structural_definition = (
    "WITHIN_FILE_CANDIDATE_CO_OCCURRENCE_ONLY"
)


# ------------------------------------------------------------
# 15. FAIL-CLOSED CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if not c09r_counts_match:
    fail_conditions.append("C09R_C07R_RECONCILIATION_FAILED")

if not candidate_universe_match:
    fail_conditions.append("CANDIDATE_UNIVERSE_MISMATCH")

if unexpected_states:
    fail_conditions.append("UNEXPECTED_STRUCTURAL_STATE")

if structural_null_count != 0:
    fail_conditions.append("STRUCTURAL_STATE_NULL")

if not pair_accounting_match:
    fail_conditions.append("PAIR_ACCOUNTING_MISMATCH")

if duplicate_pair_count != 0:
    fail_conditions.append("DUPLICATE_PAIR_RECORD")

if authorization_leaks != 0:
    fail_conditions.append("AUTHORIZATION_LEAK")

if semantic_inference_leaks != 0:
    fail_conditions.append("SEMANTIC_INFERENCE_LEAK")

if discovery_violations != 0:
    fail_conditions.append("DISCOVERY_BOUNDARY_VIOLATION")


# ------------------------------------------------------------
# 16. FINAL STATUS
# ------------------------------------------------------------

status = "PASS" if not fail_conditions else "BLOCKED"

fail_closed_triggered = bool(fail_conditions)


# ------------------------------------------------------------
# 17. AUTHORIZATION
# ------------------------------------------------------------

authorization = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY"
}


# ------------------------------------------------------------
# 18. PAYLOAD
# ------------------------------------------------------------

payload = {

    "upstream_c09r_status": c09r.get("status"),

    "c06_records": len(c06),
    "c07r_records": len(c07r),

    "c06_unique_file_variable_keys": len(c06_keys),
    "c07r_unique_file_variable_keys": len(c07r_keys),

    "only_in_c06": len(only_c06),
    "only_in_c07r": len(only_c07r),

    "candidate_universe_match": candidate_universe_match,

    "c09r_counts_match": c09r_counts_match,

    "structural_state_domain": sorted(actual_states),
    "unexpected_structural_states": sorted(unexpected_states),
    "structural_state_null_count": structural_null_count,

    "structurally_unique_candidates": c07r_unique_count,
    "structurally_nonunique_candidates": c07r_nonunique_count,

    "composite_discovery_universe":
        "OBSERVED_STRUCTURALLY_UNIQUE_FILE_VARIABLE_RECORDS",

    "unique_candidate_files":
        int(unique_candidates["relative_path"].nunique())
        if len(unique_candidates) > 0 else 0,

    "unique_candidate_records":
        len(unique_candidates),

    "expected_cooccurrence_pairs":
        expected_pair_count,

    "observed_cooccurrence_pairs":
        observed_pair_count,

    "duplicate_pair_records":
        duplicate_pair_count,

    "pair_accounting_match":
        pair_accounting_match,

    "structural_definition":
        structural_definition,

    "authorization_leaks":
        authorization_leaks,

    "semantic_inference_leaks":
        semantic_inference_leaks,

    "discovery_boundary_violations":
        discovery_violations,

    "structural_key_authorization":
        "NOT_AUTHORIZED",

    "relationship_authorization":
        "NOT_AUTHORIZED",

    "cohort_authorization":
        "NOT_AUTHORIZED",

    "construct_authorization":
        "NOT_AUTHORIZED",

    "semantic_interpretation":
        "NOT_PERFORMED",

    "semantic_inference_allowed":
        False,

    "discovery_only":
        True,

    "fail_conditions":
        fail_conditions
}


# ------------------------------------------------------------
# 19. FINAL WRAPPER
# ------------------------------------------------------------

result = {
    "oip_version": "v1.0.33",
    "cell": "C10",
    "cell_name": "STRUCTURAL_CO_OCCURRENCE_COMPOSITE_KEY_DISCOVERY",
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "status": status,
    "fail_closed_policy_active": True,
    "fail_closed_triggered": fail_closed_triggered,
    "authorization": authorization,
    "payload": payload
}


# ------------------------------------------------------------
# 20. SAVE ARTIFACTS
# ------------------------------------------------------------

cooccurrence_df.to_csv(
    OUT_LEDGER,
    index=False
)

with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    json.dump(
        payload,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(OUT_STATUS, "w", encoding="utf-8") as f:
    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 21. OUTPUT
# ------------------------------------------------------------

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C10 — STRUCTURAL CO-OCCURRENCE & COMPOSITE-KEY DISCOVERY")
print("=" * 78)

print("\nUPSTREAM")
print("-" * 78)
print("C09R status:", c09r.get("status"))

print("\nCANDIDATE RECONCILIATION")
print("-" * 78)
print("C06 records :", len(c06))
print("C07R records:", len(c07r))
print("Only in C06 :", len(only_c06))
print("Only in C07R:", len(only_c07r))
print("Universe match:", candidate_universe_match)

print("\nSTRUCTURAL DOMAIN")
print("-" * 78)
print("Actual states:", sorted(actual_states))
print("Unexpected states:", len(unexpected_states))
print("Null states:", structural_null_count)

print("\nSTRUCTURAL CANDIDATES")
print("-" * 78)
print("Structurally unique :", c07r_unique_count)
print("Structurally nonunique:", c07r_nonunique_count)

print("\nCO-OCCURRENCE")
print("-" * 78)
print("Unique candidate files :", payload["unique_candidate_files"])
print("Unique candidate records:", payload["unique_candidate_records"])
print("Expected pairs :", expected_pair_count)
print("Observed pairs :", observed_pair_count)
print("Duplicate pairs:", duplicate_pair_count)
print("Pair accounting:", pair_accounting_match)

print("\nBOUNDARY")
print("-" * 78)
print("Structural definition:", structural_definition)
print("Authorization leaks:", authorization_leaks)
print("Semantic inference leaks:", semantic_inference_leaks)
print("Discovery violations:", discovery_violations)

print("\nAUTHORIZATION")
print("-" * 78)
print("Key         :", authorization["key"])
print("Relationship:", authorization["relationship"])
print("Cohort      :", authorization["cohort"])
print("Construct   :", authorization["construct"])

print("\nFAIL-CLOSED")
print("-" * 78)
print("Fail conditions:", fail_conditions)
print("Fail-closed triggered:", fail_closed_triggered)

print("\nFINAL STATUS")
print("-" * 78)
print("C10 STATUS:", status)

print("=" * 78)

OIP v1.0.33 — ESS4
C10 — STRUCTURAL CO-OCCURRENCE & COMPOSITE-KEY DISCOVERY

UPSTREAM
------------------------------------------------------------------------------
C09R status: PASS

CANDIDATE RECONCILIATION
------------------------------------------------------------------------------
C06 records : 1044
C07R records: 1044
Only in C06 : 0
Only in C07R: 0
Universe match: True

STRUCTURAL DOMAIN
------------------------------------------------------------------------------
Actual states: ['OBSERVED_NONUNIQUE', 'OBSERVED_STRUCTURALLY_UNIQUE']
Unexpected states: 0
Null states: 0

STRUCTURAL CANDIDATES
------------------------------------------------------------------------------
Structurally unique : 23
Structurally nonunique: 1021

CO-OCCURRENCE
------------------------------------------------------------------------------
Unique candidate files : 23
Unique candidate records: 23
Expected pairs : 0
Observed pairs : 0
Duplicate pairs: 0
Pair accounting: True

BOUNDARY
---------------------

In [23]:
# ============================================================
# C10-DIAG — ACTUAL UNIQUE CANDIDATE FILE DISTRIBUTION
# ============================================================

import pandas as pd
from pathlib import Path

C07R_PROFILE = Path(
    "/kaggle/working/oip_v1_0_33/"
    "c07r_identifier_structural_profile.csv"
)

c07r = pd.read_csv(C07R_PROFILE)

# Exact state observed previously
unique_df = c07r[
    c07r["structural_uniqueness"]
    == "OBSERVED_STRUCTURALLY_UNIQUE"
].copy()

print("=" * 78)
print("OIP v1.0.33 — ESS4")
print("C10-DIAG — UNIQUE CANDIDATE FILE DISTRIBUTION")
print("=" * 78)

print("\nTOTAL UNIQUE CANDIDATES")
print("-" * 78)
print("Total:", len(unique_df))

print("\nUNIQUE CANDIDATES GROUPED BY FILE")
print("-" * 78)

grouped = (
    unique_df
    .groupby("relative_path", sort=True)["variable"]
    .apply(list)
)

print(grouped.to_string())

print("\nDISTINCT FILE COUNT")
print("-" * 78)
print("Distinct files:", unique_df["relative_path"].nunique())

print("\nFILES WITH MULTIPLE UNIQUE CANDIDATES")
print("-" * 78)

multi = grouped[grouped.apply(len) > 1]

print("Number of files:", len(multi))

if len(multi) > 0:
    print(multi.to_string())
else:
    print("None")

print("\nDIRECT PAIR COUNT")
print("-" * 78)

expected_pairs = sum(
    len(vars_) * (len(vars_) - 1) // 2
    for vars_ in grouped
)

print("Expected within-file pairs:", expected_pairs)

print("\nACCOUNTING")
print("-" * 78)
print(
    "23 candidates =",
    len(unique_df),
    "and",
    "23 distinct files =",
    unique_df["relative_path"].nunique()
)

print("=" * 78)

OIP v1.0.33 — ESS4
C10-DIAG — UNIQUE CANDIDATE FILE DISTRIBUTION

TOTAL UNIQUE CANDIDATES
------------------------------------------------------------------------------
Total: 23

UNIQUE CANDIDATES GROUPED BY FILE
------------------------------------------------------------------------------
relative_path
ETH_2018_ESS_v04_M_Stata/ETH_HouseholdGeovariables_Y4.dta    [household_id]
ETH_2018_ESS_v04_M_Stata/cons_agg_w4.dta                     [household_id]
ETH_2018_ESS_v04_M_Stata/sect01a_com_w4.dta                         [ea_id]
ETH_2018_ESS_v04_M_Stata/sect01b_com_w4.dta                         [ea_id]
ETH_2018_ESS_v04_M_Stata/sect03_com_w4.dta                          [ea_id]
ETH_2018_ESS_v04_M_Stata/sect04_com_w4.dta                          [ea_id]
ETH_2018_ESS_v04_M_Stata/sect05_com_w4.dta                          [ea_id]
ETH_2018_ESS_v04_M_Stata/sect06_com_w4.dta                          [ea_id]
ETH_2018_ESS_v04_M_Stata/sect09_com_w4.dta                          [ea_id]
ETH_2018_

In [24]:
# ============================================================
# OIP v1.0.33 — C11A
# DOCUMENTARY SOURCE INVENTORY & INTEGRITY GATE
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

print("=" * 72)
print("OIP v1.0.33 — C11A")
print("DOCUMENTARY SOURCE INVENTORY & INTEGRITY GATE")
print("=" * 72)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C00_STATUS = WORKING_DIR / "c00_source_scope.json"
C08_STATUS = WORKING_DIR / "c08_final_status.json"

OUTPUT_CSV = WORKING_DIR / "c11a_documentary_source_inventory.csv"
OUTPUT_SUMMARY = WORKING_DIR / "c11a_documentary_source_summary.json"
OUTPUT_STATUS = WORKING_DIR / "c11a_final_status.json"

# ------------------------------------------------------------
# 2. REQUIRED ARTIFACT CHECK
# ------------------------------------------------------------

required_files = {
    "C00": C00_STATUS,
    "C08": C08_STATUS,
}

missing_required = [
    name for name, path in required_files.items()
    if not path.exists()
]

if missing_required:
    raise FileNotFoundError(
        f"Required upstream artifact(s) missing: {missing_required}"
    )

# ------------------------------------------------------------
# 3. LOAD UPSTREAM STATUS
# ------------------------------------------------------------

with C00_STATUS.open("r", encoding="utf-8") as f:
    c00 = json.load(f)

with C08_STATUS.open("r", encoding="utf-8") as f:
    c08 = json.load(f)

c00_status = c00.get("status")
c08_status = c08.get("status")

print("C00 status:", c00_status)
print("C08 status:", c08_status)

if c00_status != "PASS":
    raise RuntimeError("C11A BLOCKED: C00 is not PASS.")

if c08_status != "PASS":
    raise RuntimeError("C11A BLOCKED: C08 is not PASS.")

# ------------------------------------------------------------
# 4. SOURCE ROOT
# ------------------------------------------------------------

c00_payload = c00.get("payload", {})

SOURCE_ROOT = Path(
    c00_payload.get("source_root", "")
)

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(
        f"Source root does not exist: {SOURCE_ROOT}"
    )

print("\nSOURCE ROOT:")
print(SOURCE_ROOT)

# ------------------------------------------------------------
# 5. DISCOVER PDF FILES
# ------------------------------------------------------------

pdf_paths = sorted(
    p for p in SOURCE_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() == ".pdf"
)

print("\nPDF files discovered:", len(pdf_paths))

EXPECTED_PDF_COUNT = 7

# ------------------------------------------------------------
# 6. SHA-256 FUNCTION
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

# ------------------------------------------------------------
# 7. PAGE COUNT
# ------------------------------------------------------------

page_count_errors = []

try:
    from pypdf import PdfReader
except Exception as e:
    raise RuntimeError(
        f"Unable to import pypdf: {repr(e)}"
    )

records = []

for idx, pdf_path in enumerate(pdf_paths, start=1):

    print(
        f"\n[{idx}/{len(pdf_paths)}] "
        f"{pdf_path.name}"
    )

    relative_path = str(
        pdf_path.relative_to(SOURCE_ROOT)
    )

    file_size = pdf_path.stat().st_size

    sha256 = sha256_file(pdf_path)

    page_count = None
    page_count_status = "NOT_ESTABLISHED"
    page_count_error = None

    # IMPORTANT:
    # This reads PDF structure only.
    # It does NOT extract page text.
    try:
        reader = PdfReader(
            str(pdf_path),
            strict=False
        )

        page_count = len(reader.pages)
        page_count_status = "OBSERVED"

    except Exception as e:
        page_count_error = repr(e)
        page_count_errors.append({
            "relative_path": relative_path,
            "error": repr(e)
        })

    records.append({
        "relative_path": relative_path,
        "file_name": pdf_path.name,
        "file_size_bytes": file_size,
        "sha256": sha256,
        "page_count": page_count,
        "page_count_status": page_count_status,
        "page_count_error": page_count_error,
        "text_extraction_performed": False,
        "semantic_interpretation": "NOT_PERFORMED",
        "semantic_inference_allowed": False,
    })

# ------------------------------------------------------------
# 8. DATAFRAME
# ------------------------------------------------------------

inventory_df = pd.DataFrame(records)

# ------------------------------------------------------------
# 9. INTEGRITY CHECKS
# ------------------------------------------------------------

pdf_count_match = (
    len(inventory_df) == EXPECTED_PDF_COUNT
)

zero_byte_count = int(
    (inventory_df["file_size_bytes"] == 0).sum()
)

duplicate_relative_paths = int(
    inventory_df["relative_path"].duplicated().sum()
)

duplicate_file_names = int(
    inventory_df["file_name"].duplicated().sum()
)

page_count_observed = int(
    (
        inventory_df["page_count_status"]
        == "OBSERVED"
    ).sum()
)

page_count_error_count = len(page_count_errors)

all_nonzero = (
    zero_byte_count == 0
)

no_duplicate_paths = (
    duplicate_relative_paths == 0
)

no_duplicate_names = (
    duplicate_file_names == 0
)

all_page_counts_observed = (
    page_count_observed == len(inventory_df)
)

# ------------------------------------------------------------
# 10. FAIL-CLOSED CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if not pdf_count_match:
    fail_conditions.append(
        "PDF_COUNT_MISMATCH"
    )

if not all_nonzero:
    fail_conditions.append(
        "ZERO_BYTE_PDF_PRESENT"
    )

if not no_duplicate_paths:
    fail_conditions.append(
        "DUPLICATE_RELATIVE_PATH"
    )

if not no_duplicate_names:
    fail_conditions.append(
        "DUPLICATE_FILE_NAME"
    )

if not all_page_counts_observed:
    fail_conditions.append(
        "PAGE_COUNT_NOT_ESTABLISHED"
    )

# ------------------------------------------------------------
# 11. AUTHORIZATION BOUNDARY
# ------------------------------------------------------------

authorization = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY",
}

# ------------------------------------------------------------
# 12. FINAL STATUS
# ------------------------------------------------------------

status = (
    "PASS"
    if len(fail_conditions) == 0
    else "BLOCKED"
)

fail_closed_triggered = (
    len(fail_conditions) > 0
)

timestamp_utc = datetime.now(
    timezone.utc
).isoformat()

# ------------------------------------------------------------
# 13. SUMMARY
# ------------------------------------------------------------

summary = {
    "upstream_c00_status": c00_status,
    "upstream_c08_status": c08_status,

    "source_root": str(SOURCE_ROOT),

    "expected_pdf_count": EXPECTED_PDF_COUNT,
    "observed_pdf_count": int(len(inventory_df)),
    "pdf_count_match": bool(pdf_count_match),

    "zero_byte_pdf_count": zero_byte_count,

    "duplicate_relative_path_count":
        duplicate_relative_paths,

    "duplicate_file_name_count":
        duplicate_file_names,

    "page_count_observed":
        page_count_observed,

    "page_count_error_count":
        page_count_error_count,

    "all_page_counts_observed":
        bool(all_page_counts_observed),

    "text_extraction_performed": False,

    "semantic_interpretation":
        "NOT_PERFORMED",

    "semantic_inference_allowed":
        False,

    "documentary_evidence_state":
        "INVENTORY_AND_INTEGRITY_ONLY",

    "fail_conditions":
        fail_conditions,
}

# ------------------------------------------------------------
# 14. SAVE INVENTORY
# ------------------------------------------------------------

inventory_df.to_csv(
    OUTPUT_CSV,
    index=False
)

# ------------------------------------------------------------
# 15. SAVE SUMMARY
# ------------------------------------------------------------

with OUTPUT_SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 16. FINAL WRAPPER
# ------------------------------------------------------------

final_status = {
    "oip_version": "v1.0.33",
    "cell": "C11A",
    "cell_name":
        "DOCUMENTARY SOURCE INVENTORY & INTEGRITY GATE",

    "timestamp_utc":
        timestamp_utc,

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        authorization,

    "payload":
        summary,
}

with OUTPUT_STATUS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_status,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 17. OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C11A RESULTS")
print("=" * 72)

print("PDF count:", len(inventory_df))
print("Expected PDF count:", EXPECTED_PDF_COUNT)
print("PDF count match:", pdf_count_match)

print("Zero-byte PDFs:", zero_byte_count)
print("Duplicate paths:", duplicate_relative_paths)
print("Duplicate names:", duplicate_file_names)

print(
    "Page counts observed:",
    page_count_observed,
    "/",
    len(inventory_df)
)

print(
    "Page-count errors:",
    page_count_error_count
)

print(
    "Text extraction performed:",
    False
)

print(
    "Semantic interpretation:",
    "NOT_PERFORMED"
)

print(
    "Semantic inference allowed:",
    False
)

print(
    "Fail conditions:",
    fail_conditions
)

print(
    "Fail-closed triggered:",
    fail_closed_triggered
)

print("\nC11A STATUS:", status)

print("\nArtifacts:")
print(OUTPUT_CSV)
print(OUTPUT_SUMMARY)
print(OUTPUT_STATUS)

print("=" * 72)

OIP v1.0.33 — C11A
DOCUMENTARY SOURCE INVENTORY & INTEGRITY GATE
C00 status: PASS
C08 status: PASS

SOURCE ROOT:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-33-ethiopia-ess4-evidence-audit

PDF files discovered: 7

[1/7] 2018_19_ethiopian_socioeconomic_survey_household_questionnaire.pdf

[2/7] 2018_19_ethiopian_socioeconomic_survey_post_harvest_agricultural_questionnaire.pdf

[3/7] 2018_19_ethiopian_socioeconomic_survey_post_planting_crop_cut_and_livestock_agricultural_questionnaire.pdf

[4/7] Basic Information Document Ethiopia Socioeconomic Survey 2018-19 V2.pdf

[5/7] LSMS_ESS_16-02-21.pdf

[6/7] ddi-documentation-english_microdata-3823.pdf

[7/7] manual_household_agriculture_and_community_questionnaires.pdf

C11A RESULTS
PDF count: 7
Expected PDF count: 7
PDF count match: True
Zero-byte PDFs: 0
Duplicate paths: 0
Duplicate names: 0
Page counts observed: 7 / 7
Page-count errors: 0
Text extraction performed: False
Semantic interpretation: NOT_PERFORMED
Semantic inference allowed: 

In [25]:
# ============================================================
# OIP v1.0.33 — C11B
# TARGETED DOCUMENTARY EVIDENCE DISCOVERY
# ============================================================

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from pypdf import PdfReader

print("=" * 72)
print("OIP v1.0.33 — C11B")
print("TARGETED DOCUMENTARY EVIDENCE DISCOVERY")
print("=" * 72)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C00_STATUS = WORKING_DIR / "c00_source_scope.json"
C06_STATUS = WORKING_DIR / "c06_final_status.json"
C08_STATUS = WORKING_DIR / "c08_final_status.json"
C09R_STATUS = WORKING_DIR / "c09r_final_status.json"
C10_STATUS = WORKING_DIR / "c10_final_status.json"

C06_LEDGER = WORKING_DIR / "c06_identifier_candidate_ledger.csv"
C11A_STATUS = WORKING_DIR / "c11a_final_status.json"
C11A_INVENTORY = WORKING_DIR / "c11a_documentary_source_inventory.csv"

OUTPUT_LEDGER = WORKING_DIR / "c11b_targeted_documentary_discovery.csv"
OUTPUT_SUMMARY = WORKING_DIR / "c11b_documentary_discovery_summary.json"
OUTPUT_STATUS = WORKING_DIR / "c11b_final_status.json"

# ------------------------------------------------------------
# 2. REQUIRED ARTIFACTS
# ------------------------------------------------------------

required = {
    "C00": C00_STATUS,
    "C06": C06_STATUS,
    "C08": C08_STATUS,
    "C09R": C09R_STATUS,
    "C10": C10_STATUS,
    "C06_LEDGER": C06_LEDGER,
    "C11A": C11A_STATUS,
    "C11A_INVENTORY": C11A_INVENTORY,
}

missing = [
    name for name, path in required.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        f"Required artifact(s) missing: {missing}"
    )

# ------------------------------------------------------------
# 3. LOAD UPSTREAM
# ------------------------------------------------------------

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

c00 = load_json(C00_STATUS)
c06 = load_json(C06_STATUS)
c08 = load_json(C08_STATUS)
c09r = load_json(C09R_STATUS)
c10 = load_json(C10_STATUS)
c11a = load_json(C11A_STATUS)

upstream_status = {
    "C00": c00.get("status"),
    "C06": c06.get("status"),
    "C08": c08.get("status"),
    "C09R": c09r.get("status"),
    "C10": c10.get("status"),
    "C11A": c11a.get("status"),
}

print("\nUPSTREAM STATUS")
for cell, status in upstream_status.items():
    print(f"{cell}: {status}")

if any(status != "PASS"
       for status in upstream_status.values()):
    raise RuntimeError(
        "C11B BLOCKED: one or more required upstream cells are not PASS."
    )

# ------------------------------------------------------------
# 4. SOURCE ROOT
# ------------------------------------------------------------

source_root = Path(
    c00.get("payload", {}).get("source_root", "")
)

if not source_root.exists():
    raise FileNotFoundError(
        f"Source root does not exist: {source_root}"
    )

# ------------------------------------------------------------
# 5. LOAD C06 CANDIDATE LEDGER
# ------------------------------------------------------------

candidate_df = pd.read_csv(C06_LEDGER)

required_columns = [
    "relative_path",
    "variable",
    "variable_label",
]

missing_columns = [
    col for col in required_columns
    if col not in candidate_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"C11B BLOCKED: C06 ledger missing columns: {missing_columns}"
    )

candidate_df = candidate_df[
    required_columns
].copy()

candidate_df["variable"] = (
    candidate_df["variable"]
    .fillna("")
    .astype(str)
    .str.strip()
)

candidate_df["variable_label"] = (
    candidate_df["variable_label"]
    .fillna("")
    .astype(str)
    .str.strip()
)

candidate_df["relative_path"] = (
    candidate_df["relative_path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("\nC06 candidate records:",
      len(candidate_df))

print(
    "C06 unique variables:",
    candidate_df["variable"].nunique()
)

# ------------------------------------------------------------
# 6. TARGET VOCABULARY
# ------------------------------------------------------------
#
# IMPORTANT:
# These are documentary routing terms only.
# They DO NOT authorize any construct or variable.
#
# They are based on the canonical OIP construct names and
# outcome terminology. A documentary occurrence is only
# a discovery signal.
# ------------------------------------------------------------

TARGET_TERMS = {
    "GFL": [
        "GFL",
        "Grounded Feedback Loop",
    ],

    "IDS": [
        "IDS",
        "Infrastructure Dependence",
        "Infrastructure Dependency",
        "Infrastructure Dependence Score",
        "Infrastructure Dependency Score",
    ],

    "AML": [
        "AML",
        "Adaptive Moral Logic",
    ],

    "OUTCOME": [
        "outcome",
        "outcomes",
        "dependent variable",
        "target variable",
    ],
}

# ------------------------------------------------------------
# 7. PREPARE TARGET CANDIDATE ROUTING
# ------------------------------------------------------------
#
# We do NOT search all 1044 candidates.
# We first identify candidates whose documented variable
# name or label contains an OIP routing term.
#
# This is still discovery only.
# ------------------------------------------------------------

routing_rows = []

for _, row in candidate_df.iterrows():

    variable = row["variable"]
    label = row["variable_label"]

    combined = f"{variable} {label}".lower()

    matched_targets = []

    for target, terms in TARGET_TERMS.items():

        for term in terms:

            if term.lower() in combined:
                matched_targets.append(
                    (target, term)
                )

    if matched_targets:

        seen = set()

        for target, term in matched_targets:

            key = (
                row["relative_path"],
                variable,
                target,
                term,
            )

            if key in seen:
                continue

            seen.add(key)

            routing_rows.append({
                "relative_path":
                    row["relative_path"],

                "variable":
                    variable,

                "variable_label":
                    label,

                "target_construct":
                    target,

                "routing_term":
                    term,

                "routing_basis":
                    "C06_VARIABLE_NAME_OR_LABEL_SUBSTRING",

                "routing_only":
                    True,

                "semantic_interpretation":
                    "NOT_PERFORMED",

                "semantic_inference_allowed":
                    False,
            })

routing_df = pd.DataFrame(
    routing_rows
)

print(
    "\nTargeted candidate routing records:",
    len(routing_df)
)

print(
    "Targeted unique variables:",
    routing_df["variable"].nunique()
    if len(routing_df) else 0
)

# ------------------------------------------------------------
# 8. PDF INVENTORY FROM C11A
# ------------------------------------------------------------

pdf_inventory = pd.read_csv(
    C11A_INVENTORY
)

if len(pdf_inventory) != 7:
    raise RuntimeError(
        "C11B BLOCKED: C11A PDF inventory is not 7."
    )

pdf_paths = []

for _, row in pdf_inventory.iterrows():

    rel = str(
        row["relative_path"]
    )

    pdf_path = source_root / rel

    if not pdf_path.exists():
        raise FileNotFoundError(
            f"PDF from C11A inventory missing: {pdf_path}"
        )

    pdf_paths.append(
        pdf_path
    )

# ------------------------------------------------------------
# 9. SEARCH FUNCTION
# ------------------------------------------------------------

def normalize_text(text):

    text = text or ""

    text = text.replace(
        "\x00",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

def exact_term_pattern(term):

    return re.compile(
        re.escape(term),
        flags=re.IGNORECASE
    )

# ------------------------------------------------------------
# 10. TARGETED PAGE-LEVEL DISCOVERY
# ------------------------------------------------------------
#
# No full-document concatenation.
# Each PDF is processed page by page.
#
# We record:
#   PDF
#   page
#   target
#   routing term
#   candidate
#   occurrence count
#
# This is discovery only.
# ------------------------------------------------------------

evidence_records = []
pdf_errors = []

if len(routing_df) > 0:

    for pdf_index, pdf_path in enumerate(
        pdf_paths,
        start=1
    ):

        print(
            f"\nPDF [{pdf_index}/{len(pdf_paths)}]:",
            pdf_path.name
        )

        try:

            reader = PdfReader(
                str(pdf_path),
                strict=False
            )

        except Exception as e:

            pdf_errors.append({
                "relative_path":
                    str(pdf_path.relative_to(source_root)),
                "error":
                    repr(e),
            })

            continue

        for page_index, page in enumerate(
            reader.pages,
            start=1
        ):

            try:

                text = page.extract_text() or ""

                text = normalize_text(
                    text
                )

            except KeyboardInterrupt:

                raise

            except Exception as page_error:

                pdf_errors.append({
                    "relative_path":
                        str(
                            pdf_path.relative_to(
                                source_root
                            )
                        ),

                    "page":
                        page_index,

                    "error":
                        repr(page_error),
                })

                continue

            if not text:
                continue

            # ------------------------------------------------
            # Search only routing candidates relevant to this
            # PDF. No broad 1044 x 7 regex matrix.
            # ------------------------------------------------

            for _, route in routing_df.iterrows():

                candidate_path = (
                    source_root /
                    route["relative_path"]
                )

                # Candidate belongs to a data file, not the PDF.
                # Documentary search is against PDF text only.
                # Therefore candidate routing is global and
                # documentary occurrence is recorded separately.
                #
                # To avoid repeatedly scanning irrelevant terms,
                # search only the routing term itself.
                term = route["routing_term"]

                pattern = exact_term_pattern(
                    term
                )

                matches = list(
                    pattern.finditer(text)
                )

                if not matches:
                    continue

                evidence_records.append({

                    "pdf_relative_path":
                        str(
                            pdf_path.relative_to(
                                source_root
                            )
                        ),

                    "pdf_file_name":
                        pdf_path.name,

                    "page_number":
                        page_index,

                    "target_construct":
                        route["target_construct"],

                    "routing_term":
                        term,

                    "candidate_relative_path":
                        route["relative_path"],

                    "candidate_variable":
                        route["variable"],

                    "candidate_variable_label":
                        route["variable_label"],

                    "occurrence_count":
                        len(matches),

                    "documentary_occurrence":
                        "OBSERVED",

                    "semantic_verification":
                        "NOT_PERFORMED",

                    "authorization":
                        "NOT_AUTHORIZED",

                    "discovery_only":
                        True,

                    "semantic_inference_allowed":
                        False,
                })

# ------------------------------------------------------------
# 11. RESULT DATAFRAME
# ------------------------------------------------------------

evidence_df = pd.DataFrame(
    evidence_records
)

if len(evidence_df) == 0:

    evidence_df = pd.DataFrame(
        columns=[
            "pdf_relative_path",
            "pdf_file_name",
            "page_number",
            "target_construct",
            "routing_term",
            "candidate_relative_path",
            "candidate_variable",
            "candidate_variable_label",
            "occurrence_count",
            "documentary_occurrence",
            "semantic_verification",
            "authorization",
            "discovery_only",
            "semantic_inference_allowed",
        ]
    )

# ------------------------------------------------------------
# 12. BASIC ACCOUNTING
# ------------------------------------------------------------

routing_record_count = len(routing_df)

evidence_record_count = len(
    evidence_df
)

observed_pdf_count = int(
    evidence_df[
        "pdf_relative_path"
    ].nunique()
) if evidence_record_count else 0

observed_page_count = int(
    evidence_df[
        ["pdf_relative_path", "page_number"]
    ].drop_duplicates().shape[0]
) if evidence_record_count else 0

semantic_verification_count = int(
    (
        evidence_df[
            "semantic_verification"
        ]
        == "VERIFIED"
    ).sum()
) if evidence_record_count else 0

authorization_count = int(
    (
        evidence_df[
            "authorization"
        ]
        == "AUTHORIZED"
    ).sum()
) if evidence_record_count else 0

discovery_only_violations = int(
    (
        evidence_df[
            "discovery_only"
        ] != True
    ).sum()
) if evidence_record_count else 0

semantic_inference_leaks = int(
    (
        evidence_df[
            "semantic_inference_allowed"
        ] != False
    ).sum()
) if evidence_record_count else 0

# ------------------------------------------------------------
# 13. FAIL-CLOSED CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if len(pdf_inventory) != 7:
    fail_conditions.append(
        "PDF_COUNT_MISMATCH"
    )

if len(pdf_errors) > 0:
    fail_conditions.append(
        "PDF_OR_PAGE_READ_ERROR"
    )

if discovery_only_violations > 0:
    fail_conditions.append(
        "DISCOVERY_ONLY_VIOLATION"
    )

if semantic_inference_leaks > 0:
    fail_conditions.append(
        "SEMANTIC_INFERENCE_LEAK"
    )

if semantic_verification_count > 0:
    fail_conditions.append(
        "UNAUTHORIZED_SEMANTIC_VERIFICATION"
    )

if authorization_count > 0:
    fail_conditions.append(
        "UNAUTHORIZED_AUTHORIZATION"
    )

# ------------------------------------------------------------
# 14. AUTHORIZATION BOUNDARY
# ------------------------------------------------------------

authorization = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY",
}

# ------------------------------------------------------------
# 15. FINAL STATUS
# ------------------------------------------------------------

status = (
    "PASS"
    if len(fail_conditions) == 0
    else "BLOCKED"
)

fail_closed_triggered = (
    len(fail_conditions) > 0
)

timestamp_utc = datetime.now(
    timezone.utc
).isoformat()

# ------------------------------------------------------------
# 16. SUMMARY
# ------------------------------------------------------------

summary = {

    "upstream_status":
        upstream_status,

    "source_root":
        str(source_root),

    "c06_candidate_records":
        int(len(candidate_df)),

    "c06_unique_variables":
        int(candidate_df["variable"].nunique()),

    "target_routing_records":
        int(routing_record_count),

    "target_routing_unique_variables":
        int(
            routing_df["variable"].nunique()
        ) if len(routing_df) else 0,

    "documentary_evidence_records":
        int(evidence_record_count),

    "documentary_pdfs_with_hits":
        observed_pdf_count,

    "documentary_pages_with_hits":
        observed_page_count,

    "semantic_verification_records":
        semantic_verification_count,

    "authorization_records":
        authorization_count,

    "discovery_only_violations":
        discovery_only_violations,

    "semantic_inference_leaks":
        semantic_inference_leaks,

    "pdf_or_page_errors":
        len(pdf_errors),

    "fail_conditions":
        fail_conditions,

    "documentary_occurrence_is_not_semantic_verification":
        True,

    "documentary_occurrence_is_not_authorization":
        True,

    "semantic_interpretation":
        "NOT_PERFORMED",

    "semantic_inference_allowed":
        False,

    "discovery_only":
        True,
}

# ------------------------------------------------------------
# 17. SAVE ARTIFACTS
# ------------------------------------------------------------

evidence_df.to_csv(
    OUTPUT_LEDGER,
    index=False
)

with OUTPUT_SUMMARY.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C11B",

    "cell_name":
        "TARGETED DOCUMENTARY EVIDENCE DISCOVERY",

    "timestamp_utc":
        timestamp_utc,

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        authorization,

    "payload":
        summary,
}

with OUTPUT_STATUS.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_status,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 18. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C11B RESULTS")
print("=" * 72)

print(
    "C06 candidate records:",
    len(candidate_df)
)

print(
    "Target routing records:",
    routing_record_count
)

print(
    "Target routing unique variables:",
    (
        routing_df["variable"].nunique()
        if len(routing_df)
        else 0
    )
)

print(
    "Documentary evidence records:",
    evidence_record_count
)

print(
    "PDFs with documentary hits:",
    observed_pdf_count
)

print(
    "Pages with documentary hits:",
    observed_page_count
)

print(
    "Semantic verification records:",
    semantic_verification_count
)

print(
    "Authorization records:",
    authorization_count
)

print(
    "PDF/page errors:",
    len(pdf_errors)
)

print(
    "Discovery-only violations:",
    discovery_only_violations
)

print(
    "Semantic inference leaks:",
    semantic_inference_leaks
)

print(
    "Fail conditions:",
    fail_conditions
)

print(
    "Fail-closed triggered:",
    fail_closed_triggered
)

print(
    "\nC11B STATUS:",
    status
)

print("\nArtifacts:")
print(OUTPUT_LEDGER)
print(OUTPUT_SUMMARY)
print(OUTPUT_STATUS)

print("=" * 72)

OIP v1.0.33 — C11B
TARGETED DOCUMENTARY EVIDENCE DISCOVERY

UPSTREAM STATUS
C00: PASS
C06: PASS
C08: PASS
C09R: PASS
C10: PASS
C11A: PASS

C06 candidate records: 1044
C06 unique variables: 378

Targeted candidate routing records: 0
Targeted unique variables: 0

C11B RESULTS
C06 candidate records: 1044
Target routing records: 0
Target routing unique variables: 0
Documentary evidence records: 0
PDFs with documentary hits: 0
Pages with documentary hits: 0
Semantic verification records: 0
Authorization records: 0
PDF/page errors: 0
Discovery-only violations: 0
Semantic inference leaks: 0
Fail conditions: []
Fail-closed triggered: False

C11B STATUS: PASS

Artifacts:
/kaggle/working/oip_v1_0_33/c11b_targeted_documentary_discovery.csv
/kaggle/working/oip_v1_0_33/c11b_documentary_discovery_summary.json
/kaggle/working/oip_v1_0_33/c11b_final_status.json


In [26]:
# ============================================================
# OIP v1.0.33 — C11D v2
# ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY
# SAFE PAGE-WISE EXTRACTION — FAIL-CLOSED
# ============================================================

import json
import re
import hashlib
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

print("=" * 72)
print("OIP v1.0.33 — C11D v2")
print("ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY")
print("SAFE PAGE-WISE EXTRACTION — FAIL-CLOSED")
print("=" * 72)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C00_FILE = WORKING_DIR / "c00_source_scope.json"
C08_FILE = WORKING_DIR / "c08_final_status.json"
C11A_FILE = WORKING_DIR / "c11a_final_status.json"
C11A_INV = WORKING_DIR / "c11a_documentary_source_inventory.csv"

OUT_CSV = WORKING_DIR / "c11d_domain_term_discovery.csv"
OUT_SUMMARY = WORKING_DIR / "c11d_domain_term_discovery_summary.json"
OUT_STATUS = WORKING_DIR / "c11d_final_status.json"

# ------------------------------------------------------------
# 2. STANDARD AUTHORIZATION STATE
# ------------------------------------------------------------

AUTHORIZATION = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY"
}

# ------------------------------------------------------------
# 3. SAFE JSON LOADER
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# ------------------------------------------------------------
# 4. UPSTREAM CHECK
# ------------------------------------------------------------

c00 = load_json(C00_FILE)
c08 = load_json(C08_FILE)
c11a = load_json(C11A_FILE)

print("\nUPSTREAM STATUS")
print("C00:", c00.get("status"))
print("C08:", c08.get("status"))
print("C11A:", c11a.get("status"))

upstream_ok = (
    c00.get("status") == "PASS"
    and c08.get("status") == "PASS"
    and c11a.get("status") == "PASS"
)

# ------------------------------------------------------------
# 5. SOURCE ROOT
# ------------------------------------------------------------

source_root = Path(
    c00.get("payload", {}).get("source_root", "")
)

print("\nSOURCE ROOT:")
print(source_root)

# ------------------------------------------------------------
# 6. C11A INVENTORY CHECK
# ------------------------------------------------------------

inventory_required_columns = [
    "relative_path",
    "file_name",
    "file_size_bytes",
    "sha256",
    "page_count"
]

inventory = pd.read_csv(C11A_INV)

inventory_columns_ok = all(
    c in inventory.columns
    for c in inventory_required_columns
)

pdf_inventory = inventory.copy()

if "file_name" in pdf_inventory.columns:
    pdf_inventory = pdf_inventory[
        pdf_inventory["file_name"].astype(str).str.lower().str.endswith(".pdf")
    ].copy()

pdf_count = len(pdf_inventory)

print("\nC11A PDF INVENTORY")
print("PDF files:", pdf_count)
print("Expected :", 7)

inventory_ok = (
    inventory_columns_ok
    and pdf_count == 7
)

# ------------------------------------------------------------
# 7. ESS4 DOMAIN ROUTING TERMS
# ------------------------------------------------------------
#
# IMPORTANT:
# These are ROUTING TERMS ONLY.
# They do NOT establish semantic equivalence with OIP constructs.
# No construct authorization is performed here.
# ------------------------------------------------------------

SEARCH_FAMILIES = {
    "GFL_ROUTING": [
        "shock",
        "shocks",
        "coping",
        "cope",
        "response",
        "respond",
        "recover",
        "recovery",
        "regain",
        "welfare"
    ],

    "IDS_ROUTING": [
        "electricity",
        "electric",
        "infrastructure",
        "service",
        "services",
        "access",
        "availability",
        "power",
        "outage"
    ],

    "AML_ROUTING": [
        "priority",
        "priorities",
        "rank",
        "ranking",
        "allocation",
        "allocate",
        "preference",
        "preferences",
        "trade-off",
        "tradeoff"
    ],

    "OUTCOME_ROUTING": [
        "consumption",
        "expenditure",
        "income",
        "health",
        "welfare"
    ]
}

TERM_ROWS = []

for family, terms in SEARCH_FAMILIES.items():
    for term in terms:
        TERM_ROWS.append({
            "search_family": family,
            "search_term": term
        })

print("\nSearch terms:", len(TERM_ROWS))

# ------------------------------------------------------------
# 8. EXTRACTION ENGINE DISCOVERY
# ------------------------------------------------------------

def command_exists(cmd):
    return shutil.which(cmd) is not None

HAS_PDFTOTEXT = command_exists("pdftotext")
HAS_FITZ = False

try:
    import fitz
    HAS_FITZ = True
except Exception:
    HAS_FITZ = False

print("\nEXTRACTION ENGINE CHECK")
print("pdftotext available:", HAS_PDFTOTEXT)
print("PyMuPDF available   :", HAS_FITZ)

# ------------------------------------------------------------
# 9. ENGINE SELECTION
# ------------------------------------------------------------
#
# Priority:
# 1. PyMuPDF
# 2. pdftotext
#
# pypdf is deliberately NOT used here.
# ------------------------------------------------------------

if HAS_FITZ:
    EXTRACTION_ENGINE = "PYMUPDF"
elif HAS_PDFTOTEXT:
    EXTRACTION_ENGINE = "PDFTOTEXT"
else:
    EXTRACTION_ENGINE = "NO_SAFE_ENGINE"

print("Selected engine:", EXTRACTION_ENGINE)

# ------------------------------------------------------------
# 10. REGEX
# ------------------------------------------------------------

compiled_terms = []

for row in TERM_ROWS:
    pattern = re.compile(
        r"\b" + re.escape(row["search_term"]) + r"\b",
        flags=re.IGNORECASE
    )

    compiled_terms.append({
        "search_family": row["search_family"],
        "search_term": row["search_term"],
        "pattern": pattern
    })

# ------------------------------------------------------------
# 11. SAFE TEXT EXTRACTION FUNCTIONS
# ------------------------------------------------------------

def extract_with_fitz(pdf_path, page_number):
    """
    Page number is zero-based internally.
    Returns text or raises an exception.
    """
    import fitz

    doc = fitz.open(str(pdf_path))

    try:
        if page_number >= len(doc):
            raise IndexError(
                f"Requested page {page_number + 1}, "
                f"available pages {len(doc)}"
            )

        page = doc.load_page(page_number)
        text = page.get_text("text")

        if text is None:
            text = ""

        return text

    finally:
        doc.close()


def extract_with_pdftotext(pdf_path, page_number):
    """
    Uses pdftotext through subprocess.
    Page selection is done through -f / -l.
    """
    cmd = [
        "pdftotext",
        "-f",
        str(page_number + 1),
        "-l",
        str(page_number + 1),
        "-layout",
        str(pdf_path),
        "-"
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=60
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip()
            or f"pdftotext returned code {result.returncode}"
        )

    return result.stdout or ""

# ------------------------------------------------------------
# 12. GENERIC EXTRACTION WRAPPER
# ------------------------------------------------------------

def extract_page(pdf_path, page_number):

    if EXTRACTION_ENGINE == "PYMUPDF":
        return extract_with_fitz(pdf_path, page_number)

    if EXTRACTION_ENGINE == "PDFTOTEXT":
        return extract_with_pdftotext(pdf_path, page_number)

    raise RuntimeError(
        "No safe PDF extraction engine is available."
    )

# ------------------------------------------------------------
# 13. CONTEXT SNIPPET
# ------------------------------------------------------------

def make_context(text, start, end, window=180):

    left = max(0, start - window)
    right = min(len(text), end + window)

    snippet = text[left:right]

    snippet = re.sub(
        r"\s+",
        " ",
        snippet
    ).strip()

    return snippet

# ------------------------------------------------------------
# 14. HASH CHECK AGAINST C11A INVENTORY
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

# ------------------------------------------------------------
# 15. INITIAL STATE
# ------------------------------------------------------------

records = []
pdf_errors = []
page_errors = []

processed_pdf_count = 0
processed_page_count = 0
extracted_page_count = 0
hit_count = 0

hash_mismatch_count = 0
missing_pdf_count = 0
page_count_mismatch_count = 0

# ------------------------------------------------------------
# 16. HARD BLOCK IF UPSTREAM FAILS
# ------------------------------------------------------------

if not upstream_ok:
    print("\nBLOCKED")
    print("Required upstream cell is not PASS.")

elif not inventory_ok:
    print("\nBLOCKED")
    print("C11A documentary inventory is not valid.")

elif EXTRACTION_ENGINE == "NO_SAFE_ENGINE":
    print("\nBLOCKED")
    print("No safe PDF extraction engine is available.")

else:

    # --------------------------------------------------------
    # 17. PROCESS EACH PDF
    # --------------------------------------------------------

    for idx, row in pdf_inventory.reset_index(drop=True).iterrows():

        file_name = str(row["file_name"])
        relative_path = str(row["relative_path"])

        pdf_path = source_root / relative_path

        print(
            f"\n[{idx + 1}/{pdf_count}] {file_name}"
        )

        # ----------------------------------------------------
        # FILE EXISTENCE
        # ----------------------------------------------------

        if not pdf_path.exists():

            missing_pdf_count += 1

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "SOURCE_FILE_MISSING",
                "error": "Source PDF does not exist."
            })

            print("  SOURCE FILE MISSING")
            continue

        # ----------------------------------------------------
        # HASH CHECK
        # ----------------------------------------------------

        try:

            observed_hash = sha256_file(pdf_path)
            expected_hash = str(row["sha256"])

            if observed_hash != expected_hash:

                hash_mismatch_count += 1

                pdf_errors.append({
                    "relative_path": relative_path,
                    "file_name": file_name,
                    "error_type": "SHA256_MISMATCH",
                    "expected_sha256": expected_hash,
                    "observed_sha256": observed_hash
                })

                print("  SHA256 MISMATCH")
                continue

        except Exception as e:

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "HASH_ERROR",
                "error": str(e)
            })

            print("  HASH ERROR")
            continue

        # ----------------------------------------------------
        # PAGE COUNT
        # ----------------------------------------------------

        expected_pages = int(row["page_count"])

        try:

            if EXTRACTION_ENGINE == "PYMUPDF":

                import fitz

                doc = fitz.open(str(pdf_path))

                try:
                    observed_pages = len(doc)
                finally:
                    doc.close()

            else:

                result = subprocess.run(
                    [
                        "pdfinfo",
                        str(pdf_path)
                    ],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    text=True,
                    timeout=60
                )

                if result.returncode != 0:
                    raise RuntimeError(
                        result.stderr.strip()
                    )

                page_match = re.search(
                    r"^Pages:\s+(\d+)",
                    result.stdout,
                    flags=re.MULTILINE
                )

                if not page_match:
                    raise RuntimeError(
                        "PDF page count not identified."
                    )

                observed_pages = int(
                    page_match.group(1)
                )

        except Exception as e:

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "PAGE_COUNT_ERROR",
                "error": str(e)
            })

            print("  PAGE COUNT ERROR")
            continue

        if observed_pages != expected_pages:

            page_count_mismatch_count += 1

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "PAGE_COUNT_MISMATCH",
                "expected_pages": expected_pages,
                "observed_pages": observed_pages
            })

            print(
                f"  PAGE COUNT MISMATCH: "
                f"expected={expected_pages}, "
                f"observed={observed_pages}"
            )

            continue

        processed_pdf_count += 1

        print(
            f"  Pages: {observed_pages}"
        )

        # ----------------------------------------------------
        # 18. PAGE-WISE DISCOVERY
        # ----------------------------------------------------

        for page_number in range(observed_pages):

            try:

                text = extract_page(
                    pdf_path,
                    page_number
                )

                extracted_page_count += 1

                if not text:
                    processed_page_count += 1
                    continue

                for term_item in compiled_terms:

                    family = term_item["search_family"]
                    term = term_item["search_term"]
                    pattern = term_item["pattern"]

                    for match in pattern.finditer(text):

                        snippet = make_context(
                            text,
                            match.start(),
                            match.end()
                        )

                        records.append({

                            "relative_path":
                                relative_path,

                            "file_name":
                                file_name,

                            "page_number":
                                page_number + 1,

                            "search_family":
                                family,

                            "search_term":
                                term,

                            "occurrence_text":
                                match.group(0),

                            "context_snippet":
                                snippet,

                            "documentary_occurrence":
                                "OBSERVED",

                            "semantic_verification":
                                "NOT_PERFORMED",

                            "construct_mapping":
                                "NOT_ESTABLISHED",

                            "authorization":
                                "NOT_AUTHORIZED",

                            "discovery_only":
                                True,

                            "semantic_inference_allowed":
                                False
                        })

                        hit_count += 1

                processed_page_count += 1

            except KeyboardInterrupt:

                # Never convert interruption into PASS.
                raise

            except Exception as e:

                page_errors.append({
                    "relative_path":
                        relative_path,

                    "file_name":
                        file_name,

                    "page_number":
                        page_number + 1,

                    "error_type":
                        "PAGE_EXTRACTION_ERROR",

                    "error":
                        str(e)
                })

        print(
            f"  Extracted pages: "
            f"{observed_pages}"
        )

# ------------------------------------------------------------
# 19. DISCOVERY DATAFRAME
# ------------------------------------------------------------

columns = [
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "documentary_occurrence",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "discovery_only",
    "semantic_inference_allowed"
]

discovery_df = pd.DataFrame(
    records,
    columns=columns
)

discovery_df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 20. SAFETY CHECKS
# ------------------------------------------------------------

semantic_verification_leaks = int(
    (
        discovery_df["semantic_verification"]
        != "NOT_PERFORMED"
    ).sum()
) if len(discovery_df) else 0

construct_mapping_leaks = int(
    (
        discovery_df["construct_mapping"]
        != "NOT_ESTABLISHED"
    ).sum()
) if len(discovery_df) else 0

authorization_leaks = int(
    (
        discovery_df["authorization"]
        != "NOT_AUTHORIZED"
    ).sum()
) if len(discovery_df) else 0

discovery_only_violations = int(
    (
        discovery_df["discovery_only"]
        != True
    ).sum()
) if len(discovery_df) else 0

semantic_inference_leaks = int(
    (
        discovery_df["semantic_inference_allowed"]
        != False
    ).sum()
) if len(discovery_df) else 0

# ------------------------------------------------------------
# 21. FAIL CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if not upstream_ok:
    fail_conditions.append(
        "UPSTREAM_NOT_PASS"
    )

if not inventory_ok:
    fail_conditions.append(
        "C11A_INVENTORY_INVALID"
    )

if EXTRACTION_ENGINE == "NO_SAFE_ENGINE":
    fail_conditions.append(
        "NO_SAFE_EXTRACTION_ENGINE"
    )

if missing_pdf_count > 0:
    fail_conditions.append(
        "SOURCE_PDF_MISSING"
    )

if hash_mismatch_count > 0:
    fail_conditions.append(
        "PDF_HASH_MISMATCH"
    )

if page_count_mismatch_count > 0:
    fail_conditions.append(
        "PDF_PAGE_COUNT_MISMATCH"
    )

if len(pdf_errors) > 0:
    fail_conditions.append(
        "PDF_ERRORS_PRESENT"
    )

if len(page_errors) > 0:
    fail_conditions.append(
        "PAGE_ERRORS_PRESENT"
    )

if processed_pdf_count != pdf_count:
    fail_conditions.append(
        "NOT_ALL_PDFS_PROCESSED"
    )

if processed_page_count != extracted_page_count:
    fail_conditions.append(
        "PAGE_ACCOUNTING_MISMATCH"
    )

if semantic_verification_leaks > 0:
    fail_conditions.append(
        "SEMANTIC_VERIFICATION_LEAK"
    )

if construct_mapping_leaks > 0:
    fail_conditions.append(
        "CONSTRUCT_MAPPING_LEAK"
    )

if authorization_leaks > 0:
    fail_conditions.append(
        "AUTHORIZATION_LEAK"
    )

if discovery_only_violations > 0:
    fail_conditions.append(
        "DISCOVERY_ONLY_VIOLATION"
    )

if semantic_inference_leaks > 0:
    fail_conditions.append(
        "SEMANTIC_INFERENCE_LEAK"
    )

# ------------------------------------------------------------
# 22. FINAL STATUS
# ------------------------------------------------------------

if not upstream_ok:
    status = "BLOCKED"
    fail_closed_triggered = True

elif (
    not inventory_ok
    or EXTRACTION_ENGINE == "NO_SAFE_ENGINE"
    or len(fail_conditions) > 0
):
    status = "BLOCKED"
    fail_closed_triggered = True

else:
    status = "PASS"
    fail_closed_triggered = False

# ------------------------------------------------------------
# 23. SUMMARY
# ------------------------------------------------------------

summary = {

    "oip_version": "v1.0.33",

    "cell": "C11D",

    "cell_name":
        "ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "extraction_engine":
        EXTRACTION_ENGINE,

    "upstream": {

        "c00_status":
            c00.get("status"),

        "c08_status":
            c08.get("status"),

        "c11a_status":
            c11a.get("status")
    },

    "source": {

        "source_root":
            str(source_root),

        "pdf_count":
            pdf_count,

        "expected_pdf_count":
            7,

        "processed_pdf_count":
            processed_pdf_count,

        "missing_pdf_count":
            missing_pdf_count,

        "hash_mismatch_count":
            hash_mismatch_count,

        "page_count_mismatch_count":
            page_count_mismatch_count
    },

    "processing": {

        "expected_search_terms":
            len(TERM_ROWS),

        "processed_pages":
            processed_page_count,

        "extracted_pages":
            extracted_page_count,

        "page_errors":
            len(page_errors),

        "pdf_errors":
            len(pdf_errors),

        "documentary_evidence_records":
            len(discovery_df),

        "hit_count":
            hit_count
    },

    "boundary": {

        "search_terms_are_routing_only":
            True,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_verification":
            "NOT_PERFORMED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "discovery_only":
            True,

        "semantic_inference_allowed":
            False
    },

    "fail_closed": {

        "active":
            True,

        "triggered":
            fail_closed_triggered,

        "fail_conditions":
            fail_conditions
    },

    "errors": {

        "pdf_errors":
            pdf_errors,

        "page_errors":
            page_errors
    }
}

with open(
    OUT_SUMMARY,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 24. FINAL WRAPPER
# ------------------------------------------------------------

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C11D",

    "cell_name":
        "ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        AUTHORIZATION,

    "payload": {

        "upstream_c00_status":
            c00.get("status"),

        "upstream_c08_status":
            c08.get("status"),

        "upstream_c11a_status":
            c11a.get("status"),

        "source_root":
            str(source_root),

        "extraction_engine":
            EXTRACTION_ENGINE,

        "pdf_count":
            pdf_count,

        "expected_pdf_count":
            7,

        "processed_pdf_count":
            processed_pdf_count,

        "processed_page_count":
            processed_page_count,

        "extracted_page_count":
            extracted_page_count,

        "search_term_count":
            len(TERM_ROWS),

        "documentary_evidence_records":
            len(discovery_df),

        "hit_count":
            hit_count,

        "pdf_error_count":
            len(pdf_errors),

        "page_error_count":
            len(page_errors),

        "hash_mismatch_count":
            hash_mismatch_count,

        "page_count_mismatch_count":
            page_count_mismatch_count,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_verification":
            "NOT_PERFORMED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "discovery_only":
            True,

        "semantic_inference_allowed":
            False,

        "semantic_verification_leaks":
            semantic_verification_leaks,

        "construct_mapping_leaks":
            construct_mapping_leaks,

        "authorization_leaks":
            authorization_leaks,

        "discovery_only_violations":
            discovery_only_violations,

        "semantic_inference_leaks":
            semantic_inference_leaks,

        "fail_conditions":
            fail_conditions
    }
}

with open(
    OUT_STATUS,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_status,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 25. CONSOLE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C11D v2 RESULTS")
print("=" * 72)

print("Extraction engine:", EXTRACTION_ENGINE)
print("PDF files:", pdf_count)
print("Processed PDFs:", processed_pdf_count)
print("Processed pages:", processed_page_count)
print("Extracted pages:", extracted_page_count)
print("PDF errors:", len(pdf_errors))
print("Page errors:", len(page_errors))
print("Hash mismatches:", hash_mismatch_count)
print("Page-count mismatches:", page_count_mismatch_count)
print("Search terms:", len(TERM_ROWS))
print("Documentary evidence records:", len(discovery_df))
print("Hits:", hit_count)

print("\nBOUNDARY")
print("Semantic interpretation: NOT_PERFORMED")
print("Semantic verification: NOT_PERFORMED")
print("Construct mapping: NOT_ESTABLISHED")
print("Authorization: NOT_AUTHORIZED")
print("Discovery only:", True)
print("Semantic inference allowed:", False)

print("\nFAIL-CLOSED")
print("Fail conditions:", fail_conditions)
print("Triggered:", fail_closed_triggered)

print("\nC11D v2 STATUS:", status)

print("\nArtifacts:")
print(OUT_CSV)
print(OUT_SUMMARY)
print(OUT_STATUS)

print("=" * 72)

OIP v1.0.33 — C11D v2
ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY
SAFE PAGE-WISE EXTRACTION — FAIL-CLOSED

UPSTREAM STATUS
C00: PASS
C08: PASS
C11A: PASS

SOURCE ROOT:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-33-ethiopia-ess4-evidence-audit

C11A PDF INVENTORY
PDF files: 7
Expected : 7

Search terms: 34

EXTRACTION ENGINE CHECK
pdftotext available: False
PyMuPDF available   : False
Selected engine: NO_SAFE_ENGINE

BLOCKED
No safe PDF extraction engine is available.

C11D v2 RESULTS
Extraction engine: NO_SAFE_ENGINE
PDF files: 7
Processed PDFs: 0
Processed pages: 0
Extracted pages: 0
PDF errors: 0
Page errors: 0
Hash mismatches: 0
Page-count mismatches: 0
Search terms: 34
Documentary evidence records: 0
Hits: 0

BOUNDARY
Semantic interpretation: NOT_PERFORMED
Semantic verification: NOT_PERFORMED
Construct mapping: NOT_ESTABLISHED
Authorization: NOT_AUTHORIZED
Discovery only: True
Semantic inference allowed: False

FAIL-CLOSED
Fail conditions: ['NO_SAFE_EXTRACTION_ENGINE', 'NOT_ALL_PDFS

In [27]:
# ============================================================
# OIP v1.0.33 — C11D TEST
# MINIMAL PDF EXTRACTION ENGINE TEST
# ============================================================

from pathlib import Path
import subprocess
import sys

print("=" * 72)
print("OIP v1.0.33 — C11D PDF EXTRACTION TEST")
print("=" * 72)

# ------------------------------------------------------------
# 1. SOURCE
# ------------------------------------------------------------

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/sudharsandas27/"
    "oip-v1-0-33-ethiopia-ess4-evidence-audit"
)

PDF = (
    SOURCE_ROOT /
    "2018_19_ethiopian_socioeconomic_survey_household_questionnaire.pdf"
)

print("\nPDF:")
print(PDF)

print("\nEXISTS:", PDF.exists())

if not PDF.exists():
    print("\nTEST STATUS: BLOCKED")
    print("Reason: test PDF not found.")
else:

    # --------------------------------------------------------
    # 2. CHECK INSTALLED MODULES
    # --------------------------------------------------------

    print("\nCHECKING PYTHON PDF MODULES")

    modules = [
        "fitz",
        "PyPDF2",
        "pypdf",
        "pdfplumber"
    ]

    for module in modules:

        try:
            __import__(module)
            print(module, ": AVAILABLE")
        except Exception:
            print(module, ": NOT_AVAILABLE")

    # --------------------------------------------------------
    # 3. CHECK SYSTEM TOOLS
    # --------------------------------------------------------

    print("\nCHECKING SYSTEM TOOLS")

    for command in [
        "pdftotext",
        "pdfinfo"
    ]:

        result = subprocess.run(
            ["which", command],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode == 0:
            print(command, ": AVAILABLE")
            print(" path:", result.stdout.strip())
        else:
            print(command, ": NOT_AVAILABLE")

    # --------------------------------------------------------
    # 4. TEST pypdf ONLY IF AVAILABLE
    # --------------------------------------------------------

    print("\nMINIMAL EXTRACTION TEST")

    test_success = False
    test_engine = None
    test_text_length = None
    test_error = None

    # ---- PyMuPDF ----

    try:

        import fitz

        print("\nTrying PyMuPDF on PAGE 1 only...")

        doc = fitz.open(str(PDF))

        print("PDF opened successfully.")
        print("Total pages:", len(doc))

        page = doc.load_page(0)

        text = page.get_text("text")

        test_text_length = len(text or "")
        test_engine = "PYMUPDF"
        test_success = True

        print("PAGE 1 extraction successful.")
        print("Extracted characters:", test_text_length)

        preview = (text or "").replace("\n", " ")[:300]

        print("\nTEXT PREVIEW:")
        print(preview)

        doc.close()

    except Exception as e:

        test_error = str(e)

        print(
            "\nPyMuPDF test failed:",
            repr(e)
        )

    # --------------------------------------------------------
    # 5. TEST pdftotext ONLY IF AVAILABLE
    # --------------------------------------------------------

    if not test_success:

        try:

            result = subprocess.run(
                [
                    "pdftotext",
                    "-f",
                    "1",
                    "-l",
                    "1",
                    "-layout",
                    str(PDF),
                    "-"
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=30
            )

            if result.returncode == 0:

                text = result.stdout or ""

                test_text_length = len(text)
                test_engine = "PDFTOTEXT"
                test_success = True

                print("\npdftotext PAGE 1 extraction successful.")
                print(
                    "Extracted characters:",
                    test_text_length
                )

                print("\nTEXT PREVIEW:")
                print(
                    text.replace("\n", " ")[:300]
                )

            else:

                test_error = (
                    result.stderr.strip()
                    or "pdftotext failed"
                )

                print(
                    "\npdftotext test failed:",
                    test_error
                )

        except Exception as e:

            test_error = str(e)

            print(
                "\npdftotext unavailable or failed:",
                repr(e)
            )

    # --------------------------------------------------------
    # 6. FINAL TEST RESULT
    # --------------------------------------------------------

    print("\n" + "=" * 72)

    if test_success:

        print("TEST STATUS: PASS")
        print("Working extraction engine:", test_engine)
        print("Page tested: 1")
        print("Characters extracted:", test_text_length)

    else:

        print("TEST STATUS: BLOCKED")
        print("No working safe extraction engine.")
        print("Last error:", test_error)

    print("=" * 72)

OIP v1.0.33 — C11D PDF EXTRACTION TEST

PDF:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-33-ethiopia-ess4-evidence-audit/2018_19_ethiopian_socioeconomic_survey_household_questionnaire.pdf

EXISTS: True

CHECKING PYTHON PDF MODULES
fitz : NOT_AVAILABLE
PyPDF2 : NOT_AVAILABLE
pypdf : AVAILABLE
pdfplumber : NOT_AVAILABLE

CHECKING SYSTEM TOOLS
pdftotext : NOT_AVAILABLE
pdfinfo : NOT_AVAILABLE

MINIMAL EXTRACTION TEST

PyMuPDF test failed: ModuleNotFoundError("No module named 'fitz'")

pdftotext unavailable or failed: FileNotFoundError(2, 'No such file or directory')

TEST STATUS: BLOCKED
No working safe extraction engine.
Last error: [Errno 2] No such file or directory: 'pdftotext'


In [28]:
# ============================================================
# OIP v1.0.33 — C11E
# ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY
# PYPDF PAGE-WISE + EXTRACTION QUALITY + FAIL-CLOSED
# ============================================================

import json
import re
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from pypdf import PdfReader


print("=" * 72)
print("OIP v1.0.33 — C11E")
print("ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY")
print("PYPDF PAGE-WISE + EXTRACTION QUALITY + FAIL-CLOSED")
print("=" * 72)


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

WORKING_DIR = Path("/kaggle/working/oip_v1_0_33")

C00_FILE = WORKING_DIR / "c00_source_scope.json"
C08_FILE = WORKING_DIR / "c08_final_status.json"
C11A_FILE = WORKING_DIR / "c11a_final_status.json"
C11A_INV = WORKING_DIR / "c11a_documentary_source_inventory.csv"

OUT_CSV = WORKING_DIR / "c11e_domain_term_discovery.csv"
OUT_SUMMARY = WORKING_DIR / "c11e_domain_term_discovery_summary.json"
OUT_STATUS = WORKING_DIR / "c11e_final_status.json"


# ------------------------------------------------------------
# 2. STANDARD AUTHORIZATION
# ------------------------------------------------------------

AUTHORIZATION = {
    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY"
}


# ------------------------------------------------------------
# 3. JSON LOADER
# ------------------------------------------------------------

def load_json(path):

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ------------------------------------------------------------
# 4. UPSTREAM
# ------------------------------------------------------------

c00 = load_json(C00_FILE)
c08 = load_json(C08_FILE)
c11a = load_json(C11A_FILE)

print("\nUPSTREAM STATUS")
print("C00:", c00.get("status"))
print("C08:", c08.get("status"))
print("C11A:", c11a.get("status"))

upstream_ok = (
    c00.get("status") == "PASS"
    and
    c08.get("status") == "PASS"
    and
    c11a.get("status") == "PASS"
)


# ------------------------------------------------------------
# 5. SOURCE ROOT
# ------------------------------------------------------------

source_root = Path(
    c00.get("payload", {}).get("source_root", "")
)

print("\nSOURCE ROOT:")
print(source_root)


# ------------------------------------------------------------
# 6. C11A INVENTORY
# ------------------------------------------------------------

inventory = pd.read_csv(C11A_INV)

required_inventory_columns = [
    "relative_path",
    "file_name",
    "file_size_bytes",
    "sha256",
    "page_count"
]

inventory_columns_ok = all(
    col in inventory.columns
    for col in required_inventory_columns
)

pdf_inventory = inventory[
    inventory["file_name"]
    .astype(str)
    .str.lower()
    .str.endswith(".pdf")
].copy()

pdf_count = len(pdf_inventory)

print("\nC11A PDF INVENTORY")
print("PDF files:", pdf_count)
print("Expected :", 7)


# ------------------------------------------------------------
# 7. DOMAIN ROUTING TERMS
# ------------------------------------------------------------
#
# IMPORTANT:
#
# These terms are routing terms only.
# They do NOT establish GFL / IDS / AML / OUTCOME mapping.
#
# No semantic authorization occurs in C11E.
# ------------------------------------------------------------

SEARCH_FAMILIES = {

    "GFL_ROUTING": [
        "shock",
        "shocks",
        "coping",
        "cope",
        "response",
        "respond",
        "recover",
        "recovery",
        "regain",
        "welfare"
    ],

    "IDS_ROUTING": [
        "electricity",
        "electric",
        "infrastructure",
        "service",
        "services",
        "access",
        "availability",
        "power",
        "outage"
    ],

    "AML_ROUTING": [
        "priority",
        "priorities",
        "rank",
        "ranking",
        "allocation",
        "allocate",
        "preference",
        "preferences",
        "trade-off",
        "tradeoff"
    ],

    "OUTCOME_ROUTING": [
        "consumption",
        "expenditure",
        "income",
        "health",
        "welfare"
    ]
}


TERM_ROWS = []

for family, terms in SEARCH_FAMILIES.items():

    for term in terms:

        TERM_ROWS.append({
            "search_family": family,
            "search_term": term
        })


print("\nSearch terms:", len(TERM_ROWS))


# ------------------------------------------------------------
# 8. WORD-BOUNDARY REGEX
# ------------------------------------------------------------

COMPILED_TERMS = []

for item in TERM_ROWS:

    pattern = re.compile(
        r"\b" +
        re.escape(item["search_term"]) +
        r"\b",
        flags=re.IGNORECASE
    )

    COMPILED_TERMS.append({
        "search_family": item["search_family"],
        "search_term": item["search_term"],
        "pattern": pattern
    })


# ------------------------------------------------------------
# 9. CONTEXT SNIPPET
# ------------------------------------------------------------

def make_context(text, start, end, window=220):

    left = max(0, start - window)
    right = min(len(text), end + window)

    snippet = text[left:right]

    snippet = re.sub(
        r"\s+",
        " ",
        snippet
    ).strip()

    return snippet


# ------------------------------------------------------------
# 10. STATE
# ------------------------------------------------------------

records = []

pdf_errors = []
page_errors = []

processed_pdf_count = 0
processed_page_count = 0

pages_with_text = 0
pages_without_text = 0

hit_count = 0


# ------------------------------------------------------------
# 11. PROCESS PDFs
# ------------------------------------------------------------

if not upstream_ok:

    print("\nBLOCKED")
    print("Required upstream cell is not PASS.")

elif not inventory_columns_ok:

    print("\nBLOCKED")
    print("C11A inventory schema is invalid.")

elif pdf_count != 7:

    print("\nBLOCKED")
    print("Expected exactly 7 PDFs.")

else:

    for pdf_index, row in pdf_inventory.reset_index(
        drop=True
    ).iterrows():

        file_name = str(row["file_name"])
        relative_path = str(row["relative_path"])

        pdf_path = source_root / relative_path

        print(
            f"\n[{pdf_index + 1}/{pdf_count}] "
            f"{file_name}"
        )

        # ----------------------------------------------------
        # SOURCE CHECK
        # ----------------------------------------------------

        if not pdf_path.exists():

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "SOURCE_FILE_MISSING"
            })

            print("  SOURCE FILE MISSING")
            continue


        # ----------------------------------------------------
        # PDF OPEN
        # ----------------------------------------------------

        try:

            reader = PdfReader(
                str(pdf_path),
                strict=False
            )

        except Exception as e:

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "PDF_OPEN_ERROR",
                "error": repr(e)
            })

            print("  PDF OPEN ERROR")
            print(repr(e))

            continue


        # ----------------------------------------------------
        # PAGE COUNT
        # ----------------------------------------------------

        try:

            observed_pages = len(reader.pages)

        except Exception as e:

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "PAGE_COUNT_ERROR",
                "error": repr(e)
            })

            print("  PAGE COUNT ERROR")
            continue


        expected_pages = int(
            row["page_count"]
        )

        print(
            "  Expected pages:",
            expected_pages
        )

        print(
            "  Observed pages:",
            observed_pages
        )


        if observed_pages != expected_pages:

            pdf_errors.append({
                "relative_path": relative_path,
                "file_name": file_name,
                "error_type": "PAGE_COUNT_MISMATCH",
                "expected_pages": expected_pages,
                "observed_pages": observed_pages
            })

            print("  PAGE COUNT MISMATCH")

            continue


        # ----------------------------------------------------
        # PAGE-WISE EXTRACTION
        # ----------------------------------------------------

        pdf_pages_processed = 0

        for page_number, page in enumerate(
            reader.pages,
            start=1
        ):

            try:

                text = page.extract_text()

                if text is None:
                    text = ""

                text = str(text)

                processed_page_count += 1
                pdf_pages_processed += 1

                if text.strip():

                    pages_with_text += 1

                else:

                    pages_without_text += 1


                # ------------------------------------------------
                # SEARCH IMMEDIATELY
                # ------------------------------------------------

                if not text.strip():
                    continue


                for item in COMPILED_TERMS:

                    family = item["search_family"]
                    term = item["search_term"]
                    pattern = item["pattern"]


                    for match in pattern.finditer(text):

                        snippet = make_context(
                            text,
                            match.start(),
                            match.end()
                        )


                        records.append({

                            "relative_path":
                                relative_path,

                            "file_name":
                                file_name,

                            "page_number":
                                page_number,

                            "search_family":
                                family,

                            "search_term":
                                term,

                            "occurrence_text":
                                match.group(0),

                            "context_snippet":
                                snippet,

                            "documentary_occurrence":
                                "OBSERVED",

                            "semantic_verification":
                                "NOT_PERFORMED",

                            "construct_mapping":
                                "NOT_ESTABLISHED",

                            "authorization":
                                "NOT_AUTHORIZED",

                            "discovery_only":
                                True,

                            "semantic_inference_allowed":
                                False
                        })


                        hit_count += 1


            except KeyboardInterrupt:

                raise


            except Exception as e:

                page_errors.append({

                    "relative_path":
                        relative_path,

                    "file_name":
                        file_name,

                    "page_number":
                        page_number,

                    "error_type":
                        "PAGE_EXTRACTION_ERROR",

                    "error":
                        repr(e)
                })


        # ----------------------------------------------------
        # PDF COMPLETION
        # ----------------------------------------------------

        if pdf_pages_processed == observed_pages:

            processed_pdf_count += 1

        print(
            "  Pages processed:",
            pdf_pages_processed
        )

        print(
            "  Hits so far:",
            hit_count
        )


# ------------------------------------------------------------
# 12. DISCOVERY DATAFRAME
# ------------------------------------------------------------

OUTPUT_COLUMNS = [

    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "documentary_occurrence",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "discovery_only",
    "semantic_inference_allowed"
]


discovery_df = pd.DataFrame(
    records,
    columns=OUTPUT_COLUMNS
)


discovery_df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 13. EXTRACTION QUALITY
# ------------------------------------------------------------

total_pages_attempted = (
    processed_page_count
    +
    len(page_errors)
)


if total_pages_attempted > 0:

    extraction_success_rate = (
        processed_page_count /
        total_pages_attempted
    )

else:

    extraction_success_rate = 0.0


if pages_with_text > 0:

    text_page_rate = (
        pages_with_text /
        processed_page_count
    )

else:

    text_page_rate = 0.0


if processed_page_count == 0:

    extraction_quality = "NOT_ESTABLISHED"

elif text_page_rate >= 0.50:

    extraction_quality = "ADEQUATE_FOR_ROUTING"

else:

    extraction_quality = "PARTIAL"


# ------------------------------------------------------------
# 14. PER-PDF QUALITY
# ------------------------------------------------------------

per_pdf_quality = []

for row in pdf_inventory.itertuples(
    index=False
):

    rel = str(row.relative_path)

    pdf_page_errors = [
        x for x in page_errors
        if x["relative_path"] == rel
    ]

    pdf_records = discovery_df[
        discovery_df["relative_path"] == rel
    ]

    # Count pages that produced text indirectly
    # from successful page processing.
    #
    # We use page errors only to establish incomplete
    # extraction. Exact text-page count is not reconstructed
    # from hit records because a valid page may contain no
    # search hit.

    per_pdf_quality.append({

        "relative_path":
            rel,

        "file_name":
            str(row.file_name),

        "expected_page_count":
            int(row.page_count),

        "page_extraction_errors":
            len(pdf_page_errors),

        "documentary_hit_records":
            len(pdf_records),

        "extraction_quality":
            (
                "PARTIAL"
                if len(pdf_page_errors) > 0
                else "NO_PAGE_ERRORS_OBSERVED"
            )
    })


# ------------------------------------------------------------
# 15. BOUNDARY AUDIT
# ------------------------------------------------------------

semantic_verification_leaks = int(

    (
        discovery_df["semantic_verification"]
        != "NOT_PERFORMED"
    ).sum()

) if len(discovery_df) else 0


construct_mapping_leaks = int(

    (
        discovery_df["construct_mapping"]
        != "NOT_ESTABLISHED"
    ).sum()

) if len(discovery_df) else 0


authorization_leaks = int(

    (
        discovery_df["authorization"]
        != "NOT_AUTHORIZED"
    ).sum()

) if len(discovery_df) else 0


discovery_only_violations = int(

    (
        discovery_df["discovery_only"]
        != True
    ).sum()

) if len(discovery_df) else 0


semantic_inference_leaks = int(

    (
        discovery_df["semantic_inference_allowed"]
        != False
    ).sum()

) if len(discovery_df) else 0


# ------------------------------------------------------------
# 16. FAIL CONDITIONS
# ------------------------------------------------------------

fail_conditions = []


if not upstream_ok:

    fail_conditions.append(
        "UPSTREAM_NOT_PASS"
    )


if not inventory_columns_ok:

    fail_conditions.append(
        "C11A_INVENTORY_SCHEMA_INVALID"
    )


if pdf_count != 7:

    fail_conditions.append(
        "PDF_COUNT_MISMATCH"
    )


if processed_pdf_count != pdf_count:

    fail_conditions.append(
        "NOT_ALL_PDFS_COMPLETED"
    )


if len(pdf_errors) > 0:

    fail_conditions.append(
        "PDF_ERRORS_PRESENT"
    )


if len(page_errors) > 0:

    fail_conditions.append(
        "PAGE_ERRORS_PRESENT"
    )


if semantic_verification_leaks > 0:

    fail_conditions.append(
        "SEMANTIC_VERIFICATION_LEAK"
    )


if construct_mapping_leaks > 0:

    fail_conditions.append(
        "CONSTRUCT_MAPPING_LEAK"
    )


if authorization_leaks > 0:

    fail_conditions.append(
        "AUTHORIZATION_LEAK"
    )


if discovery_only_violations > 0:

    fail_conditions.append(
        "DISCOVERY_ONLY_VIOLATION"
    )


if semantic_inference_leaks > 0:

    fail_conditions.append(
        "SEMANTIC_INFERENCE_LEAK"
    )


# ------------------------------------------------------------
# 17. FINAL STATUS
# ------------------------------------------------------------

if not upstream_ok:

    status = "BLOCKED"
    fail_closed_triggered = True

elif len(fail_conditions) > 0:

    status = "BLOCKED"
    fail_closed_triggered = True

else:

    status = "PASS"
    fail_closed_triggered = False


# ------------------------------------------------------------
# 18. SUMMARY
# ------------------------------------------------------------

summary = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C11E",

    "cell_name":
        "ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "upstream": {

        "c00_status":
            c00.get("status"),

        "c08_status":
            c08.get("status"),

        "c11a_status":
            c11a.get("status")
    },

    "source": {

        "source_root":
            str(source_root),

        "pdf_count":
            pdf_count,

        "expected_pdf_count":
            7,

        "processed_pdf_count":
            processed_pdf_count
    },

    "search": {

        "search_term_count":
            len(TERM_ROWS),

        "documentary_evidence_records":
            len(discovery_df),

        "hit_count":
            hit_count
    },

    "extraction_quality": {

        "processed_page_count":
            processed_page_count,

        "pages_with_text":
            pages_with_text,

        "pages_without_text":
            pages_without_text,

        "page_error_count":
            len(page_errors),

        "pdf_error_count":
            len(pdf_errors),

        "extraction_success_rate":
            extraction_success_rate,

        "text_page_rate":
            text_page_rate,

        "overall_quality":
            extraction_quality,

        "per_pdf":
            per_pdf_quality
    },

    "boundary": {

        "search_terms_are_routing_only":
            True,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_verification":
            "NOT_PERFORMED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "discovery_only":
            True,

        "semantic_inference_allowed":
            False
    },

    "fail_closed": {

        "active":
            True,

        "triggered":
            fail_closed_triggered,

        "fail_conditions":
            fail_conditions
    },

    "errors": {

        "pdf_errors":
            pdf_errors,

        "page_errors":
            page_errors
    }
}


with open(
    OUT_SUMMARY,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 19. FINAL STATUS WRAPPER
# ------------------------------------------------------------

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C11E",

    "cell_name":
        "ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        AUTHORIZATION,

    "payload": {

        "upstream_c00_status":
            c00.get("status"),

        "upstream_c08_status":
            c08.get("status"),

        "upstream_c11a_status":
            c11a.get("status"),

        "source_root":
            str(source_root),

        "pdf_count":
            pdf_count,

        "processed_pdf_count":
            processed_pdf_count,

        "processed_page_count":
            processed_page_count,

        "pages_with_text":
            pages_with_text,

        "pages_without_text":
            pages_without_text,

        "page_error_count":
            len(page_errors),

        "pdf_error_count":
            len(pdf_errors),

        "search_term_count":
            len(TERM_ROWS),

        "documentary_evidence_records":
            len(discovery_df),

        "hit_count":
            hit_count,

        "extraction_success_rate":
            extraction_success_rate,

        "text_page_rate":
            text_page_rate,

        "extraction_quality":
            extraction_quality,

        "semantic_interpretation":
            "NOT_PERFORMED",

        "semantic_verification":
            "NOT_PERFORMED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "discovery_only":
            True,

        "semantic_inference_allowed":
            False,

        "semantic_verification_leaks":
            semantic_verification_leaks,

        "construct_mapping_leaks":
            construct_mapping_leaks,

        "authorization_leaks":
            authorization_leaks,

        "discovery_only_violations":
            discovery_only_violations,

        "semantic_inference_leaks":
            semantic_inference_leaks,

        "fail_conditions":
            fail_conditions
    }
}


with open(
    OUT_STATUS,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_status,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 20. CONSOLE OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C11E RESULTS")
print("=" * 72)

print("PDF files:", pdf_count)
print("Processed PDFs:", processed_pdf_count)

print("Processed pages:", processed_page_count)
print("Pages with text:", pages_with_text)
print("Pages without text:", pages_without_text)

print("PDF errors:", len(pdf_errors))
print("Page errors:", len(page_errors))

print("Search terms:", len(TERM_ROWS))
print(
    "Documentary evidence records:",
    len(discovery_df)
)
print("Hits:", hit_count)

print("\nEXTRACTION QUALITY")
print(
    "Extraction success rate:",
    round(extraction_success_rate, 4)
)

print(
    "Text page rate:",
    round(text_page_rate, 4)
)

print(
    "Overall quality:",
    extraction_quality
)

print("\nBOUNDARY")
print(
    "Semantic interpretation:",
    "NOT_PERFORMED"
)

print(
    "Semantic verification:",
    "NOT_PERFORMED"
)

print(
    "Construct mapping:",
    "NOT_ESTABLISHED"
)

print(
    "Authorization:",
    "NOT_AUTHORIZED"
)

print(
    "Discovery only:",
    True
)

print(
    "Semantic inference allowed:",
    False
)

print("\nFAIL-CLOSED")
print(
    "Fail conditions:",
    fail_conditions
)

print(
    "Triggered:",
    fail_closed_triggered
)

print("\nC11E STATUS:", status)

print("\nArtifacts:")
print(OUT_CSV)
print(OUT_SUMMARY)
print(OUT_STATUS)

print("=" * 72)

OIP v1.0.33 — C11E
ESS4 DOMAIN-TERM DOCUMENTARY DISCOVERY
PYPDF PAGE-WISE + EXTRACTION QUALITY + FAIL-CLOSED

UPSTREAM STATUS
C00: PASS
C08: PASS
C11A: PASS

SOURCE ROOT:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-33-ethiopia-ess4-evidence-audit

C11A PDF INVENTORY
PDF files: 7
Expected : 7

Search terms: 34

[1/7] 2018_19_ethiopian_socioeconomic_survey_household_questionnaire.pdf
  Expected pages: 74
  Observed pages: 74
  Pages processed: 74
  Hits so far: 127

[2/7] 2018_19_ethiopian_socioeconomic_survey_post_harvest_agricultural_questionnaire.pdf
  Expected pages: 12
  Observed pages: 12
  Pages processed: 12
  Hits so far: 129

[3/7] 2018_19_ethiopian_socioeconomic_survey_post_planting_crop_cut_and_livestock_agricultural_questionnaire.pdf
  Expected pages: 34
  Observed pages: 34
  Pages processed: 34
  Hits so far: 151

[4/7] Basic Information Document Ethiopia Socioeconomic Survey 2018-19 V2.pdf
  Expected pages: 29
  Observed pages: 29
  Pages processed: 29
  Hits so far: 2

In [29]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/working/oip_v1_0_33/c11e_domain_term_discovery.csv"
)

print("C11E columns:", list(df.columns))
print()
print("First row sample:")
print(df.iloc[0].to_dict())

C11E columns: ['relative_path', 'file_name', 'page_number', 'search_family', 'search_term', 'occurrence_text', 'context_snippet', 'documentary_occurrence', 'semantic_verification', 'construct_mapping', 'authorization', 'discovery_only', 'semantic_inference_allowed']

First row sample:
{'relative_path': '2018_19_ethiopian_socioeconomic_survey_household_questionnaire.pdf', 'file_name': '2018_19_ethiopian_socioeconomic_survey_household_questionnaire.pdf', 'page_number': 5, 'search_family': 'IDS_ROUTING', 'search_term': 'services', 'occurrence_text': 'SERVICES', 'context_snippet': 'ING....................2MANUFACTURING.............3PROFESSIONAL/SCIENTIFIC/TECHNICAL ACTIVITIES......4ELECTRICITY...............5CONSTRUCTION..............6TRANSPORTATION............7BUYING AND SELLING........8FINANCIAL SERVICES........9PERSONAL SERVICES........10EDUCATION................11HEALTH...................12PUBLIC ADMINISTRATION....13OTHER (SPECIFY)..........14HOUSEWIFE.............15UNEMPLOYED.........

In [30]:
# ============================================================
# OIP v1.0.33 — C13
# C11E CATEGORY FILTERING & DOCUMENTARY EVIDENCE MAP
# EXACT C11E SCHEMA
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

BASE = Path("/kaggle/working/oip_v1_0_33")

print("=" * 72)
print("OIP v1.0.33 — C13")
print("C11E CATEGORY FILTERING & DOCUMENTARY EVIDENCE MAP")
print("=" * 72)

# ------------------------------------------------------------
# 1. LOAD C11E STATUS
# ------------------------------------------------------------

status_path = BASE / "c11e_final_status.json"
input_path = BASE / "c11e_domain_term_discovery.csv"

if not status_path.exists():
    raise FileNotFoundError(f"Missing: {status_path}")

if not input_path.exists():
    raise FileNotFoundError(f"Missing: {input_path}")

with open(status_path, "r", encoding="utf-8") as f:
    c11e_status = json.load(f)

print("\nUPSTREAM STATUS")
print("C11E:", c11e_status.get("status"))

if c11e_status.get("status") != "PASS":
    raise RuntimeError("C13 BLOCKED: C11E is not PASS.")

# ------------------------------------------------------------
# 2. LOAD C11E
# ------------------------------------------------------------

df = pd.read_csv(input_path)

print("\nC11E RECORDS:", len(df))

print("\nACTUAL SCHEMA")
print(list(df.columns))

# ------------------------------------------------------------
# 3. EXACT REQUIRED SCHEMA
# ------------------------------------------------------------

required_columns = [
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "documentary_occurrence",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "discovery_only",
    "semantic_inference_allowed"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        "C13 BLOCKED: Missing C11E columns: "
        + ", ".join(missing)
    )

# ------------------------------------------------------------
# 4. VERIFY C11E BOUNDARY
# ------------------------------------------------------------

bad_discovery = df[
    df["discovery_only"].astype(str).str.lower() != "true"
]

bad_semantic = df[
    df["semantic_verification"].astype(str) != "NOT_PERFORMED"
]

bad_mapping = df[
    df["construct_mapping"].astype(str) != "NOT_ESTABLISHED"
]

bad_authorization = df[
    df["authorization"].astype(str) != "NOT_AUTHORIZED"
]

bad_inference = df[
    df["semantic_inference_allowed"].astype(str).str.lower() != "false"
]

boundary_errors = {
    "discovery_only": len(bad_discovery),
    "semantic_verification": len(bad_semantic),
    "construct_mapping": len(bad_mapping),
    "authorization": len(bad_authorization),
    "semantic_inference_allowed": len(bad_inference)
}

print("\nC11E BOUNDARY CHECK")
for k, v in boundary_errors.items():
    print(f"{k}: violations={v}")

if any(v > 0 for v in boundary_errors.values()):
    raise RuntimeError(
        "C13 BLOCKED: C11E boundary integrity violation."
    )

# ------------------------------------------------------------
# 5. USE C11E SEARCH FAMILY DIRECTLY
# ------------------------------------------------------------
#
# IMPORTANT:
#
# C11E already recorded the routing family.
#
# Therefore C13 does NOT:
# - infer a category from snippet
# - infer a category from meaning
# - expand synonyms
# - perform fuzzy matching
# - reinterpret the question
#
# It only organizes the existing C11E discovery records.
# ------------------------------------------------------------

family_map = {
    "GFL_ROUTING": "GFL",
    "IDS_ROUTING": "IDS",
    "AML_ROUTING": "AML",
    "OUTCOME_ROUTING": "OUTCOME"
}

df["_search_family_norm"] = (
    df["search_family"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

# ------------------------------------------------------------
# 6. IDENTIFY UNEXPECTED SEARCH FAMILIES
# ------------------------------------------------------------

expected_families = set(family_map.keys())

observed_families = set(
    df["_search_family_norm"].unique()
)

unexpected_families = sorted(
    observed_families - expected_families
)

print("\nSEARCH FAMILIES")
print("Observed:", sorted(observed_families))
print("Unexpected:", unexpected_families)

# ------------------------------------------------------------
# 7. FAIL CLOSED ON UNEXPECTED FAMILY
# ------------------------------------------------------------

if unexpected_families:

    authorization = {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY"
    }

    final = {
        "oip_version": "v1.0.33",
        "cell": "C13",
        "cell_name":
            "C11E CATEGORY FILTERING & DOCUMENTARY EVIDENCE MAP",
        "timestamp_utc":
            datetime.now(timezone.utc).isoformat(),
        "status": "BLOCKED",
        "fail_closed_policy_active": True,
        "fail_closed_triggered": True,
        "authorization": authorization,
        "payload": {
            "upstream_C11E": "PASS",
            "observed_search_families":
                sorted(observed_families),
            "unexpected_search_families":
                unexpected_families,
            "boundary_errors":
                boundary_errors,
            "semantic_verification":
                "NOT_PERFORMED",
            "construct_mapping":
                "NOT_ESTABLISHED"
        }
    }

    with open(
        BASE / "c13_final_status.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            final,
            f,
            indent=2,
            ensure_ascii=False
        )

    raise RuntimeError(
        "C13 BLOCKED: Unexpected C11E search family."
    )

# ------------------------------------------------------------
# 8. CREATE CATEGORY FILES
# ------------------------------------------------------------

category_frames = {}

for family, category in family_map.items():

    sub = df[
        df["_search_family_norm"] == family
    ].copy()

    # Preserve original C11E fields.
    sub["routing_category"] = category

    # Explicitly preserve fail-closed boundary.
    sub["documentary_occurrence"] = "OBSERVED"
    sub["semantic_verification"] = "NOT_PERFORMED"
    sub["construct_mapping"] = "NOT_ESTABLISHED"
    sub["authorization"] = "NOT_AUTHORIZED"
    sub["discovery_only"] = True
    sub["semantic_inference_allowed"] = False

    sub = sub.drop(
        columns=["_search_family_norm"]
    )

    category_frames[category] = sub

# ------------------------------------------------------------
# 9. SAVE CATEGORY LEDGERS
# ------------------------------------------------------------

output_files = {
    "GFL":
        BASE / "c13_gfl_hits.csv",

    "IDS":
        BASE / "c13_ids_hits.csv",

    "AML":
        BASE / "c13_aml_hits.csv",

    "OUTCOME":
        BASE / "c13_outcome_hits.csv"
}

for category, frame in category_frames.items():

    frame.to_csv(
        output_files[category],
        index=False,
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 10. CATEGORY SUMMARY
# ------------------------------------------------------------

categories = {}

for category, frame in category_frames.items():

    pdf_pages = (
        frame[
            ["relative_path", "page_number"]
        ]
        .drop_duplicates()
    )

    pdf_distribution = (
        frame
        .groupby("file_name")
        .size()
        .sort_values(ascending=False)
        .to_dict()
    )

    term_distribution = (
        frame
        .groupby("search_term")
        .size()
        .sort_values(ascending=False)
        .to_dict()
    )

    pages_per_pdf = (
        frame
        .groupby("file_name")["page_number"]
        .nunique()
        .sort_values(ascending=False)
        .to_dict()
    )

    top10 = []

    for _, row in frame.head(10).iterrows():

        top10.append({
            "relative_path":
                row["relative_path"],

            "file_name":
                row["file_name"],

            "page_number":
                row["page_number"],

            "search_family":
                row["search_family"],

            "search_term":
                row["search_term"],

            "occurrence_text":
                row["occurrence_text"],

            "context_snippet":
                row["context_snippet"]
        })

    categories[category] = {

        "hit_count":
            int(len(frame)),

        "pdf_count":
            int(frame["file_name"].nunique()),

        "unique_pdf_page_count":
            int(len(pdf_pages)),

        "pdf_distribution":
            pdf_distribution,

        "term_distribution":
            term_distribution,

        "pages_per_pdf":
            pages_per_pdf,

        "top_10_context_records":
            top10
    }

# ------------------------------------------------------------
# 11. OVERLAP ACCOUNTING
# ------------------------------------------------------------
#
# C11E records are preserved.
# No duplicate merging is performed.
#
# A term such as "welfare" may occur in more than one
# search family if C11E originally recorded it that way.
#
# Therefore category totals are descriptive and are NOT
# interpreted as unique documentary records.
# ------------------------------------------------------------

category_total = sum(
    len(frame)
    for frame in category_frames.values()
)

# Unique original C11E records represented by source row index.
# This does not merge documentary evidence; it is only accounting.

unique_source_records = len(
    set(
        idx
        for frame in category_frames.values()
        for idx in frame.index
    )
)

# ------------------------------------------------------------
# 12. AUTHORIZATION
# ------------------------------------------------------------

authorization = {

    "construct": "NOT_AUTHORIZED",

    "predictor": "NOT_AUTHORIZED",

    "outcome": "NOT_AUTHORIZED",

    "key": "NOT_AUTHORIZED",

    "relationship": "NOT_AUTHORIZED",

    "cohort": "NOT_AUTHORIZED",

    "score": "NOT_AUTHORIZED",

    "empirical_evaluation": "NOT_READY"
}

# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

summary = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C13",

    "cell_name":
        "C11E CATEGORY FILTERING & DOCUMENTARY EVIDENCE MAP",

    "upstream": {
        "C11E": "PASS"
    },

    "input_records":
        int(len(df)),

    "observed_search_families":
        sorted(observed_families),

    "category_mapping":
        family_map,

    "categories":
        categories,

    "category_total_records":
        int(category_total),

    "unique_source_records":
        int(unique_source_records),

    "overlap_allowed":
        True,

    "duplicate_merging":
        False,

    "boundary": {

        "documentary_occurrence":
            "OBSERVED",

        "semantic_verification":
            "NOT_PERFORMED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "discovery_only":
            True,

        "semantic_inference_allowed":
            False
    },

    "authorization":
        authorization,

    "fail_conditions":
        [],

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        False
}

with open(
    BASE / "c13_category_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 14. FINAL STATUS
# ------------------------------------------------------------

final = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C13",

    "cell_name":
        "C11E CATEGORY FILTERING & DOCUMENTARY EVIDENCE MAP",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        "PASS",

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        False,

    "authorization":
        authorization,

    "payload":
        summary
}

with open(
    BASE / "c13_final_status.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 15. OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C13 RESULTS")
print("=" * 72)

print("Input C11E records:", len(df))

for category in [
    "GFL",
    "IDS",
    "AML",
    "OUTCOME"
]:

    frame = category_frames[category]

    unique_pages = (
        frame[
            ["relative_path", "page_number"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    print(
        f"{category}: "
        f"hits={len(frame)} | "
        f"PDFs={frame['file_name'].nunique()} | "
        f"pages={unique_pages}"
    )

print("\nACCOUNTING")
print("Category records:", category_total)
print("Unique source records:", unique_source_records)
print("Overlap allowed:", True)
print("Duplicate merging:", False)

print("\nBOUNDARY")
print("Semantic verification: NOT_PERFORMED")
print("Construct mapping: NOT_ESTABLISHED")
print("Authorization: NOT_AUTHORIZED")
print("Discovery only: True")
print("Semantic inference allowed: False")

print("\nFAIL-CLOSED")
print("Fail conditions: []")
print("Triggered: False")

print("\nC13 STATUS: PASS")

print("\nArtifacts:")
for path in output_files.values():
    print(path)

print(BASE / "c13_category_summary.json")
print(BASE / "c13_final_status.json")

print("=" * 72)

OIP v1.0.33 — C13
C11E CATEGORY FILTERING & DOCUMENTARY EVIDENCE MAP

UPSTREAM STATUS
C11E: PASS

C11E RECORDS: 1446

ACTUAL SCHEMA
['relative_path', 'file_name', 'page_number', 'search_family', 'search_term', 'occurrence_text', 'context_snippet', 'documentary_occurrence', 'semantic_verification', 'construct_mapping', 'authorization', 'discovery_only', 'semantic_inference_allowed']

C11E BOUNDARY CHECK
discovery_only: violations=0
semantic_verification: violations=0
construct_mapping: violations=0
authorization: violations=0
semantic_inference_allowed: violations=0

SEARCH FAMILIES
Observed: ['AML_ROUTING', 'GFL_ROUTING', 'IDS_ROUTING', 'OUTCOME_ROUTING']
Unexpected: []

C13 RESULTS
Input C11E records: 1446
GFL: hits=371 | PDFs=6 | pages=124
IDS: hits=524 | PDFs=6 | pages=176
AML: hits=20 | PDFs=5 | pages=15
OUTCOME: hits=531 | PDFs=7 | pages=197

ACCOUNTING
Category records: 1446
Unique source records: 1446
Overlap allowed: True
Duplicate merging: False

BOUNDARY
Semantic verification

In [31]:
# ============================================================
# OIP v1.0.33 — C14
# MANUAL DOCUMENTARY VERIFICATION QUEUE
# FINAL / WARNING-FREE / FAIL-CLOSED
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

BASE = Path("/kaggle/working/oip_v1_0_33")

print("=" * 72)
print("OIP v1.0.33 — C14")
print("MANUAL DOCUMENTARY VERIFICATION QUEUE")
print("=" * 72)


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------
# 2. REQUIRED INPUTS
# ------------------------------------------------------------

c13_status_path = BASE / "c13_final_status.json"

category_files = {
    "GFL": BASE / "c13_gfl_hits.csv",
    "IDS": BASE / "c13_ids_hits.csv",
    "AML": BASE / "c13_aml_hits.csv",
    "OUTCOME": BASE / "c13_outcome_hits.csv"
}

if not c13_status_path.exists():
    raise FileNotFoundError(
        f"Missing upstream artifact: {c13_status_path}"
    )

for category, path in category_files.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Missing C13 artifact for {category}: {path}"
        )


# ------------------------------------------------------------
# 3. LOAD C13 STATUS
# ------------------------------------------------------------

c13_status = load_json(c13_status_path)

print("\nUPSTREAM STATUS")
print("C13:", c13_status.get("status"))

if c13_status.get("status") != "PASS":
    raise RuntimeError(
        "C14 BLOCKED: C13 status is not PASS."
    )


# ------------------------------------------------------------
# 4. LOAD CATEGORY LEDGERS
# ------------------------------------------------------------

frames = {}

for category, path in category_files.items():

    frame = pd.read_csv(path)

    frames[category] = frame

    print(
        f"{category}: {len(frame)} records"
    )


# ------------------------------------------------------------
# 5. EXACT C13 SCHEMA CHECK
# ------------------------------------------------------------

required_columns = [
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "documentary_occurrence",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "discovery_only",
    "semantic_inference_allowed"
]

schema_errors = []

for category, frame in frames.items():

    missing = [
        col
        for col in required_columns
        if col not in frame.columns
    ]

    if missing:
        schema_errors.append(
            f"{category}: missing {missing}"
        )


if schema_errors:
    raise RuntimeError(
        "C14 BLOCKED: C13 schema mismatch: "
        + " | ".join(schema_errors)
    )


# ------------------------------------------------------------
# 6. VERIFY C13 BOUNDARY
# ------------------------------------------------------------

boundary_errors = []

for category, frame in frames.items():

    if not frame[
        "semantic_verification"
    ].astype(str).eq(
        "NOT_PERFORMED"
    ).all():

        boundary_errors.append(
            f"{category}: semantic_verification"
        )

    if not frame[
        "construct_mapping"
    ].astype(str).eq(
        "NOT_ESTABLISHED"
    ).all():

        boundary_errors.append(
            f"{category}: construct_mapping"
        )

    if not frame[
        "authorization"
    ].astype(str).eq(
        "NOT_AUTHORIZED"
    ).all():

        boundary_errors.append(
            f"{category}: authorization"
        )

    if not frame[
        "discovery_only"
    ].astype(str).str.lower().eq(
        "true"
    ).all():

        boundary_errors.append(
            f"{category}: discovery_only"
        )

    if not frame[
        "semantic_inference_allowed"
    ].astype(str).str.lower().eq(
        "false"
    ).all():

        boundary_errors.append(
            f"{category}: semantic_inference_allowed"
        )


print("\nC13 BOUNDARY CHECK")
print(
    "Violations:",
    len(boundary_errors)
)

if boundary_errors:
    raise RuntimeError(
        "C14 BLOCKED: C13 boundary violation: "
        + ", ".join(boundary_errors)
    )


# ------------------------------------------------------------
# 7. BUILD MANUAL REVIEW QUEUE
# ------------------------------------------------------------
#
# IMPORTANT:
#
# No semantic interpretation occurs here.
#
# No construct is verified.
#
# No variable is authorized.
#
# No outcome is established.
#
# C14 only creates a provenance-preserving review queue.
# ------------------------------------------------------------

queue_frames = []

for category, frame in frames.items():

    q = frame[
        [
            "relative_path",
            "file_name",
            "page_number",
            "search_family",
            "search_term",
            "occurrence_text",
            "context_snippet"
        ]
    ].copy()

    q["review_category"] = category

    q["documentary_verification"] = (
        "PENDING_REVIEW"
    )

    q["semantic_verification"] = (
        "NOT_VERIFIED"
    )

    q["construct_mapping"] = (
        "NOT_ESTABLISHED"
    )

    q["authorization"] = (
        "NOT_AUTHORIZED"
    )

    q["review_required"] = True

    q["semantic_inference_allowed"] = False

    queue_frames.append(q)


queue = pd.concat(
    queue_frames,
    ignore_index=True
)


# ------------------------------------------------------------
# 8. STABLE QUEUE ID
# ------------------------------------------------------------

queue.insert(
    0,
    "verification_queue_id",
    range(
        1,
        len(queue) + 1
    )
)


# ------------------------------------------------------------
# 9. SOURCE ACCOUNTING
# ------------------------------------------------------------

source_record_count = len(queue)

category_counts = (
    queue[
        "review_category"
    ]
    .value_counts(
        sort=False
    )
    .to_dict()
)


# ------------------------------------------------------------
# 10. PDF ACCOUNTING
# ------------------------------------------------------------

pdf_counts = {}

for category in [
    "GFL",
    "IDS",
    "AML",
    "OUTCOME"
]:

    frame = queue[
        queue["review_category"] == category
    ]

    pdf_counts[category] = int(
        frame["file_name"].nunique()
    )


# ------------------------------------------------------------
# 11. PAGE ACCOUNTING
# ------------------------------------------------------------
#
# Explicitly select columns before groupby.
# No groupby.apply().
# No pandas FutureWarning.
# ------------------------------------------------------------

page_level = (
    queue[
        [
            "review_category",
            "relative_path",
            "page_number"
        ]
    ]
    .drop_duplicates()
)

page_counts = (
    page_level
    .groupby(
        "review_category"
    )
    .size()
    .to_dict()
)

page_counts = {
    category: int(
        page_counts.get(category, 0)
    )
    for category in [
        "GFL",
        "IDS",
        "AML",
        "OUTCOME"
    ]
}


# ------------------------------------------------------------
# 12. REVIEW STATUS
# ------------------------------------------------------------

review_status_counts = (
    queue[
        "documentary_verification"
    ]
    .value_counts()
    .to_dict()
)


# ------------------------------------------------------------
# 13. PROVENANCE ACCOUNTING
# ------------------------------------------------------------

# C13 records are preserved one-for-one.
#
# No deduplication.
# No snippet merging.
# No semantic consolidation.

provenance_preserved = (
    len(queue) == sum(
        len(frame)
        for frame in frames.values()
    )
)

if not provenance_preserved:
    raise RuntimeError(
        "C14 BLOCKED: Provenance accounting mismatch."
    )


# ------------------------------------------------------------
# 14. AUTHORIZATION LOCK
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}


# ------------------------------------------------------------
# 15. BOUNDARY
# ------------------------------------------------------------

boundary = {

    "documentary_verification":
        "PENDING_REVIEW",

    "semantic_verification":
        "NOT_VERIFIED",

    "construct_mapping":
        "NOT_ESTABLISHED",

    "authorization":
        "NOT_AUTHORIZED",

    "review_required":
        True,

    "semantic_inference_allowed":
        False,

    "automatic_semantic_verification":
        False,

    "automatic_construct_mapping":
        False,

    "deduplication":
        False,

    "provenance_preserved":
        True
}


# ------------------------------------------------------------
# 16. SAVE QUEUE
# ------------------------------------------------------------

queue_path = (
    BASE /
    "c14_manual_documentary_verification_queue.csv"
)

queue.to_csv(
    queue_path,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 17. SUMMARY
# ------------------------------------------------------------

summary = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C14",

    "cell_name":
        "MANUAL DOCUMENTARY VERIFICATION QUEUE",

    "upstream": {
        "C13":
            "PASS"
    },

    "source_record_count":
        int(source_record_count),

    "category_counts":
        {
            k: int(v)
            for k, v in category_counts.items()
        },

    "pdf_counts":
        pdf_counts,

    "unique_pdf_page_counts":
        page_counts,

    "review_status":
        {
            k: int(v)
            for k, v in review_status_counts.items()
        },

    "provenance_preserved":
        provenance_preserved,

    "deduplication":
        False,

    "automatic_semantic_verification":
        False,

    "automatic_construct_mapping":
        False,

    "boundary":
        boundary,

    "authorization":
        authorization,

    "fail_conditions":
        [],

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        False
}


summary_path = (
    BASE /
    "c14_manual_verification_summary.json"
)

save_json(
    summary_path,
    summary
)


# ------------------------------------------------------------
# 18. FINAL STATUS
# ------------------------------------------------------------

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C14",

    "cell_name":
        "MANUAL DOCUMENTARY VERIFICATION QUEUE",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "PASS",

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        False,

    "authorization":
        authorization,

    "payload":
        summary
}


final_status_path = (
    BASE /
    "c14_final_status.json"
)

save_json(
    final_status_path,
    final_status
)


# ------------------------------------------------------------
# 19. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C14 RESULTS")
print("=" * 72)

print(
    "Verification queue records:",
    source_record_count
)

print("\nCATEGORY COUNTS")

for category in [
    "GFL",
    "IDS",
    "AML",
    "OUTCOME"
]:

    print(
        f"{category}: "
        f"records={category_counts.get(category, 0)} | "
        f"PDFs={pdf_counts.get(category, 0)} | "
        f"pages={page_counts.get(category, 0)}"
    )


print("\nREVIEW STATUS")

for status, count in review_status_counts.items():
    print(
        f"{status}: {count}"
    )


print("\nPROVENANCE")
print(
    "Records preserved:",
    provenance_preserved
)
print(
    "Deduplication:",
    False
)


print("\nBOUNDARY")
print(
    "Documentary verification:",
    "PENDING_REVIEW"
)
print(
    "Semantic verification:",
    "NOT_VERIFIED"
)
print(
    "Construct mapping:",
    "NOT_ESTABLISHED"
)
print(
    "Authorization:",
    "NOT_AUTHORIZED"
)
print(
    "Semantic inference allowed:",
    False
)


print("\nFAIL-CLOSED")
print(
    "Fail conditions:",
    []
)
print(
    "Triggered:",
    False
)


print("\nC14 STATUS: PASS")


print("\nARTIFACTS")
print(queue_path)
print(summary_path)
print(final_status_path)

print("=" * 72)

OIP v1.0.33 — C14
MANUAL DOCUMENTARY VERIFICATION QUEUE

UPSTREAM STATUS
C13: PASS
GFL: 371 records
IDS: 524 records
AML: 20 records
OUTCOME: 531 records

C13 BOUNDARY CHECK
Violations: 0

C14 RESULTS
Verification queue records: 1446

CATEGORY COUNTS
GFL: records=371 | PDFs=6 | pages=124
IDS: records=524 | PDFs=6 | pages=176
AML: records=20 | PDFs=5 | pages=15
OUTCOME: records=531 | PDFs=7 | pages=197

REVIEW STATUS
PENDING_REVIEW: 1446

PROVENANCE
Records preserved: True
Deduplication: False

BOUNDARY
Documentary verification: PENDING_REVIEW
Semantic verification: NOT_VERIFIED
Construct mapping: NOT_ESTABLISHED
Authorization: NOT_AUTHORIZED
Semantic inference allowed: False

FAIL-CLOSED
Fail conditions: []
Triggered: False

C14 STATUS: PASS

ARTIFACTS
/kaggle/working/oip_v1_0_33/c14_manual_documentary_verification_queue.csv
/kaggle/working/oip_v1_0_33/c14_manual_verification_summary.json
/kaggle/working/oip_v1_0_33/c14_final_status.json


In [32]:
# ============================================================
# OIP v1.0.33 — C15
# DOCUMENTARY VERIFICATION PREPARATION & EVIDENCE LOCK
# FINAL / FAIL-CLOSED
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

BASE = Path("/kaggle/working/oip_v1_0_33")

print("=" * 72)
print("OIP v1.0.33 — C15")
print("DOCUMENTARY VERIFICATION PREPARATION & EVIDENCE LOCK")
print("=" * 72)


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------
# 2. REQUIRED UPSTREAM ARTIFACTS
# ------------------------------------------------------------

status_path = BASE / "c14_final_status.json"

queue_path = (
    BASE /
    "c14_manual_documentary_verification_queue.csv"
)

if not status_path.exists():
    raise FileNotFoundError(
        f"Missing C14 status artifact: {status_path}"
    )

if not queue_path.exists():
    raise FileNotFoundError(
        f"Missing C14 queue artifact: {queue_path}"
    )


# ------------------------------------------------------------
# 3. LOAD C14
# ------------------------------------------------------------

c14 = load_json(status_path)

print("\nUPSTREAM STATUS")
print("C14:", c14.get("status"))

if c14.get("status") != "PASS":
    raise RuntimeError(
        "C15 BLOCKED: C14 status is not PASS."
    )


queue = pd.read_csv(queue_path)

print(
    "C14 queue records:",
    len(queue)
)


# ------------------------------------------------------------
# 4. EXACT C14 SCHEMA
# ------------------------------------------------------------

required_columns = [
    "verification_queue_id",
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "review_category",
    "documentary_verification",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "review_required",
    "semantic_inference_allowed"
]

missing_columns = [
    col
    for col in required_columns
    if col not in queue.columns
]

if missing_columns:
    raise RuntimeError(
        "C15 BLOCKED: Missing C14 columns: "
        + str(missing_columns)
    )


# ------------------------------------------------------------
# 5. C14 BOUNDARY VALIDATION
# ------------------------------------------------------------

boundary_errors = []

if not queue[
    "documentary_verification"
].astype(str).eq(
    "PENDING_REVIEW"
).all():
    boundary_errors.append(
        "documentary_verification"
    )

if not queue[
    "semantic_verification"
].astype(str).eq(
    "NOT_VERIFIED"
).all():
    boundary_errors.append(
        "semantic_verification"
    )

if not queue[
    "construct_mapping"
].astype(str).eq(
        "NOT_ESTABLISHED"
).all():
    boundary_errors.append(
        "construct_mapping"
    )

if not queue[
    "authorization"
].astype(str).eq(
    "NOT_AUTHORIZED"
).all():
    boundary_errors.append(
        "authorization"
    )

if not queue[
    "semantic_inference_allowed"
].astype(str).str.lower().eq(
    "false"
).all():
    boundary_errors.append(
        "semantic_inference_allowed"
    )

if not queue[
    "review_required"
].astype(str).str.lower().eq(
    "true"
).all():
    boundary_errors.append(
        "review_required"
    )


print("\nC14 BOUNDARY CHECK")
print(
    "Violations:",
    len(boundary_errors)
)

if boundary_errors:
    raise RuntimeError(
        "C15 BLOCKED: C14 boundary violation: "
        + str(boundary_errors)
    )


# ------------------------------------------------------------
# 6. QUEUE INTEGRITY
# ------------------------------------------------------------

queue_ids = queue[
    "verification_queue_id"
]

duplicate_queue_ids = int(
    queue_ids.duplicated().sum()
)

null_queue_ids = int(
    queue_ids.isna().sum()
)

expected_ids = list(
    range(
        1,
        len(queue) + 1
    )
)

observed_ids = (
    queue_ids
    .astype(int)
    .tolist()
)

queue_id_sequence_match = (
    observed_ids == expected_ids
)


if duplicate_queue_ids != 0:
    raise RuntimeError(
        "C15 BLOCKED: Duplicate verification queue IDs."
    )

if null_queue_ids != 0:
    raise RuntimeError(
        "C15 BLOCKED: Null verification queue IDs."
    )


# ------------------------------------------------------------
# 7. PROVENANCE FIELD COMPLETENESS
# ------------------------------------------------------------

provenance_fields = [
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet"
]

provenance_null_counts = {}

for col in provenance_fields:

    provenance_null_counts[col] = int(
        queue[col].isna().sum()
    )


# Page number must be present for documentary review.

if provenance_null_counts[
    "page_number"
] != 0:

    raise RuntimeError(
        "C15 BLOCKED: Missing documentary page numbers."
    )


# Source path must be present.

if provenance_null_counts[
    "relative_path"
] != 0:

    raise RuntimeError(
        "C15 BLOCKED: Missing source paths."
    )


# ------------------------------------------------------------
# 8. SOURCE REVIEW STATE
# ------------------------------------------------------------

queue["source_review_state"] = (
    "REQUIRES_SOURCE_REVIEW"
)

queue["documentary_evidence_status"] = (
    "NOT_VERIFIED"
)

queue["semantic_verification"] = (
    "NOT_VERIFIED"
)

queue["construct_mapping"] = (
    "NOT_ESTABLISHED"
)

queue["authorization"] = (
    "NOT_AUTHORIZED"
)

queue["review_required"] = True

queue["semantic_inference_allowed"] = False


# ------------------------------------------------------------
# 9. REVIEW DECISION FIELDS
# ------------------------------------------------------------
#
# These fields are intentionally empty.
#
# C15 does NOT fill them.
# Human/source verification must establish them.
# ------------------------------------------------------------

queue["source_document_verified"] = (
    "PENDING_SOURCE_REVIEW"
)

queue["question_or_section_verified"] = (
    "PENDING_SOURCE_REVIEW"
)

queue["construct_relevance_verified"] = (
    "PENDING_SOURCE_REVIEW"
)

queue["verification_note"] = (
    ""
)


# ------------------------------------------------------------
# 10. CATEGORY ACCOUNTING
# ------------------------------------------------------------

category_counts = (
    queue[
        "review_category"
    ]
    .value_counts(
        sort=False
    )
    .to_dict()
)

category_counts = {
    str(k): int(v)
    for k, v in category_counts.items()
}


# ------------------------------------------------------------
# 11. SOURCE DOCUMENT ACCOUNTING
# ------------------------------------------------------------

pdf_count = int(
    queue[
        "relative_path"
    ].nunique()
)

page_count = int(
    queue[
        [
            "relative_path",
            "page_number"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


# ------------------------------------------------------------
# 12. SEARCH-FAMILY ACCOUNTING
# ------------------------------------------------------------

search_family_counts = (
    queue[
        "search_family"
    ]
    .value_counts(
        sort=False
    )
    .to_dict()
)

search_family_counts = {
    str(k): int(v)
    for k, v in search_family_counts.items()
}


# ------------------------------------------------------------
# 13. PROVENANCE PRESERVATION
# ------------------------------------------------------------

provenance_preserved = (
    len(queue) == len(
        pd.read_csv(queue_path)
    )
)


# ------------------------------------------------------------
# 14. NO-AUTHORIZATION ASSERTION
# ------------------------------------------------------------

authorization_safe = (
    queue[
        "authorization"
    ]
    .astype(str)
    .eq("NOT_AUTHORIZED")
    .all()
)

construct_safe = (
    queue[
        "construct_mapping"
    ]
    .astype(str)
    .eq("NOT_ESTABLISHED")
    .all()
)

semantic_safe = (
    queue[
        "semantic_verification"
    ]
    .astype(str)
    .eq("NOT_VERIFIED")
    .all()
)


if not authorization_safe:
    raise RuntimeError(
        "C15 BLOCKED: Authorization state changed."
    )

if not construct_safe:
    raise RuntimeError(
        "C15 BLOCKED: Construct mapping changed."
    )

if not semantic_safe:
    raise RuntimeError(
        "C15 BLOCKED: Semantic verification changed."
    )


# ------------------------------------------------------------
# 15. FINAL BOUNDARY
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}


boundary = {

    "source_review":
        "REQUIRES_SOURCE_REVIEW",

    "documentary_evidence":
        "NOT_VERIFIED",

    "semantic_verification":
        "NOT_VERIFIED",

    "construct_mapping":
        "NOT_ESTABLISHED",

    "authorization":
        "NOT_AUTHORIZED",

    "semantic_inference_allowed":
        False,

    "automatic_authorization":
        False,

    "automatic_construct_mapping":
        False,

    "automatic_semantic_verification":
        False,

    "provenance_preserved":
        True
}


# ------------------------------------------------------------
# 16. FAIL CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if duplicate_queue_ids != 0:
    fail_conditions.append(
        "DUPLICATE_QUEUE_ID"
    )

if null_queue_ids != 0:
    fail_conditions.append(
        "NULL_QUEUE_ID"
    )

if not queue_id_sequence_match:
    fail_conditions.append(
        "QUEUE_ID_SEQUENCE_MISMATCH"
    )

if not provenance_preserved:
    fail_conditions.append(
        "PROVENANCE_NOT_PRESERVED"
    )

if not authorization_safe:
    fail_conditions.append(
        "AUTHORIZATION_LEAK"
    )

if not construct_safe:
    fail_conditions.append(
        "CONSTRUCT_AUTHORIZATION_LEAK"
    )

if not semantic_safe:
    fail_conditions.append(
        "SEMANTIC_VERIFICATION_LEAK"
    )


# ------------------------------------------------------------
# 17. STATUS
# ------------------------------------------------------------

if fail_conditions:

    status = "BLOCKED"
    fail_closed_triggered = True

else:

    status = "PASS"
    fail_closed_triggered = False


# ------------------------------------------------------------
# 18. SAVE REVIEW LEDGER
# ------------------------------------------------------------

review_ledger_path = (
    BASE /
    "c15_documentary_verification_ledger.csv"
)

queue.to_csv(
    review_ledger_path,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 19. SUMMARY
# ------------------------------------------------------------

summary = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C15",

    "cell_name":
        "DOCUMENTARY VERIFICATION PREPARATION & EVIDENCE LOCK",

    "upstream_c14_status":
        "PASS",

    "queue_records":
        int(len(queue)),

    "verification_queue_id": {
        "duplicate_count":
            duplicate_queue_ids,

        "null_count":
            null_queue_ids,

        "sequence_match":
            queue_id_sequence_match
    },

    "category_counts":
        category_counts,

    "search_family_counts":
        search_family_counts,

    "source_document_count":
        pdf_count,

    "unique_document_page_count":
        page_count,

    "provenance_null_counts":
        provenance_null_counts,

    "provenance_preserved":
        provenance_preserved,

    "source_review_state":
        "REQUIRES_SOURCE_REVIEW",

    "documentary_evidence_status":
        "NOT_VERIFIED",

    "semantic_verification":
        "NOT_VERIFIED",

    "construct_mapping":
        "NOT_ESTABLISHED",

    "automatic_semantic_verification":
        False,

    "automatic_construct_mapping":
        False,

    "automatic_authorization":
        False,

    "semantic_inference_allowed":
        False,

    "boundary":
        boundary,

    "authorization":
        authorization,

    "fail_conditions":
        fail_conditions,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered
}


summary_path = (
    BASE /
    "c15_documentary_verification_summary.json"
)

save_json(
    summary_path,
    summary
)


# ------------------------------------------------------------
# 20. FINAL STATUS ARTIFACT
# ------------------------------------------------------------

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C15",

    "cell_name":
        "DOCUMENTARY VERIFICATION PREPARATION & EVIDENCE LOCK",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        authorization,

    "payload":
        summary
}


final_status_path = (
    BASE /
    "c15_final_status.json"
)

save_json(
    final_status_path,
    final_status
)


# ------------------------------------------------------------
# 21. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C15 RESULTS")
print("=" * 72)

print(
    "Queue records:",
    len(queue)
)

print(
    "Duplicate queue IDs:",
    duplicate_queue_ids
)

print(
    "Null queue IDs:",
    null_queue_ids
)

print(
    "Queue ID sequence match:",
    queue_id_sequence_match
)

print(
    "Source documents:",
    pdf_count
)

print(
    "Unique document-pages:",
    page_count
)

print(
    "Provenance preserved:",
    provenance_preserved
)

print("\nCATEGORY COUNTS")

for category in [
    "GFL",
    "IDS",
    "AML",
    "OUTCOME"
]:

    print(
        f"{category}: "
        f"{category_counts.get(category, 0)}"
    )


print("\nBOUNDARY")

print(
    "Source review:",
    "REQUIRES_SOURCE_REVIEW"
)

print(
    "Documentary evidence:",
    "NOT_VERIFIED"
)

print(
    "Semantic verification:",
    "NOT_VERIFIED"
)

print(
    "Construct mapping:",
    "NOT_ESTABLISHED"
)

print(
    "Authorization:",
    "NOT_AUTHORIZED"
)

print(
    "Semantic inference allowed:",
    False
)

print(
    "Automatic authorization:",
    False
)


print("\nFAIL-CLOSED")

print(
    "Fail conditions:",
    fail_conditions
)

print(
    "Triggered:",
    fail_closed_triggered
)

print(
    "\nC15 STATUS:",
    status
)


print("\nARTIFACTS")

print(review_ledger_path)
print(summary_path)
print(final_status_path)

print("=" * 72)

OIP v1.0.33 — C15
DOCUMENTARY VERIFICATION PREPARATION & EVIDENCE LOCK

UPSTREAM STATUS
C14: PASS
C14 queue records: 1446

C14 BOUNDARY CHECK
Violations: 0

C15 RESULTS
Queue records: 1446
Duplicate queue IDs: 0
Null queue IDs: 0
Queue ID sequence match: True
Source documents: 7
Unique document-pages: 374
Provenance preserved: True

CATEGORY COUNTS
GFL: 371
IDS: 524
AML: 20
OUTCOME: 531

BOUNDARY
Source review: REQUIRES_SOURCE_REVIEW
Documentary evidence: NOT_VERIFIED
Semantic verification: NOT_VERIFIED
Construct mapping: NOT_ESTABLISHED
Authorization: NOT_AUTHORIZED
Semantic inference allowed: False
Automatic authorization: False

FAIL-CLOSED
Fail conditions: []
Triggered: False

C15 STATUS: PASS

ARTIFACTS
/kaggle/working/oip_v1_0_33/c15_documentary_verification_ledger.csv
/kaggle/working/oip_v1_0_33/c15_documentary_verification_summary.json
/kaggle/working/oip_v1_0_33/c15_final_status.json


In [33]:
# ============================================================
# OIP v1.0.33 — C16
# SOURCE-PAGE EXTRACTION & DOCUMENTARY EVIDENCE PACK
# FINAL CACHED VERSION
# FAIL-CLOSED / NO SEMANTIC INFERENCE
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

BASE = Path("/kaggle/working/oip_v1_0_33")

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/sudharsandas27/"
    "oip-v1-0-33-ethiopia-ess4-evidence-audit"
)

print("=" * 72)
print("OIP v1.0.33 — C16")
print("SOURCE-PAGE EXTRACTION & DOCUMENTARY EVIDENCE PACK")
print("=" * 72)


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------
# 2. INPUTS
# ------------------------------------------------------------

status_path = BASE / "c15_final_status.json"

ledger_path = (
    BASE /
    "c15_documentary_verification_ledger.csv"
)

if not status_path.exists():
    raise FileNotFoundError(
        f"Missing C15 status: {status_path}"
    )

if not ledger_path.exists():
    raise FileNotFoundError(
        f"Missing C15 ledger: {ledger_path}"
    )


# ------------------------------------------------------------
# 3. LOAD C15
# ------------------------------------------------------------

c15 = load_json(status_path)

print("\nUPSTREAM STATUS")
print("C15:", c15.get("status"))

if c15.get("status") != "PASS":
    raise RuntimeError(
        "C16 BLOCKED: C15 is not PASS."
    )

queue = pd.read_csv(
    ledger_path
)

print(
    "C15 ledger records:",
    len(queue)
)


# ------------------------------------------------------------
# 4. EXACT REQUIRED SCHEMA
# ------------------------------------------------------------

required_columns = [
    "verification_queue_id",
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "review_category",
    "source_review_state",
    "documentary_evidence_status",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "review_required",
    "semantic_inference_allowed"
]

missing_columns = [
    c for c in required_columns
    if c not in queue.columns
]

if missing_columns:
    raise RuntimeError(
        "C16 BLOCKED: Missing C15 columns: "
        + str(missing_columns)
    )


# ------------------------------------------------------------
# 5. C15 BOUNDARY CHECK
# ------------------------------------------------------------

boundary_errors = []

checks = {

    "source_review_state":
        queue[
            "source_review_state"
        ].astype(str).eq(
            "REQUIRES_SOURCE_REVIEW"
        ),

    "documentary_evidence_status":
        queue[
            "documentary_evidence_status"
        ].astype(str).eq(
            "NOT_VERIFIED"
        ),

    "semantic_verification":
        queue[
            "semantic_verification"
        ].astype(str).eq(
            "NOT_VERIFIED"
        ),

    "construct_mapping":
        queue[
            "construct_mapping"
        ].astype(str).eq(
            "NOT_ESTABLISHED"
        ),

    "authorization":
        queue[
            "authorization"
        ].astype(str).eq(
            "NOT_AUTHORIZED"
        ),

    "semantic_inference_allowed":
        queue[
            "semantic_inference_allowed"
        ].astype(str).str.lower().eq(
            "false"
        )
}

for name, result in checks.items():

    if not result.all():
        boundary_errors.append(name)


print("\nC15 BOUNDARY CHECK")
print(
    "Violations:",
    len(boundary_errors)
)

if boundary_errors:
    raise RuntimeError(
        "C16 BLOCKED: "
        + str(boundary_errors)
    )


# ------------------------------------------------------------
# 6. PDF READER
# ------------------------------------------------------------

try:
    from pypdf import PdfReader
except Exception as e:
    raise RuntimeError(
        "C16 BLOCKED: pypdf unavailable."
    ) from e


# ------------------------------------------------------------
# 7. SOURCE RESOLUTION
# ------------------------------------------------------------

def resolve_source(relative_path):

    rel = Path(
        str(relative_path)
    )

    candidates = [
        SOURCE_ROOT / rel,
        SOURCE_ROOT / rel.name
    ]

    for path in candidates:

        if (
            path.exists()
            and path.is_file()
        ):
            return path

    return None


# ------------------------------------------------------------
# 8. PDF CACHE
# ------------------------------------------------------------

reader_cache = {}

def get_reader(relative_path):

    if relative_path in reader_cache:
        return reader_cache[
            relative_path
        ]

    source_path = resolve_source(
        relative_path
    )

    if source_path is None:

        reader_cache[
            relative_path
        ] = (None, None)

        return None, None

    try:

        reader = PdfReader(
            str(source_path)
        )

        reader_cache[
            relative_path
        ] = (
            reader,
            source_path
        )

        return (
            reader,
            source_path
        )

    except Exception:

        reader_cache[
            relative_path
        ] = (
            None,
            source_path
        )

        return (
            None,
            source_path
        )


# ------------------------------------------------------------
# 9. PAGE TEXT CACHE
# ------------------------------------------------------------

page_text_cache = {}

def get_page_text(
    relative_path,
    reader,
    page_number
):

    cache_key = (
        relative_path,
        int(page_number)
    )

    if cache_key in page_text_cache:
        return (
            page_text_cache[
                cache_key
            ],
            "CACHE"
        )

    page = reader.pages[
        int(page_number) - 1
    ]

    text = (
        page.extract_text()
        or ""
    )

    text = str(text)

    page_text_cache[
        cache_key
    ] = text

    return (
        text,
        "EXTRACTED"
    )


# ------------------------------------------------------------
# 10. EXTRACTION
# ------------------------------------------------------------

records = []

source_missing = 0
read_errors = 0
page_out_of_range = 0
pages_extracted = 0
empty_text_pages = 0
cache_hits = 0
unique_pages_processed = 0

for _, row in queue.iterrows():

    relative_path = str(
        row["relative_path"]
    )

    page_number = int(
        row["page_number"]
    )

    reader, source_path = get_reader(
        relative_path
    )

    extraction_status = (
        "NOT_EXTRACTED"
    )

    extraction_mode = (
        "NONE"
    )

    page_text = ""

    pdf_page_count = None

    error_message = ""

    if reader is None:

        if source_path is None:

            source_missing += 1

            extraction_status = (
                "SOURCE_MISSING"
            )

        else:

            read_errors += 1

            extraction_status = (
                "PDF_READ_ERROR"
            )

    else:

        try:

            pdf_page_count = len(
                reader.pages
            )

            if (
                page_number < 1
                or page_number > pdf_page_count
            ):

                page_out_of_range += 1

                extraction_status = (
                    "PAGE_OUT_OF_RANGE"
                )

            else:

                (
                    page_text,
                    extraction_mode
                ) = get_page_text(
                    relative_path,
                    reader,
                    page_number
                )

                if extraction_mode == "CACHE":

                    cache_hits += 1

                else:

                    unique_pages_processed += 1

                    pages_extracted += 1

                if not page_text.strip():

                    empty_text_pages += 1

                    extraction_status = (
                        "PAGE_EXTRACTED_EMPTY"
                    )

                else:

                    extraction_status = (
                        "PAGE_EXTRACTED"
                    )

        except Exception as e:

            read_errors += 1

            extraction_status = (
                "PAGE_EXTRACTION_ERROR"
            )

            error_message = str(e)


    records.append({

        "verification_queue_id":
            int(
                row[
                    "verification_queue_id"
                ]
            ),

        "relative_path":
            relative_path,

        "file_name":
            str(
                row["file_name"]
            ),

        "page_number":
            page_number,

        "pdf_page_count":
            pdf_page_count,

        "search_family":
            str(
                row["search_family"]
            ),

        "search_term":
            str(
                row["search_term"]
            ),

        "occurrence_text":
            str(
                row["occurrence_text"]
            ),

        "context_snippet":
            str(
                row["context_snippet"]
            ),

        "review_category":
            str(
                row["review_category"]
            ),

        "source_path_resolution":
            (
                "RESOLVED"
                if source_path is not None
                else "NOT_RESOLVED"
            ),

        "source_page_extraction":
            extraction_status,

        "extraction_mode":
            extraction_mode,

        "page_text":
            page_text,

        "page_text_length":
            len(page_text),

        "documentary_verification":
            "NOT_VERIFIED",

        "semantic_verification":
            "NOT_VERIFIED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "review_required":
            True,

        "semantic_inference_allowed":
            False,

        "error_message":
            error_message
    })


evidence_pack = pd.DataFrame(
    records
)


# ------------------------------------------------------------
# 11. ACCOUNTING
# ------------------------------------------------------------

queue_count = len(queue)

pack_count = len(
    evidence_pack
)

record_accounting_match = (
    queue_count == pack_count
)

if not record_accounting_match:
    raise RuntimeError(
        "C16 BLOCKED: record accounting mismatch."
    )


resolved_count = int(
    evidence_pack[
        "source_path_resolution"
    ].eq(
        "RESOLVED"
    ).sum()
)

not_resolved_count = int(
    evidence_pack[
        "source_path_resolution"
    ].eq(
        "NOT_RESOLVED"
    ).sum()
)

extracted_count = int(
    evidence_pack[
        "source_page_extraction"
    ].isin([
        "PAGE_EXTRACTED",
        "PAGE_EXTRACTED_EMPTY"
    ]).sum()
)

text_count = int(
    evidence_pack[
        "source_page_extraction"
    ].eq(
        "PAGE_EXTRACTED"
    ).sum()
)

empty_count = int(
    evidence_pack[
        "source_page_extraction"
    ].eq(
        "PAGE_EXTRACTED_EMPTY"
    ).sum()
)

failed_count = int(
    evidence_pack[
        "source_page_extraction"
    ].isin([
        "SOURCE_MISSING",
        "PDF_READ_ERROR",
        "PAGE_OUT_OF_RANGE",
        "PAGE_EXTRACTION_ERROR"
    ]).sum()
)


# ------------------------------------------------------------
# 12. PROVENANCE
# ------------------------------------------------------------

provenance_fields = [
    "verification_queue_id",
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet"
]

provenance_nulls = {
    col: int(
        evidence_pack[col].isna().sum()
    )
    for col in provenance_fields
}


# ------------------------------------------------------------
# 13. SAFETY ASSERTIONS
# ------------------------------------------------------------

semantic_safe = (
    evidence_pack[
        "semantic_verification"
    ].astype(str).eq(
        "NOT_VERIFIED"
    ).all()
)

construct_safe = (
    evidence_pack[
        "construct_mapping"
    ].astype(str).eq(
        "NOT_ESTABLISHED"
    ).all()
)

authorization_safe = (
    evidence_pack[
        "authorization"
    ].astype(str).eq(
        "NOT_AUTHORIZED"
    ).all()
)

inference_safe = (
    evidence_pack[
        "semantic_inference_allowed"
    ].astype(str).str.lower().eq(
        "false"
    ).all()
)


# ------------------------------------------------------------
# 14. FAIL CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if not record_accounting_match:
    fail_conditions.append(
        "RECORD_ACCOUNTING_MISMATCH"
    )

if page_out_of_range != 0:
    fail_conditions.append(
        "PAGE_OUT_OF_RANGE"
    )

if not semantic_safe:
    fail_conditions.append(
        "SEMANTIC_VERIFICATION_LEAK"
    )

if not construct_safe:
    fail_conditions.append(
        "CONSTRUCT_MAPPING_LEAK"
    )

if not authorization_safe:
    fail_conditions.append(
        "AUTHORIZATION_LEAK"
    )

if not inference_safe:
    fail_conditions.append(
        "SEMANTIC_INFERENCE_LEAK"
    )


# ------------------------------------------------------------
# 15. STATUS
# ------------------------------------------------------------

if fail_conditions:

    status = "BLOCKED"
    fail_closed_triggered = True

else:

    status = "PASS"
    fail_closed_triggered = False


# ------------------------------------------------------------
# 16. SAVE PACK
# ------------------------------------------------------------

pack_path = (
    BASE /
    "c16_source_page_evidence_pack.csv"
)

evidence_pack.to_csv(
    pack_path,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 17. SUMMARY
# ------------------------------------------------------------

category_counts = (
    evidence_pack[
        "review_category"
    ]
    .value_counts(
        sort=False
    )
    .to_dict()
)

category_counts = {
    str(k): int(v)
    for k, v in category_counts.items()
}


summary = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C16",

    "cell_name":
        "SOURCE-PAGE EXTRACTION & DOCUMENTARY EVIDENCE PACK",

    "upstream_c15_status":
        "PASS",

    "queue_record_count":
        int(queue_count),

    "evidence_pack_record_count":
        int(pack_count),

    "record_accounting_match":
        record_accounting_match,

    "source_documents":
        int(
            evidence_pack[
                "relative_path"
            ].nunique()
        ),

    "resolved_sources":
        resolved_count,

    "not_resolved_sources":
        not_resolved_count,

    "pages_extracted":
        extracted_count,

    "unique_pages_extracted":
        int(unique_pages_processed),

    "cache_hits":
        int(cache_hits),

    "pages_with_text":
        text_count,

    "pages_with_empty_text":
        empty_count,

    "failed_extraction":
        failed_count,

    "source_missing":
        int(source_missing),

    "pdf_read_errors":
        int(read_errors),

    "page_out_of_range":
        int(page_out_of_range),

    "category_counts":
        category_counts,

    "provenance_null_counts":
        provenance_nulls,

    "documentary_verification":
        "NOT_VERIFIED",

    "semantic_verification":
        "NOT_VERIFIED",

    "construct_mapping":
        "NOT_ESTABLISHED",

    "authorization":
        "NOT_AUTHORIZED",

    "semantic_inference_allowed":
        False,

    "automatic_semantic_verification":
        False,

    "automatic_construct_mapping":
        False,

    "automatic_authorization":
        False,

    "fail_conditions":
        fail_conditions,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered
}


summary_path = (
    BASE /
    "c16_source_page_evidence_summary.json"
)

save_json(
    summary_path,
    summary
)


# ------------------------------------------------------------
# 18. FINAL STATUS
# ------------------------------------------------------------

authorization = {

    "construct":
        "NOT_AUTHORIZED",

    "predictor":
        "NOT_AUTHORIZED",

    "outcome":
        "NOT_AUTHORIZED",

    "key":
        "NOT_AUTHORIZED",

    "relationship":
        "NOT_AUTHORIZED",

    "cohort":
        "NOT_AUTHORIZED",

    "score":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}


final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C16",

    "cell_name":
        "SOURCE-PAGE EXTRACTION & DOCUMENTARY EVIDENCE PACK",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        authorization,

    "payload":
        summary
}


final_status_path = (
    BASE /
    "c16_final_status.json"
)

save_json(
    final_status_path,
    final_status
)


# ------------------------------------------------------------
# 19. OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C16 RESULTS")
print("=" * 72)

print(
    "C15 ledger records:",
    queue_count
)

print(
    "Evidence pack records:",
    pack_count
)

print(
    "Record accounting:",
    record_accounting_match
)

print(
    "Source documents:",
    evidence_pack[
        "relative_path"
    ].nunique()
)

print(
    "Resolved sources:",
    resolved_count
)

print(
    "Pages extracted:",
    extracted_count
)

print(
    "Unique pages extracted:",
    unique_pages_processed
)

print(
    "Cache hits:",
    cache_hits
)

print(
    "Pages with text:",
    text_count
)

print(
    "Pages with empty text:",
    empty_count
)

print(
    "Failed extraction:",
    failed_count
)

print("\nCATEGORY COUNTS")

for category in [
    "GFL",
    "IDS",
    "AML",
    "OUTCOME"
]:

    print(
        f"{category}: "
        f"{category_counts.get(category, 0)}"
    )


print("\nBOUNDARY")

print(
    "Documentary verification:",
    "NOT_VERIFIED"
)

print(
    "Semantic verification:",
    "NOT_VERIFIED"
)

print(
    "Construct mapping:",
    "NOT_ESTABLISHED"
)

print(
    "Authorization:",
    "NOT_AUTHORIZED"
)

print(
    "Semantic inference allowed:",
    False
)

print(
    "Automatic authorization:",
    False
)


print("\nFAIL-CLOSED")

print(
    "Fail conditions:",
    fail_conditions
)

print(
    "Triggered:",
    fail_closed_triggered
)

print(
    "\nC16 STATUS:",
    status
)


print("\nARTIFACTS")

print(pack_path)
print(summary_path)
print(final_status_path)

print("=" * 72)

OIP v1.0.33 — C16
SOURCE-PAGE EXTRACTION & DOCUMENTARY EVIDENCE PACK

UPSTREAM STATUS
C15: PASS
C15 ledger records: 1446

C15 BOUNDARY CHECK
Violations: 0

C16 RESULTS
C15 ledger records: 1446
Evidence pack records: 1446
Record accounting: True
Source documents: 7
Resolved sources: 1446
Pages extracted: 1446
Unique pages extracted: 374
Cache hits: 1072
Pages with text: 1446
Pages with empty text: 0
Failed extraction: 0

CATEGORY COUNTS
GFL: 371
IDS: 524
AML: 20
OUTCOME: 531

BOUNDARY
Documentary verification: NOT_VERIFIED
Semantic verification: NOT_VERIFIED
Construct mapping: NOT_ESTABLISHED
Authorization: NOT_AUTHORIZED
Semantic inference allowed: False
Automatic authorization: False

FAIL-CLOSED
Fail conditions: []
Triggered: False

C16 STATUS: PASS

ARTIFACTS
/kaggle/working/oip_v1_0_33/c16_source_page_evidence_pack.csv
/kaggle/working/oip_v1_0_33/c16_source_page_evidence_summary.json
/kaggle/working/oip_v1_0_33/c16_final_status.json


In [34]:
# ============================================================
# OIP v1.0.33 — C17
# DOCUMENTARY COMPONENT VERIFICATION MATRIX
# FINAL / FAIL-CLOSED / SOURCE-LOCKED
#
# Purpose:
#   1. Audit C13 category routing and overlap
#   2. Verify C15 schema dependency
#   3. Build component-level documentary verification matrix
#
# Important:
#   No keyword hit is treated as semantic proof.
#   No automatic construct verification.
#   No construct authorization.
#   No score.
#   No empirical evaluation.
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

BASE = Path("/kaggle/working/oip_v1_0_33")

print("=" * 72)
print("OIP v1.0.33 — C17")
print("DOCUMENTARY COMPONENT VERIFICATION MATRIX")
print("=" * 72)


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


# ------------------------------------------------------------
# 2. REQUIRED ARTIFACTS
# ------------------------------------------------------------

required_files = [

    BASE / "c13_final_status.json",

    BASE / "c13_gfl_hits.csv",

    BASE / "c13_ids_hits.csv",

    BASE / "c13_aml_hits.csv",

    BASE / "c13_outcome_hits.csv",

    BASE / "c15_final_status.json",

    BASE / "c15_documentary_verification_ledger.csv",

    BASE / "c16_final_status.json",

    BASE / "c16_source_page_evidence_pack.csv"
]

missing_files = [
    str(p)
    for p in required_files
    if not p.exists()
]

if missing_files:

    raise RuntimeError(
        "C17 BLOCKED: Missing required artifacts:\n"
        + "\n".join(missing_files)
    )


# ------------------------------------------------------------
# 3. UPSTREAM STATUS
# ------------------------------------------------------------

c13_status = load_json(
    BASE / "c13_final_status.json"
)

c15_status = load_json(
    BASE / "c15_final_status.json"
)

c16_status = load_json(
    BASE / "c16_final_status.json"
)

print("\nUPSTREAM STATUS")

print(
    "C13:",
    c13_status.get("status")
)

print(
    "C15:",
    c15_status.get("status")
)

print(
    "C16:",
    c16_status.get("status")
)

if c13_status.get("status") != "PASS":
    raise RuntimeError(
        "C17 BLOCKED: C13 is not PASS."
    )

if c15_status.get("status") != "PASS":
    raise RuntimeError(
        "C17 BLOCKED: C15 is not PASS."
    )

if c16_status.get("status") != "PASS":
    raise RuntimeError(
        "C17 BLOCKED: C16 is not PASS."
    )


# ------------------------------------------------------------
# 4. LOAD C13 CATEGORY FILES
# ------------------------------------------------------------

gfl = pd.read_csv(
    BASE / "c13_gfl_hits.csv"
)

ids = pd.read_csv(
    BASE / "c13_ids_hits.csv"
)

aml = pd.read_csv(
    BASE / "c13_aml_hits.csv"
)

outcome = pd.read_csv(
    BASE / "c13_outcome_hits.csv"
)

category_frames = {
    "GFL": gfl,
    "IDS": ids,
    "AML": aml,
    "OUTCOME": outcome
}


print("\nC13 CATEGORY RECORDS")

for category, frame in category_frames.items():

    print(
        f"{category}: {len(frame)}"
    )


# ------------------------------------------------------------
# 5. C13 SCHEMA RESOLUTION
# ------------------------------------------------------------

c13_required = [
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet"
]

c13_schema_errors = {}

for category, frame in category_frames.items():

    missing = [
        col
        for col in c13_required
        if col not in frame.columns
    ]

    if missing:

        c13_schema_errors[
            category
        ] = missing


if c13_schema_errors:

    raise RuntimeError(
        "C17 BLOCKED: C13 schema mismatch: "
        + str(c13_schema_errors)
    )


# ------------------------------------------------------------
# 6. CATEGORY OVERLAP AUDIT
# ------------------------------------------------------------
#
# We do NOT assume that overlap exists or does not exist.
#
# A source occurrence is identified structurally by:
#
#   relative_path
#   page_number
#   search_term
#   occurrence_text
#   context_snippet
#
# Category membership is audited directly from the actual
# category CSVs.
# ------------------------------------------------------------

overlap_frames = []

for category, frame in category_frames.items():

    temp = frame[
        [
            "relative_path",
            "page_number",
            "search_term",
            "occurrence_text",
            "context_snippet"
        ]
    ].copy()

    temp[
        "category"
    ] = category

    overlap_frames.append(
        temp
    )

all_category_records = pd.concat(
    overlap_frames,
    ignore_index=True
)

overlap_key = [
    "relative_path",
    "page_number",
    "search_term",
    "occurrence_text",
    "context_snippet"
]

category_membership = (
    all_category_records[
        overlap_key + ["category"]
    ]
    .drop_duplicates()
    .groupby(
        overlap_key,
        dropna=False
    )["category"]
    .agg(
        lambda x: sorted(
            set(x)
        )
    )
    .reset_index()
)

category_membership[
    "category_count"
] = category_membership[
    "category"
].apply(len)

overlapping_occurrence_count = int(
    (
        category_membership[
            "category_count"
        ] > 1
    ).sum()
)

single_category_occurrence_count = int(
    (
        category_membership[
            "category_count"
        ] == 1
    ).sum()
)

category_overlap_detected = (
    overlapping_occurrence_count > 0
)


# ------------------------------------------------------------
# 7. WELFARE-SPECIFIC AUDIT
# ------------------------------------------------------------

def count_term(frame, term):

    if "search_term" not in frame.columns:
        return 0

    return int(
        frame[
            "search_term"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            term.lower()
        )
        .sum()
    )


welfare_gfl = count_term(
    gfl,
    "welfare"
)

welfare_outcome = count_term(
    outcome,
    "welfare"
)

welfare_cross_category = 0

if welfare_gfl > 0 and welfare_outcome > 0:

    welfare_gfl_rows = gfl[
        gfl[
            "search_term"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("welfare")
    ]

    welfare_outcome_rows = outcome[
        outcome[
            "search_term"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("welfare")
    ]

    gkeys = set(
        tuple(x)
        for x in welfare_gfl_rows[
            overlap_key
        ].astype(str).values
    )

    okeys = set(
        tuple(x)
        for x in welfare_outcome_rows[
            overlap_key
        ].astype(str).values
    )

    welfare_cross_category = len(
        gkeys.intersection(okeys)
    )


# ------------------------------------------------------------
# 8. TOTAL CATEGORY ACCOUNTING
# ------------------------------------------------------------

category_total = sum(
    len(frame)
    for frame in category_frames.values()
)

c13_total_from_status = int(
    c13_status[
        "payload"
    ].get(
        "category_records",
        category_total
    )
)

category_sum_matches_c13 = (
    category_total ==
    c13_total_from_status
)


# ------------------------------------------------------------
# 9. C15 SCHEMA DEPENDENCY AUDIT
# ------------------------------------------------------------

c15 = pd.read_csv(
    BASE /
    "c15_documentary_verification_ledger.csv"
)

c15_required = [
    "verification_queue_id",
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "review_category",
    "source_review_state",
    "documentary_evidence_status",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "review_required",
    "semantic_inference_allowed"
]

c15_missing = [
    col
    for col in c15_required
    if col not in c15.columns
]

c15_schema_verified = (
    len(c15_missing) == 0
)


# ------------------------------------------------------------
# 10. C16 SCHEMA AUDIT
# ------------------------------------------------------------

c16 = pd.read_csv(
    BASE /
    "c16_source_page_evidence_pack.csv"
)

c16_required = [
    "verification_queue_id",
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "review_category",
    "source_path_resolution",
    "source_page_extraction",
    "page_text",
    "documentary_verification",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "semantic_inference_allowed"
]

c16_missing = [
    col
    for col in c16_required
    if col not in c16.columns
]

c16_schema_verified = (
    len(c16_missing) == 0
)


# ------------------------------------------------------------
# 11. C16 ACCOUNTING
# ------------------------------------------------------------

c16_record_count = len(c16)

c15_record_count = len(c15)

c13_record_count = category_total

c13_c15_match = (
    c13_record_count ==
    c15_record_count
)

c15_c16_match = (
    c15_record_count ==
    c16_record_count
)


# ------------------------------------------------------------
# 12. PAGE-TEXT AVAILABILITY
# ------------------------------------------------------------

page_text_available = int(
    c16[
        "page_text"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

page_text_missing = int(
    c16_record_count -
    page_text_available
)


# ------------------------------------------------------------
# 13. COMPONENT FRAMEWORK
# ------------------------------------------------------------
#
# These are REVIEW COMPONENT SLOTS.
#
# They are NOT automatically inferred from keywords.
# Each component starts as:
#
#   NOT_VERIFIED
#
# The matrix preserves documentary provenance and leaves
# semantic verification to source-based review.
# ------------------------------------------------------------

component_definitions = {

    "GFL": [
        "shock_problem",
        "consequence",
        "response_decision",
        "subsequent_behavioral_change",
        "persistence_repeated_correction",
        "temporal_ordering"
    ],

    "IDS": [
        "infrastructure_external_system",
        "decision_reasoning_action",
        "explicit_dependency",
        "dependency_direction",
        "temporal_contextual_link"
    ],

    "AML": [
        "life_preserving_objective",
        "competing_objective",
        "explicit_priority_tradeoff",
        "adaptive_response",
        "contextual_temporal_evidence"
    ]
}


# ------------------------------------------------------------
# 14. COMPONENT MATRIX
# ------------------------------------------------------------

component_records = []

for construct, components in (
    component_definitions.items()
):

    for component in components:

        component_records.append({

            "construct":
                construct,

            "component":
                component,

            "documentary_status":
                "NOT_VERIFIED",

            "source_evidence_required":
                True,

            "source_evidence_found":
                "NOT_ESTABLISHED",

            "semantic_verification":
                "NOT_VERIFIED",

            "construct_mapping":
                "NOT_ESTABLISHED",

            "authorization":
                "NOT_AUTHORIZED",

            "automatic_keyword_authorization":
                False,

            "manual_source_review_required":
                True,

            "semantic_inference_allowed":
                False
        })


component_matrix = pd.DataFrame(
    component_records
)


# ------------------------------------------------------------
# 15. CONSTRUCT-LEVEL COVERAGE
# ------------------------------------------------------------

construct_summary = []

for construct, components in (
    component_definitions.items()
):

    total_components = len(
        components
    )

    verified_components = 0

    construct_summary.append({

        "construct":
            construct,

        "total_components":
            total_components,

        "verified_components":
            verified_components,

        "coverage_percent":
            0.0,

        "construct_status":
            "PENDING_EVIDENCE",

        "authorization":
            "NOT_AUTHORIZED"
    })


construct_summary = pd.DataFrame(
    construct_summary
)


# ------------------------------------------------------------
# 16. OUTCOME VERIFICATION STATE
# ------------------------------------------------------------

outcome_state = {

    "documentary_presence":
        "NOT_VERIFIED",

    "independence":
        "NOT_VERIFIED",

    "temporal_ordering":
        "NOT_VERIFIED",

    "outcome_authorization":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY"
}


# ------------------------------------------------------------
# 17. GLOBAL BOUNDARY CHECK
# ------------------------------------------------------------

boundary_errors = []

if not c15_schema_verified:
    boundary_errors.append(
        "C15_SCHEMA_MISMATCH"
    )

if not c16_schema_verified:
    boundary_errors.append(
        "C16_SCHEMA_MISMATCH"
    )

if not c13_c15_match:
    boundary_errors.append(
        "C13_C15_RECORD_MISMATCH"
    )

if not c15_c16_match:
    boundary_errors.append(
        "C15_C16_RECORD_MISMATCH"
    )

if page_text_missing != 0:
    boundary_errors.append(
        "PAGE_TEXT_MISSING"
    )

if not category_sum_matches_c13:
    boundary_errors.append(
        "C13_CATEGORY_ACCOUNTING_MISMATCH"
    )


# ------------------------------------------------------------
# 18. AUTHORIZATION SAFETY
# ------------------------------------------------------------

authorization_safe = (
    c16[
        "authorization"
    ]
    .astype(str)
    .eq(
        "NOT_AUTHORIZED"
    )
    .all()
)

semantic_safe = (
    c16[
        "semantic_verification"
    ]
    .astype(str)
    .eq(
        "NOT_VERIFIED"
    )
    .all()
)

construct_safe = (
    c16[
        "construct_mapping"
    ]
    .astype(str)
    .eq(
        "NOT_ESTABLISHED"
    )
    .all()
)

inference_safe = (
    c16[
        "semantic_inference_allowed"
    ]
    .astype(str)
    .str.lower()
    .eq(
        "false"
    )
    .all()
)

if not authorization_safe:
    boundary_errors.append(
        "AUTHORIZATION_LEAK"
    )

if not semantic_safe:
    boundary_errors.append(
        "SEMANTIC_VERIFICATION_LEAK"
    )

if not construct_safe:
    boundary_errors.append(
        "CONSTRUCT_MAPPING_LEAK"
    )

if not inference_safe:
    boundary_errors.append(
        "SEMANTIC_INFERENCE_LEAK"
    )


# ------------------------------------------------------------
# 19. IMPORTANT INTERPRETATION LOCK
# ------------------------------------------------------------

interpretation_lock = {

    "keyword_hit_is_semantic_proof":
        False,

    "category_membership_is_construct_proof":
        False,

    "page_text_presence_is_construct_proof":
        False,

    "component_status_auto_verified":
        False,

    "construct_authorization_from_keywords":
        False,

    "outcome_authorization":
        False,

    "score_authorization":
        False,

    "empirical_evaluation_ready":
        False
}


# ------------------------------------------------------------
# 20. FAIL-CLOSED STATUS
# ------------------------------------------------------------

if boundary_errors:

    status = "BLOCKED"

    fail_closed_triggered = True

else:

    status = "PASS"

    fail_closed_triggered = False


# ------------------------------------------------------------
# 21. SAVE CATEGORY OVERLAP AUDIT
# ------------------------------------------------------------

overlap_audit = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C17",

    "category_records":
        int(category_total),

    "category_sum_matches_c13":
        category_sum_matches_c13,

    "single_category_occurrences":
        single_category_occurrence_count,

    "overlapping_occurrences":
        overlapping_occurrence_count,

    "category_overlap_detected":
        category_overlap_detected,

    "welfare_in_gfl":
        welfare_gfl,

    "welfare_in_outcome":
        welfare_outcome,

    "welfare_cross_category_same_occurrence":
        welfare_cross_category,

    "deduplication_performed":
        False,

    "semantic_interpretation":
        "NOT_PERFORMED",

    "authorization":
        "NOT_AUTHORIZED"
}

overlap_path = (
    BASE /
    "c17_category_overlap_audit.json"
)

save_json(
    overlap_path,
    overlap_audit
)


# ------------------------------------------------------------
# 22. SAVE COMPONENT MATRIX
# ------------------------------------------------------------

component_path = (
    BASE /
    "c17_documentary_component_matrix.csv"
)

component_matrix.to_csv(
    component_path,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 23. SAVE CONSTRUCT SUMMARY
# ------------------------------------------------------------

construct_path = (
    BASE /
    "c17_construct_coverage_summary.csv"
)

construct_summary.to_csv(
    construct_path,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 24. SUMMARY
# ------------------------------------------------------------

summary = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C17",

    "cell_name":
        "DOCUMENTARY COMPONENT VERIFICATION MATRIX",

    "upstream": {

        "C13":
            c13_status.get("status"),

        "C15":
            c15_status.get("status"),

        "C16":
            c16_status.get("status")
    },

    "records": {

        "C13":
            int(c13_record_count),

        "C15":
            int(c15_record_count),

        "C16":
            int(c16_record_count),

        "C13_C15_match":
            c13_c15_match,

        "C15_C16_match":
            c15_c16_match
    },

    "schema": {

        "C15_verified":
            c15_schema_verified,

        "C15_missing":
            c15_missing,

        "C16_verified":
            c16_schema_verified,

        "C16_missing":
            c16_missing
    },

    "category_overlap":

        overlap_audit,

    "source_page_text": {

        "available_records":
            page_text_available,

        "missing_records":
            page_text_missing
    },

    "construct_component_matrix": {

        "GFL":
            6,

        "IDS":
            5,

        "AML":
            5,

        "total_components":
            16,

        "verified_components":
            0
    },

    "construct_status": {

        "GFL":
            "PENDING_EVIDENCE",

        "IDS":
            "PENDING_EVIDENCE",

        "AML":
            "PENDING_EVIDENCE"
    },

    "outcome": outcome_state,

    "interpretation_lock":
        interpretation_lock,

    "boundary_errors":
        boundary_errors,

    "authorization": {

        "construct":
            "NOT_AUTHORIZED",

        "predictor":
            "NOT_AUTHORIZED",

        "outcome":
            "NOT_AUTHORIZED",

        "key":
            "NOT_AUTHORIZED",

        "relationship":
            "NOT_AUTHORIZED",

        "cohort":
            "NOT_AUTHORIZED",

        "score":
            "NOT_AUTHORIZED",

        "empirical_evaluation":
            "NOT_READY"
    },

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered
}


summary_path = (
    BASE /
    "c17_documentary_component_summary.json"
)

save_json(
    summary_path,
    summary
)


# ------------------------------------------------------------
# 25. FINAL STATUS
# ------------------------------------------------------------

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C17",

    "cell_name":
        "DOCUMENTARY COMPONENT VERIFICATION MATRIX",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization": {

        "construct":
            "NOT_AUTHORIZED",

        "predictor":
            "NOT_AUTHORIZED",

        "outcome":
            "NOT_AUTHORIZED",

        "key":
            "NOT_AUTHORIZED",

        "relationship":
            "NOT_AUTHORIZED",

        "cohort":
            "NOT_AUTHORIZED",

        "score":
            "NOT_AUTHORIZED",

        "empirical_evaluation":
            "NOT_READY"
    },

    "payload":
        summary
}

final_status_path = (
    BASE /
    "c17_final_status.json"
)

save_json(
    final_status_path,
    final_status
)


# ------------------------------------------------------------
# 26. OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C17 RESULTS")
print("=" * 72)

print(
    "C13 records:",
    c13_record_count
)

print(
    "C15 records:",
    c15_record_count
)

print(
    "C16 records:",
    c16_record_count
)

print(
    "C13 → C15 match:",
    c13_c15_match
)

print(
    "C15 → C16 match:",
    c15_c16_match
)

print("\nSCHEMA")

print(
    "C15 schema verified:",
    c15_schema_verified
)

print(
    "C16 schema verified:",
    c16_schema_verified
)

print("\nCATEGORY OVERLAP")

print(
    "Single-category occurrences:",
    single_category_occurrence_count
)

print(
    "Overlapping occurrences:",
    overlapping_occurrence_count
)

print(
    "Category overlap detected:",
    category_overlap_detected
)

print(
    "Welfare in GFL:",
    welfare_gfl
)

print(
    "Welfare in OUTCOME:",
    welfare_outcome
)

print(
    "Welfare same-occurrence overlap:",
    welfare_cross_category
)

print("\nSOURCE PAGE TEXT")

print(
    "Available:",
    page_text_available
)

print(
    "Missing:",
    page_text_missing
)

print("\nCOMPONENT MATRIX")

print(
    "GFL components:",
    6
)

print(
    "IDS components:",
    5
)

print(
    "AML components:",
    5
)

print(
    "Total components:",
    16
)

print(
    "Automatically verified:",
    0
)

print("\nCONSTRUCT STATUS")

print(
    "GFL:",
    "PENDING_EVIDENCE"
)

print(
    "IDS:",
    "PENDING_EVIDENCE"
)

print(
    "AML:",
    "PENDING_EVIDENCE"
)

print("\nBOUNDARY")

print(
    "Keyword = semantic proof:",
    False
)

print(
    "Page text = construct proof:",
    False
)

print(
    "Automatic component verification:",
    False
)

print(
    "Construct authorization:",
    "NOT_AUTHORIZED"
)

print(
    "Outcome authorization:",
    "NOT_AUTHORIZED"
)

print(
    "Empirical evaluation:",
    "NOT_READY"
)

print("\nFAIL-CLOSED")

print(
    "Fail conditions:",
    boundary_errors
)

print(
    "Triggered:",
    fail_closed_triggered
)

print(
    "\nC17 STATUS:",
    status
)

print("\nARTIFACTS")

print(overlap_path)
print(component_path)
print(construct_path)
print(summary_path)
print(final_status_path)

print("=" * 72)

OIP v1.0.33 — C17
DOCUMENTARY COMPONENT VERIFICATION MATRIX

UPSTREAM STATUS
C13: PASS
C15: PASS
C16: PASS

C13 CATEGORY RECORDS
GFL: 371
IDS: 524
AML: 20
OUTCOME: 531

C17 RESULTS
C13 records: 1446
C15 records: 1446
C16 records: 1446
C13 → C15 match: True
C15 → C16 match: True

SCHEMA
C15 schema verified: True
C16 schema verified: True

CATEGORY OVERLAP
Single-category occurrences: 1412
Overlapping occurrences: 17
Category overlap detected: True
Welfare in GFL: 17
Welfare in OUTCOME: 17
Welfare same-occurrence overlap: 17

SOURCE PAGE TEXT
Available: 1446
Missing: 0

COMPONENT MATRIX
GFL components: 6
IDS components: 5
AML components: 5
Total components: 16
Automatically verified: 0

CONSTRUCT STATUS
GFL: PENDING_EVIDENCE
IDS: PENDING_EVIDENCE
AML: PENDING_EVIDENCE

BOUNDARY
Keyword = semantic proof: False
Page text = construct proof: False
Automatic component verification: False
Construct authorization: NOT_AUTHORIZED
Outcome authorization: NOT_AUTHORIZED
Empirical evaluation: NOT_RE

In [35]:
# ============================================================
# OIP v1.0.33 — C18
# SOURCE-LOCKED DOCUMENTARY CONCEPT COVERAGE AUDIT
# ============================================================

import json
import re
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

BASE = Path("/kaggle/working/oip_v1_0_33")

C13_STATUS = BASE / "c13_final_status.json"
C15_STATUS = BASE / "c15_final_status.json"
C16_STATUS = BASE / "c16_final_status.json"

C16_EVIDENCE = BASE / "c16_source_page_evidence_pack.csv"

OUT_MATRIX = BASE / "c18_source_locked_concept_coverage.csv"
OUT_SUMMARY = BASE / "c18_construct_coverage_summary.json"
OUT_FINAL = BASE / "c18_final_status.json"


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def fail_closed_authorization():
    return {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY",
    }


def normalized_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower()


def phrase_present(text, phrase):
    """
    Literal case-insensitive phrase/term discovery only.

    No stemming
    No synonym expansion
    No fuzzy matching
    No semantic inference
    """
    return phrase.lower() in text


# ------------------------------------------------------------
# 2. LOAD UPSTREAM STATUS
# ------------------------------------------------------------

c13 = load_json(C13_STATUS)
c15 = load_json(C15_STATUS)
c16 = load_json(C16_STATUS)

c13_status = c13.get("status")
c15_status = c15.get("status")
c16_status = c16.get("status")

print("=" * 72)
print("OIP v1.0.33 — C18")
print("SOURCE-LOCKED DOCUMENTARY CONCEPT COVERAGE AUDIT")
print("=" * 72)

print()
print("UPSTREAM STATUS")
print("C13:", c13_status)
print("C15:", c15_status)
print("C16:", c16_status)


# ------------------------------------------------------------
# 3. UPSTREAM GATE
# ------------------------------------------------------------

upstream_fail = []

if c13_status != "PASS":
    upstream_fail.append("C13_NOT_PASS")

if c15_status != "PASS":
    upstream_fail.append("C15_NOT_PASS")

if c16_status != "PASS":
    upstream_fail.append("C16_NOT_PASS")


# ------------------------------------------------------------
# 4. LOAD C16 EVIDENCE
# ------------------------------------------------------------

required_c16_columns = [
    "verification_queue_id",
    "relative_path",
    "file_name",
    "page_number",
    "search_family",
    "search_term",
    "occurrence_text",
    "context_snippet",
    "review_category",
    "source_path_resolution",
    "source_page_extraction",
    "page_text",
    "documentary_verification",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "semantic_inference_allowed",
]


c16_df = pd.DataFrame()

if C16_EVIDENCE.exists():
    c16_df = pd.read_csv(C16_EVIDENCE)


schema_missing = [
    c for c in required_c16_columns
    if c not in c16_df.columns
]

schema_verified = len(schema_missing) == 0

print()
print("C16 EVIDENCE")
print("Records:", len(c16_df))
print("Schema verified:", schema_verified)

if schema_missing:
    print("Missing columns:", schema_missing)


# ------------------------------------------------------------
# 5. SOURCE-LOCKED OIP CONCEPT DEFINITIONS
# ------------------------------------------------------------
#
# IMPORTANT:
#
# These are NOT new OIP components.
# They are documentary search concepts derived directly
# from the source-locked construct definitions.
#
# Literal discovery only.
# No synonym expansion.
# No semantic interpretation.
# ------------------------------------------------------------

source_locked_concepts = {

    "GFL": {
        "source_definition":
            "Grounded Feedback Loop: Real-world outcome correction.",

        "concepts": {

            "real_world":
                [
                    "real-world",
                    "real world",
                ],

            "outcome":
                [
                    "outcome",
                    "physical outcomes",
                    "real-world consequences",
                ],

            "error_correction":
                [
                    "error correction",
                    "error_correction",
                    "correction",
                    "outcome correction",
                ],

            "direct_feedback":
                [
                    "direct feedback",
                    "outcome feedback",
                    "feedback",
                ],

            "behavioral_update":
                [
                    "behavioral update",
                    "behavioral_update",
                ],

            "decision_testing":
                [
                    "decisions cannot be tested",
                    "tested against physical outcomes",
                    "test against physical outcomes",
                ],
        },
    },

    "IDS": {
        "source_definition":
            "Infrastructure Dependency Score: Degree to which reasoning requires external systems.",

        "concepts": {

            "reasoning":
                [
                    "reasoning",
                ],

            "requires":
                [
                    "requires",
                    "require",
                ],

            "external_systems":
                [
                    "external systems",
                    "external technological systems",
                ],

            "infrastructure_dependency":
                [
                    "infrastructure dependency",
                    "dependency",
                ],

            "function_without_external_systems":
                [
                    "function without external technological systems",
                    "ability to function without external technological systems",
                ],
        },
    },

    "AML": {
        "source_definition":
            "Adaptive Moral Logic: Life-preserving prioritization.",

        "concepts": {

            "life_preservation":
                [
                    "life-preserving",
                    "life preserving",
                    "life preservation",
                ],

            "ecosystem_stabilization":
                [
                    "ecosystem-stabilizing",
                    "ecosystem stabilizing",
                ],

            "prestige":
                [
                    "prestige",
                ],

            "profit":
                [
                    "profit",
                ],

            "abstract_efficiency":
                [
                    "abstract efficiency",
                ],

            "survival_and_resource_collapse":
                [
                    "system survival probability",
                    "resource collapse risk",
                ],

            "moral_priority":
                [
                    "moral priority",
                ],
        },
    },
}


# ------------------------------------------------------------
# 6. SOURCE-LOCKED SEARCH
# ------------------------------------------------------------

records = []

if schema_verified and len(c16_df) > 0:

    working = c16_df.copy()

    working["__page_text__"] = (
        working["page_text"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    for construct, construct_info in source_locked_concepts.items():

        source_definition = construct_info["source_definition"]

        for concept, search_terms in construct_info["concepts"].items():

            for term in search_terms:

                mask = working["__page_text__"].apply(
                    lambda x: phrase_present(x, term)
                )

                matched = working.loc[mask].copy()

                for _, row in matched.iterrows():

                    records.append({
                        "verification_queue_id":
                            row["verification_queue_id"],

                        "relative_path":
                            row["relative_path"],

                        "file_name":
                            row["file_name"],

                        "page_number":
                            row["page_number"],

                        "review_category":
                            row["review_category"],

                        "construct":
                            construct,

                        "source_locked_concept":
                            concept,

                        "source_definition":
                            source_definition,

                        "source_search_term":
                            term,

                        "page_text_match":
                            "OBSERVED",

                        "documentary_evidence_status":
                            "CANDIDATE_EVIDENCE_PRESENT",

                        "semantic_verification":
                            "NOT_VERIFIED",

                        "construct_mapping":
                            "NOT_ESTABLISHED",

                        "authorization":
                            "NOT_AUTHORIZED",

                        "discovery_only":
                            True,

                        "semantic_inference_allowed":
                            False,
                    })


coverage_df = pd.DataFrame(records)


# ------------------------------------------------------------
# 7. EMPTY RESULT SAFETY
# ------------------------------------------------------------

if coverage_df.empty:

    coverage_df = pd.DataFrame(columns=[
        "verification_queue_id",
        "relative_path",
        "file_name",
        "page_number",
        "review_category",
        "construct",
        "source_locked_concept",
        "source_definition",
        "source_search_term",
        "page_text_match",
        "documentary_evidence_status",
        "semantic_verification",
        "construct_mapping",
        "authorization",
        "discovery_only",
        "semantic_inference_allowed",
    ])


# ------------------------------------------------------------
# 8. STRUCTURAL DUPLICATE AUDIT
# ------------------------------------------------------------

coverage_key = [
    "verification_queue_id",
    "construct",
    "source_locked_concept",
    "source_search_term",
]

if len(coverage_df) > 0:
    duplicate_rows = int(
        coverage_df.duplicated(
            subset=coverage_key,
            keep=False
        ).sum()
    )
else:
    duplicate_rows = 0


# ------------------------------------------------------------
# 9. CONSTRUCT COVERAGE SUMMARY
# ------------------------------------------------------------

summary_rows = []

for construct, construct_info in source_locked_concepts.items():

    concepts = list(construct_info["concepts"].keys())

    construct_df = coverage_df[
        coverage_df["construct"] == construct
    ]

    concept_rows = []

    for concept in concepts:

        concept_df = construct_df[
            construct_df["source_locked_concept"] == concept
        ]

        hit_count = len(concept_df)

        unique_pages = (
            concept_df[
                ["relative_path", "page_number"]
            ]
            .drop_duplicates()
            .shape[0]
        )

        unique_documents = (
            concept_df["relative_path"]
            .nunique()
            if len(concept_df) > 0
            else 0
        )

        if hit_count > 0:
            discovery_state = "CANDIDATE_EVIDENCE_PRESENT"
        else:
            discovery_state = "NOT_IDENTIFIED_BY_LITERAL_DISCOVERY"

        concept_rows.append({
            "concept": concept,
            "hit_count": hit_count,
            "unique_pages": unique_pages,
            "unique_documents": unique_documents,
            "discovery_state": discovery_state,
        })

    concepts_with_hits = sum(
        1
        for x in concept_rows
        if x["hit_count"] > 0
    )

    total_concepts = len(concepts)

    if concepts_with_hits == 0:
        construct_discovery_state = "NOT_IDENTIFIED_BY_LITERAL_DISCOVERY"
    else:
        construct_discovery_state = "CANDIDATE_EVIDENCE_PRESENT"

    summary_rows.append({
        "construct": construct,
        "source_definition":
            construct_info["source_definition"],
        "concept_count":
            total_concepts,
        "concepts_with_literal_hits":
            concepts_with_hits,
        "coverage_ratio":
            (
                concepts_with_hits / total_concepts
                if total_concepts > 0
                else 0.0
            ),
        "discovery_state":
            construct_discovery_state,
        "documentary_verification":
            "NOT_VERIFIED",
        "semantic_verification":
            "NOT_VERIFIED",
        "construct_mapping":
            "NOT_ESTABLISHED",
        "authorization":
            "NOT_AUTHORIZED",
        "concept_details":
            concept_rows,
    })


# ------------------------------------------------------------
# 10. SOURCE PAGE TEXT AUDIT
# ------------------------------------------------------------

if len(c16_df) > 0:

    page_text_available = int(
        c16_df["page_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    page_text_missing = (
        len(c16_df) - page_text_available
    )

else:

    page_text_available = 0
    page_text_missing = 0


# ------------------------------------------------------------
# 11. BOUNDARY AUDIT
# ------------------------------------------------------------

boundary_violations = []

if len(coverage_df) > 0:

    if (
        coverage_df["semantic_verification"]
        .astype(str)
        .ne("NOT_VERIFIED")
        .any()
    ):
        boundary_violations.append(
            "SEMANTIC_VERIFICATION_LEAK"
        )

    if (
        coverage_df["construct_mapping"]
        .astype(str)
        .ne("NOT_ESTABLISHED")
        .any()
    ):
        boundary_violations.append(
            "CONSTRUCT_MAPPING_LEAK"
        )

    if (
        coverage_df["authorization"]
        .astype(str)
        .ne("NOT_AUTHORIZED")
        .any()
    ):
        boundary_violations.append(
            "AUTHORIZATION_LEAK"
        )

    if (
        coverage_df["semantic_inference_allowed"]
        .astype(bool)
        .any()
    ):
        boundary_violations.append(
            "SEMANTIC_INFERENCE_LEAK"
        )

    if (
        coverage_df["discovery_only"]
        .astype(bool)
        .eq(False)
        .any()
    ):
        boundary_violations.append(
            "DISCOVERY_ONLY_VIOLATION"
        )


# ------------------------------------------------------------
# 12. FAIL-CLOSED CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

fail_conditions.extend(upstream_fail)

if not schema_verified:
    fail_conditions.append("C16_SCHEMA_INVALID")

if len(c16_df) == 0:
    fail_conditions.append("C16_EVIDENCE_EMPTY")

if page_text_missing > 0:
    fail_conditions.append("PAGE_TEXT_MISSING")

if duplicate_rows > 0:
    fail_conditions.append("DUPLICATE_COVERAGE_RECORDS")

fail_conditions.extend(boundary_violations)


# ------------------------------------------------------------
# 13. STATUS
# ------------------------------------------------------------

status = "PASS" if len(fail_conditions) == 0 else "BLOCKED"

fail_closed_triggered = len(fail_conditions) > 0


# ------------------------------------------------------------
# 14. SAVE COVERAGE MATRIX
# ------------------------------------------------------------

coverage_df.to_csv(
    OUT_MATRIX,
    index=False
)


# ------------------------------------------------------------
# 15. SAVE CONSTRUCT SUMMARY
# ------------------------------------------------------------

summary_payload = {
    "oip_version": "v1.0.33",
    "cell": "C18",
    "cell_name":
        "SOURCE-LOCKED DOCUMENTARY CONCEPT COVERAGE AUDIT",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "upstream_status": {
        "C13": c13_status,
        "C15": c15_status,
        "C16": c16_status,
    },

    "source_locked_constructs": summary_rows,

    "evidence_records":
        int(len(coverage_df)),

    "unique_queue_records":
        (
            int(
                coverage_df["verification_queue_id"]
                .nunique()
            )
            if len(coverage_df) > 0
            else 0
        ),

    "duplicate_coverage_records":
        duplicate_rows,

    "page_text": {
        "available": page_text_available,
        "missing": page_text_missing,
    },

    "semantic_verification":
        "NOT_VERIFIED",

    "construct_mapping":
        "NOT_ESTABLISHED",

    "authorization":
        "NOT_AUTHORIZED",

    "discovery_only":
        True,

    "semantic_inference_allowed":
        False,

    "automatic_authorization":
        False,

    "fail_closed": {
        "active": True,
        "triggered": fail_closed_triggered,
        "fail_conditions": fail_conditions,
    },
}


with open(
    OUT_SUMMARY,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary_payload,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 16. FINAL STATUS WRAPPER
# ------------------------------------------------------------

final_payload = {
    "oip_version": "v1.0.33",
    "cell": "C18",
    "cell_name":
        "SOURCE-LOCKED DOCUMENTARY CONCEPT COVERAGE AUDIT",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        fail_closed_triggered,

    "authorization":
        fail_closed_authorization(),

    "payload": {
        "upstream_c13_status":
            c13_status,

        "upstream_c15_status":
            c15_status,

        "upstream_c16_status":
            c16_status,

        "c16_records":
            int(len(c16_df)),

        "c16_schema_verified":
            schema_verified,

        "c16_schema_missing":
            schema_missing,

        "coverage_records":
            int(len(coverage_df)),

        "duplicate_coverage_records":
            duplicate_rows,

        "page_text_available":
            page_text_available,

        "page_text_missing":
            page_text_missing,

        "constructs_analyzed":
            list(source_locked_concepts.keys()),

        "source_locked_discovery_only":
            True,

        "semantic_verification":
            "NOT_VERIFIED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "automatic_authorization":
            False,

        "semantic_inference_allowed":
            False,

        "boundary_violations":
            boundary_violations,

        "fail_conditions":
            fail_conditions,
    },
}


with open(
    OUT_FINAL,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_payload,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 17. CONSOLE OUTPUT
# ------------------------------------------------------------

print()
print("=" * 72)
print("C18 RESULTS")
print("=" * 72)

print("C16 records:", len(c16_df))
print("C16 schema verified:", schema_verified)

print()
print("SOURCE-LOCKED CONCEPT DISCOVERY")
print("Coverage records:", len(coverage_df))
print("Duplicate coverage records:", duplicate_rows)

print()
print("PAGE TEXT")
print("Available:", page_text_available)
print("Missing:", page_text_missing)

print()
print("CONSTRUCT DISCOVERY")

for row in summary_rows:
    print(
        f"{row['construct']}: "
        f"{row['concepts_with_literal_hits']}/"
        f"{row['concept_count']} concepts with literal hits | "
        f"{row['discovery_state']}"
    )

print()
print("BOUNDARY")
print("Semantic verification:", "NOT_VERIFIED")
print("Construct mapping:", "NOT_ESTABLISHED")
print("Authorization:", "NOT_AUTHORIZED")
print("Automatic authorization:", False)
print("Semantic inference:", False)

print()
print("FAIL-CLOSED")
print("Fail conditions:", fail_conditions)
print("Triggered:", fail_closed_triggered)

print()
print("C18 STATUS:", status)

print()
print("ARTIFACTS")
print(OUT_MATRIX)
print(OUT_SUMMARY)
print(OUT_FINAL)

print("=" * 72)

OIP v1.0.33 — C18
SOURCE-LOCKED DOCUMENTARY CONCEPT COVERAGE AUDIT

UPSTREAM STATUS
C13: PASS
C15: PASS
C16: PASS

C16 EVIDENCE
Records: 1446
Schema verified: True

C18 RESULTS
C16 records: 1446
C16 schema verified: True

SOURCE-LOCKED CONCEPT DISCOVERY
Coverage records: 133
Duplicate coverage records: 0

PAGE TEXT
Available: 1446
Missing: 0

CONSTRUCT DISCOVERY
GFL: 3/6 concepts with literal hits | CANDIDATE_EVIDENCE_PRESENT
IDS: 2/5 concepts with literal hits | CANDIDATE_EVIDENCE_PRESENT
AML: 1/7 concepts with literal hits | CANDIDATE_EVIDENCE_PRESENT

BOUNDARY
Semantic verification: NOT_VERIFIED
Construct mapping: NOT_ESTABLISHED
Authorization: NOT_AUTHORIZED
Automatic authorization: False
Semantic inference: False

FAIL-CLOSED
Fail conditions: []
Triggered: False

C18 STATUS: PASS

ARTIFACTS
/kaggle/working/oip_v1_0_33/c18_source_locked_concept_coverage.csv
/kaggle/working/oip_v1_0_33/c18_construct_coverage_summary.json
/kaggle/working/oip_v1_0_33/c18_final_status.json


In [36]:
# ============================================================
# OIP v1.0.33 — C18R
# C18 CONSISTENCY AND VOCABULARY LOCK
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone


BASE = Path("/kaggle/working/oip_v1_0_33")

C17_SUMMARY = BASE / "c17_documentary_component_summary.json"
C18_SUMMARY = BASE / "c18_construct_coverage_summary.json"
C18_STATUS = BASE / "c18_final_status.json"

OUT_AUDIT = BASE / "c18r_consistency_audit.json"
OUT_FINAL = BASE / "c18r_final_status.json"


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def auth_state():
    return {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY",
    }


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

c17 = load_json(C17_SUMMARY)
c18 = load_json(C18_SUMMARY)
c18_status = load_json(C18_STATUS)


print("=" * 72)
print("OIP v1.0.33 — C18R")
print("C18 CONSISTENCY AND VOCABULARY LOCK")
print("=" * 72)


# ------------------------------------------------------------
# 3. EXTRACT C17 INFORMATION
# ------------------------------------------------------------

c17_payload = c17.get("payload", {})

c17_component_counts = {
    "GFL": c17_payload.get("gfl_components"),
    "IDS": c17_payload.get("ids_components"),
    "AML": c17_payload.get("aml_components"),
}

# C17 may store the counts under another structure.
# Inspect recursively without assuming a semantic source.
if not all(v is not None for v in c17_component_counts.values()):

    for key, value in c17.items():

        if isinstance(value, dict):

            for k, v in value.items():

                lk = str(k).lower()

                if lk == "gfl_components":
                    c17_component_counts["GFL"] = v

                elif lk == "ids_components":
                    c17_component_counts["IDS"] = v

                elif lk == "aml_components":
                    c17_component_counts["AML"] = v


# ------------------------------------------------------------
# 4. EXTRACT C18 INFORMATION
# ------------------------------------------------------------

c18_constructs = c18.get("source_locked_constructs", [])

if not c18_constructs:
    c18_constructs = c18.get("payload", {}).get(
        "source_locked_constructs", []
    )

c18_component_counts = {}

for item in c18_constructs:

    if isinstance(item, dict):

        construct = item.get("construct")

        if construct:

            c18_component_counts[construct] = item.get(
                "concept_count"
            )


# ------------------------------------------------------------
# 5. PRINT COUNTS
# ------------------------------------------------------------

print()
print("COMPONENT / CONCEPT COUNT RECONCILIATION")
print()

for construct in ["GFL", "IDS", "AML"]:

    print(
        construct,
        "| C17:",
        c17_component_counts.get(construct),
        "| C18:",
        c18_component_counts.get(construct)
    )


# ------------------------------------------------------------
# 6. INTERNAL CONSISTENCY CHECK
# ------------------------------------------------------------

count_mismatches = []

for construct in ["GFL", "IDS", "AML"]:

    c17_count = c17_component_counts.get(construct)
    c18_count = c18_component_counts.get(construct)

    if (
        c17_count is not None
        and c18_count is not None
        and c17_count != c18_count
    ):

        count_mismatches.append({
            "construct": construct,
            "c17_count": c17_count,
            "c18_count": c18_count,
        })


# ------------------------------------------------------------
# 7. IMPORTANT SOURCE BOUNDARY
# ------------------------------------------------------------
#
# Neither C17 nor C18 is allowed to establish that its
# component/concept list is an authoritative OIP definition.
#
# Therefore:
#
# C17 count != C18 count
# does NOT mean one of them is scientifically correct.
#
# It means the review schema is inconsistent.
# ------------------------------------------------------------

component_schema_consistent = (
    len(count_mismatches) == 0
)


# ------------------------------------------------------------
# 8. VOCABULARY AUDIT
# ------------------------------------------------------------

# C04 canonical vocabulary states established in the
# current OIP v1.0.33 pipeline.

canonical_evidence_states = {
    "OBSERVED",
    "DOCUMENTED",
    "NOT_DOCUMENTED",
    "VERIFIED",
    "NOT_VERIFIED",
    "CANDIDATE",
    "NOT_ESTABLISHED",
}

canonical_authorization_states = {
    "AUTHORIZED",
    "NOT_AUTHORIZED",
}

canonical_readiness_states = {
    "READY",
    "NOT_READY",
}


noncanonical_terms = {
    "CANDIDATE_EVIDENCE_PRESENT"
}


vocabulary_issue_detected = False

c18_text = json.dumps(
    c18,
    ensure_ascii=False
)

for term in noncanonical_terms:

    if term in c18_text:
        vocabulary_issue_detected = True


# ------------------------------------------------------------
# 9. TERMINOLOGY REPAIR RULE
# ------------------------------------------------------------

#
# C18 literal search results are discovery observations.
#
# They must NOT create a new authorization/evidence state.
#
# Therefore:
#
# CANDIDATE_EVIDENCE_PRESENT
#        ↓
# CANDIDATE
#
# with the separate field:
#
# discovery_basis = LITERAL_PAGE_TEXT_MATCH
#
# ------------------------------------------------------------

terminology_replacement = {
    "CANDIDATE_EVIDENCE_PRESENT":
        "CANDIDATE"
}


# ------------------------------------------------------------
# 10. SEMANTIC BOUNDARY
# ------------------------------------------------------------

semantic_boundary = {
    "literal_hit_equals_semantic_proof": False,
    "literal_hit_equals_documentary_verification": False,
    "page_text_equals_construct_proof": False,
    "component_hit_equals_construct_authorization": False,
    "automatic_construct_authorization": False,
    "automatic_outcome_authorization": False,
    "score_authorization": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY",
}


# ------------------------------------------------------------
# 11. QUALIFICATION BOUNDARY
# ------------------------------------------------------------

#
# C18/C18R cannot declare:
#
# ESS4 QUALIFICATION FAILED
#
# solely from literal coverage.
#
# It can only establish that documentary semantic
# verification has not yet been completed.
# ------------------------------------------------------------

qualification_statement = (
    "DOCUMENTARY_CONCEPT_DISCOVERY_COMPLETED; "
    "SEMANTIC_CONSTRUCT_VERIFICATION_NOT_ESTABLISHED"
)


# ------------------------------------------------------------
# 12. FAIL CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if not C17_SUMMARY.exists():
    fail_conditions.append("C17_SUMMARY_MISSING")

if not C18_SUMMARY.exists():
    fail_conditions.append("C18_SUMMARY_MISSING")

if not C18_STATUS.exists():
    fail_conditions.append("C18_STATUS_MISSING")

if len(count_mismatches) > 0:
    fail_conditions.append(
        "C17_C18_COMPONENT_SCHEMA_MISMATCH"
    )


# Vocabulary issue is recorded as a repair item,
# not as semantic evidence failure.

repair_items = []

if vocabulary_issue_detected:
    repair_items.append(
        "REPLACE_NONCANONICAL_CANDIDATE_EVIDENCE_PRESENT_WITH_CANDIDATE"
    )


# ------------------------------------------------------------
# 13. STATUS
# ------------------------------------------------------------

#
# Because the review schema is inconsistent, C18 cannot
# be promoted as a clean canonical component-coverage
# result until repaired.
#

if len(fail_conditions) == 0:
    status = "PASS"
    triggered = False
else:
    status = "BLOCKED"
    triggered = True


# ------------------------------------------------------------
# 14. SAVE AUDIT
# ------------------------------------------------------------

audit_payload = {

    "oip_version": "v1.0.33",

    "cell": "C18R",

    "cell_name":
        "C18 CONSISTENCY AND VOCABULARY LOCK",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "c17_component_counts":
        c17_component_counts,

    "c18_component_counts":
        c18_component_counts,

    "count_mismatches":
        count_mismatches,

    "component_schema_consistent":
        component_schema_consistent,

    "noncanonical_terms_detected":
        sorted(list(noncanonical_terms))
        if vocabulary_issue_detected
        else [],

    "terminology_replacement":
        terminology_replacement,

    "semantic_boundary":
        semantic_boundary,

    "qualification_statement":
        qualification_statement,

    "repair_items":
        repair_items,

    "authorization":
        auth_state(),

    "fail_closed": {
        "active": True,
        "triggered": triggered,
        "fail_conditions": fail_conditions,
    },
}


with open(
    OUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_payload,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 15. FINAL STATUS
# ------------------------------------------------------------

final_payload = {

    "oip_version": "v1.0.33",

    "cell": "C18R",

    "cell_name":
        "C18 CONSISTENCY AND VOCABULARY LOCK",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        triggered,

    "authorization":
        auth_state(),

    "payload": audit_payload,
}


with open(
    OUT_FINAL,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_payload,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 16. OUTPUT
# ------------------------------------------------------------

print()
print("=" * 72)
print("C18R RESULTS")
print("=" * 72)

print(
    "C17 GFL:",
    c17_component_counts.get("GFL")
)

print(
    "C17 IDS:",
    c17_component_counts.get("IDS")
)

print(
    "C17 AML:",
    c17_component_counts.get("AML")
)

print()

print(
    "C18 GFL:",
    c18_component_counts.get("GFL")
)

print(
    "C18 IDS:",
    c18_component_counts.get("IDS")
)

print(
    "C18 AML:",
    c18_component_counts.get("AML")
)

print()
print("Schema consistent:", component_schema_consistent)

print(
    "Count mismatches:",
    len(count_mismatches)
)

print(
    "Noncanonical terminology detected:",
    vocabulary_issue_detected
)

print()
print("SEMANTIC BOUNDARY")
print("Literal hit = semantic proof:", False)
print("Literal hit = documentary verification:", False)
print("Page text = construct proof:", False)
print("Automatic authorization:", False)

print()
print("QUALIFICATION BOUNDARY")
print(qualification_statement)

print()
print("FAIL-CLOSED")
print("Fail conditions:", fail_conditions)
print("Triggered:", triggered)

print()
print("C18R STATUS:", status)

print()
print("ARTIFACTS")
print(OUT_AUDIT)
print(OUT_FINAL)

print("=" * 72)

OIP v1.0.33 — C18R
C18 CONSISTENCY AND VOCABULARY LOCK

COMPONENT / CONCEPT COUNT RECONCILIATION

GFL | C17: None | C18: 6
IDS | C17: None | C18: 5
AML | C17: None | C18: 7

C18R RESULTS
C17 GFL: None
C17 IDS: None
C17 AML: None

C18 GFL: 6
C18 IDS: 5
C18 AML: 7

Schema consistent: True
Count mismatches: 0
Noncanonical terminology detected: True

SEMANTIC BOUNDARY
Literal hit = semantic proof: False
Literal hit = documentary verification: False
Page text = construct proof: False
Automatic authorization: False

QUALIFICATION BOUNDARY
DOCUMENTARY_CONCEPT_DISCOVERY_COMPLETED; SEMANTIC_CONSTRUCT_VERIFICATION_NOT_ESTABLISHED

FAIL-CLOSED
Fail conditions: []
Triggered: False

C18R STATUS: PASS

ARTIFACTS
/kaggle/working/oip_v1_0_33/c18r_consistency_audit.json
/kaggle/working/oip_v1_0_33/c18r_final_status.json


In [37]:
# ============================================================
# OIP v1.0.33 — C19
# C17/C18 COMPONENT SCHEMA RECONCILIATION
# ============================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone


BASE = Path("/kaggle/working/oip_v1_0_33")

C17_MATRIX = BASE / "c17_documentary_component_matrix.csv"
C17_SUMMARY = BASE / "c17_documentary_component_summary.json"
C18_COVERAGE = BASE / "c18_source_locked_concept_coverage.csv"
C18_SUMMARY = BASE / "c18_construct_coverage_summary.json"
C18_STATUS = BASE / "c18_final_status.json"
C18R_STATUS = BASE / "c18r_final_status.json"

OUT_AUDIT = BASE / "c19_component_schema_reconciliation.json"
OUT_FINAL = BASE / "c19_final_status.json"


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_payload(obj):
    return obj.get("payload", {})


def auth_state():
    return {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY",
    }


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

c17_summary = load_json(C17_SUMMARY)
c18_summary = load_json(C18_SUMMARY)
c18_status = load_json(C18_STATUS)
c18r_status = load_json(C18R_STATUS)

c17_matrix = pd.read_csv(C17_MATRIX)
c18_coverage = pd.read_csv(C18_COVERAGE)


print("=" * 72)
print("OIP v1.0.33 — C19")
print("C17/C18 COMPONENT SCHEMA RECONCILIATION")
print("=" * 72)


# ------------------------------------------------------------
# 3. UPSTREAM STATUS
# ------------------------------------------------------------

print()
print("UPSTREAM STATUS")
print("C17:", c17_summary.get("status"))
print("C18:", c18_status.get("status"))
print("C18R:", c18r_status.get("status"))


# ------------------------------------------------------------
# 4. C17 MATRIX SCHEMA
# ------------------------------------------------------------

print()
print("C17 MATRIX SCHEMA")
print(list(c17_matrix.columns))

required_c17_columns = {
    "construct",
    "component",
}

missing_c17_columns = sorted(
    required_c17_columns - set(c17_matrix.columns)
)

c17_schema_verified = (
    len(missing_c17_columns) == 0
)

print("C17 schema verified:", c17_schema_verified)

if missing_c17_columns:
    print("Missing C17 columns:", missing_c17_columns)


# ------------------------------------------------------------
# 5. ACTUAL C17 COMPONENT COUNTS
# ------------------------------------------------------------

c17_counts = {}

if c17_schema_verified:

    c17_counts = (
        c17_matrix[
            ["construct", "component"]
        ]
        .dropna()
        .drop_duplicates()
        .groupby("construct")
        .size()
        .to_dict()
    )


print()
print("ACTUAL C17 COMPONENT COUNTS")

for construct in ["GFL", "IDS", "AML"]:

    print(
        construct,
        ":",
        c17_counts.get(construct, 0)
    )


# ------------------------------------------------------------
# 6. C18 COVERAGE SCHEMA
# ------------------------------------------------------------

print()
print("C18 COVERAGE SCHEMA")
print(list(c18_coverage.columns))

required_c18_columns = {
    "construct",
    "source_locked_concept",
}

missing_c18_columns = sorted(
    required_c18_columns - set(c18_coverage.columns)
)

c18_schema_verified = (
    len(missing_c18_columns) == 0
)

print("C18 schema verified:", c18_schema_verified)

if missing_c18_columns:
    print("Missing C18 columns:", missing_c18_columns)


# ------------------------------------------------------------
# 7. ACTUAL C18 CONCEPT COUNTS
# ------------------------------------------------------------

c18_counts = {}

if c18_schema_verified:

    c18_counts = (
        c18_coverage[
            ["construct", "source_locked_concept"]
        ]
        .dropna()
        .drop_duplicates()
        .groupby("construct")
        .size()
        .to_dict()
    )


print()
print("ACTUAL C18 SOURCE-LOCKED CONCEPT COUNTS")

for construct in ["GFL", "IDS", "AML"]:

    print(
        construct,
        ":",
        c18_counts.get(construct, 0)
    )


# ------------------------------------------------------------
# 8. RECONCILIATION
# ------------------------------------------------------------

reconciliation = []

for construct in ["GFL", "IDS", "AML"]:

    c17_count = int(c17_counts.get(construct, 0))
    c18_count = int(c18_counts.get(construct, 0))

    reconciliation.append({
        "construct": construct,
        "c17_component_count": c17_count,
        "c18_concept_count": c18_count,
        "count_difference": c18_count - c17_count,
        "count_match": c17_count == c18_count,
    })


recon_df = pd.DataFrame(reconciliation)


print()
print("RECONCILIATION")

print(recon_df.to_string(index=False))


# ------------------------------------------------------------
# 9. IMPORTANT INTERPRETATION LOCK
# ------------------------------------------------------------

#
# C17 "component" and C18 "source_locked_concept" are not
# automatically equivalent.
#
# Therefore even if counts match numerically, this Cell does
# NOT establish semantic equivalence.
#
# If counts differ, schema inconsistency exists.
#
# If counts match, only numerical reconciliation is
# established.
# ------------------------------------------------------------

count_mismatch = (
    not bool(recon_df["count_match"].all())
)


semantic_equivalence_established = False


# ------------------------------------------------------------
# 10. AML-SPECIFIC CHECK
# ------------------------------------------------------------

aml_c17 = int(
    c17_counts.get("AML", 0)
)

aml_c18 = int(
    c18_counts.get("AML", 0)
)

aml_count_reconciled = (
    aml_c17 == aml_c18
)

print()
print("AML RECONCILIATION")
print("C17 AML:", aml_c17)
print("C18 AML:", aml_c18)
print("AML count reconciled:", aml_count_reconciled)


# ------------------------------------------------------------
# 11. C18 TERMINOLOGY CHECK
# ------------------------------------------------------------

noncanonical_term_present = False

for column in c18_coverage.columns:

    if c18_coverage[column].dtype == object:

        if (
            c18_coverage[column]
            .astype(str)
            .str.contains(
                "CANDIDATE_EVIDENCE_PRESENT",
                regex=False,
                na=False
            )
            .any()
        ):
            noncanonical_term_present = True


print()
print("VOCABULARY")
print(
    "CANDIDATE_EVIDENCE_PRESENT detected:",
    noncanonical_term_present
)


# ------------------------------------------------------------
# 12. C18R STATUS CHECK
# ------------------------------------------------------------

c18r_payload = get_payload(c18r_status)

c18r_mismatches = c18r_payload.get(
    "count_mismatches",
    []
)

print()
print("C18R recorded mismatches:")
print(c18r_mismatches)


# ------------------------------------------------------------
# 13. QUALIFICATION BOUNDARY
# ------------------------------------------------------------

qualification_state = (
    "SCHEMA_RECONCILIATION_COMPLETE"
    if not count_mismatch
    else
    "SCHEMA_RECONCILIATION_BLOCKED"
)

print()
print("QUALIFICATION STATE")
print(qualification_state)

print()
print("SEMANTIC EQUIVALENCE ESTABLISHED:", False)
print("CONSTRUCT VERIFICATION:", "NOT_ESTABLISHED")
print("CONSTRUCT AUTHORIZATION:", "NOT_AUTHORIZED")


# ------------------------------------------------------------
# 14. FAIL CONDITIONS
# ------------------------------------------------------------

fail_conditions = []

if c17_summary.get("status") != "PASS":
    fail_conditions.append("C17_NOT_PASS")

if c18_status.get("status") != "PASS":
    fail_conditions.append("C18_NOT_PASS")

if c18r_status.get("status") != "PASS":
    fail_conditions.append("C18R_NOT_PASS")

if not c17_schema_verified:
    fail_conditions.append("C17_SCHEMA_NOT_VERIFIED")

if not c18_schema_verified:
    fail_conditions.append("C18_SCHEMA_NOT_VERIFIED")

if count_mismatch:
    fail_conditions.append(
        "C17_C18_COUNT_RECONCILIATION_MISMATCH"
    )


# ------------------------------------------------------------
# 15. FINAL STATUS
# ------------------------------------------------------------

if fail_conditions:

    status = "BLOCKED"
    triggered = True

else:

    status = "PASS"
    triggered = False


# ------------------------------------------------------------
# 16. AUDIT PAYLOAD
# ------------------------------------------------------------

audit_payload = {

    "oip_version": "v1.0.33",

    "cell": "C19",

    "cell_name":
        "C17/C18 COMPONENT SCHEMA RECONCILIATION",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "upstream": {
        "c17": c17_summary.get("status"),
        "c18": c18_status.get("status"),
        "c18r": c18r_status.get("status"),
    },

    "c17_schema_verified":
        c17_schema_verified,

    "c18_schema_verified":
        c18_schema_verified,

    "c17_component_counts":
        c17_counts,

    "c18_concept_counts":
        c18_counts,

    "reconciliation":
        reconciliation,

    "count_mismatch":
        count_mismatch,

    "aml_count_reconciled":
        aml_count_reconciled,

    "semantic_equivalence_established":
        semantic_equivalence_established,

    "noncanonical_term_present":
        noncanonical_term_present,

    "qualification_state":
        qualification_state,

    "semantic_boundary": {
        "literal_hit_equals_semantic_proof": False,
        "concept_equals_construct": False,
        "count_match_equals_semantic_equivalence": False,
        "construct_verification":
            "NOT_ESTABLISHED",
        "construct_authorization":
            "NOT_AUTHORIZED",
    },

    "authorization":
        auth_state(),

    "fail_closed": {
        "active": True,
        "triggered": triggered,
        "fail_conditions": fail_conditions,
    },
}


# ------------------------------------------------------------
# 17. SAVE AUDIT
# ------------------------------------------------------------

with open(
    OUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_payload,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 18. FINAL STATUS
# ------------------------------------------------------------

final_status = {

    "oip_version": "v1.0.33",

    "cell": "C19",

    "cell_name":
        "C17/C18 COMPONENT SCHEMA RECONCILIATION",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        triggered,

    "authorization":
        auth_state(),

    "payload":
        audit_payload,
}


with open(
    OUT_FINAL,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_status,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 19. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("=" * 72)
print("C19 STATUS")
print("=" * 72)

print("C17 schema verified:", c17_schema_verified)
print("C18 schema verified:", c18_schema_verified)

print(
    "C17 counts:",
    c17_counts
)

print(
    "C18 counts:",
    c18_counts
)

print(
    "Count mismatch:",
    count_mismatch
)

print(
    "AML reconciled:",
    aml_count_reconciled
)

print(
    "Semantic equivalence:",
    semantic_equivalence_established
)

print(
    "Construct verification:",
    "NOT_ESTABLISHED"
)

print(
    "Construct authorization:",
    "NOT_AUTHORIZED"
)

print()
print("Fail conditions:", fail_conditions)
print("Triggered:", triggered)
print("C19 STATUS:", status)

print()
print("ARTIFACTS")
print(OUT_AUDIT)
print(OUT_FINAL)

print("=" * 72)

OIP v1.0.33 — C19
C17/C18 COMPONENT SCHEMA RECONCILIATION

UPSTREAM STATUS
C17: None
C18: PASS
C18R: PASS

C17 MATRIX SCHEMA
['construct', 'component', 'documentary_status', 'source_evidence_required', 'source_evidence_found', 'semantic_verification', 'construct_mapping', 'authorization', 'automatic_keyword_authorization', 'manual_source_review_required', 'semantic_inference_allowed']
C17 schema verified: True

ACTUAL C17 COMPONENT COUNTS
GFL : 6
IDS : 5
AML : 5

C18 COVERAGE SCHEMA
['verification_queue_id', 'relative_path', 'file_name', 'page_number', 'review_category', 'construct', 'source_locked_concept', 'source_definition', 'source_search_term', 'page_text_match', 'documentary_evidence_status', 'semantic_verification', 'construct_mapping', 'authorization', 'discovery_only', 'semantic_inference_allowed']
C18 schema verified: True

ACTUAL C18 SOURCE-LOCKED CONCEPT COUNTS
GFL : 3
IDS : 2
AML : 1

RECONCILIATION
construct  c17_component_count  c18_concept_count  count_difference  coun

In [38]:
# ============================================================
# OIP v1.0.33 — C20
# FINAL DOCUMENTARY CONSTRUCT GATE
# ============================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone


BASE = Path("/kaggle/working/oip_v1_0_33")

C17_STATUS = BASE / "c17_final_status.json"
C17_MATRIX = BASE / "c17_documentary_component_matrix.csv"

C18_STATUS = BASE / "c18_final_status.json"
C18_COVERAGE = BASE / "c18_source_locked_concept_coverage.csv"

C18R_STATUS = BASE / "c18r_final_status.json"
C19_STATUS = BASE / "c19_final_status.json"


OUT_COVERAGE = BASE / "c20_construct_gate_summary.csv"
OUT_AUDIT = BASE / "c20_final_construct_gate_audit.json"
OUT_FINAL = BASE / "c20_final_status.json"


# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def auth_state():
    return {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY",
    }


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

c17_status = load_json(C17_STATUS)
c18_status = load_json(C18_STATUS)
c18r_status = load_json(C18R_STATUS)

c17 = pd.read_csv(C17_MATRIX)
c18 = pd.read_csv(C18_COVERAGE)


print("=" * 72)
print("OIP v1.0.33 — C20")
print("FINAL DOCUMENTARY CONSTRUCT GATE")
print("=" * 72)


# ------------------------------------------------------------
# 3. UPSTREAM STATUS
# ------------------------------------------------------------

print()
print("UPSTREAM STATUS")
print("C17:", c17_status.get("status"))
print("C18:", c18_status.get("status"))
print("C18R:", c18r_status.get("status"))
print("C19:", load_json(C19_STATUS).get("status"))


# ------------------------------------------------------------
# 4. SCHEMA CHECK
# ------------------------------------------------------------

required_c17 = {
    "construct",
    "component",
    "documentary_status",
    "source_evidence_required",
    "source_evidence_found",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "automatic_keyword_authorization",
    "manual_source_review_required",
    "semantic_inference_allowed",
}

required_c18 = {
    "verification_queue_id",
    "relative_path",
    "file_name",
    "page_number",
    "review_category",
    "construct",
    "source_locked_concept",
    "source_definition",
    "source_search_term",
    "page_text_match",
    "documentary_evidence_status",
    "semantic_verification",
    "construct_mapping",
    "authorization",
    "discovery_only",
    "semantic_inference_allowed",
}


missing_c17 = sorted(
    required_c17 - set(c17.columns)
)

missing_c18 = sorted(
    required_c18 - set(c18.columns)
)

c17_schema_verified = len(missing_c17) == 0
c18_schema_verified = len(missing_c18) == 0


print()
print("C17 schema verified:", c17_schema_verified)
print("C18 schema verified:", c18_schema_verified)


# ------------------------------------------------------------
# 5. C17 COMPONENT INVENTORY
# ------------------------------------------------------------

c17_components = {}

if c17_schema_verified:

    c17_components = (
        c17[
            ["construct", "component"]
        ]
        .dropna()
        .drop_duplicates()
        .groupby("construct")
        .size()
        .to_dict()
    )


print()
print("C17 DISCOVERY MATRIX COUNTS")

for construct in ["GFL", "IDS", "AML"]:

    print(
        construct,
        ":",
        int(c17_components.get(construct, 0))
    )


# ------------------------------------------------------------
# 6. C18 LITERAL-HIT COUNTS
# ------------------------------------------------------------

c18_literal_hits = {}

if c18_schema_verified:

    c18_literal_hits = (
        c18[
            ["construct", "source_locked_concept"]
        ]
        .dropna()
        .drop_duplicates()
        .groupby("construct")
        .size()
        .to_dict()
    )


print()
print("C18 LITERAL-HIT CONCEPT COUNTS")

for construct in ["GFL", "IDS", "AML"]:

    print(
        construct,
        ":",
        int(c18_literal_hits.get(construct, 0))
    )


# ------------------------------------------------------------
# 7. IMPORTANT UNIT BOUNDARY
# ------------------------------------------------------------

#
# C17:
#   documentary component review matrix
#
# C18:
#   source-locked literal concept coverage records
#
# These are NOT declared equivalent measurement units.
#
# Therefore:
#
# C17 count - C18 count
#
# MUST NOT be interpreted as:
#
# "documented components missing"
#
# and MUST NOT be used to calculate construct validity.
# ------------------------------------------------------------

unit_equivalence_established = False


# ------------------------------------------------------------
# 8. CONSTRUCT DOCUMENTARY STATE
# ------------------------------------------------------------

construct_rows = []

for construct in ["GFL", "IDS", "AML"]:

    component_count = int(
        c17_components.get(construct, 0)
    )

    literal_hit_count = int(
        c18_literal_hits.get(construct, 0)
    )

    if literal_hit_count > 0:

        discovery_state = (
            "CANDIDATE"
        )

    else:

        discovery_state = (
            "NOT_ESTABLISHED"
        )

    construct_rows.append({

        "construct":
            construct,

        "c17_component_records":
            component_count,

        "c18_literal_hit_concepts":
            literal_hit_count,

        "literal_discovery_state":
            discovery_state,

        "documentary_verification":
            "NOT_VERIFIED",

        "semantic_verification":
            "NOT_VERIFIED",

        "construct_mapping":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",

        "qualification":
            "NOT_ESTABLISHED",

        "manual_source_review_required":
            True,

        "automatic_authorization":
            False,

        "semantic_inference_allowed":
            False,
    })


coverage_df = pd.DataFrame(
    construct_rows
)


# ------------------------------------------------------------
# 9. OUTCOME GATE
# ------------------------------------------------------------

outcome_state = {

    "outcome_documentary_verification":
        "NOT_VERIFIED",

    "outcome_independence":
        "NOT_VERIFIED",

    "temporal_validity":
        "NOT_VERIFIED",

    "synthetic_outcome":
        "NOT_AUTHORIZED",

    "outcome_authorization":
        "NOT_AUTHORIZED",
}


# ------------------------------------------------------------
# 10. SCORE GATE
# ------------------------------------------------------------

score_state = {

    "construct_authorization":
        "NOT_AUTHORIZED",

    "outcome_authorization":
        "NOT_AUTHORIZED",

    "score_authorization":
        "NOT_AUTHORIZED",

    "empirical_evaluation":
        "NOT_READY",
}


# ------------------------------------------------------------
# 11. FAIL-CLOSED CONDITIONS
# ------------------------------------------------------------

fail_conditions = []


if c17_status.get("status") != "PASS":
    fail_conditions.append("C17_NOT_PASS")


if c18_status.get("status") != "PASS":
    fail_conditions.append("C18_NOT_PASS")


if c18r_status.get("status") != "PASS":
    fail_conditions.append("C18R_NOT_PASS")


if not c17_schema_verified:
    fail_conditions.append("C17_SCHEMA_NOT_VERIFIED")


if not c18_schema_verified:
    fail_conditions.append("C18_SCHEMA_NOT_VERIFIED")


# ------------------------------------------------------------
# 12. FINAL QUALIFICATION DECISION
# ------------------------------------------------------------

#
# No construct is authorized because:
#
# semantic verification       = NOT_VERIFIED
# construct mapping           = NOT_ESTABLISHED
#
# This Cell does NOT claim that ESS4 has "failed" as a dataset.
#
# It establishes that the OIP construct authorization gate
# remains closed for this audit state.
# ------------------------------------------------------------

construct_authorization = "NOT_AUTHORIZED"

qualification_state = (
    "FAIL_CLOSED_CONSTRUCT_AUTHORIZATION_NOT_ESTABLISHED"
)


# ------------------------------------------------------------
# 13. STATUS
# ------------------------------------------------------------

#
# The Cell itself executes successfully if all required
# upstream artifacts and schemas are present.
#
# Fail-closed is the RESULTING GOVERNANCE STATE,
# not a Python execution failure.
# ------------------------------------------------------------

if fail_conditions:

    status = "BLOCKED"
    triggered = True

else:

    status = "PASS"
    triggered = False


# ------------------------------------------------------------
# 14. AUDIT PAYLOAD
# ------------------------------------------------------------

audit_payload = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C20",

    "cell_name":
        "FINAL DOCUMENTARY CONSTRUCT GATE",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "upstream": {

        "c17":
            c17_status.get("status"),

        "c18":
            c18_status.get("status"),

        "c18r":
            c18r_status.get("status"),

    },

    "schema": {

        "c17_verified":
            c17_schema_verified,

        "c18_verified":
            c18_schema_verified,

        "missing_c17":
            missing_c17,

        "missing_c18":
            missing_c18,

    },

    "c17_component_counts":
        c17_components,

    "c18_literal_hit_concept_counts":
        c18_literal_hits,

    "unit_equivalence_established":
        unit_equivalence_established,

    "construct_summary":
        construct_rows,

    "outcome_state":
        outcome_state,

    "score_state":
        score_state,

    "qualification_state":
        qualification_state,

    "interpretation_lock": {

        "literal_hit_equals_documented":
            False,

        "literal_hit_equals_verified":
            False,

        "literal_hit_equals_construct_validity":
            False,

        "c17_count_equals_c18_hit_count":
            False,

        "count_difference_equals_missing_components":
            False,

        "construct_authorization_from_literal_hits":
            False,

        "automatic_authorization":
            False,

        "semantic_inference":
            False,

    },

    "authorization":
        auth_state(),

    "fail_closed": {

        "active":
            True,

        "triggered":
            triggered,

        "fail_conditions":
            fail_conditions,

    },
}


# ------------------------------------------------------------
# 15. SAVE COVERAGE SUMMARY
# ------------------------------------------------------------

coverage_df.to_csv(
    OUT_COVERAGE,
    index=False
)


# ------------------------------------------------------------
# 16. SAVE AUDIT
# ------------------------------------------------------------

with open(
    OUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_payload,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 17. FINAL STATUS
# ------------------------------------------------------------

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C20",

    "cell_name":
        "FINAL DOCUMENTARY CONSTRUCT GATE",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        triggered,

    "authorization":
        auth_state(),

    "payload":
        audit_payload,
}


with open(
    OUT_FINAL,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_status,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 18. OUTPUT
# ------------------------------------------------------------

print()
print("=" * 72)
print("C20 RESULTS")
print("=" * 72)

print()
print("CONSTRUCT DISCOVERY")

for row in construct_rows:

    print(
        row["construct"],
        "| C17 component records:",
        row["c17_component_records"],
        "| C18 literal-hit concepts:",
        row["c18_literal_hit_concepts"],
        "| Discovery:",
        row["literal_discovery_state"],
        "| Verification:",
        row["semantic_verification"],
        "| Authorization:",
        row["authorization"],
    )


print()
print("UNIT EQUIVALENCE ESTABLISHED:",
      unit_equivalence_established)

print()
print("CONSTRUCT AUTHORIZATION:",
      construct_authorization)

print("OUTCOME AUTHORIZATION:",
      outcome_state["outcome_authorization"])

print("SCORE AUTHORIZATION:",
      score_state["score_authorization"])

print("EMPIRICAL EVALUATION:",
      score_state["empirical_evaluation"])

print()
print("QUALIFICATION STATE:")
print(qualification_state)

print()
print("INTERPRETATION LOCK")
print("Literal hit = documented:", False)
print("Literal hit = verified:", False)
print("Literal hit = construct validity:", False)
print("Automatic authorization:", False)
print("Semantic inference:", False)

print()
print("FAIL-CLOSED")
print("Fail conditions:", fail_conditions)
print("Triggered:", triggered)

print()
print("C20 STATUS:", status)

print()
print("ARTIFACTS")
print(OUT_COVERAGE)
print(OUT_AUDIT)
print(OUT_FINAL)

print("=" * 72)

OIP v1.0.33 — C20
FINAL DOCUMENTARY CONSTRUCT GATE

UPSTREAM STATUS
C17: PASS
C18: PASS
C18R: PASS
C19: BLOCKED

C17 schema verified: True
C18 schema verified: True

C17 DISCOVERY MATRIX COUNTS
GFL : 6
IDS : 5
AML : 5

C18 LITERAL-HIT CONCEPT COUNTS
GFL : 3
IDS : 2
AML : 1

C20 RESULTS

CONSTRUCT DISCOVERY
GFL | C17 component records: 6 | C18 literal-hit concepts: 3 | Discovery: CANDIDATE | Verification: NOT_VERIFIED | Authorization: NOT_AUTHORIZED
IDS | C17 component records: 5 | C18 literal-hit concepts: 2 | Discovery: CANDIDATE | Verification: NOT_VERIFIED | Authorization: NOT_AUTHORIZED
AML | C17 component records: 5 | C18 literal-hit concepts: 1 | Discovery: CANDIDATE | Verification: NOT_VERIFIED | Authorization: NOT_AUTHORIZED

UNIT EQUIVALENCE ESTABLISHED: False

CONSTRUCT AUTHORIZATION: NOT_AUTHORIZED
OUTCOME AUTHORIZATION: NOT_AUTHORIZED
SCORE AUTHORIZATION: NOT_AUTHORIZED
EMPIRICAL EVALUATION: NOT_READY

QUALIFICATION STATE:
FAIL_CLOSED_CONSTRUCT_AUTHORIZATION_NOT_ESTABLISHED

In [39]:
# ============================================================
# OIP v1.0.33 — C21
# FINAL DATASET QUALIFICATION & FAIL-CLOSED RELEASE GATE
# ============================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone


BASE = Path("/kaggle/working/oip_v1_0_33")

# ------------------------------------------------------------
# UPSTREAM ARTIFACTS
# ------------------------------------------------------------

STATUS_FILES = {
    "C00": BASE / "c00_source_scope.json",
    "C01": BASE / "c01_vocabulary_lock.json",
    "C02": BASE / "c02_integrity_status.json",
    "C03": BASE / "c03_final_status.json",
    "C04": BASE / "c04_final_status.json",
    "C05": BASE / "c05_final_status.json",
    "C06": BASE / "c06_final_status.json",
    "C07": BASE / "c07_final_status.json",
    "C07R": BASE / "c07r_final_status.json",
    "C08": BASE / "c08_final_status.json",
    "C09R": BASE / "c09r_final_status.json",
    "C10": BASE / "c10_final_status.json",
    "C11A": BASE / "c11a_final_status.json",
    "C11E": BASE / "c11e_final_status.json",
    "C13": BASE / "c13_final_status.json",
    "C14": BASE / "c14_final_status.json",
    "C15": BASE / "c15_final_status.json",
    "C16": BASE / "c16_final_status.json",
    "C17": BASE / "c17_final_status.json",
    "C18": BASE / "c18_final_status.json",
    "C18R": BASE / "c18r_final_status.json",
    "C19": BASE / "c19_final_status.json",
    "C20": BASE / "c20_final_status.json",
}


OUT_SUMMARY = BASE / "c21_final_dataset_qualification_summary.csv"
OUT_AUDIT = BASE / "c21_final_dataset_qualification_audit.json"
OUT_FINAL = BASE / "c21_final_status.json"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_payload(obj):
    value = obj.get("payload", {})
    return value if isinstance(value, dict) else {}


def auth_state():
    return {
        "construct": "NOT_AUTHORIZED",
        "predictor": "NOT_AUTHORIZED",
        "outcome": "NOT_AUTHORIZED",
        "key": "NOT_AUTHORIZED",
        "relationship": "NOT_AUTHORIZED",
        "cohort": "NOT_AUTHORIZED",
        "score": "NOT_AUTHORIZED",
        "empirical_evaluation": "NOT_READY",
    }


print("=" * 72)
print("OIP v1.0.33 — C21")
print("FINAL DATASET QUALIFICATION & FAIL-CLOSED RELEASE GATE")
print("=" * 72)


# ------------------------------------------------------------
# 1. LOAD ALL AVAILABLE STATUS ARTIFACTS
# ------------------------------------------------------------

statuses = {}
missing_artifacts = []

for cell, path in STATUS_FILES.items():

    if not path.exists():

        missing_artifacts.append(cell)

        continue

    try:
        statuses[cell] = load_json(path)

    except Exception as e:

        missing_artifacts.append(
            f"{cell}:READ_ERROR"
        )


print()
print("STATUS ARTIFACTS")
print("Expected:", len(STATUS_FILES))
print("Loaded:", len(statuses))
print("Missing / unreadable:", missing_artifacts)


# ------------------------------------------------------------
# 2. ACTUAL STATUS REGISTER
# ------------------------------------------------------------

status_register = []

for cell in STATUS_FILES:

    if cell not in statuses:

        status_register.append({
            "cell": cell,
            "status": "NOT_AVAILABLE",
        })

    else:

        status_register.append({
            "cell": cell,
            "status": statuses[cell].get(
                "status",
                "NOT_AVAILABLE"
            ),
        })


status_df = pd.DataFrame(status_register)


print()
print("STATUS REGISTER")

for _, row in status_df.iterrows():

    print(
        row["cell"],
        ":",
        row["status"]
    )


# ------------------------------------------------------------
# 3. C21 DOES NOT REWRITE HISTORICAL CELL RESULTS
# ------------------------------------------------------------

#
# Important:
#
# C19 = BLOCKED
#
# is preserved as an audit fact.
#
# C21 does NOT silently convert C19 BLOCKED into PASS.
#
# The final qualification state is therefore determined
# separately from execution status.
# ------------------------------------------------------------


c19_status = (
    statuses.get("C19", {})
    .get("status")
)

c20_status = (
    statuses.get("C20", {})
    .get("status")
)


# ------------------------------------------------------------
# 4. STRUCTURAL / DOCUMENTARY EXECUTION STATE
# ------------------------------------------------------------

required_pass_cells = [
    "C00",
    "C01",
    "C02",
    "C03",
    "C04",
    "C05",
    "C06",
    "C07R",
    "C08",
    "C09R",
    "C10",
    "C11A",
    "C11E",
    "C13",
    "C14",
    "C15",
    "C16",
    "C17",
    "C18",
    "C18R",
    "C20",
]


upstream_execution_failures = []

for cell in required_pass_cells:

    actual = (
        statuses.get(cell, {})
        .get("status")
    )

    if actual != "PASS":

        upstream_execution_failures.append({
            "cell": cell,
            "status": actual,
        })


# ------------------------------------------------------------
# 5. CONSTRUCT STATE FROM C20
# ------------------------------------------------------------

c20_payload = get_payload(
    statuses.get("C20", {})
)

c20_construct_summary = (
    c20_payload.get(
        "construct_summary",
        []
    )
)

construct_states = {}

for row in c20_construct_summary:

    if not isinstance(row, dict):
        continue

    construct = row.get("construct")

    if construct:

        construct_states[construct] = {
            "literal_discovery_state":
                row.get(
                    "literal_discovery_state",
                    "NOT_ESTABLISHED"
                ),

            "documentary_verification":
                row.get(
                    "documentary_verification",
                    "NOT_VERIFIED"
                ),

            "semantic_verification":
                row.get(
                    "semantic_verification",
                    "NOT_VERIFIED"
                ),

            "construct_mapping":
                row.get(
                    "construct_mapping",
                    "NOT_ESTABLISHED"
                ),

            "authorization":
                row.get(
                    "authorization",
                    "NOT_AUTHORIZED"
                ),
        }


print()
print("CONSTRUCT STATES")

for construct in ["GFL", "IDS", "AML"]:

    state = construct_states.get(
        construct,
        {}
    )

    print(
        construct,
        "| discovery:",
        state.get(
            "literal_discovery_state",
            "NOT_ESTABLISHED"
        ),
        "| semantic:",
        state.get(
            "semantic_verification",
            "NOT_VERIFIED"
        ),
        "| mapping:",
        state.get(
            "construct_mapping",
            "NOT_ESTABLISHED"
        ),
        "| authorization:",
        state.get(
            "authorization",
            "NOT_AUTHORIZED"
        ),
    )


# ------------------------------------------------------------
# 6. OUTCOME / SCORE / EMPIRICAL STATE
# ------------------------------------------------------------

outcome_authorization = "NOT_AUTHORIZED"
score_authorization = "NOT_AUTHORIZED"
empirical_evaluation = "NOT_READY"

if isinstance(c20_payload.get("outcome_state"), dict):

    outcome_authorization = c20_payload[
        "outcome_state"
    ].get(
        "outcome_authorization",
        "NOT_AUTHORIZED"
    )

if isinstance(c20_payload.get("score_state"), dict):

    score_authorization = c20_payload[
        "score_state"
    ].get(
        "score_authorization",
        "NOT_AUTHORIZED"
    )

    empirical_evaluation = c20_payload[
        "score_state"
    ].get(
        "empirical_evaluation",
        "NOT_READY"
    )


# ------------------------------------------------------------
# 7. MANDATORY QUALIFICATION GATES
# ------------------------------------------------------------

#
# C21 does not infer PASS/FAIL for gates that have not been
# explicitly established by the upstream audit.
#
# Unknown remains NOT_ESTABLISHED.
# ------------------------------------------------------------

qualification_gates = {

    "provenance_integrity":
        "ESTABLISHED",

    "temporal_validity":
        "NOT_ESTABLISHED",

    "construct_measurement_evidence":
        "NOT_ESTABLISHED",

    "outcome_independence":
        "NOT_ESTABLISHED",

    "no_synthetic_outcome_construction":
        "NOT_ESTABLISHED",

    "leakage_control":
        "NOT_ESTABLISHED",
}


# ------------------------------------------------------------
# 8. VERIFY EXPLICIT EVIDENCE FOR SYNTHETIC OUTCOME
# ------------------------------------------------------------

#
# C20 explicitly carries the score/outcome authorization
# boundary. No synthetic outcome authorization is created.
#

synthetic_outcome_authorization = "NOT_AUTHORIZED"


# ------------------------------------------------------------
# 9. FINAL AUTHORIZATION STATE
# ------------------------------------------------------------

final_authorization = auth_state()

final_authorization[
    "construct"
] = "NOT_AUTHORIZED"

final_authorization[
    "outcome"
] = outcome_authorization

final_authorization[
    "score"
] = score_authorization

final_authorization[
    "empirical_evaluation"
] = empirical_evaluation


# ------------------------------------------------------------
# 10. FINAL QUALIFICATION STATE
# ------------------------------------------------------------

#
# Because required qualification gates remain
# NOT_ESTABLISHED, the dataset is not authorized for
# OIP score calculation or empirical evaluation.
#
# This is a governance state, not a claim that the dataset
# itself is invalid.
# ------------------------------------------------------------

unresolved_gates = [
    gate
    for gate, state in qualification_gates.items()
    if state != "ESTABLISHED"
]


final_qualification_state = (
    "FAIL_CLOSED_QUALIFICATION_NOT_ESTABLISHED"
)


release_state = (
    "RELEASE_AS_FAIL_CLOSED_AUDIT_ONLY"
)


# ------------------------------------------------------------
# 11. FINAL CLAIM DISCIPLINE
# ------------------------------------------------------------

claims_allowed = [

    "SOURCE_INTEGRITY_AUDITED",

    "METADATA_AND_STRUCTURAL_DISCOVERY_AUDITED",

    "DOCUMENTARY_OCCURRENCE_DISCOVERY_COMPLETED",

    "SEMANTIC_CONSTRUCT_VERIFICATION_NOT_ESTABLISHED",

    "CONSTRUCT_AUTHORIZATION_NOT_ESTABLISHED",

    "OUTCOME_AUTHORIZATION_NOT_ESTABLISHED",

    "SCORE_NOT_AUTHORIZED",

    "EMPIRICAL_EVALUATION_NOT_READY",

    "FAIL_CLOSED_STATE_ACTIVE",
]


claims_not_authorized = [

    "OIP_EMPIRICALLY_VALIDATED",

    "OIP_SCIENTIFICALLY_PROVEN",

    "OIP_CAUSALLY_VALIDATED",

    "OIP_SUPERIOR",

    "ESS4_PROVES_OIP",

    "HIGH_PERFORMANCE_VALIDATES_OIP",

]


# ------------------------------------------------------------
# 12. EXECUTION STATUS
# ------------------------------------------------------------

#
# C19 remains BLOCKED in the historical audit record.
#
# It is not overwritten.
#
# C21 itself can still PASS because its task is to produce
# the final qualification/release state while preserving
# the earlier BLOCKED artifact.
# ------------------------------------------------------------

c21_fail_conditions = []

if missing_artifacts:

    c21_fail_conditions.append(
        "REQUIRED_STATUS_ARTIFACT_MISSING"
    )

if upstream_execution_failures:

    c21_fail_conditions.append(
        "REQUIRED_UPSTREAM_EXECUTION_NOT_PASS"
    )


if c20_status != "PASS":

    c21_fail_conditions.append(
        "C20_NOT_PASS"
    )


if c21_fail_conditions:

    c21_status = "BLOCKED"
    c21_triggered = True

else:

    c21_status = "PASS"
    c21_triggered = False


# ------------------------------------------------------------
# 13. FINAL SUMMARY TABLE
# ------------------------------------------------------------

summary_rows = [

    {
        "gate":
            "Provenance / Integrity",

        "state":
            qualification_gates[
                "provenance_integrity"
            ],

        "authorization":
            "NOT_AUTHORIZED",
    },

    {
        "gate":
            "Temporal Validity",

        "state":
            qualification_gates[
                "temporal_validity"
            ],

        "authorization":
            "NOT_AUTHORIZED",
    },

    {
        "gate":
            "Construct Measurement Evidence",

        "state":
            qualification_gates[
                "construct_measurement_evidence"
            ],

        "authorization":
            "NOT_AUTHORIZED",
    },

    {
        "gate":
            "Outcome Independence",

        "state":
            qualification_gates[
                "outcome_independence"
            ],

        "authorization":
            "NOT_AUTHORIZED",
    },

    {
        "gate":
            "No Synthetic Outcome Construction",

        "state":
            qualification_gates[
                "no_synthetic_outcome_construction"
            ],

        "authorization":
            synthetic_outcome_authorization,
    },

    {
        "gate":
            "Leakage Control",

        "state":
            qualification_gates[
                "leakage_control"
            ],

        "authorization":
            "NOT_AUTHORIZED",
    },

    {
        "gate":
            "Construct Authorization",

        "state":
            "NOT_ESTABLISHED",

        "authorization":
            "NOT_AUTHORIZED",
    },

    {
        "gate":
            "Outcome Authorization",

        "state":
            "NOT_ESTABLISHED",

        "authorization":
            outcome_authorization,
    },

    {
        "gate":
            "Score Authorization",

        "state":
            "NOT_AUTHORIZED",

        "authorization":
            score_authorization,
    },

    {
        "gate":
            "Empirical Evaluation",

        "state":
            "NOT_READY",

        "authorization":
            empirical_evaluation,
    },

]


summary_df = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# 14. SAVE SUMMARY CSV
# ------------------------------------------------------------

summary_df.to_csv(
    OUT_SUMMARY,
    index=False
)


# ------------------------------------------------------------
# 15. FINAL AUDIT PAYLOAD
# ------------------------------------------------------------

audit_payload = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C21",

    "cell_name":
        "FINAL DATASET QUALIFICATION & FAIL-CLOSED RELEASE GATE",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "upstream_status_register":
        status_register,

    "c19_historical_status":
        c19_status,

    "c20_status":
        c20_status,

    "construct_states":
        construct_states,

    "qualification_gates":
        qualification_gates,

    "unresolved_gates":
        unresolved_gates,

    "outcome_authorization":
        outcome_authorization,

    "score_authorization":
        score_authorization,

    "empirical_evaluation":
        empirical_evaluation,

    "synthetic_outcome_authorization":
        synthetic_outcome_authorization,

    "final_authorization":
        final_authorization,

    "final_qualification_state":
        final_qualification_state,

    "release_state":
        release_state,

    "claims_allowed":
        claims_allowed,

    "claims_not_authorized":
        claims_not_authorized,

    "interpretation_lock": {

        "literal_hit_equals_documentary_verification":
            False,

        "documentary_occurrence_equals_construct_validity":
            False,

        "construct_discovery_equals_construct_authorization":
            False,

        "dataset_audit_equals_empirical_validation":
            False,

        "fail_closed_equals_dataset_invalid":
            False,

        "c19_blocked_overwritten":
            False,

    },

    "fail_closed": {

        "active":
            True,

        "triggered":
            True,

        "qualification_not_established":
            True,

        "fail_conditions":
            c21_fail_conditions,

    },

}


# ------------------------------------------------------------
# 16. SAVE AUDIT JSON
# ------------------------------------------------------------

with open(
    OUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_payload,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 17. FINAL STATUS JSON
# ------------------------------------------------------------

final_status = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C21",

    "cell_name":
        "FINAL DATASET QUALIFICATION & FAIL-CLOSED RELEASE GATE",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        c21_status,

    "fail_closed_policy_active":
        True,

    "fail_closed_triggered":
        True,

    "authorization":
        final_authorization,

    "payload":
        audit_payload,

}


with open(
    OUT_FINAL,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_status,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 18. FINAL CONSOLE OUTPUT
# ------------------------------------------------------------

print()
print("=" * 72)
print("C21 FINAL RESULTS")
print("=" * 72)

print()
print("C19 HISTORICAL STATUS:", c19_status)
print("C20 STATUS:", c20_status)

print()
print("CONSTRUCT AUTHORIZATION")
print("GFL:", final_authorization["construct"])
print("IDS:", final_authorization["construct"])
print("AML:", final_authorization["construct"])

print()
print("OUTCOME AUTHORIZATION:",
      outcome_authorization)

print("SCORE AUTHORIZATION:",
      score_authorization)

print("EMPIRICAL EVALUATION:",
      empirical_evaluation)

print()
print("QUALIFICATION GATES")

for gate, state in qualification_gates.items():

    print(
        gate,
        ":",
        state
    )


print()
print("UNRESOLVED GATES:")
print(unresolved_gates)

print()
print("FINAL QUALIFICATION STATE:")
print(final_qualification_state)

print()
print("RELEASE STATE:")
print(release_state)

print()
print("CLAIM DISCIPLINE")
print("Empirical validation claim: NOT_AUTHORIZED")
print("Scientific proof claim: NOT_AUTHORIZED")
print("Causal validation claim: NOT_AUTHORIZED")
print("Score claim: NOT_AUTHORIZED")

print()
print("FAIL-CLOSED")
print("Active:", True)
print("Triggered:", True)
print("Qualification established:", False)

print()
print("C21 STATUS:", c21_status)

print()
print("ARTIFACTS")
print(OUT_SUMMARY)
print(OUT_AUDIT)
print(OUT_FINAL)

print("=" * 72)

OIP v1.0.33 — C21
FINAL DATASET QUALIFICATION & FAIL-CLOSED RELEASE GATE

STATUS ARTIFACTS
Expected: 23
Loaded: 23
Missing / unreadable: []

STATUS REGISTER
C00 : PASS
C01 : PASS
C02 : PASS
C03 : PASS
C04 : PASS
C05 : PASS
C06 : PASS
C07 : PASS
C07R : PASS
C08 : PASS
C09R : PASS
C10 : PASS
C11A : PASS
C11E : PASS
C13 : PASS
C14 : PASS
C15 : PASS
C16 : PASS
C17 : PASS
C18 : PASS
C18R : PASS
C19 : BLOCKED
C20 : PASS

CONSTRUCT STATES
GFL | discovery: CANDIDATE | semantic: NOT_VERIFIED | mapping: NOT_ESTABLISHED | authorization: NOT_AUTHORIZED
IDS | discovery: CANDIDATE | semantic: NOT_VERIFIED | mapping: NOT_ESTABLISHED | authorization: NOT_AUTHORIZED
AML | discovery: CANDIDATE | semantic: NOT_VERIFIED | mapping: NOT_ESTABLISHED | authorization: NOT_AUTHORIZED

C21 FINAL RESULTS

C19 HISTORICAL STATUS: BLOCKED
C20 STATUS: PASS

CONSTRUCT AUTHORIZATION
GFL: NOT_AUTHORIZED
IDS: NOT_AUTHORIZED
AML: NOT_AUTHORIZED

OUTCOME AUTHORIZATION: NOT_AUTHORIZED
SCORE AUTHORIZATION: NOT_AUTHORIZED
EMP

In [40]:
# ============================================================
# OIP v1.0.33 — C21R
# STATUS FILE COMPLETENESS AUDIT
# READ-ONLY VERIFICATION
# ============================================================

from pathlib import Path
import json

BASE = Path("/kaggle/working/oip_v1_0_33")

print("=" * 72)
print("OIP v1.0.33 — C21R")
print("STATUS FILE COMPLETENESS AUDIT")
print("=" * 72)

# ------------------------------------------------------------
# 1. ACTUAL STATUS FILES
# ------------------------------------------------------------

files = sorted(
    BASE.glob("c*_final_status.json"),
    key=lambda p: p.name
)

print()
print("TOTAL c*_final_status.json FILES:", len(files))
print()

for f in files:
    try:
        data = json.loads(
            f.read_text(encoding="utf-8")
        )

        print(
            f"{f.name:35s}",
            "→",
            data.get("cell", "CELL_NOT_DECLARED"),
            "|",
            data.get("status", "STATUS_NOT_DECLARED")
        )

    except Exception as e:

        print(
            f"{f.name:35s}",
            "→ READ_ERROR:",
            type(e).__name__
        )


# ------------------------------------------------------------
# 2. IMPORTANT CELLS SPECIFICALLY CHECKED
# ------------------------------------------------------------

check_cells = [
    "C00",
    "C01",
    "C02",
    "C03",
    "C04",
    "C05",
    "C06",
    "C07",
    "C07R",
    "C08",
    "C09R",
    "C10",
    "C11A",
    "C11B",
    "C11C",
    "C11D",
    "C11E",
    "C12",
    "C13",
    "C13R",
    "C14",
    "C15",
    "C16",
    "C17",
    "C18",
    "C18R",
    "C19",
    "C20",
    "C21",
]


print()
print("=" * 72)
print("TARGET CELL CHECK")
print("=" * 72)

actual_cells = {}

for f in files:

    try:

        data = json.loads(
            f.read_text(encoding="utf-8")
        )

        cell = data.get("cell")

        if cell:

            actual_cells[cell] = {
                "file": f.name,
                "status": data.get(
                    "status",
                    "STATUS_NOT_DECLARED"
                )
            }

    except Exception:
        pass


missing_cells = []
present_cells = []

for cell in check_cells:

    if cell in actual_cells:

        present_cells.append(cell)

        print(
            cell,
            "→",
            actual_cells[cell]["status"],
            "|",
            actual_cells[cell]["file"]
        )

    else:

        missing_cells.append(cell)

        print(
            cell,
            "→ MISSING"
        )


# ------------------------------------------------------------
# 3. DUPLICATE CELL DECLARATIONS
# ------------------------------------------------------------

cell_files = {}

for f in files:

    try:

        data = json.loads(
            f.read_text(encoding="utf-8")
        )

        cell = data.get("cell")

        if cell:

            cell_files.setdefault(
                cell,
                []
            ).append(f.name)

    except Exception:
        pass


duplicates = {
    cell: names
    for cell, names in cell_files.items()
    if len(names) > 1
}


# ------------------------------------------------------------
# 4. C21 REGISTER CROSS-CHECK
# ------------------------------------------------------------

c21_path = BASE / "c21_final_status.json"

c21_register_cells = []

if c21_path.exists():

    c21 = json.loads(
        c21_path.read_text(
            encoding="utf-8"
        )
    )

    register = (
        c21
        .get("payload", {})
        .get("upstream_status_register", [])
    )

    for row in register:

        if isinstance(row, dict):

            cell = row.get("cell")

            if cell:
                c21_register_cells.append(cell)


actual_cell_set = set(actual_cells)
c21_register_set = set(c21_register_cells)

actual_not_in_c21 = sorted(
    actual_cell_set - c21_register_set
)

c21_not_in_actual = sorted(
    c21_register_set - actual_cell_set
)


# ------------------------------------------------------------
# 5. FINAL DIAGNOSTIC OUTPUT
# ------------------------------------------------------------

print()
print("=" * 72)
print("C21R RESULTS")
print("=" * 72)

print()
print("Actual status files:", len(files))

print(
    "Target cells present:",
    len(present_cells),
    "/",
    len(check_cells)
)

print(
    "Target cells missing:",
    missing_cells
)

print()
print(
    "Duplicate cell declarations:",
    duplicates
)

print()
print(
    "ACTUAL CELLS NOT IN C21 REGISTER:"
)
print(actual_not_in_c21)

print()
print(
    "C21 REGISTER CELLS NOT FOUND AS ACTUAL CELL:"
)
print(c21_not_in_actual)

print()
print("SPECIFIC CHECK")

for cell in [
    "C11B",
    "C11C",
    "C11D",
    "C12",
    "C13",
    "C13R",
]:

    if cell in actual_cells:

        print(
            cell,
            "→ PRESENT |",
            actual_cells[cell]["status"],
            "|",
            actual_cells[cell]["file"]
        )

    else:

        print(
            cell,
            "→ NOT_PRESENT"
        )

print()
print("=" * 72)

OIP v1.0.33 — C21R
STATUS FILE COMPLETENESS AUDIT

TOTAL c*_final_status.json FILES: 23

c03_final_status.json               → C03 | PASS
c04_final_status.json               → C04 | PASS
c05_final_status.json               → C05 | PASS
c06_final_status.json               → C06 | PASS
c07_final_status.json               → C07 | PASS
c07r_final_status.json              → C07R | PASS
c08_final_status.json               → C08 | PASS
c09r_final_status.json              → C09R | PASS
c10_final_status.json               → C10 | PASS
c11a_final_status.json              → C11A | PASS
c11b_final_status.json              → C11B | PASS
c11d_final_status.json              → C11D | BLOCKED
c11e_final_status.json              → C11E | PASS
c13_final_status.json               → C13 | PASS
c14_final_status.json               → C14 | PASS
c15_final_status.json               → C15 | PASS
c16_final_status.json               → C16 | PASS
c17_final_status.json               → C17 | PASS
c18_final_status.jso

In [41]:
# ============================================================
# OIP v1.0.33 — C21R2
# CORRECTED FINAL CELL REGISTER & COMPLETENESS GATE
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone

BASE = Path("/kaggle/working/oip_v1_0_33")

print("=" * 72)
print("OIP v1.0.33 — C21R2")
print("CORRECTED FINAL CELL REGISTER & COMPLETENESS GATE")
print("=" * 72)


# ------------------------------------------------------------
# 1. ACTUAL ARTIFACT → CELL MAPPING
# ------------------------------------------------------------

artifact_map = {

    "C00": "c00_source_scope.json",
    "C01": "c01_vocabulary_lock.json",
    "C02": "c02_integrity_status.json",

    "C03": "c03_final_status.json",
    "C04": "c04_final_status.json",
    "C05": "c05_final_status.json",
    "C06": "c06_final_status.json",

    "C07": "c07_final_status.json",
    "C07R": "c07r_final_status.json",

    "C08": "c08_final_status.json",

    "C09R": "c09r_final_status.json",
    "C10": "c10_final_status.json",

    "C11A": "c11a_final_status.json",
    "C11B": "c11b_final_status.json",

    # C11C intentionally absent
    "C11D": "c11d_final_status.json",
    "C11E": "c11e_final_status.json",

    # C12 intentionally absent

    "C13": "c13_final_status.json",

    # C13R intentionally absent

    "C14": "c14_final_status.json",
    "C15": "c15_final_status.json",
    "C16": "c16_final_status.json",
    "C17": "c17_final_status.json",
    "C18": "c18_final_status.json",
    "C18R": "c18r_final_status.json",
    "C19": "c19_final_status.json",
    "C20": "c20_final_status.json",
    "C21": "c21_final_status.json",
}


# ------------------------------------------------------------
# 2. CELLS THAT HAVE NO VERIFIED STATUS ARTIFACT
# ------------------------------------------------------------

not_available_cells = {

    "C11C": {
        "state": "NOT_RUN",
        "reason": "No C11C status artifact exists in working directory."
    },

    "C12": {
        "state": "NOT_RUN",
        "reason": "No C12 status artifact exists in working directory."
    },

    "C13R": {
        "state": "NOT_RUN",
        "reason": "No C13R status artifact exists in working directory."
    },
}


# ------------------------------------------------------------
# 3. OPTIONAL HISTORICAL / CORRECTED RELATIONSHIPS
# ------------------------------------------------------------

supersession = {

    "C07": {
        "superseded_by": "C07R",
        "reason":
            "C07 used variable-level structural grouping; "
            "C07R established the corrected file-variable-level audit."
    },

    "C09R": {
        "superseded_by": None,
        "reason":
            "C09R is the accepted corrected reconciliation artifact."
    },

    "C13": {
        "superseded_by": None,
        "reason":
            "C13 is the actual available category-routing artifact."
    },

}


# ------------------------------------------------------------
# 4. READ ACTUAL ARTIFACT STATUSES
# ------------------------------------------------------------

actual_register = []

missing_artifacts = []

for cell, filename in artifact_map.items():

    path = BASE / filename

    if not path.exists():

        missing_artifacts.append({
            "cell": cell,
            "expected_file": filename,
        })

        actual_register.append({
            "cell": cell,
            "artifact": filename,
            "artifact_exists": False,
            "status": "NOT_AVAILABLE",
        })

        continue

    try:

        data = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        declared_cell = data.get(
            "cell",
            "NOT_DECLARED"
        )

        status = data.get(
            "status",
            "STATUS_NOT_DECLARED"
        )

        actual_register.append({
            "cell": cell,
            "artifact": filename,
            "artifact_exists": True,
            "declared_cell": declared_cell,
            "status": status,
        })

    except Exception as e:

        actual_register.append({
            "cell": cell,
            "artifact": filename,
            "artifact_exists": True,
            "declared_cell": "READ_ERROR",
            "status": "READ_ERROR",
        })


# ------------------------------------------------------------
# 5. CHECK DECLARED CELL CONSISTENCY
# ------------------------------------------------------------

cell_declaration_mismatches = []

for row in actual_register:

    if not row["artifact_exists"]:
        continue

    if row.get("declared_cell") == "NOT_DECLARED":
        continue

    if row.get("declared_cell") != row["cell"]:

        cell_declaration_mismatches.append({
            "expected_cell": row["cell"],
            "declared_cell": row.get(
                "declared_cell"
            ),
            "artifact": row["artifact"],
        })


# ------------------------------------------------------------
# 6. REQUIRED FINAL CELLS
# ------------------------------------------------------------

canonical_cells = [
    "C00",
    "C01",
    "C02",
    "C03",
    "C04",
    "C05",
    "C06",
    "C07",
    "C07R",
    "C08",
    "C09R",
    "C10",
    "C11A",
    "C11B",
    "C11D",
    "C11E",
    "C13",
    "C14",
    "C15",
    "C16",
    "C17",
    "C18",
    "C18R",
    "C19",
    "C20",
    "C21",
]


# ------------------------------------------------------------
# 7. BUILD COMPLETE MANIFEST REGISTER
# ------------------------------------------------------------

manifest_rows = []

for cell in canonical_cells:

    row = next(
        (
            x
            for x in actual_register
            if x["cell"] == cell
        ),
        None
    )

    if row is None:

        manifest_rows.append({
            "cell": cell,
            "execution_state": "NOT_AVAILABLE",
            "status_artifact": "NOT_AVAILABLE",
            "status": "NOT_AVAILABLE",
            "superseded_by": None,
        })

    else:

        manifest_rows.append({
            "cell": cell,
            "execution_state":
                "EXECUTED"
                if row["artifact_exists"]
                else "NOT_AVAILABLE",

            "status_artifact":
                row["artifact"],

            "status":
                row["status"],

            "superseded_by":
                supersession
                .get(cell, {})
                .get("superseded_by"),
        })


for cell, info in not_available_cells.items():

    manifest_rows.append({
        "cell": cell,
        "execution_state": info["state"],
        "status_artifact": "NOT_PRESENT",
        "status": info["state"],
        "superseded_by": None,
    })


# ------------------------------------------------------------
# 8. SORT MANIFEST NUMERICALLY
# ------------------------------------------------------------

def cell_sort_key(cell):

    raw = cell.replace("C", "")

    digits = ""

    for char in raw:

        if char.isdigit():
            digits += char

    number = int(digits) if digits else 999

    suffix = raw[len(str(number)):]

    return (
        number,
        suffix
    )


manifest_rows = sorted(
    manifest_rows,
    key=lambda x: cell_sort_key(
        x["cell"]
    )
)


# ------------------------------------------------------------
# 9. STATUS COUNTS
# ------------------------------------------------------------

status_counts = {}

for row in manifest_rows:

    state = row["status"]

    status_counts[state] = (
        status_counts.get(state, 0) + 1
    )


executed_cells = [
    row
    for row in manifest_rows
    if row["execution_state"] == "EXECUTED"
]


pass_cells = [
    row
    for row in executed_cells
    if row["status"] == "PASS"
]


blocked_cells = [
    row
    for row in executed_cells
    if row["status"] == "BLOCKED"
]


not_run_cells = [
    row
    for row in manifest_rows
    if row["status"] == "NOT_RUN"
]


# ------------------------------------------------------------
# 10. C19 PRESERVATION
# ------------------------------------------------------------

c19_row = next(
    (
        row
        for row in manifest_rows
        if row["cell"] == "C19"
    ),
    None
)

c19_preserved = (
    c19_row is not None
    and c19_row["status"] == "BLOCKED"
)


# ------------------------------------------------------------
# 11. C21 FINAL GOVERNANCE STATE
# ------------------------------------------------------------

final_authorization = {

    "construct": "NOT_AUTHORIZED",
    "predictor": "NOT_AUTHORIZED",
    "outcome": "NOT_AUTHORIZED",
    "key": "NOT_AUTHORIZED",
    "relationship": "NOT_AUTHORIZED",
    "cohort": "NOT_AUTHORIZED",
    "score": "NOT_AUTHORIZED",
    "empirical_evaluation": "NOT_READY",
}


final_qualification = (
    "FAIL_CLOSED_QUALIFICATION_NOT_ESTABLISHED"
)

release_state = (
    "RELEASE_AS_FAIL_CLOSED_AUDIT_ONLY"
)


# ------------------------------------------------------------
# 12. CLAIM BOUNDARY
# ------------------------------------------------------------

claims_not_authorized = [

    "EMPIRICAL_VALIDATION",

    "PREDICTIVE_VALIDITY",

    "CAUSAL_VALIDITY",

    "UNIVERSAL_VALIDITY",

    "COMMERCIAL_VALIDITY",

    "SCIENTIFIC_PROOF",

    "SUPERIORITY_CLAIM",

]


claims_permitted = [

    "SOURCE_LOCKED_AUDIT_EXECUTED",

    "FAIL_CLOSED_DISCIPLINE_MAINTAINED",

    "CONSTRUCT_AUTHORIZATION_WITHHELD",

    "OUTCOME_AUTHORIZATION_WITHHELD",

    "SCORE_AUTHORIZATION_WITHHELD",

    "EMPIRICAL_EVALUATION_NOT_READY",

]


# ------------------------------------------------------------
# 13. COMPLETENESS FAIL CONDITIONS
# ------------------------------------------------------------

fail_conditions = []


if missing_artifacts:

    fail_conditions.append(
        "EXPECTED_ARTIFACT_MISSING"
    )


if cell_declaration_mismatches:

    fail_conditions.append(
        "CELL_DECLARATION_MISMATCH"
    )


if not c19_preserved:

    fail_conditions.append(
        "C19_BLOCKED_STATE_NOT_PRESERVED"
    )


# C11C, C12 and C13R are explicitly recorded as NOT_RUN.
# Their absence is therefore not treated as a hidden PASS.

unexpected_not_available = [
    cell
    for cell in canonical_cells
    if cell not in [
        row["cell"]
        for row in actual_register
        if row["artifact_exists"]
    ]
]


# ------------------------------------------------------------
# 14. C21R2 EXECUTION STATUS
# ------------------------------------------------------------

if fail_conditions:

    c21r2_status = "BLOCKED"

else:

    c21r2_status = "PASS"


# ------------------------------------------------------------
# 15. SAVE MANIFEST JSON
# ------------------------------------------------------------

manifest_payload = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C21R2",

    "cell_name":
        "CORRECTED FINAL CELL REGISTER & COMPLETENESS GATE",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "actual_status_artifacts":
        actual_register,

    "complete_manifest_register":
        manifest_rows,

    "not_available_cells":
        not_available_cells,

    "supersession":
        supersession,

    "status_counts":
        status_counts,

    "executed_cell_count":
        len(executed_cells),

    "pass_cell_count":
        len(pass_cells),

    "blocked_cell_count":
        len(blocked_cells),

    "not_run_cell_count":
        len(not_run_cells),

    "c19_blocked_preserved":
        c19_preserved,

    "final_authorization":
        final_authorization,

    "final_qualification_state":
        final_qualification,

    "release_state":
        release_state,

    "claims_permitted":
        claims_permitted,

    "claims_not_authorized":
        claims_not_authorized,

    "fail_conditions":
        fail_conditions,

    "interpretation_lock": {

        "missing_status_artifact_equals_failure":
            False,

        "missing_status_artifact_equals_pass":
            False,

        "not_run_equals_pass":
            False,

        "blocked_equals_successful_analysis":
            False,

        "construct_authorized":
            False,

        "score_calculated":
            False,

        "empirical_validation_claimed":
            False,

    },

    "manifest_only":
        True,

    "new_analysis_performed":
        False,

}


OUT_JSON = (
    BASE /
    "c21r2_corrected_final_cell_register.json"
)

OUT_JSON.write_text(
    json.dumps(
        manifest_payload,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 16. CONSOLE
# ------------------------------------------------------------

print()
print("=" * 72)
print("C21R2 RESULTS")
print("=" * 72)

print()
print("Actual status artifacts found:",
      len(files := list(BASE.glob("c*_final_status.json"))))

print(
    "Canonical executed cells:",
    len(executed_cells)
)

print(
    "PASS:",
    len(pass_cells)
)

print(
    "BLOCKED:",
    len(blocked_cells)
)

print(
    "NOT_RUN:",
    len(not_run_cells)
)

print()
print("C11B:",
      next(
          row["status"]
          for row in manifest_rows
          if row["cell"] == "C11B"
      ))

print("C11D:",
      next(
          row["status"]
          for row in manifest_rows
          if row["cell"] == "C11D"
      ))

print("C11C:",
      next(
          row["status"]
          for row in manifest_rows
          if row["cell"] == "C11C"
      ))

print("C12:",
      next(
          row["status"]
          for row in manifest_rows
          if row["cell"] == "C12"
      ))

print("C13:",
      next(
          row["status"]
          for row in manifest_rows
          if row["cell"] == "C13"
      ))

print("C13R:",
      next(
          row["status"]
          for row in manifest_rows
          if row["cell"] == "C13R"
      ))

print()
print("C19 BLOCKED PRESERVED:",
      c19_preserved)

print()
print("CELL DECLARATION MISMATCHES:")
print(cell_declaration_mismatches)

print()
print("FINAL AUTHORIZATION")
print(json.dumps(
    final_authorization,
    indent=2
))

print()
print("FINAL QUALIFICATION:")
print(final_qualification)

print()
print("RELEASE STATE:")
print(release_state)

print()
print("FAIL CONDITIONS:")
print(fail_conditions)

print()
print("C21R2 STATUS:",
      c21r2_status)

print()
print("ARTIFACT:")
print(OUT_JSON)

print("=" * 72)

OIP v1.0.33 — C21R2
CORRECTED FINAL CELL REGISTER & COMPLETENESS GATE

C21R2 RESULTS

Actual status artifacts found: 23
Canonical executed cells: 26
PASS: 24
BLOCKED: 2
NOT_RUN: 3

C11B: PASS
C11D: BLOCKED
C11C: NOT_RUN
C12: NOT_RUN
C13: PASS
C13R: NOT_RUN

C19 BLOCKED PRESERVED: True

CELL DECLARATION MISMATCHES:
[]

FINAL AUTHORIZATION
{
  "construct": "NOT_AUTHORIZED",
  "predictor": "NOT_AUTHORIZED",
  "outcome": "NOT_AUTHORIZED",
  "key": "NOT_AUTHORIZED",
  "relationship": "NOT_AUTHORIZED",
  "cohort": "NOT_AUTHORIZED",
  "score": "NOT_AUTHORIZED",
  "empirical_evaluation": "NOT_READY"
}

FINAL QUALIFICATION:
FAIL_CLOSED_QUALIFICATION_NOT_ESTABLISHED

RELEASE STATE:
RELEASE_AS_FAIL_CLOSED_AUDIT_ONLY

FAIL CONDITIONS:
[]

C21R2 STATUS: PASS

ARTIFACT:
/kaggle/working/oip_v1_0_33/c21r2_corrected_final_cell_register.json


In [42]:
# ============================================================
# OIP v1.0.33 — C22
# FINAL AUDIT MANIFEST & REPRODUCIBILITY LOCK
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone


BASE = Path("/kaggle/working/oip_v1_0_33")

OUT_MANIFEST = BASE / "c22_final_audit_manifest.json"
OUT_HASHES = BASE / "c22_artifact_sha256.csv"
OUT_MANIFEST_HASH = BASE / "c22_manifest_sha256.txt"


print("=" * 72)
print("OIP v1.0.33 — C22")
print("FINAL AUDIT MANIFEST & REPRODUCIBILITY LOCK")
print("=" * 72)


# ------------------------------------------------------------
# 1. HASH FUNCTION
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# 2. CANONICAL CELL REGISTER
# ------------------------------------------------------------

cell_register = [

    {
        "cell": "C00",
        "artifact": "c00_source_scope.json",
        "status": "PASS"
    },

    {
        "cell": "C01",
        "artifact": "c01_vocabulary_lock.json",
        "status": "PASS"
    },

    {
        "cell": "C02",
        "artifact": "c02_integrity_status.json",
        "status": "PASS"
    },

    {
        "cell": "C03",
        "artifact": "c03_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C04",
        "artifact": "c04_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C05",
        "artifact": "c05_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C06",
        "artifact": "c06_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C07",
        "artifact": "c07_final_status.json",
        "status": "PASS",
        "superseded_by": "C07R"
    },

    {
        "cell": "C07R",
        "artifact": "c07r_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C08",
        "artifact": "c08_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C09R",
        "artifact": "c09r_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C10",
        "artifact": "c10_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C11A",
        "artifact": "c11a_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C11B",
        "artifact": "c11b_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C11C",
        "artifact": None,
        "status": "NOT_RUN"
    },

    {
        "cell": "C11D",
        "artifact": "c11d_final_status.json",
        "status": "BLOCKED"
    },

    {
        "cell": "C11E",
        "artifact": "c11e_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C12",
        "artifact": None,
        "status": "NOT_RUN"
    },

    {
        "cell": "C13",
        "artifact": "c13_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C13R",
        "artifact": None,
        "status": "NOT_RUN"
    },

    {
        "cell": "C14",
        "artifact": "c14_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C15",
        "artifact": "c15_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C16",
        "artifact": "c16_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C17",
        "artifact": "c17_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C18",
        "artifact": "c18_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C18R",
        "artifact": "c18r_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C19",
        "artifact": "c19_final_status.json",
        "status": "BLOCKED"
    },

    {
        "cell": "C20",
        "artifact": "c20_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C21",
        "artifact": "c21_final_status.json",
        "status": "PASS"
    },

    {
        "cell": "C21R2",
        "artifact": "c21r2_corrected_final_cell_register.json",
        "status": "PASS"
    },

]


# ------------------------------------------------------------
# 3. VERIFY REGISTERED ARTIFACTS
# ------------------------------------------------------------

artifact_records = []

missing_registered_artifacts = []

hash_errors = []

for row in cell_register:

    artifact = row.get("artifact")

    if artifact is None:

        artifact_records.append({
            "cell": row["cell"],
            "artifact": None,
            "status": row["status"],
            "exists": False,
            "sha256": None,
            "size_bytes": None,
        })

        continue


    path = BASE / artifact

    if not path.exists():

        missing_registered_artifacts.append(
            artifact
        )

        artifact_records.append({
            "cell": row["cell"],
            "artifact": artifact,
            "status": row["status"],
            "exists": False,
            "sha256": None,
            "size_bytes": None,
        })

        continue


    try:

        digest = sha256_file(path)

        size = path.stat().st_size

        artifact_records.append({
            "cell": row["cell"],
            "artifact": artifact,
            "status": row["status"],
            "exists": True,
            "sha256": digest,
            "size_bytes": size,
        })

    except Exception as e:

        hash_errors.append({
            "artifact": artifact,
            "error": type(e).__name__,
        })


# ------------------------------------------------------------
# 4. VERIFY STATUS ARTIFACT DECLARATIONS
# ------------------------------------------------------------

declaration_mismatches = []

for record in artifact_records:

    if not record["exists"]:
        continue

    artifact = record["artifact"]

    path = BASE / artifact

    if not artifact.endswith(".json"):
        continue

    try:

        data = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        declared_cell = data.get(
            "cell"
        )

        expected_cell = record[
            "cell"
        ]

        if declared_cell != expected_cell:

            declaration_mismatches.append({
                "artifact": artifact,
                "expected_cell": expected_cell,
                "declared_cell": declared_cell,
            })

    except Exception as e:

        declaration_mismatches.append({
            "artifact": artifact,
            "expected_cell": record["cell"],
            "declared_cell": "READ_ERROR",
        })


# ------------------------------------------------------------
# 5. STATUS COUNTS
# ------------------------------------------------------------

status_counts = {}

for row in cell_register:

    status = row["status"]

    status_counts[status] = (
        status_counts.get(status, 0) + 1
    )


# ------------------------------------------------------------
# 6. FINAL GOVERNANCE STATE
# ------------------------------------------------------------

final_authorization = {

    "construct": "NOT_AUTHORIZED",

    "predictor": "NOT_AUTHORIZED",

    "outcome": "NOT_AUTHORIZED",

    "key": "NOT_AUTHORIZED",

    "relationship": "NOT_AUTHORIZED",

    "cohort": "NOT_AUTHORIZED",

    "score": "NOT_AUTHORIZED",

    "empirical_evaluation": "NOT_READY",

}


qualification_state = (
    "FAIL_CLOSED_QUALIFICATION_NOT_ESTABLISHED"
)

release_state = (
    "RELEASE_AS_FAIL_CLOSED_AUDIT_ONLY"
)


# ------------------------------------------------------------
# 7. QUALIFICATION GATES
# ------------------------------------------------------------

qualification_gates = {

    "provenance_integrity":
        "ESTABLISHED",

    "temporal_validity":
        "NOT_ESTABLISHED",

    "construct_measurement_evidence":
        "NOT_ESTABLISHED",

    "outcome_independence":
        "NOT_ESTABLISHED",

    "no_synthetic_outcome_construction":
        "NOT_ESTABLISHED",

    "leakage_control":
        "NOT_ESTABLISHED",

}


# ------------------------------------------------------------
# 8. EXPLICIT INTERPRETATION OF
#    NO SYNTHETIC OUTCOME CONSTRUCTION
# ------------------------------------------------------------

synthetic_outcome_gate_note = {

    "state":
        "NOT_ESTABLISHED",

    "does_not_mean":
        "A synthetic outcome was constructed.",

    "meaning":
        "This qualification gate was not formally "
        "established by the completed evidence chain.",

    "authorization":
        "NOT_AUTHORIZED",

    "synthetic_outcome_created":
        "NOT_ESTABLISHED",

}


# ------------------------------------------------------------
# 9. CLAIM BOUNDARY
# ------------------------------------------------------------

claims_not_claimed = [

    "EMPIRICAL_VALIDATION",

    "PREDICTIVE_VALIDITY",

    "CAUSAL_VALIDITY",

    "UNIVERSAL_VALIDITY",

    "COMMERCIAL_VALIDITY",

    "SCIENTIFIC_PROOF",

    "SUPERIORITY",

]


claims_permitted = [

    "SOURCE_LOCKED_FAIL_CLOSED_EVIDENCE_AUDIT_COMPLETED",

    "CONSTRUCT_AUTHORIZATION_WITHHELD",

    "OUTCOME_AUTHORIZATION_WITHHELD",

    "SCORE_AUTHORIZATION_WITHHELD",

    "EMPIRICAL_EVALUATION_NOT_READY",

]


# ------------------------------------------------------------
# 10. DATASET IDENTITY
# ------------------------------------------------------------

dataset_identity = {

    "dataset_reference":
        "ETH_2018_ESS_v04_M",

    "source_scope":
        "75 files",

    "stata_files":
        68,

    "pdf_files":
        7,

    "other_files":
        0,

    "source_authority":
        "RAW_SOURCE_ONLY",

}


# ------------------------------------------------------------
# 11. REPRODUCIBILITY STATEMENT
# ------------------------------------------------------------

reproducibility_statement = {

    "protocol":
        "OIP v1.0.33",

    "dataset":
        "ETH_2018_ESS_v04_M",

    "source_files":
        75,

    "stata_files":
        68,

    "pdf_files":
        7,

    "canonical_register_cells":
        len(cell_register),

    "executed_cells":
        sum(
            1
            for row in cell_register
            if row["status"]
            in ["PASS", "BLOCKED"]
        ),

    "pass_cells":
        status_counts.get(
            "PASS",
            0
        ),

    "blocked_cells":
        status_counts.get(
            "BLOCKED",
            0
        ),

    "not_run_cells":
        status_counts.get(
            "NOT_RUN",
            0
        ),

    "final_qualification":
        qualification_state,

    "release":
        release_state,

}


# ------------------------------------------------------------
# 12. C19 PRESERVATION
# ------------------------------------------------------------

c19_record = next(
    (
        row
        for row in cell_register
        if row["cell"] == "C19"
    ),
    None
)

c19_preserved = (
    c19_record is not None
    and c19_record["status"] == "BLOCKED"
)


# ------------------------------------------------------------
# 13. MANIFEST VALIDATION STATE
# ------------------------------------------------------------

manifest_fail_conditions = []

if missing_registered_artifacts:

    manifest_fail_conditions.append(
        "REGISTERED_ARTIFACT_MISSING"
    )

if declaration_mismatches:

    manifest_fail_conditions.append(
        "CELL_DECLARATION_MISMATCH"
    )

if hash_errors:

    manifest_fail_conditions.append(
        "ARTIFACT_HASH_ERROR"
    )

if not c19_preserved:

    manifest_fail_conditions.append(
        "C19_BLOCKED_STATE_NOT_PRESERVED"
    )


manifest_verification = (
    "PASS"
    if not manifest_fail_conditions
    else "BLOCKED"
)


# ------------------------------------------------------------
# 14. WRITE MANIFEST
# ------------------------------------------------------------

manifest = {

    "oip_version":
        "v1.0.33",

    "cell":
        "C22",

    "cell_name":
        "FINAL AUDIT MANIFEST & REPRODUCIBILITY LOCK",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "manifest_verification":
        manifest_verification,

    "manifest_only":
        True,

    "new_analysis_performed":
        False,

    "new_construct_authorization":
        False,

    "new_score_calculation":
        False,

    "new_empirical_evaluation":
        False,

    "dataset_identity":
        dataset_identity,

    "cell_register":
        cell_register,

    "artifact_records":
        artifact_records,

    "missing_registered_artifacts":
        missing_registered_artifacts,

    "declaration_mismatches":
        declaration_mismatches,

    "hash_errors":
        hash_errors,

    "status_counts":
        status_counts,

    "reproducibility":
        reproducibility_statement,

    "qualification_gates":
        qualification_gates,

    "synthetic_outcome_gate":
        synthetic_outcome_gate_note,

    "final_authorization":
        final_authorization,

    "qualification_state":
        qualification_state,

    "release_state":
        release_state,

    "claims_permitted":
        claims_permitted,

    "claims_not_claimed":
        claims_not_claimed,

    "c19_blocked_preserved":
        c19_preserved,

    "manifest_fail_conditions":
        manifest_fail_conditions,

    "interpretation_lock": {

        "artifact_hash_equals_semantic_validity":
            False,

        "audit_completion_equals_empirical_validation":
            False,

        "literal_discovery_equals_construct_authorization":
            False,

        "not_established_equals_synthetic_outcome_created":
            False,

        "missing_status_artifact_equals_pass":
            False,

        "blocked_status_overwritten":
            False,

        "construct_authorized":
            False,

        "score_authorized":
            False,

    },

}


OUT_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 15. HASH ALL REGISTERED ARTIFACTS
# ------------------------------------------------------------

import csv

with open(
    OUT_HASHES,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "cell",
            "artifact",
            "status",
            "exists",
            "size_bytes",
            "sha256",
        ]
    )

    writer.writeheader()

    for record in artifact_records:

        writer.writerow(record)


# ------------------------------------------------------------
# 16. HASH THE MANIFEST ITSELF
# ------------------------------------------------------------

manifest_digest = sha256_file(
    OUT_MANIFEST
)

OUT_MANIFEST_HASH.write_text(
    manifest_digest + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------
# 17. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("=" * 72)
print("C22 FINAL RESULTS")
print("=" * 72)

print()
print(
    "Manifest verification:",
    manifest_verification
)

print(
    "Registered cells:",
    len(cell_register)
)

print(
    "PASS:",
    status_counts.get("PASS", 0)
)

print(
    "BLOCKED:",
    status_counts.get("BLOCKED", 0)
)

print(
    "NOT_RUN:",
    status_counts.get("NOT_RUN", 0)
)

print()
print(
    "Missing registered artifacts:",
    missing_registered_artifacts
)

print(
    "Declaration mismatches:",
    declaration_mismatches
)

print(
    "Hash errors:",
    hash_errors
)

print()
print(
    "C19 BLOCKED PRESERVED:",
    c19_preserved
)

print()
print("DATASET")
print(
    "Reference:",
    dataset_identity["dataset_reference"]
)

print(
    "Source files:",
    dataset_identity["source_scope"]
)

print(
    "Stata:",
    dataset_identity["stata_files"]
)

print(
    "PDF:",
    dataset_identity["pdf_files"]
)

print()
print("FINAL AUTHORIZATION")

for key, value in final_authorization.items():

    print(
        key,
        ":",
        value
    )

print()
print(
    "FINAL QUALIFICATION:",
    qualification_state
)

print(
    "RELEASE:",
    release_state
)

print()
print("CLAIMS NOT CLAIMED")

for claim in claims_not_claimed:

    print(
        "-",
        claim
    )

print()
print(
    "Synthetic outcome gate:",
    synthetic_outcome_gate_note["state"]
)

print(
    "Interpretation:",
    synthetic_outcome_gate_note["meaning"]
)

print()
print(
    "MANIFEST SHA-256:",
    manifest_digest
)

print()
print("ARTIFACTS")
print(OUT_MANIFEST)
print(OUT_HASHES)
print(OUT_MANIFEST_HASH)

print()
print(
    "C22 STATUS:",
    "PASS"
    if manifest_verification == "PASS"
    else "BLOCKED"
)

print("=" * 72)

OIP v1.0.33 — C22
FINAL AUDIT MANIFEST & REPRODUCIBILITY LOCK

C22 FINAL RESULTS

Manifest verification: PASS
Registered cells: 30
PASS: 25
BLOCKED: 2
NOT_RUN: 3

Missing registered artifacts: []
Declaration mismatches: []
Hash errors: []

C19 BLOCKED PRESERVED: True

DATASET
Reference: ETH_2018_ESS_v04_M
Source files: 75 files
Stata: 68
PDF: 7

FINAL AUTHORIZATION
construct : NOT_AUTHORIZED
predictor : NOT_AUTHORIZED
outcome : NOT_AUTHORIZED
key : NOT_AUTHORIZED
relationship : NOT_AUTHORIZED
cohort : NOT_AUTHORIZED
score : NOT_AUTHORIZED
empirical_evaluation : NOT_READY

FINAL QUALIFICATION: FAIL_CLOSED_QUALIFICATION_NOT_ESTABLISHED
RELEASE: RELEASE_AS_FAIL_CLOSED_AUDIT_ONLY

CLAIMS NOT CLAIMED
- EMPIRICAL_VALIDATION
- PREDICTIVE_VALIDITY
- CAUSAL_VALIDITY
- UNIVERSAL_VALIDITY
- COMMERCIAL_VALIDITY
- SCIENTIFIC_PROOF
- SUPERIORITY

Synthetic outcome gate: NOT_ESTABLISHED
Interpretation: This qualification gate was not formally established by the completed evidence chain.

MANIFEST SHA